# IFRS S1/S2 report generation notebook — with extracted requirements integration

This notebook contains the four core IFRS S1/S2-style sections plus the General Requirements section, split cell by cell for debugging.

New integration added:
- loads the extracted IFRS S1/S2 + Commercial Banks requirements catalog generated by the extraction notebook;
- builds compact section-specific requirements packs;
- injects those packs into the Writer, Judge and Reviser prompts;
- adds a lightweight IFRS requirement coverage pre-check to each Judge step;
- keeps bank-specific facts strictly grounded in the section payload evidence.

Run order:
1. Run `ifrs_requirements_S1_S2_CommercialBanks_v5.ipynb` to create `data/requirements/ifrs_bank_requirements.json`.
2. Run this generation notebook.
3. If the requirements file is not found, this notebook still runs using the manually coded section requirements only, but the requirement packs will be disabled.

Updated: includes Emirates clean style guide integration for writing style only.

Updated: includes public wording polish rules for final-report generation.


## 1. Setup, imports, environment variables and Azure OpenAI helpers


In [1]:
# ============================================================
# GOVERNANCE + STRATEGY SECTION GENERATORS
# Role-based Azure OpenAI REST endpoints
# Writer: GPT-5.1 | Judge: GPT-5.2 (LLM-only evaluation) | Reviser: GPT-5.1
# ============================================================

import os
import json
import re
import urllib.request
import urllib.error
import time
import random
from typing import TypedDict, Literal
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from langgraph.graph import StateGraph, START, END

# ── ENV LOADING ──────────────────────────────────────────────
env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"Loaded .env from: {env_path}")
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")

# Shared-key fallback.
# When all three deployments belong to the same Azure resource, one Azure key
# can authenticate all roles. The code also accepts any existing role-specific
# key as the shared fallback, which prevents unnecessary configuration failures.
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")

_shared_key_fallback = (
    AZURE_OPENAI_API_KEY
    or os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or os.getenv("AZURE_OPENAI_REVISER_API_KEY")
)

AZURE_OPENAI_WRITER_API_KEY = (
    os.getenv("AZURE_OPENAI_WRITER_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_JUDGE_API_KEY = (
    os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
    or _shared_key_fallback
)
AZURE_OPENAI_REVISER_API_KEY = (
    os.getenv("AZURE_OPENAI_REVISER_API_KEY")
    or _shared_key_fallback
)

# Full Azure chat-completions deployment URLs.
# AZURE_OPENAI_CHAT_URL is retained as a backward-compatible writer fallback.
AZURE_OPENAI_WRITER_URL = (
    os.getenv("AZURE_OPENAI_WRITER_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)
AZURE_OPENAI_JUDGE_URL = os.getenv("AZURE_OPENAI_JUDGE_URL")
AZURE_OPENAI_REVISER_URL = os.getenv("AZURE_OPENAI_REVISER_URL")


def _clean_url(value: str | None) -> str | None:
    if not value:
        return None
    return value.strip().strip('"').strip("'")


AZURE_OPENAI_WRITER_URL = _clean_url(AZURE_OPENAI_WRITER_URL)
AZURE_OPENAI_JUDGE_URL = _clean_url(AZURE_OPENAI_JUDGE_URL)
AZURE_OPENAI_REVISER_URL = _clean_url(AZURE_OPENAI_REVISER_URL)


def validate_role_config() -> None:
    required = {
        "AZURE_OPENAI_WRITER_API_KEY": AZURE_OPENAI_WRITER_API_KEY,
        "AZURE_OPENAI_JUDGE_API_KEY": AZURE_OPENAI_JUDGE_API_KEY,
        "AZURE_OPENAI_REVISER_API_KEY": AZURE_OPENAI_REVISER_API_KEY,
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }

    missing = [name for name, value in required.items() if not value]
    if missing:
        loaded_flags = {
            "shared_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "writer_key_loaded": bool(AZURE_OPENAI_WRITER_API_KEY),
            "judge_key_loaded": bool(AZURE_OPENAI_JUDGE_API_KEY),
            "reviser_key_loaded": bool(AZURE_OPENAI_REVISER_API_KEY),
            "writer_url_loaded": bool(AZURE_OPENAI_WRITER_URL),
            "judge_url_loaded": bool(AZURE_OPENAI_JUDGE_URL),
            "reviser_url_loaded": bool(AZURE_OPENAI_REVISER_URL),
        }
        raise ValueError(
            "Missing role-based Azure configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded configuration flags (keys are never printed):\n"
            + json.dumps(loaded_flags, indent=2)
            + "\n\nRequired .env configuration when all deployments use the same Azure resource:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_WRITER_URL=<full GPT-5.1 deployment URL>\n"
              "AZURE_OPENAI_JUDGE_URL=<full GPT-5.2 deployment URL>\n"
              "AZURE_OPENAI_REVISER_URL=<full GPT-5.1 deployment URL>\n\n"
              "Use role-specific API keys only when a deployment belongs to a different Azure resource."
        )

    for name, url in {
        "AZURE_OPENAI_WRITER_URL": AZURE_OPENAI_WRITER_URL,
        "AZURE_OPENAI_JUDGE_URL": AZURE_OPENAI_JUDGE_URL,
        "AZURE_OPENAI_REVISER_URL": AZURE_OPENAI_REVISER_URL,
    }.items():
        if not url.startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS Azure deployment URL: {url!r}")


validate_role_config()

print("Role-based Azure OpenAI configuration loaded")
print("Writer endpoint (GPT-5.1):", AZURE_OPENAI_WRITER_URL[:90] + "...")
print("Judge endpoint  (GPT-5.2):", AZURE_OPENAI_JUDGE_URL[:90] + "...")
print("Reviser endpoint (GPT-5.1):", AZURE_OPENAI_REVISER_URL[:90] + "...")


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: list[dict],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: float | None = None,
    use_max_completion_tokens: bool = False,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 6,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Retries transient 500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries `max_completion_tokens`.
    - If the gateway returns 400/500, retries using `max_tokens`, because some
      enterprise proxies do not yet forward `max_completion_tokens` correctly.
    - Does not expose API keys in errors.
    """

    preferred_field = (
        "max_completion_tokens" if use_max_completion_tokens else "max_tokens"
    )
    token_fields = [preferred_field]
    if preferred_field == "max_completion_tokens":
        token_fields.append("max_tokens")

    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {url}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                transient = exc.code in {500, 502, 503, 504}
                rate_limited = exc.code == 429
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if rate_limited and attempt < max_attempts:
                    retry_after = None
                    try:
                        ra = exc.headers.get("Retry-After") if exc.headers else None
                        if ra is not None:
                            retry_after = float(str(ra).strip())
                    except (TypeError, ValueError):
                        retry_after = None
                    wait = (
                        retry_after
                        if retry_after is not None
                        else (2 ** attempt) * 2 + random.random()
                    )
                    wait = min(wait, 90)
                    print(
                        f"{request_label}: rate limited (429); retrying attempt "
                        f"{attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; "
                        f"retrying attempt {attempt + 1}/{max_attempts} "
                        f"in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

            except (ConnectionError, TimeoutError, OSError) as exc:
                # Catches transient connection resets ([WinError 10054]),
                # socket timeouts and similar gateway drops that are NOT wrapped
                # as HTTPError/URLError when they occur mid-response.
                last_error = RuntimeError(
                    f"{request_label} connection reset/timeout.\n"
                    f"Endpoint: {url!r}\n"
                    f"Error: {type(exc).__name__}: {exc}"
                )
                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection reset/timeout "
                        f"({type(exc).__name__}); retrying attempt "
                        f"{attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue
                raise last_error from exc

    raise last_error or RuntimeError(
        f"{request_label} request failed for an unknown reason."
    )

def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )
    return content


def call_writer_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 writer."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_WRITER_URL,
        api_key=AZURE_OPENAI_WRITER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=5600,
        use_max_completion_tokens=True,
        temperature=0,
        request_label="GPT-5.1 writer",
    )
    return _extract_message_content(data)


def _extract_json_object(text: str) -> str:
    """
    Extract the outermost JSON object from model output.
    Handles accidental markdown fences or leading/trailing commentary.
    """
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def call_judge_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """
    GPT-5.2 judge with JSON-safe retry logic.

    First call:
    - Requests valid JSON mode.
    - Uses a larger output allowance to avoid truncation.

    On invalid/truncated JSON:
    - Sends the returned content back to GPT-5.2 for JSON repair.
    - Requests a concise, complete JSON object only.
    """
    data = _azure_chat_completion(
        url=AZURE_OPENAI_JUDGE_URL,
        api_key=AZURE_OPENAI_JUDGE_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=3600,
        use_max_completion_tokens=True,
        temperature=0,
        json_mode=True,
        request_label="GPT-5.2 judge",
    )

    content = _extract_message_content(data)
    candidate = _extract_json_object(content)

    try:
        return json.loads(candidate)

    except json.JSONDecodeError:
        print("GPT-5.2 judge returned incomplete/invalid JSON. Attempting JSON repair...")

        repair_system = (
            "You repair malformed or truncated JSON. "
            "Return one complete valid JSON object only. "
            "Preserve the original meaning, scores, checklist values, issues, and fixes. "
            "Keep strings concise. Do not add markdown fences or commentary."
        )

        repair_user = f"""
Repair the following malformed or truncated judge output into one complete valid JSON object.

Requirements:
- Keep the same top-level fields when present.
- Finish incomplete strings and arrays conservatively.
- Limit each issue/fix string to at most 35 words.
- Limit arrays to the 6 most important items.
- Return JSON only.

MALFORMED OUTPUT:
{content}
""".strip()

        repaired_data = _azure_chat_completion(
            url=AZURE_OPENAI_JUDGE_URL,
            api_key=AZURE_OPENAI_JUDGE_API_KEY,
            messages=[
                {"role": "system", "content": repair_system},
                {"role": "user", "content": repair_user},
            ],
            max_output_tokens=3200,
            use_max_completion_tokens=True,
            temperature=0,
            json_mode=True,
            request_label="GPT-5.2 judge JSON repair",
        )

        repaired_content = _extract_message_content(repaired_data)
        repaired_candidate = _extract_json_object(repaired_content)

        try:
            return json.loads(repaired_candidate)
        except json.JSONDecodeError as exc:
            raise ValueError(
                "GPT-5.2 judge failed to return valid JSON even after repair.\n"
                f"Original output preview:\n{content[:4000]}\n\n"
                f"Repair output preview:\n{repaired_content[:4000]}"
            ) from exc


def call_reviser_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 reviser."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_REVISER_URL,
        api_key=AZURE_OPENAI_REVISER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=5600,
        use_max_completion_tokens=True,
        temperature=0,
        request_label="GPT-5.1 reviser",
    )
    return _extract_message_content(data)




def call_strategy_writer_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 writer with a higher output ceiling for long Strategy sections."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_WRITER_URL,
        api_key=AZURE_OPENAI_WRITER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=6500,
        use_max_completion_tokens=True,
        temperature=0,
        request_label="GPT-5.1 writer (strategy)",
    )
    return _extract_message_content(data)


def call_strategy_reviser_llm(system_prompt: str, user_prompt: str) -> str:
    """GPT-5.1 reviser with a higher output ceiling for long Strategy sections."""
    data = _azure_chat_completion(
        url=AZURE_OPENAI_REVISER_URL,
        api_key=AZURE_OPENAI_REVISER_API_KEY,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=6500,
        use_max_completion_tokens=True,
        temperature=0,
        request_label="GPT-5.1 reviser (strategy)",
    )
    return _extract_message_content(data)


print("Role-specific LLM helper functions ready")


Loaded .env from: c:\Users\BV426BP\Documents\IFRS Data\.env
Role-based Azure OpenAI configuration loaded
Writer endpoint (GPT-5.1): https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.1/chat/comp...
Judge endpoint  (GPT-5.2): https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.2/chat/comp...
Reviser endpoint (GPT-5.1): https://eyq-incubator.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gpt-5.1/chat/comp...
Role-specific LLM helper functions ready


## 2. Load Governance payload and optional risk-register patch


In [2]:
# ── LOAD GOVERNANCE, STRATEGY, RISK MANAGEMENT AND METRICS & TARGETS PAYLOADS ──

def find_payload_file(filename: str) -> Path | None:
    search_dirs = [
        Path.cwd(),
        Path.cwd() / "Data",
        Path.cwd() / "payloads",
        Path.cwd().parent / "payloads",
        Path.cwd().parent / "Data",
        Path("/mnt/data"),
    ]
    for base in search_dirs:
        candidate = base / filename
        if candidate.exists():
            return candidate
    return None


def load_json_file(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


GOVERNANCE_PAYLOAD_PATH = find_payload_file("payload_BANK01_governance.json") or find_payload_file("payload_BANK01.json")
STRATEGY_PAYLOAD_PATH = find_payload_file("payload_BANK01_strategy.json") or find_payload_file("payload_BANK01.json")
RISK_MANAGEMENT_PAYLOAD_PATH = find_payload_file("payload_BANK01_risk_management.json") or find_payload_file("payload_BANK01_strategy.json") or find_payload_file("payload_BANK01.json")
METRICS_TARGETS_PAYLOAD_PATH = find_payload_file("payload_BANK01_metrics_targets.json") or find_payload_file("payload_BANK01_strategy.json") or find_payload_file("payload_BANK01.json")

required_paths = {
    "Governance": GOVERNANCE_PAYLOAD_PATH,
    "Strategy": STRATEGY_PAYLOAD_PATH,
    "Risk Management": RISK_MANAGEMENT_PAYLOAD_PATH,
    "Metrics and Targets": METRICS_TARGETS_PAYLOAD_PATH,
}
missing = [name for name, path in required_paths.items() if path is None]
if missing:
    raise FileNotFoundError(f"Missing payload files for: {missing}")

governance_payload = load_json_file(GOVERNANCE_PAYLOAD_PATH)
strategy_payload = load_json_file(STRATEGY_PAYLOAD_PATH)
risk_management_payload = load_json_file(RISK_MANAGEMENT_PAYLOAD_PATH)
metrics_targets_payload = load_json_file(METRICS_TARGETS_PAYLOAD_PATH)

if "climate_risk_register" not in governance_payload:
    if "climate_risk_register" in risk_management_payload:
        governance_payload["climate_risk_register"] = risk_management_payload["climate_risk_register"]
        print("Patched Governance payload: climate_risk_register from Risk Management.")
    elif "climate_risk_register" in strategy_payload:
        governance_payload["climate_risk_register"] = strategy_payload["climate_risk_register"]
        print("Patched Governance payload: climate_risk_register from Strategy.")

if "climate_risk_register" not in risk_management_payload and "climate_risk_register" in strategy_payload:
    risk_management_payload["climate_risk_register"] = strategy_payload["climate_risk_register"]
    print("Patched Risk Management payload: climate_risk_register from Strategy.")

if "climate_scenarios" not in risk_management_payload and "climate_scenarios" in strategy_payload:
    risk_management_payload["climate_scenarios"] = strategy_payload["climate_scenarios"]
    print("Patched Risk Management payload: climate_scenarios from Strategy.")

for key in ["targets", "target_summary", "reporting_kpis", "financial_summary"]:
    if key not in metrics_targets_payload and key in strategy_payload:
        metrics_targets_payload[key] = strategy_payload[key]
        print(f"Patched Metrics and Targets payload: {key} from Strategy.")

payload = governance_payload
PAYLOAD_PATH = GOVERNANCE_PAYLOAD_PATH
bank_name = governance_payload["bank"]["bank_name"]

print(f"Loaded Governance: {GOVERNANCE_PAYLOAD_PATH}")
print(f"Loaded Strategy: {STRATEGY_PAYLOAD_PATH}")
print(f"Loaded Risk Management: {RISK_MANAGEMENT_PAYLOAD_PATH}")
print(f"Loaded Metrics and Targets: {METRICS_TARGETS_PAYLOAD_PATH}")
print("Bank:", bank_name)
print("Metrics and Targets keys:", list(metrics_targets_payload.keys()))

Patched Governance payload: climate_risk_register from Risk Management.
Patched Risk Management payload: climate_scenarios from Strategy.
Loaded Governance: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_governance.json
Loaded Strategy: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_strategy.json
Loaded Risk Management: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_risk_management.json
Loaded Metrics and Targets: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\payloads\payload_BANK01_metrics_targets.json
Bank: Eurolux Universal Bank AG
Metrics and Targets keys: ['metadata', 'bank', 'financial_summary', 'scope1', 'scope2', 'scope3_travel', 'financed_emissions', 'financed_emissions_equity', 'financed_emissions_sovereign', 'targets', 'carbon_credits', 'internal_carbon_price', 'reporting_kpis']


## IFRS S1/S2 requirements integration — final-polished output compatible

Loads the new deterministic `ifrs_requirements_kb_outputs_final` artifacts and keeps legacy requirement-file compatibility.

In [3]:
# ── IFRS S1/S2 REQUIREMENTS INTEGRATION — NEW FINAL-POLISHED + LEGACY COMPATIBLE ──
# New supported output from: ifrs_s1_s2_requirements_final_polished.ipynb
#   gen_data/IFRS/ifrs_requirements_kb_outputs_final/ifrs_s1_s2_generation_requirements.json
#   gen_data/IFRS/ifrs_requirements_kb_outputs_final/section_by_section_requirements/json/*_requirements.json
# Legacy supported outputs are still accepted for backward compatibility.
#
# Use policy:
# - IFRS requirements guide disclosure coverage and topic selection.
# - Bank payload evidence remains the only source for bank-specific facts.
# - Missing evidence must become a disclosed boundary; never invent payload facts to satisfy IFRS topics.

from collections import defaultdict
from pathlib import Path
import csv
import hashlib
import json
import os
import re

IFRS_REQUIREMENTS_JSON_PATH = globals().get("IFRS_REQUIREMENTS_JSON_PATH") or os.getenv("IFRS_REQUIREMENTS_JSON_PATH")
IFRS_REQUIREMENTS_DIR_PATH = globals().get("IFRS_REQUIREMENTS_DIR_PATH") or os.getenv("IFRS_REQUIREMENTS_DIR_PATH")

_PROJECT_ROOT_CANDIDATES = []
for _base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path("/mnt/data")]:
    if _base not in _PROJECT_ROOT_CANDIDATES:
        _PROJECT_ROOT_CANDIDATES.append(_base)

IFRS_REQUIREMENTS_PATH_CANDIDATES = []
if IFRS_REQUIREMENTS_JSON_PATH:
    IFRS_REQUIREMENTS_PATH_CANDIDATES.append(Path(IFRS_REQUIREMENTS_JSON_PATH))

for _base in _PROJECT_ROOT_CANDIDATES:
    IFRS_REQUIREMENTS_PATH_CANDIDATES.extend([
        # New final-polished deterministic generation-ready exports
        _base / "gen_data" / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_generation_requirements.json",
        _base / "gen_data" / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_generation_requirements.csv",
        _base / "gen_data" / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_requirements_kb_final.json",
        _base / "gen_data" / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_requirements_kb_final.csv",
        # Legacy locations
        _base / "gen_data" / "requirements" / "requirements_kb_data.json",
        _base / "gen_data" / "requirements" / "ifrs_requirements.json",
        _base / "gen_data" / "requirements" / "ifrs_bank_requirements.json",
        _base / "data" / "requirements" / "requirements_kb_data.json",
        _base / "data" / "requirements" / "ifrs_requirements.json",
        _base / "data" / "requirements" / "ifrs_bank_requirements.json",
        _base / "requirements_kb_data.json",
        _base / "ifrs_requirements.json",
        _base / "ifrs_bank_requirements.json",
    ])

# Remove duplicates while preserving order.
_seen_paths = set()
IFRS_REQUIREMENTS_PATH_CANDIDATES = [
    p for p in IFRS_REQUIREMENTS_PATH_CANDIDATES
    if not (str(p) in _seen_paths or _seen_paths.add(str(p)))
]

REPORT_SECTION_ALIASES = {
    "general_requirements": "general_requirements",
    "general requirements": "general_requirements",
    "general": "general_requirements",
    "governance": "governance",
    "strategy": "strategy",
    "risk_management": "risk_management",
    "risk management": "risk_management",
    "metrics_targets": "metrics_targets",
    "metrics and targets": "metrics_targets",
    "metrics_and_targets": "metrics_targets",
    "metrics & targets": "metrics_targets",
    "industry_metrics": "industry_metrics",
    "industry metrics": "industry_metrics",
    "commercial banking": "industry_metrics",
    "commercial_banking": "industry_metrics",
}

SECTION_NAMES = [
    "general_requirements",
    "governance",
    "strategy",
    "risk_management",
    "metrics_targets",
    "industry_metrics",
]

SECTION_TITLE_TO_KEY = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_targets",
}


def _safe_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        txt = value.strip()
        if not txt:
            return []
        # New CSV exports may store lists as JSON strings.
        if (txt.startswith("[") and txt.endswith("]")) or (txt.startswith("{") and txt.endswith("}")):
            try:
                loaded = json.loads(txt)
                return _safe_list(loaded)
            except Exception:
                pass
        # Avoid splitting normal sentences; split only obvious tag fields.
        if "," in txt and len(txt) < 200 and not any(ch in txt for ch in ".;:"):
            return [x.strip() for x in txt.split(",") if x.strip()]
        return [txt]
    return [value]


def _normalise_section_name(section: str | None) -> str:
    if not section:
        return "other"
    raw = str(section).strip()
    if raw in SECTION_TITLE_TO_KEY:
        return SECTION_TITLE_TO_KEY[raw]
    key = raw.lower().replace("-", "_").replace("/", "_").strip()
    key = re.sub(r"\s+", " ", key)
    return REPORT_SECTION_ALIASES.get(key, REPORT_SECTION_ALIASES.get(key.replace(" ", "_"), key.replace(" ", "_")))


def _shorten_text(text: str, max_chars: int = 420) -> str:
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 1].rstrip() + "…"


def _stable_req_id(req: dict) -> str:
    base = "|".join([
        str(req.get("standard", "")),
        str(req.get("paragraph") or req.get("paragraph_id") or ""),
        str(req.get("source_authority", "")),
        str(req.get("requirement_text") or req.get("clean_requirement_text") or "")[:160],
    ])
    return hashlib.md5(base.encode("utf-8")).hexdigest()[:10]


def _to_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    if isinstance(value, (int, float)):
        return bool(value)
    return str(value).strip().lower() in {"1", "true", "yes", "y", "mandatory", "shall"}


def _normalise_standard(value: str | None) -> str:
    txt = str(value or "").strip()
    up = txt.upper().replace("IFRS", "IFRS ")
    up = re.sub(r"\s+", " ", up).strip()
    if up in {"S1", "IFRS S1"}:
        return "IFRS S1"
    if up in {"S2", "IFRS S2"}:
        return "IFRS S2"
    if "S1" in up and "S2" not in up:
        return "IFRS S1"
    if "S2" in up:
        return "IFRS S2"
    return txt or "IFRS"


def _normalise_obligation(value, mandatory=None) -> str:
    raw = str(value or "").strip().lower()
    if raw in {"shall", "mandatory", "must", "required", "prohibition", "prohibited"}:
        return "shall"
    if raw in {"should", "supporting_guidance", "guidance"}:
        return "should"
    if raw in {"may", "optional", "relief", "relief_or_optional"}:
        return "may"
    if _to_bool(mandatory):
        return "shall"
    return "should"


def _safe_json_value(value):
    if isinstance(value, str):
        txt = value.strip()
        if (txt.startswith("[") and txt.endswith("]")) or (txt.startswith("{") and txt.endswith("}")):
            try:
                return json.loads(txt)
            except Exception:
                return value
    return value


def _normalise_requirement_record(req: dict) -> dict:
    """Normalise both the new final-polished schema and the legacy generation schema."""
    req = {k: _safe_json_value(v) for k, v in dict(req).items()}

    raw_report_sections = []
    raw_report_sections.extend(_safe_list(req.get("report_sections")))
    raw_report_sections.extend(_safe_list(req.get("report_section")))
    raw_report_sections.extend(_safe_list(req.get("section")))
    raw_report_sections.extend(_safe_list(req.get("section_key")))
    raw_report_sections.extend(_safe_list(req.get("primary_section")))

    report_sections = []
    for value in raw_report_sections:
        sec = _normalise_section_name(value)
        if sec in SECTION_NAMES and sec not in report_sections:
            report_sections.append(sec)

    # Some IFRS S2 Appendix B / Commercial Banking rows should be available as bank-focused metrics guidance.
    evidence_tags = [str(x).strip().lower() for x in _safe_list(req.get("evidence_tags"))]
    banking_relevance = str(req.get("banking_relevance") or "").strip().lower()
    if not report_sections and ("commercial_banking" in evidence_tags or banking_relevance == "high"):
        report_sections = ["metrics_targets", "industry_metrics"]

    primary_section = report_sections[0] if report_sections else "other"

    text = (
        req.get("requirement_text")
        or req.get("clean_requirement_text")
        or req.get("text")
        or req.get("source_paragraph_text")
        or ""
    )

    standard = _normalise_standard(req.get("standard") or req.get("source_doc"))
    paragraph = str(req.get("paragraph") or req.get("paragraph_id") or req.get("para_id") or "").strip()
    mandatory = _to_bool(req.get("mandatory")) or str(req.get("generation_bucket") or "").lower() == "must_disclose_leaf"
    obligation = _normalise_obligation(req.get("obligation_type"), mandatory=mandatory)

    # The final polished notebook extracts official IFRS S1/S2 rows, not SASB-style industry guidance.
    source_authority = req.get("source_authority") or req.get("authority") or "core_standard"
    if str(source_authority).strip().lower() in {"core", "ifrs", "official_standard", "standard"}:
        source_authority = "core_standard"

    applies_to_banks = _to_bool(req.get("applies_to_banks")) or banking_relevance in {"high", "medium"} or "commercial_banking" in evidence_tags
    applicability = req.get("applicability") or ("commercial_banking" if applies_to_banks else "general")

    out = dict(req)
    out.update({
        "requirement_id": req.get("requirement_id") or _stable_req_id(req),
        "standard": standard,
        "paragraph": paragraph,
        "paragraph_id": paragraph,
        "source_doc": req.get("source_doc") or standard,
        "source_authority": source_authority,
        "obligation_type": obligation,
        "mandatory": mandatory,
        "applicability": applicability,
        "applies_to_banks": applies_to_banks,
        "metric_type": req.get("metric_type"),
        "requirement_text": str(text).strip(),
        "clean_requirement_text": str(req.get("clean_requirement_text") or text).strip(),
        "source_paragraph_text": str(req.get("source_paragraph_text") or text).strip(),
        "page_start": req.get("page_start") or req.get("page"),
        "page_end": req.get("page_end") or req.get("page"),
        "section": primary_section,
        "report_sections": report_sections,
        "evidence_tags": evidence_tags,
        "banking_relevance": banking_relevance or req.get("banking_relevance"),
    })
    return out


def _records_from_section_json(data) -> list[dict]:
    out = []
    if not isinstance(data, dict):
        return out
    section_key = _normalise_section_name(data.get("section_key") or data.get("section_title"))
    standards = data.get("standards", {})
    if isinstance(standards, dict):
        for standard, block in standards.items():
            reqs = block.get("requirements", []) if isinstance(block, dict) else []
            for r in _safe_list(reqs):
                if isinstance(r, dict):
                    rr = dict(r)
                    rr.setdefault("standard", standard)
                    rr.setdefault("report_section", data.get("section_title") or section_key)
                    rr.setdefault("section", section_key)
                    out.append(rr)
    return out


def _flatten_requirements_json(data) -> list[dict]:
    """
    Accepts:
    - new final-polished flat generation JSON list;
    - new section_by_section_requirements JSON object;
    - legacy flat list;
    - legacy requirements_kb_data.json with by_report_section/by_primary_section.
    """
    if isinstance(data, list):
        return [_normalise_requirement_record(r) for r in data if isinstance(r, dict)]

    if isinstance(data, dict):
        # New section export object.
        section_records = _records_from_section_json(data)
        if section_records:
            return [_normalise_requirement_record(r) for r in section_records]

        # Legacy / generic wrappers.
        for key in ["requirements", "generation_requirements", "records", "items"]:
            if isinstance(data.get(key), list):
                return [_normalise_requirement_record(r) for r in data[key] if isinstance(r, dict)]

        seen = set()
        out = []
        for grouping_key in ["by_report_section", "by_primary_section", "sections"]:
            group = data.get(grouping_key, {})
            if isinstance(group, dict):
                for section_name, reqs in group.items():
                    for r in _safe_list(reqs):
                        if not isinstance(r, dict):
                            continue
                        rr = dict(r)
                        rr.setdefault("report_section", section_name)
                        nr = _normalise_requirement_record(rr)
                        key = nr["requirement_id"]
                        if key not in seen:
                            seen.add(key)
                            out.append(nr)
        return out

    return []


def _load_csv_records(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def _load_single_requirements_path(path: Path) -> list[dict]:
    suffix = path.suffix.lower()
    if suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            return _flatten_requirements_json(json.load(f))
    if suffix == ".csv":
        return [_normalise_requirement_record(r) for r in _load_csv_records(path)]
    if suffix == ".jsonl":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return [_normalise_requirement_record(r) for r in rows if isinstance(r, dict)]
    return []


def _section_json_candidates() -> list[Path]:
    dirs = []
    if IFRS_REQUIREMENTS_DIR_PATH:
        dirs.append(Path(IFRS_REQUIREMENTS_DIR_PATH))
    for _base in _PROJECT_ROOT_CANDIDATES:
        dirs.extend([
            _base / "gen_data" / "IFRS" / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json",
            _base / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json",
        ])
    out = []
    for d in dirs:
        if d.exists() and d.is_dir():
            out.extend(sorted(d.glob("*_requirements.json")))
    return out


def load_extracted_ifrs_requirements() -> tuple[list[dict], Path | None]:
    checked_paths = []

    for path in IFRS_REQUIREMENTS_PATH_CANDIDATES:
        path = Path(path)
        checked_paths.append(str(path))
        if path.exists() and path.is_file():
            reqs = _load_single_requirements_path(path)
            if reqs:
                print(f"Loaded IFRS requirements from: {path}")
                print(f"Extracted requirement records: {len(reqs)}")
                return reqs, path
            print(f"Found requirements file but no records could be parsed: {path}")

    # Fallback: load all new section-by-section JSON files if the global file is not present.
    section_paths = _section_json_candidates()
    checked_paths.extend(str(p) for p in section_paths)
    if section_paths:
        all_reqs = []
        seen = set()
        for path in section_paths:
            reqs = _load_single_requirements_path(path)
            for r in reqs:
                key = r.get("requirement_id") or _stable_req_id(r)
                if key not in seen:
                    seen.add(key)
                    all_reqs.append(r)
        if all_reqs:
            source = section_paths[0].parent
            print(f"Loaded IFRS section-by-section requirements from: {source}")
            print(f"Extracted requirement records: {len(all_reqs)}")
            return all_reqs, source

    print("No IFRS requirements file found. Checked these paths:")
    for checked_path in checked_paths:
        print(" -", checked_path)
    return [], None


EXTRACTED_IFRS_REQUIREMENTS, IFRS_REQUIREMENTS_SOURCE_PATH = load_extracted_ifrs_requirements()
IFRS_REQUIREMENTS_AVAILABLE = len(EXTRACTED_IFRS_REQUIREMENTS) > 0

IFRS_REQUIREMENTS_BY_REPORT_SECTION = defaultdict(list)
for req in EXTRACTED_IFRS_REQUIREMENTS:
    for section in req.get("report_sections", []):
        IFRS_REQUIREMENTS_BY_REPORT_SECTION[section].append(req)


def _paragraph_sort_key(pid):
    text = str(pid or "")
    m = re.match(r"^([A-Z]?)(\d+)([A-Z]?)", text)
    if not m:
        return (99, 10**9, 99, text)
    prefix, number, suffix = m.groups()
    prefix_rank = {"": 0, "B": 1, "C": 2, "D": 3, "E": 4}.get(prefix, 99)
    suffix_rank = ord(suffix) - ord("A") + 1 if suffix else 0
    return (prefix_rank, int(number), suffix_rank, text)


def _req_sort_key(req: dict):
    authority_order = {"core_standard": 0, "industry_guidance": 1}
    obligation_order = {"shall": 0, "should": 1, "may": 2}
    standard_order = {"IFRS S1": 0, "S1": 0, "IFRS S2": 1, "S2": 1, "S2_IBG_CB": 2}
    return (
        authority_order.get(req.get("source_authority"), 9),
        obligation_order.get(req.get("obligation_type"), 9),
        standard_order.get(req.get("standard"), 9),
        req.get("page_start") or 9999,
        _paragraph_sort_key(req.get("paragraph")),
        str(req.get("requirement_id") or ""),
    )


def get_section_ifrs_requirements(
    section: str,
    include_industry_guidance: bool = True,
    mandatory_core_only: bool = False,
    banks_only: bool = False,
) -> list[dict]:
    section = _normalise_section_name(section)
    reqs = list(IFRS_REQUIREMENTS_BY_REPORT_SECTION.get(section, []))

    # Metrics & Targets benefits from any banking-specific rows tagged by the extraction notebook.
    if section == "metrics_targets":
        reqs.extend(IFRS_REQUIREMENTS_BY_REPORT_SECTION.get("industry_metrics", []))

    if not include_industry_guidance:
        reqs = [r for r in reqs if r.get("source_authority") != "industry_guidance"]

    if mandatory_core_only:
        reqs = [
            r for r in reqs
            if r.get("source_authority") == "core_standard"
            and r.get("obligation_type") == "shall"
        ]

    if banks_only:
        reqs = [
            r for r in reqs
            if r.get("applies_to_banks") is True
            or r.get("applicability") in {"commercial_banking", "general_financial_sector"}
            or r.get("source_authority") == "industry_guidance"
        ]

    seen = set()
    deduped = []
    for r in sorted(reqs, key=_req_sort_key):
        key = r.get("requirement_id") or _stable_req_id(r)
        if key not in seen:
            seen.add(key)
            deduped.append(r)

    return deduped


def build_ifrs_requirements_pack(section: str, max_core: int = 28, max_banking: int = 10, max_other: int = 8) -> dict:
    """
    Builds a compact, prompt-safe requirements pack.
    Core IFRS S1/S2 requirements are mandatory disclosure guidance.
    Banking-relevant rows guide commercial-bank specificity, but do not authorize unsupported bank facts.
    """
    section = _normalise_section_name(section)
    reqs = get_section_ifrs_requirements(section, include_industry_guidance=True)

    core_mandatory = [
        r for r in reqs
        if r.get("source_authority") == "core_standard"
        and r.get("obligation_type") == "shall"
    ]

    banking_relevant = [
        r for r in reqs
        if r.get("applies_to_banks") is True
        or str(r.get("banking_relevance") or "").lower() in {"high", "medium"}
        or "commercial_banking" in [str(x).lower() for x in _safe_list(r.get("evidence_tags"))]
        or r.get("source_authority") == "industry_guidance"
    ]

    other_core = [
        r for r in reqs
        if r.get("source_authority") == "core_standard"
        and r.get("obligation_type") != "shall"
    ]

    def compact(r: dict) -> dict:
        return {
            "requirement_id": r.get("requirement_id"),
            "standard": r.get("standard"),
            "paragraph": r.get("paragraph") or r.get("paragraph_id"),
            "authority": r.get("source_authority"),
            "obligation": r.get("obligation_type"),
            "applicability": r.get("applicability"),
            "banking_relevance": r.get("banking_relevance"),
            "evidence_tags": _safe_list(r.get("evidence_tags"))[:8],
            "text": _shorten_text(r.get("requirement_text") or r.get("clean_requirement_text") or "", 460),
        }

    return {
        "section": section,
        "requirements_source_loaded": IFRS_REQUIREMENTS_AVAILABLE,
        "source_path": str(IFRS_REQUIREMENTS_SOURCE_PATH) if IFRS_REQUIREMENTS_SOURCE_PATH else None,
        "counts": {
            "total_for_section": len(reqs),
            "core_mandatory_total": len(core_mandatory),
            "banking_relevant_total": len(banking_relevant),
            "core_other_total": len(other_core),
            "core_mandatory_in_prompt": min(len(core_mandatory), max_core),
            "banking_relevant_in_prompt": min(len(banking_relevant), max_banking),
        },
        "core_mandatory_requirements": [compact(r) for r in core_mandatory[:max_core]],
        "banking_relevant_requirements": [compact(r) for r in banking_relevant[:max_banking]],
        # Legacy key retained so existing print/meta logic does not break.
        "commercial_banks_guidance": [compact(r) for r in banking_relevant[:max_banking]],
        "other_core_requirements": [compact(r) for r in other_core[:max_other]],
        "omitted_from_prompt": {
            "core_mandatory_omitted": max(0, len(core_mandatory) - max_core),
            "banking_relevant_omitted": max(0, len(banking_relevant) - max_banking),
            "core_other_omitted": max(0, len(other_core) - max_other),
        },
    }


def build_ifrs_requirements_text(section: str) -> str:
    pack = build_ifrs_requirements_pack(section)

    if not pack["requirements_source_loaded"]:
        return (
            "No extracted IFRS requirements catalog was found at runtime. "
            "Run ifrs_s1_s2_requirements_final_polished.ipynb first to create "
            "gen_data/IFRS/ifrs_requirements_kb_outputs_final/ifrs_s1_s2_generation_requirements.json. "
            "Proceed using the manually coded section requirements and bank evidence only."
        )

    lines = []
    lines.append(f"Section: {pack['section']}")
    lines.append(f"Requirements source: {pack['source_path']}")
    lines.append(f"Counts: {json.dumps(pack['counts'], ensure_ascii=False)}")

    lines.append("\nCore IFRS S1/S2 mandatory requirements to address:")
    if pack["core_mandatory_requirements"]:
        for r in pack["core_mandatory_requirements"]:
            ref = f"{r['standard']} ¶{r['paragraph']}".replace("¶None", "").strip()
            lines.append(f"- [{r['requirement_id']} | {ref}] {r['text']}")
    else:
        lines.append("- No core mandatory requirements extracted for this section.")

    lines.append("\nBanking-relevant IFRS requirements / application points:")
    if pack["banking_relevant_requirements"]:
        for r in pack["banking_relevant_requirements"]:
            ref = f"{r['standard']} ¶{r['paragraph']}".replace("¶None", "").strip()
            tag_txt = ", ".join([str(x) for x in r.get("evidence_tags", []) if str(x).strip()][:4])
            suffix = f" Tags: {tag_txt}." if tag_txt else ""
            lines.append(f"- [{r['requirement_id']} | {ref}] {r['text']}{suffix}")
    else:
        lines.append("- No additional banking-specific rows extracted for this section.")

    if pack["other_core_requirements"]:
        lines.append("\nOther IFRS guidance/context rows, lower priority than mandatory requirements:")
        for r in pack["other_core_requirements"]:
            ref = f"{r['standard']} ¶{r['paragraph']}".replace("¶None", "").strip()
            lines.append(f"- [{r['requirement_id']} | {ref}] {r['text']}")

    omitted = pack["omitted_from_prompt"]
    if any(omitted.values()):
        lines.append(
            f"\nNote: omitted for prompt length — "
            f"core mandatory: {omitted['core_mandatory_omitted']}, "
            f"banking-relevant: {omitted['banking_relevant_omitted']}, "
            f"other core: {omitted['core_other_omitted']}."
        )

    lines.append(
        "\nRequirement-to-evidence rule: cover these topics only to the extent supported by the bank payload. "
        "Where payload evidence is absent, disclose the boundary instead of inventing facts."
    )
    return "\n".join(lines)


def _keywords_from_requirement_text(text: str) -> list[str]:
    words = re.findall(r"[A-Za-z][A-Za-z\-]{4,}", str(text).lower())
    stop = {
        "shall", "disclose", "disclosure", "entity", "entities", "information", "about",
        "requirements", "required", "including", "include", "would", "could", "should",
        "general", "financial", "climate", "related", "sustainability", "reporting",
        "effects", "risks", "opportunities", "terms", "period", "standard", "paragraph",
    }
    terms = []
    for w in words:
        w = w.strip("-")
        if w not in stop and w not in terms:
            terms.append(w)
    return terms[:8]


def check_ifrs_requirement_coverage(section_text: str, section: str, max_requirements: int = 34) -> dict:
    """
    Lightweight pre-check for judge context.
    It is intentionally non-fatal because true coverage requires evidence-aware judgement.
    """
    section = _normalise_section_name(section)
    if not IFRS_REQUIREMENTS_AVAILABLE:
        return {
            "enabled": False,
            "reason": "No extracted IFRS requirements catalog loaded.",
            "source_path": None,
        }

    reqs = get_section_ifrs_requirements(section, include_industry_guidance=True)
    core_mandatory = [
        r for r in reqs
        if r.get("source_authority") == "core_standard"
        and r.get("obligation_type") == "shall"
    ][:max_requirements]

    text_l = str(section_text or "").lower()
    boundary_markers = [
        "not available", "not currently available", "does not currently disclose",
        "not specified", "limited", "boundary", "outside the scope", "has not yet been",
        "not quantified", "not disclosed", "unavailable",
    ]
    has_boundary_language = any(marker in text_l for marker in boundary_markers)
    results = []

    for r in core_mandatory:
        terms = _keywords_from_requirement_text(r.get("requirement_text") or r.get("clean_requirement_text") or "")
        hits = [t for t in terms if t in text_l]
        coverage_ratio = len(hits) / max(1, len(terms))
        if coverage_ratio >= 0.35:
            status = "likely_addressed"
        elif has_boundary_language:
            status = "possibly_addressed_or_boundary_disclosed"
        else:
            status = "possibly_missing"

        results.append({
            "requirement_id": r.get("requirement_id"),
            "standard": r.get("standard"),
            "paragraph": r.get("paragraph") or r.get("paragraph_id"),
            "status": status,
            "keyword_hits": hits,
            "keywords_checked": terms,
            "requirement_preview": _shorten_text(r.get("requirement_text") or r.get("clean_requirement_text") or "", 240),
        })

    return {
        "enabled": True,
        "section": section,
        "source_path": str(IFRS_REQUIREMENTS_SOURCE_PATH),
        "core_mandatory_checked": len(core_mandatory),
        "likely_addressed": sum(1 for x in results if x["status"] == "likely_addressed"),
        "possibly_addressed_or_boundary_disclosed": sum(1 for x in results if x["status"] == "possibly_addressed_or_boundary_disclosed"),
        "possibly_missing": sum(1 for x in results if x["status"] == "possibly_missing"),
        "items": results,
        "note": (
            "This is a heuristic pre-check for the LLM judge. "
            "It should guide review but should not override evidence-based judgement."
        ),
    }


def build_all_ifrs_requirement_packs_summary() -> dict:
    return {
        section: build_ifrs_requirements_pack(section)
        for section in ["general_requirements", "governance", "strategy", "risk_management", "metrics_targets"]
    }


IFRS_REQUIREMENTS_PACKS_SUMMARY = build_all_ifrs_requirement_packs_summary()

print("IFRS requirements integration ready.")
print("Requirements loaded:", IFRS_REQUIREMENTS_AVAILABLE)
print("Source:", IFRS_REQUIREMENTS_SOURCE_PATH)
for section, pack in IFRS_REQUIREMENTS_PACKS_SUMMARY.items():
    print(
        f"- {section}: total={pack['counts']['total_for_section']}, "
        f"core_mandatory={pack['counts']['core_mandatory_total']}, "
        f"banking_relevant={pack['counts']['banking_relevant_total']}"
    )


Loaded IFRS requirements from: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\ifrs_s1_s2_generation_requirements.json
Extracted requirement records: 361
IFRS requirements integration ready.
Requirements loaded: True
Source: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\ifrs_s1_s2_generation_requirements.json
- general_requirements: total=108, core_mandatory=108, banking_relevant=7
- governance: total=15, core_mandatory=15, banking_relevant=0
- strategy: total=70, core_mandatory=70, banking_relevant=57
- risk_management: total=17, core_mandatory=17, banking_relevant=9
- metrics_targets: total=151, core_mandatory=151, banking_relevant=123


## Reference report style system integration — style only

Loads the new multi-file `style_system/` artifacts and keeps legacy single-file style-guide compatibility. The style source is never treated as bank evidence.

In [4]:
# ── STYLE SYSTEM INTEGRATION — NEW style_system/ + LEGACY COMPATIBLE ──
# New supported output from: 01_style_extraction_from_reference_report_FULL.ipynb
#   gen_data/style/style_system/
#     global_style_guide.json
#     layout_style_guide.json
#     table_patterns/table_patterns.json
#     language_rules/no_copying_rules.md
#     language_rules/forbidden_reference_terms.json
#     section_style_guides/*_style.json
#     section_blueprints/*_blueprint.json
# Legacy single-file clean style guide is still accepted.
#
# Use policy:
# - The style system controls tone, structure, formatting and PDF layout only.
# - It is never factual evidence.
# - Target-bank payload evidence and IFRS requirements override style guidance.

from pathlib import Path
import json
import os
import re

STYLE_SYSTEM_DIR_PATH = globals().get("STYLE_SYSTEM_DIR_PATH") or os.getenv("STYLE_SYSTEM_DIR_PATH")
STYLE_GUIDE_JSON_PATH = globals().get("STYLE_GUIDE_JSON_PATH") or os.getenv("STYLE_GUIDE_JSON_PATH")

STYLE_SECTION_ALIASES = {
    "general": "general_requirements",
    "general_requirements": "general_requirements",
    "general requirements": "general_requirements",
    "governance": "governance",
    "strategy": "strategy",
    "risk_management": "risk_management",
    "risk management": "risk_management",
    "metrics_targets": "metrics_and_targets",
    "metrics_and_targets": "metrics_and_targets",
    "metrics and targets": "metrics_and_targets",
    "metrics & targets": "metrics_and_targets",
}

STYLE_SECTION_TITLES = {
    "general_requirements": "General Requirements",
    "governance": "Governance",
    "strategy": "Strategy",
    "risk_management": "Risk Management",
    "metrics_and_targets": "Metrics and Targets",
}


def _normalise_style_section(section: str | None) -> str:
    raw = str(section or "general_requirements").strip()
    key = raw.lower().replace("-", "_").replace("/", "_")
    key = re.sub(r"\s+", " ", key)
    return STYLE_SECTION_ALIASES.get(key, STYLE_SECTION_ALIASES.get(key.replace(" ", "_"), key.replace(" ", "_")))


def _read_json_if_exists(path: Path, default=None):
    try:
        if path and path.exists() and path.is_file():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    except Exception as exc:
        print(f"Could not read JSON style artifact {path}: {exc}")
    return default


def _read_text_if_exists(path: Path, default="") -> str:
    try:
        if path and path.exists() and path.is_file():
            return path.read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        print(f"Could not read text style artifact {path}: {exc}")
    return default


def _project_root_candidates_for_style() -> list[Path]:
    roots = []
    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path("/mnt/data")]:
        if base not in roots:
            roots.append(base)
    # Common case: generation notebook is in notebooks/ and style output is under notebooks/gen_data/style/style_system.
    for base in list(roots):
        candidate = base / "notebooks"
        if candidate.exists() and candidate not in roots:
            roots.append(candidate)
    return roots


def _style_system_candidates() -> list[Path]:
    out = []
    if STYLE_SYSTEM_DIR_PATH:
        out.append(Path(STYLE_SYSTEM_DIR_PATH))
    for base in _project_root_candidates_for_style():
        out.extend([
            base / "gen_data" / "style" / "style_system",
            base / "style" / "style_system",
            base / "style_system",
        ])
    seen = set()
    return [p for p in out if not (str(p) in seen or seen.add(str(p)))]


def _legacy_style_json_candidates() -> list[Path]:
    out = []
    if STYLE_GUIDE_JSON_PATH:
        out.append(Path(STYLE_GUIDE_JSON_PATH))
    for base in _project_root_candidates_for_style():
        out.extend([
            base / "gen_data" / "style" / "emirates_nbd_style_reference_clean.json",
            base / "style" / "emirates_nbd_style_reference_clean.json",
            base / "emirates_nbd_style_reference_clean.json",
        ])
    seen = set()
    return [p for p in out if not (str(p) in seen or seen.add(str(p)))]


def load_style_reference() -> tuple[dict | None, Path | None]:
    checked = []

    for root in _style_system_candidates():
        checked.append(str(root))
        if root.exists() and root.is_dir():
            global_style = _read_json_if_exists(root / "global_style_guide.json", {}) or {}
            layout_style = _read_json_if_exists(root / "layout_style_guide.json", {}) or {}
            table_patterns = _read_json_if_exists(root / "table_patterns" / "table_patterns.json", {}) or {}
            forbidden_terms = _read_json_if_exists(root / "language_rules" / "forbidden_reference_terms.json", []) or []
            no_copying_rules = _read_text_if_exists(root / "language_rules" / "no_copying_rules.md", "")

            section_style_guides = {}
            section_blueprints = {}
            for slug in STYLE_SECTION_TITLES:
                section_style_guides[slug] = _read_json_if_exists(
                    root / "section_style_guides" / f"{slug}_style.json", {}
                ) or {}
                section_blueprints[slug] = _read_json_if_exists(
                    root / "section_blueprints" / f"{slug}_blueprint.json", {}
                ) or {}

            loaded_any = bool(global_style or layout_style or table_patterns or any(section_style_guides.values()))
            if loaded_any:
                ref = {
                    "source_type": "style_system",
                    "style_system_dir": str(root),
                    "global_style_guide": global_style,
                    "layout_style_guide": layout_style,
                    "table_patterns": table_patterns,
                    "forbidden_reference_terms": forbidden_terms,
                    "no_copying_rules": no_copying_rules,
                    "section_style_guides": section_style_guides,
                    "section_blueprints": section_blueprints,
                }
                print(f"Loaded style system from: {root}")
                print("Loaded section style guides:", [k for k, v in section_style_guides.items() if v])
                return ref, root

    for path in _legacy_style_json_candidates():
        checked.append(str(path))
        if path.exists() and path.is_file():
            ref = _read_json_if_exists(path, None)
            if isinstance(ref, dict):
                ref.setdefault("source_type", "legacy_clean_style_json")
                print(f"Loaded legacy clean style guide from: {path}")
                return ref, path

    print("No reusable style artifact found. Checked these paths:")
    for item in checked:
        print(" -", item)
    return None, None


STYLE_REFERENCE, STYLE_SOURCE_PATH = load_style_reference()


def _style_take(items, n=8, max_chars=300):
    if items is None:
        return []
    if isinstance(items, str):
        # Preserve markdown no-copying files as short rule lines.
        lines = [re.sub(r"^[-#\s]+", "", x).strip() for x in items.splitlines()]
        items = [x for x in lines if x]
    if isinstance(items, dict):
        items = [f"{k}: {v}" for k, v in items.items()]
    if not isinstance(items, list):
        items = [items]
    cleaned = []
    for item in items:
        text = re.sub(r"\s+", " ", str(item or "")).strip()
        if not text:
            continue
        if len(text) > max_chars:
            text = text[:max_chars].rstrip() + "..."
        cleaned.append(text)
        if len(cleaned) >= n:
            break
    return cleaned


def _bullet_lines(items):
    items = [x for x in items if str(x).strip()]
    return "\n".join(f"- {x}" for x in items) if items else "- Not specified in style guide."


def _format_named_value(name: str, value) -> list[str]:
    if value is None or value == "":
        return []
    if isinstance(value, list):
        vals = ", ".join(_style_take(value, 8, 120))
        return [f"{name}: {vals}"] if vals else []
    if isinstance(value, dict):
        vals = "; ".join(_style_take(value, 5, 160))
        return [f"{name}: {vals}"] if vals else []
    return [f"{name}: {str(value).strip()}"]


def _build_style_context_from_style_system(section: str, max_items: int = 8) -> str:
    slug = _normalise_style_section(section)
    title = STYLE_SECTION_TITLES.get(slug, slug.replace("_", " ").title())

    global_style = STYLE_REFERENCE.get("global_style_guide", {}) if STYLE_REFERENCE else {}
    layout_style = STYLE_REFERENCE.get("layout_style_guide", {}) if STYLE_REFERENCE else {}
    table_patterns = STYLE_REFERENCE.get("table_patterns", {}) if STYLE_REFERENCE else {}
    section_style = STYLE_REFERENCE.get("section_style_guides", {}).get(slug, {}) if STYLE_REFERENCE else {}
    blueprint = STYLE_REFERENCE.get("section_blueprints", {}).get(slug, {}) if STYLE_REFERENCE else {}
    forbidden_terms = _style_take(STYLE_REFERENCE.get("forbidden_reference_terms", []), 40, 120) if STYLE_REFERENCE else []
    no_copying = _style_take(STYLE_REFERENCE.get("no_copying_rules", ""), max_items, 260) if STYLE_REFERENCE else []

    global_voice_lines = []
    for label, key in [
        ("Report voice", "report_voice"),
        ("Tone", "tone"),
        ("Point of view", "point_of_view"),
    ]:
        global_voice_lines.extend(_format_named_value(label, global_style.get(key)))

    section_profile = []
    for label, key in [
        ("Purpose", "purpose"),
        ("Tone", "tone"),
        ("Level of detail", "level_of_detail"),
        ("Paragraph style", "paragraph_style"),
        ("Sentence style", "sentence_style"),
        ("Preferred evidence style", "preferred_evidence_style"),
        ("Table usage", "table_usage"),
        ("Figure usage", "figure_usage"),
        ("PDF layout usage", "pdf_layout_usage"),
        ("Missing data language", "how_to_discuss_missing_data"),
    ]:
        section_profile.extend(_format_named_value(label, section_style.get(key)))

    section_avoid = _style_take(section_style.get("what_to_avoid", []), max_items)
    verbs = _style_take(section_style.get("preferred_disclosure_verbs", []), max_items, 80)

    layout_rules = []
    section_layout = layout_style.get("section_layout_rules", {}) if isinstance(layout_style, dict) else {}
    layout_rules.extend(_style_take(section_layout.get(title, []), max_items))
    layout_rules.extend(_style_take(layout_style.get("pdf_readability_rules", []), 6))

    table_rules = []
    if isinstance(table_patterns, dict):
        table_rules.extend(_style_take(table_patterns.get("general_table_rules", []), 4))
        table_key_candidates = [
            f"{slug}_tables",
            "metrics_and_targets_tables" if slug == "metrics_and_targets" else None,
            "governance_tables" if slug == "governance" else None,
            "strategy_tables" if slug == "strategy" else None,
            "risk_management_tables" if slug == "risk_management" else None,
        ]
        for key in [k for k in table_key_candidates if k]:
            table_rules.extend(_style_take(table_patterns.get(key, []), 4))

    return f"""
STYLE GUIDE USAGE POLICY:
- Use this style system for writing style, tone, section flow, formatting and PDF layout only.
- Do not use it as factual evidence.
- Do not copy wording, numbers, facts, entities, committees, images or distinctive design from the reference report.
- If style guidance conflicts with bank payload evidence or IFRS requirements, payload evidence and IFRS requirements win.

STYLE SOURCE:
{STYLE_SOURCE_PATH}

FORBIDDEN REFERENCE TERMS:
{", ".join(forbidden_terms) if forbidden_terms else "No forbidden terms file loaded."}

GLOBAL REPORT VOICE:
{_bullet_lines(global_voice_lines)}

GLOBAL PARAGRAPH AND SENTENCE RULES:
{_bullet_lines(_style_take(global_style.get("paragraph_rules", []), max_items) + _style_take(global_style.get("sentence_rules", []), max_items))}

DISCLOSURE LANGUAGE RULES:
{_bullet_lines(_style_take(global_style.get("disclosure_language_rules", []), max_items))}

EVIDENCE, TRACEABILITY AND MISSING-DATA LANGUAGE RULES:
{_bullet_lines(_style_take(global_style.get("evidence_and_traceability_rules", []), max_items) + _style_take(global_style.get("missing_data_language_rules", []), max_items))}

SECTION STYLE PROFILE — {title}:
{_bullet_lines(section_profile)}

PREFERRED DISCLOSURE VERBS:
{_bullet_lines(verbs)}

RECOMMENDED SECTION BLUEPRINT:
- Recommended subsections: {", ".join(_style_take(blueprint.get("recommended_subsections", []), 12, 100)) or "Not specified."}
- Recommended order: {", ".join(_style_take(blueprint.get("recommended_order", []), 12, 100)) or "Not specified."}
- Required narrative blocks: {", ".join(_style_take(blueprint.get("required_narrative_blocks", []), 12, 120)) or "Not specified."}
- Optional narrative blocks: {", ".join(_style_take(blueprint.get("optional_narrative_blocks", []), 8, 120)) or "Not specified."}

OUTPUT STRUCTURE RULES:
{_bullet_lines(_style_take(blueprint.get("output_structure_rules", []), max_items))}

TABLE / FIGURE / PDF LAYOUT RULES:
{_bullet_lines(table_rules + layout_rules)}

SECTION-SPECIFIC AVOID RULES:
{_bullet_lines(section_avoid)}

NO-COPYING RULES:
{_bullet_lines(no_copying)}
""".strip()


def _build_style_context_from_legacy(section: str, max_items: int = 8) -> str:
    # Backward-compatible path for the old emirates_nbd_style_reference_clean.json schema.
    section_key = "general" if section == "general_requirements" else section
    guide = STYLE_REFERENCE.get("style_guide", {}) if STYLE_REFERENCE else {}

    identity_policy = guide.get("identity_policy", {})
    global_rules = _style_take(guide.get("global_style_rules", []), max_items)
    copying_controls = _style_take(guide.get("copying_risk_controls", []), max_items)
    quality_checks = _style_take(guide.get("quality_checks_for_generated_reports", []), max_items)

    tone = guide.get("tone_profile", {})
    tone_lines = []
    if tone.get("primary_tone"):
        tone_lines.append(f"Primary tone: {tone.get('primary_tone')}")
    if tone.get("secondary_tones"):
        tone_lines.append("Secondary tones: " + ", ".join(_style_take(tone.get("secondary_tones", []), 6)))
    if tone.get("avoid"):
        tone_lines.append("Avoid: " + ", ".join(_style_take(tone.get("avoid", []), 6)))

    section_guidance = guide.get("section_guidance", {}).get(section_key, {})
    section_flow = _style_take(section_guidance.get("recommended_flow", []), max_items)
    section_notes = _style_take(section_guidance.get("style_notes", []), 4)

    formatting = guide.get("formatting_conventions", {})
    formatting_rules = []
    for key in ["headings", "paragraphs", "lists", "tables", "numbers_units_dates"]:
        formatting_rules.extend(_style_take(formatting.get(key, []), 3))
    formatting_rules = formatting_rules[:12]

    language_patterns = guide.get("language_patterns", {})
    selected_patterns = []
    legacy_pattern_keys = {
        "general": ["compliance_and_reporting_basis", "methodology_assurance_limitations", "evidence_boundary"],
        "general_requirements": ["compliance_and_reporting_basis", "methodology_assurance_limitations", "evidence_boundary"],
        "governance": ["governance_oversight", "methodology_assurance_limitations", "evidence_boundary"],
        "strategy": ["strategy_value_chain", "methodology_assurance_limitations", "evidence_boundary"],
        "risk_management": ["risk_management_process", "methodology_assurance_limitations", "evidence_boundary"],
        "metrics_and_targets": ["metrics_targets", "financed_emissions", "methodology_assurance_limitations", "evidence_boundary"],
        "metrics_targets": ["metrics_targets", "financed_emissions", "methodology_assurance_limitations", "evidence_boundary"],
    }
    for key in legacy_pattern_keys.get(section, legacy_pattern_keys.get(section_key, [])):
        values = _style_take(language_patterns.get(key, []), max_items)
        if values:
            selected_patterns.append(f"{key}:")
            selected_patterns.extend([f"- {v}" for v in values])

    forbidden_terms = _style_take(identity_policy.get("forbidden_reference_identity_terms", []), 30, max_chars=120)

    return f"""
STYLE GUIDE USAGE POLICY:
- Use this guide for writing style, tone, structure, flow and formatting only.
- Do not use this guide as factual evidence.
- Do not copy wording from the reference report.
- Do not mention the reference bank or its named committees.
- If this guide conflicts with payload evidence or IFRS requirements, payload evidence and IFRS requirements win.

IDENTITY POLICY:
{identity_policy.get("rule", "Use the target bank identity from payload only.")}

FORBIDDEN REFERENCE TERMS:
{", ".join(forbidden_terms) if forbidden_terms else "None loaded."}

TONE PROFILE:
{chr(10).join(tone_lines) if tone_lines else "Formal, technical, bank-reporting style."}

GLOBAL STYLE RULES:
{_bullet_lines(global_rules)}

SECTION PURPOSE:
{section_guidance.get("purpose", "Use the section purpose defined by the notebook requirements.")}

RECOMMENDED SECTION FLOW:
{_bullet_lines(section_flow)}

SECTION STYLE NOTES:
{_bullet_lines(section_notes)}

RELEVANT LANGUAGE PATTERNS:
{chr(10).join(selected_patterns) if selected_patterns else "- Use formal IFRS S1/S2 bank-reporting language."}

FORMATTING CONVENTIONS:
{_bullet_lines(formatting_rules)}

COPYING RISK CONTROLS:
{_bullet_lines(copying_controls)}

STYLE QUALITY CHECKS:
{_bullet_lines(quality_checks)}
""".strip()


def build_style_context(section: str, max_items: int = 8) -> str:
    """
    Builds a compact section-specific style block.

    Priority logic:
    1. Payload evidence = facts allowed to say.
    2. IFRS requirements = topics to cover.
    3. Style system = how to write and format it.
    """
    if STYLE_REFERENCE is None:
        return (
            "No external style guide loaded. Use the notebook's default formal IFRS S1/S2 "
            "bank-reporting style. Do not invent facts."
        )

    if STYLE_REFERENCE.get("source_type") == "style_system":
        return _build_style_context_from_style_system(section, max_items=max_items)
    return _build_style_context_from_legacy(section, max_items=max_items)


def _get_forbidden_reference_terms() -> list[str]:
    if STYLE_REFERENCE is None:
        return []
    if STYLE_REFERENCE.get("source_type") == "style_system":
        return [str(x).strip() for x in STYLE_REFERENCE.get("forbidden_reference_terms", []) if str(x).strip()]
    return [
        str(x).strip()
        for x in STYLE_REFERENCE.get("style_guide", {})
        .get("identity_policy", {})
        .get("forbidden_reference_identity_terms", [])
        if str(x).strip()
    ]


def check_reference_identity_leak(text: str) -> dict:
    """Detect accidental leakage of reference-report identity terms into generated output."""
    forbidden = _get_forbidden_reference_terms()
    if not forbidden:
        return {"passed": True, "matches": []}

    lower_text = (text or "").lower()
    matches = []
    for term in forbidden:
        if term and term.lower() in lower_text:
            matches.append(term)

    return {
        "passed": len(matches) == 0,
        "matches": sorted(set(matches)),
    }


print("Style guide integration ready.")
print("Style loaded:", STYLE_REFERENCE is not None)
print("Source:", STYLE_SOURCE_PATH)
print("Example style block preview:")
print(build_style_context("governance")[:1200])


Loaded style system from: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system
Loaded section style guides: ['general_requirements', 'governance', 'strategy', 'risk_management', 'metrics_and_targets']
Style guide integration ready.
Style loaded: True
Source: c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system
Example style block preview:
STYLE GUIDE USAGE POLICY:
- Use this style system for writing style, tone, section flow, formatting and PDF layout only.
- Do not use it as factual evidence.
- Do not copy wording, numbers, facts, entities, committees, images or distinctive design from the reference report.
- If style guidance conflicts with bank payload evidence or IFRS requirements, payload evidence and IFRS requirements win.

STYLE SOURCE:
c:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system

FORBIDDEN REFERENCE TERMS:
Emirates NBD, Emirates NBD Group, DenizBank, Emirat

## Section completeness checks


In [5]:
# ── SECTION COMPLETENESS AND PUBLIC-REPORT QUALITY CHECKS ───
# These checks prevent truncated sections from being approved or assembled.

SECTION_REQUIRED_HEADINGS = {
    "general": [
        "#### Understanding the Bank’s approach towards implementing IFRS S1 and IFRS S2 requirements",
        "#### Fair presentation",
        "#### Connected information",
        "#### Comparative information",
        "#### Timing and location of disclosure",
        "#### Reporting entity, business model and value chain",
        "#### Sources of guidance",
        "#### Statement of compliance",
        "#### Materiality assessment",
    ],
    "general_requirements": [
        "#### Understanding the Bank’s approach towards implementing IFRS S1 and IFRS S2 requirements",
        "#### Fair presentation",
        "#### Connected information",
        "#### Comparative information",
        "#### Timing and location of disclosure",
        "#### Reporting entity, business model and value chain",
        "#### Sources of guidance",
        "#### Statement of compliance",
        "#### Materiality assessment",
    ],
    "governance": [
        "#### Overview",
        "#### The role of the Board of Directors",
        "#### Board committees and climate-related oversight",
        "#### Management responsibility for climate-related risks and opportunities",
        "#### Skills, competencies and remuneration",
        "#### Governance decisions, controls and evidence boundaries",
    ],
    "strategy": [
        "#### Overview",
        "#### Sustainability risks and opportunities across the value chain",
        "#### Strategic management of climate-related risks and opportunities",
        "#### Climate-related effects on business model, value chain and decision-making",
        "#### Financial effects and resource allocation",
        "#### Climate resilience and scenario analysis",
        "#### Strategy limitations and evidence boundaries",
    ],
    "risk_management": [
        "#### Sustainability-related risk management overview",
        "#### Upstream sustainability risks",
        "#### Risk management across internal operations",
        "#### Downstream sustainability risks",
        "#### Climate risk management",
        "#### Risk management limitations and evidence boundaries",
    ],
    "metrics_and_targets": [
        "#### Overview",
        "#### Sustainability-related metrics and targets",
        "#### Greenhouse gas emissions",
        "#### Managing exposure towards financed emissions",
        "#### Climate-related financial metrics and resource allocation",
        "#### Climate targets and progress",
        "#### Metrics and targets limitations and evidence boundaries",
    ],
}

SECTION_FINAL_BOUNDARY_HEADINGS = {
    "general": "#### Materiality assessment",
    "general_requirements": "#### Materiality assessment",
    "governance": "#### Governance decisions, controls and evidence boundaries",
    "strategy": "#### Strategy limitations and evidence boundaries",
    "risk_management": "#### Risk management limitations and evidence boundaries",
    "metrics_and_targets": "#### Metrics and targets limitations and evidence boundaries",
}

SECTION_MIN_FINAL_WORDS = {
    "general": 60,
    "general_requirements": 60,
    "governance": 90,
    "strategy": 80,
    "risk_management": 70,
    "metrics_and_targets": 100,
}


def _last_nonempty_line(text: str) -> str:
    lines = [line.strip() for line in (text or "").splitlines() if line.strip()]
    return lines[-1] if lines else ""


def _looks_mid_sentence(text: str) -> bool:
    t = text or ""
    # Deterministic exhibits (tables/figures) are appended after the prose; evaluate the
    # prose ending so a trailing table row or figure caption is not mistaken for truncation.
    marker = "#### Supporting data and exhibits"
    if marker in t:
        t = t.split(marker, 1)[0]
    last = _last_nonempty_line(t)
    if not last:
        return True

    # Structural endings that are complete (markdown table rows, figure captions, images).
    if last.startswith("|") or last.endswith("|"):
        return False
    if last.endswith("}") or last.startswith("!["):
        return False

    # Headings or bullet starters are incomplete endings.
    if last.startswith("#") or last.endswith(":") or last.endswith("-"):
        return True

    # Markdown bullets are okay only if they end with punctuation.
    terminal_ok = last.endswith((".", "!", "?", ")", "]", ".”", "’"))
    if not terminal_ok:
        return True

    # Very common truncation fragments seen in outputs.
    bad_fragments = [
        "ifrs s",
        "for 2024, eurolux",
        "available evidence for the 2024",
        "with available",
        "based on available",
    ]
    return last.lower().strip() in bad_fragments


def check_section_completeness(text: str, section: str) -> dict:
    """
    Strict completeness check used before approving/assembling a section.
    """
    text = text or ""
    failures, warnings = [], []

    _machinery = [p for p in (
        "available evidence", "available documentation", "documentation reviewed",
        "source data", "internally generated analytical outputs", "evidence package",
        "in the available", "based on available", "available source data",
    ) if p in (text or "").lower()]
    if _machinery:
        warnings.append({
            "check": "report_machinery_language_present",
            "details": ("Rewrite in the Bank's own voice; remove report-machinery phrases: "
                        + ", ".join(sorted(set(_machinery)))),
        })

    required = SECTION_REQUIRED_HEADINGS.get(section, [])
    missing = [h for h in required if h not in text]
    if missing:
        failures.append({
            "check": "section_missing_required_headings",
            "details": missing,
        })

    final_heading = SECTION_FINAL_BOUNDARY_HEADINGS.get(section)
    if final_heading:
        if final_heading not in text:
            failures.append({
                "check": "section_final_boundary_heading_missing",
                "details": final_heading,
            })
        else:
            final_text = text.split(final_heading, 1)[1]
            word_count = len(final_text.split())
            min_words = SECTION_MIN_FINAL_WORDS.get(section, 60)
            if word_count < min_words:
                failures.append({
                    "check": "section_final_boundary_too_short_or_truncated",
                    "details": f"{final_heading} has {word_count} words; expected at least {min_words}.",
                })

    if _looks_mid_sentence(text):
        failures.append({
            "check": "section_ends_mid_sentence",
            "details": _last_nonempty_line(text)[-240:],
        })

    # Scan PROSE only: strip appended deterministic exhibits/appendices and image
    # attribute-list blocks so figure filenames / caption attrs are not mis-read as
    # raw field syntax (consistent with the senior-QA raw-field gate).
    _prose = text
    for _m in ("#### Supporting data and exhibits", "\n## Appendices", "\n## Appendix"):
        if _m in _prose:
            _prose = _prose.split(_m, 1)[0]
    _prose = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", _prose)   # markdown images
    _prose = re.sub(r"\{:[^}]*\}", "", _prose)               # {: .attr } caption blocks

    if re.search(r"\b[A-Za-z_]+=[A-Za-z0-9_.-]+", _prose):
        warnings.append({
            "check": "raw_field_syntax_visible",
            "details": "Raw key=value syntax appears in final narrative. Convert field names into public report wording.",
        })

    if re.search(r"\b(target_value_pct_reduction|baseline_value|third_party_validated|latest_paris_alignment|gross_or_net|planned_carbon_credits_pct|target_framework)\b", _prose, flags=re.IGNORECASE):
        warnings.append({
            "check": "raw_payload_field_names_visible",
            "details": "Raw target field names appear in final narrative.",
        })

    if "check_public_report_wording" in globals():
        public_wording = check_public_report_wording(text)
        failures.extend(public_wording.get("failures", []))
        warnings.extend(public_wording.get("warnings", []))

    return {
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def assert_section_complete(text: str, section: str) -> None:
    result = check_section_completeness(text, section)
    if not result["passed"]:
        raise RuntimeError(
            f"Section '{section}' is incomplete and cannot be assembled into the full report.\n"
            + json.dumps(result, indent=2, ensure_ascii=False)
        )


def section_completion_failure_items(text: str, section: str) -> tuple[list, list]:
    result = check_section_completeness(text, section)
    return result.get("failures", []), result.get("warnings", [])


print("Section completeness checks ready.")

Section completeness checks ready.


## Public report wording polish rules


In [6]:
# ── PUBLIC REPORT WORDING POLISH RULES ──────────────────────
# These rules keep the report public-facing instead of exposing project/data-field language.

RAW_TO_PUBLIC_REPLACEMENTS = {
    "SBTi_1.5C": "SBTi 1.5°C pathway",
    "UNEP_FI": "UNEP FI",
    "on_track": "on track",
    "technology_removal": "technology-removal credits",
    "pct reduction vs baseline": "percentage reduction versus baseline",
    "scope1_and_2": "Scope 1 and Scope 2",
    "scope3_cat15": "Scope 3 Category 15",
    "scope_3_cat15_financed": "Scope 3 Category 15 financed emissions",
    "all_scopes": "all scopes",
    "all_kyoto_7": "all Kyoto greenhouse gases",
    "whole_entity": "whole entity",
    "absolute_reduction": "absolute reduction",
    "intensity_reduction": "intensity reduction",
    "net_zero": "net zero",
    "tco2e_per_meur_lending": "tCO₂e per EUR million lending",
    "tco2e_absolute": "absolute tCO₂e",
    "baseline_value": "baseline value",
    "target_value_pct_reduction": "target percentage reduction",
    "third_party_validated": "third-party validation field",
    "latest_paris_alignment": "Paris-alignment field",
    "gross_or_net": "gross or net basis",
    "planned_carbon_credits_pct": "planned carbon credits percentage",
    "target_framework": "target framework",
    "Target ID": "Target reference",
    "ERM integration flags": "ERM integration indicators",
    "ERM integration flag": "ERM integration indicator",
    "source-data flags": "source-data indicators",
    "source data flags": "source data indicates",
    "flags the bank’s disclosures": "indicates that the Bank’s disclosures are",
    "flags the Bank’s disclosures": "indicates that the Bank’s disclosures are",
}


PUBLIC_REPORT_FORBIDDEN_TERMS = [
    "generated sections",
    "synthetic or generated sections",
    "synthetic or generated sections and data",
    "third party validation field=true",
    "third-party validation field=true",
    "validation_body",
    "target framework=",
    "synthetic data",
    "generated data",
    "synthetic or generated data",
    "payload",
    ".json",
    "notebook",
    "cell_",
    "outputs/",
    "third_party_validated=true",
    "latest_paris_alignment=true",
    "status=",
    "progress=",
]


PUBLIC_REPORT_WARNING_TERMS = [
    "SBTi_1.5C",
    "UNEP_FI",
    "technology_removal",
    "on_track",
    "baseline_value",
    "target_value_pct_reduction",
    "third_party_validated",
    "latest_paris_alignment",
    "gross_or_net",
    "planned_carbon_credits_pct",
    "target_framework",
    "ERM integration flags",
]


def polish_public_report_wording(text: str) -> str:
    """
    Light deterministic cleanup for known raw technical tokens that should not appear
    in the public Markdown report. This does not add facts; it only changes wording.
    """
    if not text:
        return text

    out = text

    # Replace known raw tokens with public wording.
    for raw, public in RAW_TO_PUBLIC_REPLACEMENTS.items():
        out = out.replace(raw, public)

    # Public-facing replacement for the internal data-generation wording.
    out = re.sub(
        r"\bsynthetic or generated data\b",
        "modelled estimates, internally generated analytical outputs and available source data",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\bsynthetic data\b",
        "modelled estimates and available source data",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\bgenerated data\b",
        "internally generated analytical outputs",
        out,
        flags=re.IGNORECASE,
    )

    # Soften CSRD if it appears as a strong unsupported regime claim.
    out = re.sub(
        r"\bthe applicable CSRD regime\b",
        "sustainability reporting requirements that may apply to banks operating in the European regulatory environment",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\bapplicable CSRD regime\b",
        "European sustainability reporting environment",
        out,
        flags=re.IGNORECASE,
    )

    # Improve raw boolean/status phrasing if it slips through.
    out = re.sub(
        r"target record field indicates validation by ([A-Za-z0-9 ._\-]+)",
        lambda m: "target record indicates that third-party validation is recorded, with "
        + m.group(1).replace("_", " ")
        + " identified as the validation body",
        out,
        flags=re.IGNORECASE,
    )

    out = re.sub(
        r"target record field reports status [“\"]?on track[”\"]?",
        "target record reports the target as on track",
        out,
        flags=re.IGNORECASE,
    )



    # Remove public-report blocker language that exposes the report as synthetic/generated.
    out = re.sub(
        r"This report is prepared using synthetic or generated sections and data where indicated in the evidence base\.\s*"
        r"Not all underlying methodology records are available, and some disclosures are based on modelled estimates or incomplete methodologies\.",
        "Parts of the report rely on modelled estimates, internally generated analytical outputs and available source data. "
        "Not all underlying methodology records are available, and some disclosures are based on modelled estimates or incomplete methodologies.",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"This report is prepared using synthetic or generated sections and data where indicated in the evidence base\.",
        "Parts of the report rely on modelled estimates, internally generated analytical outputs and available source data.",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"synthetic or generated sections and data",
        "modelled estimates, internally generated analytical outputs and available source data",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"synthetic or generated sections",
        "internally generated analytical outputs",
        out,
        flags=re.IGNORECASE,
    )

    # Remove duplicated wording introduced by earlier public replacements.
    out = re.sub(
        r"modelled estimates,\s*internally generated analytical outputs and available source data and modelled estimates",
        "modelled estimates, internally generated analytical outputs and available source data",
        out,
        flags=re.IGNORECASE,
    )

    # Convert raw target framework syntax into report wording.
    out = re.sub(
        r"the target record indicates target framework=([A-Za-z0-9 .°&/+-]+)",
        lambda m: "the target record identifies " + m.group(1).strip().replace("_", " ") + " as the target framework",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"target framework=([A-Za-z0-9 .°&/+-]+)",
        lambda m: m.group(1).strip().replace("_", " ") + " as the target framework",
        out,
        flags=re.IGNORECASE,
    )

    # Convert raw validation field wording into public report wording.
    out = re.sub(
        r"the target record indicates third-party validation field=true and names ([A-Za-z0-9 ._\-]+) as validation body",
        lambda m: "the target record records third-party validation, with "
        + m.group(1).replace("_", " ")
        + " identified as the validation body",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"the target record indicates third-party validation field=true and names ([A-Za-z0-9 ._\-]+) as validation_body",
        lambda m: "the target record records third-party validation, with "
        + m.group(1).replace("_", " ")
        + " identified as the validation body",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"third-party validation field=true",
        "third-party validation is recorded",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"names ([A-Za-z0-9 ._\-]+) as validation_body",
        lambda m: "identifies " + m.group(1).replace("_", " ") + " as the validation body",
        out,
        flags=re.IGNORECASE,
    )
    out = out.replace("validation_body", "validation body")

    # Normalise duplicated spaces introduced by replacements.
    out = re.sub(r"[ \t]{2,}", " ", out)
    return out


def check_public_report_wording(text: str) -> dict:
    text_l = (text or "").lower()
    failures = []
    warnings = []

    found_forbidden = [term for term in PUBLIC_REPORT_FORBIDDEN_TERMS if term.lower() in text_l]
    if found_forbidden:
        failures.append({
            "check": "public_report_forbidden_internal_terms",
            "details": found_forbidden,
        })

    found_warnings = [term for term in PUBLIC_REPORT_WARNING_TERMS if term.lower() in text_l]
    if found_warnings:
        warnings.append({
            "check": "public_report_raw_terms_warning",
            "details": found_warnings,
        })

    if re.search(r"target\s+framework=|validation_body|third-party validation field=true", text or "", flags=re.IGNORECASE):
        failures.append({
            "check": "raw_target_field_syntax_visible",
            "details": "Raw target framework or validation field syntax appears in final narrative.",
        })

    # Catch visible snake_case tokens in narrative.
    snake_case_tokens = sorted(set(re.findall(r"\b[a-z]+_[a-z0-9_]+\b", text or "")))
    allowed = {"tco2e"}  # keep very limited allowance; normally tCO₂e should be used instead
    snake_case_tokens = [x for x in snake_case_tokens if x.lower() not in allowed]
    if snake_case_tokens:
        warnings.append({
            "check": "snake_case_tokens_visible",
            "details": snake_case_tokens[:25],
        })

    return {
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


print("Public report wording polish helpers ready.")

# Wrap the base public-polish function with the senior final polish if available.
_BASE_POLISH_PUBLIC_REPORT_WORDING = polish_public_report_wording

def polish_public_report_wording(text: str) -> str:
    out = _BASE_POLISH_PUBLIC_REPORT_WORDING(text)
    if "polish_public_report_wording_final" in globals():
        out = polish_public_report_wording_final(out)
    return out

Public report wording polish helpers ready.


## Senior final report QA gates


In [7]:
# ── SENIOR FINAL REPORT QA GATES ─────────────────────────────
# Senior-grade guardrails:
# - Do not assemble failed / unapproved sections.
# - Do not allow internal judge/revision wording into the public report.
# - Do not allow raw field syntax or synthetic/generated report language.
# - Enforce section-specific issues found during expert review.

SENIOR_PUBLIC_FORBIDDEN_TERMS = [
    "synthetic data",
    "generated data",
    "synthetic or generated data",
    "synthetic or generated sections",
    "generated sections",
    "ai-produced",
    "payload",
    ".json",
    "notebook",
    "cell_",
    "outputs/",
    "assessed as requiring revision",
    "requires revision",
    "revision_required",
    "judge feedback",
    "deterministic failure",
    "failed section",
    "target framework=",
    "validation_body",
    "third-party validation field=true",
    "SBTi_1.5",
]

SENIOR_RAW_FIELD_PATTERNS = [
    r"\b[a-z]+_[a-z0-9_]*\b",
    r"\b[A-Za-z ]+=\w+",
]


def _extract_markdown_section(markdown_text: str, section_title: str) -> str:
    pattern = rf"^###\s+{re.escape(section_title)}\s*$"
    match = re.search(pattern, markdown_text or "", flags=re.IGNORECASE | re.MULTILINE)
    if not match:
        return ""
    start = match.start()
    next_match = re.search(r"^###\s+", markdown_text[match.end():], flags=re.MULTILINE)
    if not next_match:
        return markdown_text[start:].strip()
    end = match.end() + next_match.start()
    return markdown_text[start:end].strip()


def _extract_subsection(section_text: str, heading: str, next_heading: str | None = None) -> str:
    if heading not in section_text:
        return ""
    part = section_text.split(heading, 1)[1]
    if next_heading and next_heading in part:
        part = part.split(next_heading, 1)[0]
    else:
        next_h = re.search(r"\n####\s+", part)
        if next_h:
            part = part[:next_h.start()]
    return part.strip()


def polish_public_report_wording_final(text: str) -> str:
    """
    Final deterministic cleanup that only changes wording, never facts.
    It is intentionally conservative and targets known failure modes.
    """
    out = text or ""

    # Remove internal/artificial report wording.
    out = re.sub(
        r"This report is prepared using synthetic or generated sections and data where indicated in the evidence base\.\s*",
        "",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\bsynthetic or generated sections and data\b",
        "modelled estimates, internally generated analytical outputs and available source data",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\bsynthetic or generated data\b|\bgenerated data\b|\bsynthetic data\b",
        "modelled estimates and available source data",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\bgenerated sections\b|\bsynthetic or generated sections\b",
        "analytical report sections",
        out,
        flags=re.IGNORECASE,
    )

    # Remove internal judge/revision language from public report.
    out = re.sub(
        r"\n?-?\s*The Metrics and targets section is in scope but has been assessed as requiring revision,[^.]*\.\s*",
        "\n",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"\n?[^.\n]*assessed as requiring revision[^.\n]*\.\s*",
        "\n",
        out,
        flags=re.IGNORECASE,
    )

    # Fix repeated/awkward modelled-estimate wording.
    out = re.sub(
        r"Parts of the report rely on modelled estimates, internally generated analytical outputs and available source data, particularly with respect to climate scenario analysis and financed emissions\.\s*modelled estimates, internally generated analytical outputs and available source data is used in the preparation of this report, especially in sections involving scenario analysis, modelled estimates, and where direct measurement is not available\.",
        "Parts of the report rely on modelled estimates, internally generated analytical outputs and available source data, particularly for climate scenario analysis and financed emissions. Where methodologies or data sources are incomplete, the relevant evidence boundaries are disclosed in the applicable sections.",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"modelled estimates,\s*modelled estimates,\s*internally generated analytical outputs and available source data,\s*and analytical outputs",
        "modelled estimates, internally generated analytical outputs and available source data",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(
        r"modelled estimates,\s*internally generated analytical outputs and available source data is used",
        "modelled estimates, internally generated analytical outputs and available source data are used",
        out,
        flags=re.IGNORECASE,
    )

    # Strategy raw target wording and contradiction fixes.
    out = out.replace("SBTi_1.5°C", "SBTi 1.5°C pathway")
    out = out.replace("SBTi_1.5C", "SBTi 1.5°C pathway")
    out = re.sub(
        r"Target information is available in summary form only, covering target type, scope, status, target year, framework, and progress; detailed target assumptions, baselines, milestones, validation bodies, gross versus net treatment, and planned carbon-credit use are not described\.",
        "Target information in the Strategy evidence is available in summary form only. Detailed target assumptions, calculation methods, validation evidence, gross-versus-net treatment and carbon-credit governance are described only where available in the Metrics and targets section.",
        out,
        flags=re.IGNORECASE,
    )

    # Senior wording cleanup for risk management overstatements.
    out = re.sub(
        r"integrates climate-related and other sustainability-related risks into its overall approach to risk management primarily through",
        "manages climate-related risks primarily through",
        out,
        flags=re.IGNORECASE,
    )
    out = re.sub(r"\bhazard mapping tools\b", "flood mapping overlay", out, flags=re.IGNORECASE)
    out = re.sub(r"\bhazard mapping overlays\b", "flood mapping overlays", out, flags=re.IGNORECASE)

    # Metrics: avoid assurance vocabulary in the Metrics section by using verification/validation wording.
    metrics = _extract_markdown_section(out, "Metrics and Targets")
    if metrics:
        metrics_polished = metrics
        metrics_polished = re.sub(r"\bassurance statement\b", "verification statement", metrics_polished, flags=re.IGNORECASE)
        metrics_polished = re.sub(r"\bindependent assurance\b", "independent verification", metrics_polished, flags=re.IGNORECASE)
        metrics_polished = re.sub(r"\bassurance of current-year emissions or progress\b", "verification of current-year emissions or progress", metrics_polished, flags=re.IGNORECASE)
        metrics_polished = re.sub(r"\bindependently assured outcome\b", "independently verified outcome", metrics_polished, flags=re.IGNORECASE)
        out = out.replace(metrics, metrics_polished)

    # Normalize whitespace while preserving markdown.
    out = re.sub(r"[ \t]{2,}", " ", out)
    out = re.sub(r"\n{3,}", "\n\n", out)
    return out.strip()


def senior_public_report_quality_check(markdown_text: str, metadata: dict | None = None) -> dict:
    failures = []
    warnings = []
    text = markdown_text or ""
    text_l = text.lower()

    # 1. No failed/unapproved sections in metadata.
    if metadata:
        section_meta = metadata.get("sections", {})
        for section_key, info in section_meta.items():
            meta = (info or {}).get("metadata", {}) or {}
            approved = meta.get("approved")
            status = str(meta.get("status", "")).lower()
            approval_status = str(meta.get("approval_status", "")).lower()
            score = meta.get("overall_score")
            # 'needs_human_review' sections are accepted into the report (signed off via the
            # data-limitations & expert-review log). Only genuinely failed or revision-required
            # sections block the final report.
            hard_fail = (status in ("failed", "error")) or (approval_status in ("revision_required", "rejected"))
            if hard_fail:
                failures.append({
                    "check": "unapproved_section_in_full_report",
                    "section": section_key,
                    "details": {
                        "approved": approved,
                        "status": status,
                        "overall_score": score,
                        "approval_status": meta.get("approval_status"),
                    },
                })
            try:
                if score is not None and float(score) < 7:
                    failures.append({
                        "check": "section_score_below_threshold",
                        "section": section_key,
                        "details": score,
                    })
            except Exception:
                pass

    # 2. Public forbidden terms.
    found_terms = [term for term in SENIOR_PUBLIC_FORBIDDEN_TERMS if term.lower() in text_l]
    if found_terms:
        failures.append({"check": "senior_public_forbidden_terms", "details": sorted(set(found_terms))})

    # 3. Raw target or field syntax (ignore markdown image references / figure file paths
    #    and attribute-list blocks, whose snake_case filenames/classes are not prose).
    scan_text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    scan_text = re.sub(r"\{:[^}]*\}", " ", scan_text)
    raw_matches = []
    for pattern in SENIOR_RAW_FIELD_PATTERNS:
        raw_matches.extend(re.findall(pattern, scan_text))
    raw_matches = sorted(set(m for m in raw_matches if m not in {"tCO2e"}))
    # Permit all-caps framework acronyms with no underscore.
    raw_matches = [m for m in raw_matches if "_" in m or "=" in m]
    if raw_matches:
        failures.append({"check": "raw_field_or_key_value_syntax", "details": raw_matches[:30]})

    # 4. General Requirements checks.
    general = _extract_markdown_section(text, "General Requirements")
    if general:
        if "modelled estimates, internally generated analytical outputs and available source data are used" in general and "Parts of the report rely on" in general:
            warnings.append({
                "check": "general_requirements_possible_repetition",
                "details": "Repeated modelled-estimates preparation wording may still be present.",
            })
        if re.search(r"metrics and targets section[^.\n]*(requiring revision|revision)", general, flags=re.IGNORECASE):
            failures.append({
                "check": "internal_revision_language_in_general_requirements",
                "details": "General Requirements contains internal evaluation wording.",
            })
        if re.search(r"own operations[^.\n]*(heat|flooding|wildfire)", general, flags=re.IGNORECASE):
            warnings.append({
                "check": "own_operations_hazard_specificity_may_exceed_evidence",
                "details": "Use generic physical disruption wording unless hazard-level own-operations evidence exists.",
            })

    # 5. Strategy contradiction/raw framework checks.
    strategy = _extract_markdown_section(text, "Strategy")
    if strategy:
        if "SBTi_1.5" in strategy:
            failures.append({"check": "raw_strategy_framework_token", "details": "SBTi_1.5 token visible."})
        if "baselines, milestones, validation bodies" in strategy and "are not described" in strategy:
            failures.append({
                "check": "strategy_metrics_targets_contradiction",
                "details": "Strategy says target details are not described, while Metrics and targets may describe them.",
            })
        if "ensuring that climate-related considerations are embedded" in strategy:
            warnings.append({
                "check": "strategy_overstrong_embedding_language",
                "details": "Replace with evidence-bound wording such as 'supports incorporation'.",
            })
        if "technology readiness is assumed to be medium to high" in strategy.lower():
            warnings.append({
                "check": "scenario_technology_readiness_range_too_narrow",
                "details": "Available evidence supports low/medium/high range by scenario type.",
            })

    # 6. Metrics section checks.
    metrics = _extract_markdown_section(text, "Metrics and Targets")
    if metrics:
        ghg = _extract_subsection(
            metrics,
            "#### Greenhouse gas emissions",
            "#### Managing exposure towards financed emissions",
        )
        if "| Target | Scope | Baseline Year" in ghg or "Target parameters (not current-year emissions)" in ghg:
            failures.append({
                "check": "target_parameter_table_in_ghg_subsection",
                "details": "Move operational target baseline/milestone values out of Greenhouse gas emissions into Climate targets and progress.",
            })
        if re.search(r"\bassur(?:e|ed|ance)\b", metrics, flags=re.IGNORECASE):
            warnings.append({
                "check": "metrics_assurance_vocabulary_present",
                "details": "Prefer validation/verification wording in Metrics and targets.",
            })
        if "monitored by management" in metrics and "available evidence does not specify" not in metrics.lower():
            warnings.append({
                "check": "metrics_governance_linkage_boundary_missing",
                "details": "Add boundary if target-setting oversight body/remuneration linkage is not evidenced here.",
            })

    return {
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


print("Senior final report QA gates ready.")

Senior final report QA gates ready.


In [8]:
# ── SHARED FINALIZE-GATE + ROUTING (workstream 1, rev.2) ─────
# Targets a 'good 9'. Publishable requires: score>=TARGET_SCORE, no unsupported
# claims, no MATERIAL available-but-unused evidence, and zero deterministic failures.
# Optional omissions and correctly-disclosed boundaries do NOT block publish.
# rev.2 adds: (a) material-vs-optional omission handling, (b) a hardened reviser
# brief that forbids raw tokens/provenance leakage, (c) best-draft (non-regression)
# selection so iteration can never ship a draft worse than an earlier one.

from __future__ import annotations

TARGET_SCORE = 9
DEFAULT_MAX_REVISIONS = 3

# Tokens/wording that must never reach the public report. Used to guard the reviser.
PUBLIC_WORDING_FORBIDDEN_HINTS = [
    "raw field names", "snake_case (e.g. latest_paris_alignment, on_track, all_kyoto_7)",
    "key=value syntax (e.g. third_party_validated=true)",
    "internal IDs or codes (e.g. SCN0002, TGT001, FN-CB-410a.2, SBTi_1.5C)",
    "data-provenance terms (e.g. 'synthetic', 'generated data', 'payload')",
]


def _norm_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [v for v in value if str(v).strip()]
    text = str(value).strip()
    return [text] if text else []


def _coerce_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def term_in_claiming_context(text_l: str, term: str) -> bool:
    """True only if `term` appears in a sentence that ASSERTS it, not one that
    denies or bounds it. Prevents deterministic checks from flagging correctly
    disclosed boundaries (e.g. 'baselines are not described', 'not subject to assurance')."""
    import re as _re
    term = term.lower()
    denial_cues = (
        "not ", "no ", "without", "outside", "absence of", "cannot", "are not",
        "is not", "does not", "do not", "unavailable", "not available",
        "not described", "not disclosed", "not specified", "not provided",
        "not evidenced", "limited to", "rather than", "outside the scope", "not yet", "not covered",
    )
    for sent in _re.split(r"(?<=[.;:])\s+", text_l):
        if term in sent and not any(cue in sent for cue in denial_cues):
            return True
    return False


def _material_unsupported(judge_result: dict):
    """Material unsupported claims block publish. Prefer the judge's material/minor
    split; fall back to the flat unsupported_claims only if the split is absent."""
    jr = judge_result or {}
    if ("material_unsupported_claims" in jr) or ("minor_wording_to_tighten" in jr):
        return _norm_list(jr.get("material_unsupported_claims"))
    return _norm_list(jr.get("unsupported_claims"))


def _material_omitted(judge_result: dict):
    """Material omissions block publish. Prefer the judge's material/optional split;
    fall back to the flat available_evidence_omitted only if the split is absent."""
    jr = judge_result or {}
    if ("material_evidence_omitted" in jr) or ("optional_evidence_omitted" in jr):
        return _norm_list(jr.get("material_evidence_omitted"))
    return _norm_list(jr.get("available_evidence_omitted"))


def evaluate_finalize_gate(judge_result: dict,
                           deterministic_checks: dict | None = None,
                           target_score: int = TARGET_SCORE) -> dict:
    judge_result = judge_result or {}
    det = deterministic_checks or judge_result.get("deterministic_prechecks", {}) or {}

    score = _coerce_int(judge_result.get("overall_score"), 0)
    unsupported = _material_unsupported(judge_result)
    omitted = _material_omitted(judge_result)
    det_failures = det.get("failures", []) or []
    failure_count = det.get("failure_count", len(det_failures))

    reasons = []
    if score < target_score:
        reasons.append(f"overall_score {score} < target {target_score}")
    if unsupported:
        reasons.append(f"{len(unsupported)} unsupported claim(s) must be removed or reframed")
    if omitted:
        reasons.append(f"{len(omitted)} material evidence item(s) must be incorporated")
    if failure_count:
        reasons.append(f"{failure_count} deterministic failure(s) must be resolved")

    return {
        "publishable": (len(reasons) == 0),
        "met_target_score": score >= target_score,
        "score": score,
        "target_score": target_score,
        "blocking_reasons": reasons,
        "unsupported_claims": unsupported,
        "evidence_omitted": omitted,
        "deterministic_failures": det_failures,
    }


def finalize_quality_key(judge_result: dict):
    """Higher tuple == better draft. Used for best-draft (non-regression) selection."""
    jr = judge_result or {}
    gate = evaluate_finalize_gate(jr, jr.get("deterministic_prechecks", {}))
    return (
        1 if gate["publishable"] else 0,
        gate["score"],
        -len(gate["deterministic_failures"]),
        -len(gate["unsupported_claims"]),
        -len(gate["evidence_omitted"]),
    )


def _strip_best(jr: dict) -> dict:
    return {k: v for k, v in (jr or {}).items() if k != "_best_so_far"}


def track_best(state: dict, judge_result: dict) -> dict:
    """Return the best (draft, judge_result) seen so far, stored inside judge_result
    under '_best_so_far'. Called from each judge node; needs no extra state channel."""
    candidate = {
        "draft": state.get("draft", ""),
        "judge_result": _strip_best(judge_result),
        "revision_count": state.get("revision_count", 0),
    }
    prev_best = (state.get("judge_result", {}) or {}).get("_best_so_far")
    if prev_best is None or finalize_quality_key(candidate["judge_result"]) > finalize_quality_key(prev_best["judge_result"]):
        return candidate
    return prev_best


def route_after_judge_generic(state: dict,
                              target_score: int = TARGET_SCORE,
                              revise_label: str = "revise") -> str:
    judge = state.get("judge_result", {}) or {}
    cur = evaluate_finalize_gate(judge, judge.get("deterministic_prechecks", {}), target_score)

    best = judge.get("_best_so_far")
    best_publishable = bool(
        best and evaluate_finalize_gate(
            best["judge_result"], best["judge_result"].get("deterministic_prechecks", {}), target_score
        )["publishable"]
    )

    revision_count = state.get("revision_count", 0)
    max_revisions = state.get("max_revisions", DEFAULT_MAX_REVISIONS)

    if cur["publishable"] or best_publishable:
        return "finalize"
    if revision_count >= max_revisions:
        return "finalize"
    return revise_label


def finalize_state(state: dict, target_score: int = TARGET_SCORE) -> dict:
    """Finalize on the BEST draft seen across iterations, not necessarily the last."""
    judge_now = state.get("judge_result", {}) or {}
    best = judge_now.get("_best_so_far")

    if best:
        draft = best["draft"]
        judge = dict(best["judge_result"])
    else:
        draft = state.get("draft", "")
        judge = _strip_best(judge_now)

    det = judge.get("deterministic_prechecks", {})
    gate = evaluate_finalize_gate(judge, det, target_score)
    boundaries = _norm_list(judge.get("correctly_disclosed_evidence_boundaries"))

    if gate["publishable"]:
        status = "approved_with_limitations" if boundaries else "approved"
        approved = True
    else:
        status = "needs_human_review"
        approved = False

    judge["approval_status"] = status
    judge["approved"] = approved
    judge["finalize_gate"] = gate

    return {
        **state,
        "judge_result": judge,
        "final_section": draft,
        "draft": draft,
        "status": "approved" if approved else "needs_human_review",
    }


def build_reviser_focus(judge_result: dict) -> str:
    """Prioritised revision brief. Feeds the reviser the MATERIAL omissions and
    unsupported claims it previously never saw, and forbids raw-token leakage."""
    judge_result = judge_result or {}
    gate = judge_result.get("finalize_gate") or evaluate_finalize_gate(judge_result)

    blocks = []

    blocks.append(
        "REVISION STYLE: Make the MINIMAL edits needed to address the points below. Change only the "
        "specific words or sentences flagged; do NOT rewrite unaffected sentences, do NOT add new facts, "
        "claims or subsections, and preserve all correct content. Introducing new unsupported claims while "
        "fixing others is the main cause of score regressions."
    )

    if gate["unsupported_claims"]:
        blocks.append(
            "REMOVE OR REFRAME THESE UNSUPPORTED CLAIMS (do not invent evidence; reframe "
            "to what the evidence supports or state the boundary):\n"
            + "\n".join(f"- {c}" for c in gate["unsupported_claims"])
        )

    if gate["evidence_omitted"]:
        blocks.append(
            "INCORPORATE THIS MATERIAL EVIDENCE THAT WAS OMITTED. It is present in the "
            "compact evidence. Express each item as natural report prose with figures in "
            "plain language. Do NOT paste raw field names, codes, record IDs or labels:\n"
            + "\n".join(f"- {c}" for c in gate["evidence_omitted"])
        )

    det_fail = gate["deterministic_failures"]
    if det_fail:
        blocks.append(
            "RESOLVE THESE DETERMINISTIC FAILURES (mechanical, must be fixed):\n"
            + "\n".join(f"- {f.get('check')}: {f.get('details')}" for f in det_fail)
        )

    required_fixes = _norm_list(judge_result.get("required_fixes"))
    if required_fixes:
        blocks.append("ADDITIONAL JUDGE-REQUIRED FIXES:\n" + "\n".join(f"- {f}" for f in required_fixes))

    checklist = judge_result.get("checklist", {}) or {}
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
    if false_items:
        blocks.append("FAILED CHECKLIST ITEMS:\n" + "\n".join(f"- {item}" for item in false_items))

    # Standing guard: every revision must preserve public-report quality.
    blocks.append(
        "PUBLIC-WORDING GUARD — do not violate while revising:\n"
        "- Never introduce: " + "; ".join(PUBLIC_WORDING_FORBIDDEN_HINTS) + ".\n"
        "- Translate every figure, status or flag into formal report prose.\n"
        "- Do not duplicate decision/topic content across subsections.\n"
        "- Adding evidence must not introduce unsupported or stronger claims, and must "
        "not push optional detail at the cost of readability."
    )

    if len(blocks) == 1:  # only the guard
        return "No blocking issues were returned; tighten wording and specificity only.\n\n" + blocks[0]
    return "\n\n".join(blocks)


# ── HONEST SCORE FLOOR (best-of-N reseed) ────────────────────────────────────
# Raises the *realized* score floor without altering any score. A section that is
# capable of ACCEPTABLE_FLOOR_SCORE is re-run from a fresh draft when it lands below
# the floor, and the genuinely best result is kept (same ranking as track_best). A
# section that truly cannot reach the floor on the available evidence still finalizes
# at its true best and is surfaced for human review — nothing is clamped or faked.
ACCEPTABLE_FLOOR_SCORE = 8
FLOOR_MAX_ATTEMPTS = 2  # extra full re-draft attempts; total runs = 1 + FLOOR_MAX_ATTEMPTS


def _finalized_score(result: dict) -> int:
    jr = (result or {}).get("judge_result", {}) or {}
    gate = jr.get("finalize_gate") or evaluate_finalize_gate(jr, jr.get("deterministic_prechecks", {}))
    return _coerce_int(gate.get("score"), 0)


def run_section_with_floor(graph, initial_state, *, floor: int = ACCEPTABLE_FLOOR_SCORE,
                           max_attempts: int = FLOOR_MAX_ATTEMPTS, label: str = "section") -> dict:
    """Invoke the section graph; if the finalized score is below `floor`, re-run from a
    fresh draft and keep the best result (best-of-N reseed)."""
    import copy
    best = graph.invoke(copy.deepcopy(initial_state))
    best_score = _finalized_score(best)
    used = 0
    while best_score < floor and used < max_attempts:
        used += 1
        print(f"\n[floor] {label}: finalized {best_score} < floor {floor} — reseed {used}/{max_attempts}")
        cand = graph.invoke(copy.deepcopy(initial_state))
        if finalize_quality_key(cand.get("judge_result", {}) or {}) > finalize_quality_key(best.get("judge_result", {}) or {}):
            best, best_score = cand, _finalized_score(cand)
    note = "meets floor" if best_score >= floor else "BELOW FLOOR — surfaced for human review"
    print(f"\n[floor] {label}: final score {best_score} (floor {floor}, {used} reseed(s)) — {note}")
    return best


## 3. Governance evidence extraction helpers


In [9]:
# ── GOVERNANCE EVIDENCE EXTRACTOR ────────────────────────────
# Data-aware behaviour:
# - The Governance payload is the source of Governance facts.
# - This uploaded Governance payload contains governance, board_minutes and reporting_kpis.
# - It does not always contain climate_risk_register. If risk-register rows are absent,
#   the Management Responsibility subsection must be limited to the management committee,
#   board reporting frequency, ERM integration flag and major-transaction climate check.
# - The writer must not invent formal committee charters, trade-offs, escalation thresholds,
#   skills adequacy assessments, or assurance over financed emissions.

def _is_present(value) -> bool:
    return value is not None and str(value).strip().lower() not in {"", "nan", "none", "null"}


def _safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _normalise_text(value) -> str:
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _format_reporting_frequency(value: str | None) -> str | None:
    if not _is_present(value):
        return None
    return str(value).replace("_", "-").lower()


def extract_management_process_evidence(payload: dict, year: int = 2024) -> dict:
    """
    Summarise Management Responsibility evidence.

    Preferred source: climate_risk_register, when present.
    Fallback source: governance table fields in the Governance payload.
    """
    risks = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
    ]

    gov_records = payload.get("governance", [])
    gov_2024 = next(
        (
            r for r in gov_records
            if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year
        ),
        {},
    )

    governance_controls_available = any(
        _is_present(gov_2024.get(field))
        for field in [
            "management_committee_name",
            "climate_risk_reporting_to_board",
            "erm_integration_flag",
            "major_transactions_climate_check",
        ]
    )

    if not risks:
        return {
            "risk_register_available": False,
            "governance_controls_available": governance_controls_available,
            "reporting_year": year,
            "management_committee_name": gov_2024.get("management_committee_name"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "formal_escalation_thresholds_available": False,
            "message": (
                "The available Governance evidence does not contain climate_risk_register records. "
                "Management Responsibility can be described only using governance-level evidence."
            ),
            "process_flow_instruction": (
                "Do not claim a risk-register workflow, risk counts, risk categories, scenario links, "
                "monitoring frequencies by risk, or mitigation actions unless risk-register evidence is present. "
                "Use the available governance controls only: management committee name, board reporting frequency, "
                "ERM integration flag and major-transaction climate check."
            ),
        }

    frequencies = sorted({
        str(r.get("monitoring_frequency"))
        for r in risks
        if _is_present(r.get("monitoring_frequency"))
    })
    risk_categories = sorted({
        str(r.get("risk_category"))
        for r in risks
        if _is_present(r.get("risk_category"))
    })
    risk_ratings = sorted({
        str(r.get("risk_rating"))
        for r in risks
        if _is_present(r.get("risk_rating"))
    })
    scenario_links = sorted({
        str(r.get("scenario_analysis_link"))
        for r in risks
        if _is_present(r.get("scenario_analysis_link"))
    })
    mitigation_actions = sorted({
        str(r.get("mitigation_actions"))
        for r in risks
        if _is_present(r.get("mitigation_actions"))
    })

    integrated_count = sum(1 for r in risks if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks if r.get("changed_since_prior_period") is True)

    rating_priority = {"critical": 4, "high": 3, "medium": 2, "low": 1}
    sorted_risks = sorted(
        risks,
        key=lambda r: (
            rating_priority.get(str(r.get("risk_rating", "")).lower(), 0),
            float(r.get("financial_impact_meur") or 0),
        ),
        reverse=True,
    )

    material_risk_examples = []
    for r in sorted_risks[:5]:
        material_risk_examples.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "financial_impact_meur": r.get("financial_impact_meur"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
        })

    return {
        "risk_register_available": True,
        "governance_controls_available": governance_controls_available,
        "reporting_year": year,
        "management_committee_name": gov_2024.get("management_committee_name"),
        "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
        "erm_integration_flag": gov_2024.get("erm_integration_flag"),
        "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
        "risk_count": len(risks),
        "erm_integrated_count": integrated_count,
        "changed_since_prior_period_count": changed_count,
        "monitoring_frequencies": frequencies,
        "risk_categories": risk_categories,
        "risk_ratings": risk_ratings,
        "scenario_analysis_links": scenario_links,
        "mitigation_actions": mitigation_actions[:8],
        "material_risk_examples": material_risk_examples,
        "formal_escalation_thresholds_available": False,
        "process_flow_instruction": (
            "Write management responsibility as a process flow only if risk-register evidence is present: "
            "risk identification/register, classification by category/time horizon/rating, monitoring frequency, "
            "scenario links, mitigation actions and ERM integration. Do not invent formal escalation thresholds."
        ),
    }


def extract_governance_evidence(payload: dict) -> dict:
    gov_records = payload.get("governance", [])
    board_minutes = payload.get("board_minutes", [])
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})
    metadata = payload.get("metadata", {})

    reporting_year = int(metadata.get("reporting_year", 2024))

    gov_by_year = {
        str(r["reporting_year"]): r
        for r in gov_records
        if isinstance(r, dict) and "reporting_year" in r
    }

    gov_trend = []
    for year in ["2022", "2023", "2024"]:
        if year in gov_by_year:
            g = gov_by_year[year]
            gov_trend.append({
                "year": int(year),
                "esg_committee_meetings": g.get("esg_committee_meetings_per_year"),
                "board_climate_expertise_pct": g.get("board_climate_expertise_pct"),
                "ceo_esg_compensation_pct": g.get("ceo_esg_compensation_pct"),
                "all_exec_climate_remuneration_pct": g.get("all_exec_climate_remuneration_pct"),
                "climate_on_board_agenda_pct": g.get("climate_on_board_agenda_pct"),
                "management_committee_name": g.get("management_committee_name"),
                "board_full_meeting_frequency": g.get("board_full_meeting_frequency"),
            })

    PRIORITY_TOPICS = [
        "esg report", "scenario analysis", "transition plan", "net-zero",
        "net zero", "target", "carbon credit", "physical risk", "green finance"
    ]

    minutes_2024 = [
        m for m in board_minutes
        if isinstance(m, dict)
        and _safe_int(m.get("reporting_year")) == reporting_year
        and m.get("decision_made_flag") is True
        and _is_present(m.get("decision_summary"))
        and _is_present(m.get("meeting_id"))
    ]

    # Group by decision text so the same decision is not treated as two separate
    # Board/committee decisions when it appears in multiple meeting records.
    grouped_decisions = {}
    for m in minutes_2024:
        key = re.sub(r"\s+", " ", str(m.get("decision_summary", "")).lower().strip())
        if key not in grouped_decisions:
            grouped_decisions[key] = {
                "decision": _normalise_text(m.get("decision_summary")),
                "dates": set(),
                "committees": set(),
                "committee_types": set(),
                "topics_discussed": set(),
                "meeting_ids": set(),
                "ifrs_evidence_paras": set(),
            }

        grouped_decisions[key]["dates"].add(m.get("meeting_date"))
        grouped_decisions[key]["committees"].add(m.get("committee_name"))
        grouped_decisions[key]["committee_types"].add(m.get("committee_type"))
        grouped_decisions[key]["meeting_ids"].add(m.get("meeting_id"))
        grouped_decisions[key]["ifrs_evidence_paras"].add(m.get("ifrs_s2_para_evidence"))
        topics = str(m.get("climate_topics_discussed") or "").split("|")
        grouped_decisions[key]["topics_discussed"].update(t for t in topics if _is_present(t))

    selected_decisions = []
    for item in grouped_decisions.values():
        dates = sorted(x for x in item["dates"] if _is_present(x))
        committees = sorted(x for x in item["committees"] if _is_present(x))
        topics = sorted(x for x in item["topics_discussed"] if _is_present(x))
        meeting_ids = sorted(x for x in item["meeting_ids"] if _is_present(x))
        decision = item["decision"]
        score = sum(1 for term in PRIORITY_TOPICS if term in decision.lower() or term in " ".join(topics).lower())
        selected_decisions.append({
            "primary_date": dates[0] if dates else None,
            "dates": dates,
            "committees": committees,
            "committee_types": sorted(x for x in item["committee_types"] if _is_present(x)),
            "topics_discussed": topics,
            "decision": decision,
            "ifrs_evidence_paras": sorted(x for x in item["ifrs_evidence_paras"] if _is_present(x)),
            "internal_refs": [f"[REF:{mid}]" for mid in meeting_ids],
            "decision_group_score": score,
        })

    selected_decisions = sorted(
        selected_decisions,
        key=lambda d: (-d.get("decision_group_score", 0), d.get("primary_date") or "")
    )[:6]

    gov_2024 = gov_by_year.get(str(reporting_year), {})

    # Conservative evidence-gap assessment.
    governance_instrument_fields = [
        "committee_charter", "committee_terms_of_reference", "board_mandate",
        "esg_committee_mandate", "formal_climate_mandate",
        "governance_policy_reference", "committee_charter_climate_mandate",
    ]
    formal_mandate_available = any(_is_present(gov_2024.get(f)) for f in governance_instrument_fields)

    tradeoff_terms = [
        "tradeoff", "trade-off", "capital allocation", "profitability",
        "implementation cost", "risk appetite", "competing priority", "competing priorities"
    ]
    tradeoff_decisions = [
        m for m in minutes_2024
        if any(
            term in str(m.get("decision_summary", "")).lower()
            or term in str(m.get("climate_topics_discussed", "")).lower()
            for term in tradeoff_terms
        )
    ]
    board_tradeoff_evidence_available = len(tradeoff_decisions) > 0

    skills_process_fields = [
        "skills_matrix", "skills_assessment_process", "board_skills_review",
        "skills_adequacy_assessment", "director_training_frequency",
        "training_hours", "skills_gap_analysis",
    ]
    skills_adequacy_process_available = any(_is_present(gov_2024.get(f)) for f in skills_process_fields)

    assurance_scope = str(gov_2024.get("assurance_scope", ""))
    financed_emissions_2024 = reporting_kpis.get("financed_emissions_2024_tco2e")
    if isinstance(financed_emissions_2024, (int, float)):
        financed_emissions_2024 = round(financed_emissions_2024)  # avoid false-precision passthrough
    financed_in_scope = "financed" in assurance_scope.lower() or "scope 3" in assurance_scope.lower()

    assurance_scope_limitation = {
        "assurance_scope": assurance_scope,
        "external_assurance": gov_2024.get("external_assurance"),
        "provider": gov_2024.get("assurance_provider"),
        "standard": gov_2024.get("assurance_standard"),
        "financed_emissions_2024_tco2e": financed_emissions_2024,
        "financed_emissions_in_scope": financed_in_scope,
        "instruction": (
            "State that assurance covers only the stated scope. If the stated scope is Scope 1 and 2 emissions, "
            "do not imply financed emissions or other Scope 3 categories are assured. For a bank, explicitly clarify "
            "that financed emissions are outside the stated assurance scope based on available evidence."
        ),
    }

    management_evidence = extract_management_process_evidence(payload, year=reporting_year)

    return {
        "bank": {
            "name": bank.get("bank_name"),
            "bank_id": bank.get("bank_id"),
            "country": bank.get("country"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "reporting_year": reporting_year,
        "comparative_years": metadata.get("comparative_years", [2022, 2023]),
        "payload_profile": {
            "source_payload": "governance",
            "top_level_keys": list(payload.keys()),
            "climate_risk_register_in_payload": "climate_risk_register" in payload,
            "board_minutes_count": len(board_minutes),
            "governance_record_count": len(gov_records),
        },
        "governance_2024": {
            "board_size": gov_2024.get("board_size"),
            "independent_directors_pct": gov_2024.get("independent_directors_pct"),
            "esg_committee_exists": gov_2024.get("esg_committee_exists"),
            "esg_committee_meetings_per_year": gov_2024.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": gov_2024.get("board_climate_expertise_pct"),
            "ceo_compensation_esg_linked": gov_2024.get("ceo_compensation_esg_linked"),
            "ceo_esg_compensation_pct": gov_2024.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": gov_2024.get("all_exec_climate_remuneration_pct"),
            "climate_risk_reporting_to_board": gov_2024.get("climate_risk_reporting_to_board"),
            "climate_on_board_agenda_pct": gov_2024.get("climate_on_board_agenda_pct"),
            "board_full_meeting_frequency": gov_2024.get("board_full_meeting_frequency"),
            "management_committee_name": gov_2024.get("management_committee_name"),
            "erm_integration_flag": gov_2024.get("erm_integration_flag"),
            "skills_development_programme": gov_2024.get("skills_development_programme"),
            "major_transactions_climate_check": gov_2024.get("major_transactions_climate_check"),
            "external_assurance": gov_2024.get("external_assurance"),
            "assurance_provider": gov_2024.get("assurance_provider"),
            "assurance_scope": gov_2024.get("assurance_scope"),
            "assurance_standard": gov_2024.get("assurance_standard"),
            "tcfd_aligned": gov_2024.get("tcfd_aligned"),
            "ifrs_s2_aligned": gov_2024.get("ifrs_s2_aligned"),
        },
        "governance_trend": gov_trend,
        "management_process_evidence": management_evidence,
        "board_decisions_2024": selected_decisions,
        "assurance_context": {
            "financed_emissions_2024_tco2e": financed_emissions_2024,
            "assurance_scope": assurance_scope,
            "financed_emissions_in_scope": financed_in_scope,
        },
        "strict_governance_evidence": {
            "formal_governance_mandate_available": formal_mandate_available,
            "formal_governance_mandate_instruction": (
                "Do not claim the ESG & Sustainability Committee has a formal climate mandate unless charter "
                "or terms-of-reference evidence is provided. If no formal instrument is available, say that "
                "available documentation evidences committee activity and meeting frequency but does not include "
                "the committee charter or terms of reference."
            ),
            "board_tradeoff_evidence_available": board_tradeoff_evidence_available,
            "tradeoff_decisions": tradeoff_decisions[:3],
            "board_tradeoff_instruction": (
                "Discuss board trade-offs only if explicit trade-off evidence exists. If not, state that the board "
                "decision evidence identifies climate-related decisions but does not describe specific trade-offs "
                "such as profitability, capital allocation, implementation cost, risk appetite or competing strategic priorities."
            ),
            "skills_adequacy_process_available": skills_adequacy_process_available,
            "skills_adequacy_instruction": (
                "Use the board climate expertise percentage and skills development programme as outcome/activity evidence. "
                "Do not invent a formal skills adequacy assessment process."
            ),
            "assurance_scope_limitation": assurance_scope_limitation,
        },
        "interpretation_notes": {
            "climate_on_board_agenda_pct": (
                "This figure represents the percentage of board meetings during the year where climate-related topics "
                "appeared on the agenda. It does not mean percentage of agenda time devoted to climate."
            ),
            "management_committee_names": (
                "Committee names are recorded by year only. The evidence does not prove that one committee evolved into, "
                "replaced, or was renamed as another. State the 2024 committee name and, if comparative names are used, "
                "present them neutrally."
            ),
            "major_transactions_climate_check": (
                "A true value indicates evidence of climate checks for major transactions; it does not prove a formal mandatory policy."
            ),
            "board_decision_traceability": (
                "Grouped decisions include internal refs for audit traceability. Do not print these refs in the final report."
            ),
            "avoid_duplication": (
                "Do not list the same board decisions twice. Board oversight should summarise decision governance; "
                "the detailed dated list belongs only in the Board and committee decisions subsection."
            ),
            "carbon_credit_boundary": (
                "Carbon credit procurement may be described only as a specific 2024 budget approval decision unless recurring topic evidence is available."
            ),
            "alignment_boundary": (
                "TCFD/IFRS S2 alignment flags show source-data alignment status only. Do not claim governance oversight supports or ensures alignment unless an alignment control process is evidenced."
            ),
        },
    }


evidence = extract_governance_evidence(governance_payload)

# ── IFRS coverage: items required by IFRS S1/S2 but genuinely ABSENT from the data.
# These are recorded in metadata only (NOT written into the section, NOT scored).
def _find_flag(obj, name):
    if isinstance(obj, dict):
        if name in obj:
            return obj[name]
        for v in obj.values():
            r = _find_flag(v, name)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _find_flag(v, name)
            if r is not None:
                return r
    return None

# Map each evidence availability flag to the IFRS requirement topic it satisfies (by keyword).
# The IFRS reference text/standard/paragraph are EXTRACTED from the requirements catalog,
# not hard-coded here.
GOVERNANCE_FLAG_KEYWORDS = {
    "formal_governance_mandate_available":    ["mandate", "terms of reference", "charter", "responsibilit"],
    "formal_escalation_thresholds_available": ["escalat"],
    "board_tradeoff_evidence_available":      ["trade-off", "tradeoff", "trade off"],
    "skills_adequacy_process_available":      ["skill", "competenc"],
}

def governance_missing_ifrs_items(ev):
    """IFRS core-mandatory governance requirements NOT backed by source data.
    Requirement id/standard/paragraph/text are pulled from the extracted IFRS catalog."""
    try:
        reqs = get_section_ifrs_requirements("governance", include_industry_guidance=False)
    except Exception:
        reqs = []
    core = [r for r in reqs
            if r.get("source_authority") == "core_standard" and r.get("obligation_type") == "shall"]
    out, seen = [], set()
    for r in core:
        txt = (str(r.get("requirement_text", "")) + " " + str(r.get("paragraph", ""))).lower()
        for flag, kws in GOVERNANCE_FLAG_KEYWORDS.items():
            if any(k in txt for k in kws) and not _find_flag(ev, flag):
                rid = r.get("requirement_id") or f"{r.get('standard')}|{r.get('paragraph')}"
                if rid in seen:
                    continue
                seen.add(rid)
                out.append({
                    "requirement_id":   r.get("requirement_id"),
                    "standard":         r.get("standard"),
                    "paragraph":        r.get("paragraph"),
                    "requirement_text": r.get("requirement_text"),
                    "reason":           "required by IFRS but not present in source data",
                })
                break
    return out

_erm_n = _find_flag(evidence, "erm_integrated_count")
_risk_n = _find_flag(evidence, "risk_count")
evidence["approved_phrasing"] = {
    "erm_integration": (
        f"{_erm_n} of {_risk_n} recorded climate-related risks carry an ERM-integration flag; "
        "enterprise-wide integration into the enterprise risk management framework is not evidenced."
        if _erm_n is not None and _risk_n is not None else
        "selected climate-related risks carry an ERM-integration flag; enterprise-wide ERM integration is not evidenced."
    ),
    "governance_metrics": "governance indicators the Bank reports (climate-on-agenda %, board climate-expertise %, climate-linked remuneration %)",
    "transaction_checks": "the Bank performed climate checks for major transactions in 2024",
}

governance_missing_ifrs = governance_missing_ifrs_items(evidence)
print(f"IFRS items required but missing in data (metadata-only): {len(governance_missing_ifrs)}")
for _m in governance_missing_ifrs:
    print("  -", _m.get("standard"), _m.get("paragraph"), "-", (_m.get("requirement_text") or "")[:80])

print(f"Evidence extracted for: {evidence['bank']['name']}")
print(f"Governance payload risk register present: {evidence['payload_profile']['climate_risk_register_in_payload']}")
print(f"Board decisions selected: {len(evidence['board_decisions_2024'])}")
print(f"Trend years: {[t['year'] for t in evidence['governance_trend']]}")
print(f"Management evidence risk register available: {evidence['management_process_evidence'].get('risk_register_available')}")
print("Strict governance evidence flags:")
for k, v in evidence["strict_governance_evidence"].items():
    if isinstance(v, bool):
        print(f"- {k}: {v}")
print("Grouped decisions:")
for d in evidence["board_decisions_2024"]:
    print(f"- {d.get('primary_date')} | {', '.join(d.get('committees', []))} | {d.get('decision')}")

IFRS items required but missing in data (metadata-only): 6
  - IFRS S1 27 - To achieve this objective, an entity shall disclose information about — (a) the 
  - IFRS S1 27 - To achieve this objective, an entity shall disclose information about — (a) the 
  - IFRS S1 27 - To achieve this objective, an entity shall disclose information about — (a) the 
  - IFRS S2 6 - To achieve this objective, an entity shall disclose information about — (a) the 
  - IFRS S2 6 - To achieve this objective, an entity shall disclose information about — (a) the 
  - IFRS S2 6 - To achieve this objective, an entity shall disclose information about — (a) the 
Evidence extracted for: Eurolux Universal Bank AG
Governance payload risk register present: True
Board decisions selected: 5
Trend years: [2022, 2023, 2024]
Management evidence risk register available: True
Strict governance evidence flags:
- formal_governance_mandate_available: False
- board_tradeoff_evidence_available: False
- skills_adequacy_process_a

## 4. Build compact Governance evidence


In [10]:
# ── GOVERNANCE EVIDENCE AVAILABILITY + SAVING ────────────────
# The raw Governance payload remains the source of truth.
# The compact evidence is the agent-ready input used by Writer/Judge/Reviser.

def is_scope1_scope2_only(scope: str | None) -> bool:
    if not scope:
        return False
    s = str(scope).lower()
    s = re.sub(r"\s+", " ", s)
    has_scope1 = bool(re.search(r"\bscope\s*1\b|\bscope1\b", s))
    has_scope2 = bool(
        re.search(r"\bscope\s*2\b|\bscope2\b", s)
        or re.search(r"\bscope\s*1\s*(and|&)\s*2\b", s)
        or re.search(r"\bscope\s*1\s*(and|&)\s*scope\s*2\b", s)
    )
    has_scope3_or_financed = bool(
        re.search(r"\bscope\s*3\b|\bscope3\b|financed emissions|category 15|cat\.?\s*15", s)
    )
    return has_scope1 and has_scope2 and not has_scope3_or_financed

def build_governance_availability_profile(evidence: dict) -> dict:
    gov = evidence.get("governance_2024", {})
    trend = evidence.get("governance_trend", [])
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})
    decisions = evidence.get("board_decisions_2024", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    trend_metrics = {
        "esg_committee_meetings": [
            row.get("esg_committee_meetings") for row in trend
            if present(row.get("esg_committee_meetings"))
        ],
        "board_climate_expertise_pct": [
            row.get("board_climate_expertise_pct") for row in trend
            if present(row.get("board_climate_expertise_pct"))
        ],
        "ceo_esg_compensation_pct": [
            row.get("ceo_esg_compensation_pct") for row in trend
            if present(row.get("ceo_esg_compensation_pct"))
        ],
        "all_exec_climate_remuneration_pct": [
            row.get("all_exec_climate_remuneration_pct") for row in trend
            if present(row.get("all_exec_climate_remuneration_pct"))
        ],
        "climate_on_board_agenda_pct": [
            row.get("climate_on_board_agenda_pct") for row in trend
            if present(row.get("climate_on_board_agenda_pct"))
        ],
        "board_full_meeting_frequency": [
            row.get("board_full_meeting_frequency") for row in trend
            if present(row.get("board_full_meeting_frequency"))
        ],
    }

    assurance_scope = str(gov.get("assurance_scope") or "").lower()
    financed_emissions = evidence.get("assurance_context", {}).get(
        "financed_emissions_2024_tco2e"
    )

    risk_register_available = bool(management.get("risk_register_available"))
    governance_controls_available = bool(management.get("governance_controls_available"))

    return {
        "board_core_metrics_available": all(
            present(gov.get(field))
            for field in [
                "board_size",
                "independent_directors_pct",
                "esg_committee_meetings_per_year",
                "climate_risk_reporting_to_board",
                "climate_on_board_agenda_pct",
            ]
        ),
        "trend_metrics_available": {
            name: len(values) >= 2
            for name, values in trend_metrics.items()
        },
        "selected_decision_count": len(decisions),
        "board_decisions_available": len(decisions) > 0,
        "formal_governance_mandate_available": bool(
            strict.get("formal_governance_mandate_available")
        ),
        "board_tradeoff_evidence_available": bool(
            strict.get("board_tradeoff_evidence_available")
        ),
        "management_process_available": risk_register_available or governance_controls_available,
        "management_risk_register_available": risk_register_available,
        "management_governance_controls_available": governance_controls_available,
        "formal_escalation_thresholds_available": bool(
            management.get("formal_escalation_thresholds_available", False)
        ),
        "skills_outcome_metrics_available": present(
            gov.get("board_climate_expertise_pct")
        ),
        "skills_development_programme_available": bool(
            gov.get("skills_development_programme")
        ),
        "skills_adequacy_process_available": bool(
            strict.get("skills_adequacy_process_available")
        ),
        "remuneration_evidence_available": (
            present(gov.get("ceo_esg_compensation_pct"))
            and present(gov.get("all_exec_climate_remuneration_pct"))
        ),
        "assurance_evidence_available": all(
            present(gov.get(field))
            for field in [
                "external_assurance",
                "assurance_provider",
                "assurance_scope",
                "assurance_standard",
            ]
        ),
        "assurance_scope_is_scope1_scope2_only": is_scope1_scope2_only(gov.get("assurance_scope")),
        "financed_emissions_available_for_scope_context": present(
            financed_emissions
        ),
        "payload_boundary": {
            "governance_payload_has_climate_risk_register": evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"),
            "governance_payload_tables": evidence.get("payload_profile", {}).get("top_level_keys", []),
        },
        "writer_policy": {
            "use_available_evidence": (
                "Use every material governance evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material governance requirement is not supported by the available evidence, state the boundary once in the relevant subsection. Do not invent the missing process or control."
            ),
            "management_process_boundary": (
                "If climate_risk_register is unavailable, describe only the management committee, board reporting frequency, ERM integration flag and major-transaction climate check. If it is available, describe the risk-register process but do not invent formal escalation thresholds."
            ),
            "wording": (
                "Write in the Bank's own voice and state facts directly. Do not refer to the report's evidence base: never use 'available evidence', 'available documentation', 'documentation reviewed', 'source data', 'internally generated analytical outputs' or 'payload' in the final disclosure. Frame genuine gaps as the Bank's position - 'the Bank does not currently disclose X', 'X is outside the scope of this report', or 'X has not yet been quantified'."
            ),
        },
    }


def add_governance_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_governance_availability_profile(evidence)
    source_tables = [
        "bank",
        "governance",
        "board_minutes",
        "reporting_kpis",
    ]
    if evidence.get("payload_profile", {}).get("climate_risk_register_in_payload"):
        source_tables.append("climate_risk_register")

    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": source_tables,
        "selected_board_decision_refs": [
            item.get("internal_ref")
            for item in evidence.get("board_decisions_2024", [])
            if item.get("internal_ref")
        ],
        "management_risk_refs": [
            item.get("risk_id")
            for item in evidence.get("management_process_evidence", {}).get(
                "material_risk_examples", []
            )
            if item.get("risk_id")
        ],
    }
    return evidence


evidence = add_governance_traceability(evidence, GOVERNANCE_PAYLOAD_PATH)

raw_governance_payload = {
    key: governance_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "governance",
        "board_minutes",
        "climate_risk_register",
        "reporting_kpis",
    ]
    if key in governance_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_GOVERNANCE_PATH = output_dir / "payload_BANK01_governance_raw.json"
COMPACT_GOVERNANCE_PATH = output_dir / "compact_governance_evidence_BANK01.json"

with open(RAW_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_governance_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_GOVERNANCE_PATH, "w", encoding="utf-8") as f:
    json.dump(evidence, f, indent=2, ensure_ascii=False)

print("Governance evidence prepared")
print(f"- Raw Governance payload: {RAW_GOVERNANCE_PATH}")
print(f"- Compact Governance evidence: {COMPACT_GOVERNANCE_PATH}")
print("- Availability profile:")
print(json.dumps(evidence["availability_profile"], indent=2, ensure_ascii=False))

Governance evidence prepared
- Raw Governance payload: outputs\payload_BANK01_governance_raw.json
- Compact Governance evidence: outputs\compact_governance_evidence_BANK01.json
- Availability profile:
{
  "board_core_metrics_available": true,
  "trend_metrics_available": {
    "esg_committee_meetings": true,
    "board_climate_expertise_pct": true,
    "ceo_esg_compensation_pct": true,
    "all_exec_climate_remuneration_pct": true,
    "climate_on_board_agenda_pct": true,
    "board_full_meeting_frequency": true
  },
  "selected_decision_count": 5,
  "board_decisions_available": true,
  "formal_governance_mandate_available": false,
  "board_tradeoff_evidence_available": false,
  "management_process_available": true,
  "management_risk_register_available": true,
  "management_governance_controls_available": true,
  "formal_escalation_thresholds_available": false,
  "skills_outcome_metrics_available": true,
  "skills_development_programme_available": true,
  "skills_adequacy_process_avai

## 5. Add traceability and save Governance evidence


In [11]:
# ── STATE DEFINITION ─────────────────────────────────────────
class GovernanceState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

## 6. Governance state definition


In [12]:
# ── GOVERNANCE REQUIREMENTS ─────────────────────────────────
# IFRS S1/S2 references are used internally only. Final text must not show paragraph tags.
# Structure is adapted to a dedicated banking IFRS S1/S2 disclosure report style.

IFRS_GOVERNANCE_REQUIREMENTS = """
STRICT GOVERNANCE DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Final section title and headings must be exactly:
### Governance
#### Overview
#### The role of the Board of Directors
#### Board committees and climate-related oversight
#### Management responsibility for climate-related risks and opportunities
#### Skills, competencies and remuneration
#### Governance decisions, controls and evidence boundaries

Decision placement and anti-duplication rules:
- Do not mention specific 2024 decision topics in the Board or Committee subsections.
- In "The role of the Board of Directors" and "Board committees and climate-related oversight", refer only generally to documented 2024 Board and committee decisions.
- Place all decision dates and specific decision topics only in "Governance decisions, controls and evidence boundaries".
- Decision topics that must not appear before the final subsection include: net-zero target revision, transition plan endorsement/update, climate scenario analysis methodology approval, carbon credit procurement budget, ESG report approval, TCFD review, executive remuneration ESG KPI calibration.

Precision wording rules:
- Avoid broad phrases such as "multi-layer governance structure", "risk and control processes", "governance and control processes", "focused review and challenge", "strengthening competence", and "operates a climate governance structure" unless immediately tied to evidenced mechanisms.
- Prefer evidence-anchored wording such as: "Based on available evidence, climate-related governance is reflected in Board oversight, ESG & Sustainability Committee activity, executive management responsibility, semi-annual Board reporting, ERM integration flags, climate checks for major transactions and a climate risk register."
- Do not describe alignment indicators in the Overview. Discuss TCFD/IFRS S2 alignment flags only in the final boundaries subsection.
- When mentioning alignment flags, explicitly state that source data includes alignment indicators and that available documentation does not evidence a separate governance control process designed to verify or assure that alignment.

Specific wording controls:
- Keep semi-annual Board reporting separate from individual Board/committee decision evidence. Do not imply that net-zero target revisions, transition-plan updates or scenario-methodology approvals were part of the semi-annual reporting pack unless that linkage is explicitly evidenced.
- Use evidence-anchored risk-register wording such as "uses a climate risk register" or "documents climate-related risks in a climate risk register". Avoid "maintains a dedicated climate risk register" because it implies an ownership/control process not directly evidenced.
- Avoid broad wording such as "supporting processes embedded in risk management and control frameworks". Prefer specific evidenced wording such as "supporting processes reflected in ERM integration flags, climate risk reporting and climate checks for major transactions".

Presentation style:
- Write like a dedicated bank IFRS S1/S2 sustainability disclosure report, not like a checklist answer.
- Start with an overview of how climate-related governance is organised across the Board, committees, management and control functions.
- Use smooth report-style wording.
- Do not overuse the phrase "available evidence" in the main narrative. Use "source data", "available documentation" or "documentation reviewed" mainly for limitations and evidence boundaries.

Internal IFRS S2 Governance coverage map:
1. Governance mandate / terms of reference / role descriptions:
   - If formal mandate evidence is available, describe it.
   - If unavailable, state that available documentation evidences governance activity and meeting frequency but does not include a separate climate-specific charter, terms of reference or formal mandate.
2. Board oversight:
   - Use board size, independent-director percentage, full Board meeting frequency and climate agenda frequency.
   - Use the three-year trend for full Board meetings and climate-on-board-agenda frequency.
   - Explain that Board oversight is evidenced through agenda coverage, reporting frequency, Board/committee decisions and governance KPIs.
3. Board committees and climate-related oversight:
   - Use ESG Committee meeting counts and trends.
   - Use grouped 2024 Board/committee decisions without duplicating a long dated list in earlier subsections.
   - Use target oversight and transition-plan/scenario-methodology approvals only where evidenced.
4. Skills and competencies:
   - Use board climate expertise percentage and the skills development programme.
   - If no formal skills adequacy assessment is available, state this boundary.
5. Remuneration:
   - Use CEO ESG-linked compensation percentage and all-executive climate-linked remuneration percentage.
   - Include the three-year trend for remuneration metrics where available.
6. Trade-offs:
   - Use only explicit trade-off evidence.
   - If unavailable, state that documented decisions do not describe quantified or specific Board-level trade-offs.
7. Management responsibility:
   - Identify the management committee responsible for climate-related risks and opportunities.
   - Use reporting frequency to the Board, ERM integration flag, major-transaction climate check and climate risk register evidence where available.
8. Management controls and procedures:
   - If risk register evidence is available, describe the process at a summary level: risk identification/register, categorisation, monitoring frequency, scenario links, mitigation actions and ERM integration.
   - If unavailable, limit the description to the management committee, Board reporting, ERM integration and major-transaction climate checks.
   - In both cases, do not invent formal escalation thresholds.
9. Assurance and controls:
   - Describe assurance only for the stated scope.
   - If assurance scope is Scope 1 and Scope 2 emissions, state that financed emissions / Scope 3 are outside the stated assurance scope when financed-emissions context is available.
10. Evidence boundaries:
   - The final subsection must include concise boundaries for unavailable formal mandate/charter evidence, Board trade-off evidence, skills adequacy assessment, escalation thresholds, and assurance scope where applicable.

Data-aware coverage requirements:
1. Use the Governance evidence as the source of governance facts.
2. If climate_risk_register is present, Management responsibility must describe the evidenced risk-register process.
3. If climate_risk_register is absent, Management responsibility must be limited to management committee name, climate risk reporting frequency, ERM integration flag and major-transaction climate check flag.
4. Do not invent formal committee charters, terms of reference, formal governance mandates, formal escalation thresholds, Board trade-off analysis or formal skills adequacy assessment.
5. Use year-on-year trends for available metrics: ESG committee meetings 5→6→7; board climate expertise 33.5%→35.5%→37.5%; CEO ESG-linked compensation 6.3%→7.8%→9.3%; all-executive climate-linked remuneration 8.7%→15.3%→15.1%; climate agenda frequency 69.8%→73.1%→72.6%; full Board meetings 4→7→8.
6. Carbon credit procurement may be described only as a specific 2024 budget approval decision unless recurring topic evidence is available.
7. Alignment flags may be disclosed only as source-data flags. Do not claim governance oversight supports or ensures TCFD/IFRS S2 alignment unless an alignment control process is evidenced.
8. Do not include visible IFRS paragraph references, meeting IDs, internal refs or bracketed IFRS tags in the final output.
9. Do not use the technical word "payload" in the final report.
""".strip()

print("Governance requirements ready — Emirates-style structure")

Governance requirements ready — Emirates-style structure


## 7. Governance requirements — Emirates-style structure


In [13]:
# ── GOVERNANCE WRITER PROMPT ─────────────────────────────────
WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Governance section of a dedicated IFRS S1/S2-style climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style similar to a bank IFRS S1/S2 disclosure report.

Evidence rules:
- Translate raw data field names into public report wording; avoid snake_case and key=value syntax in the final section.
- Never end the section mid-sentence; complete every required subsection, especially the final evidence-boundary subsection.
- Use the clean style guide only for writing style, tone, structure and formatting; never use it as factual evidence.
- Use only the compact Governance evidence supplied by the user prompt.
- Do not invent missing governance policies, committee charters, trade-offs, escalation thresholds, risk-register workflows, assurance coverage or skills assessment processes.
- When evidence is missing, state any genuine gap plainly in the Bank's own voice as the Bank's position - for example "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references, meeting IDs, internal references or bracketed evidence tags.
- Do not use markdown tables.

Style rules:
- Write like a report section, not like an audit checklist.
Anti-duplication and precision rules:
- In the Overview, Board and Committee subsections, do not list or name specific 2024 decision topics such as net-zero target revisions, transition plan endorsements, climate scenario methodology, carbon credit procurement budget, ESG report approval, TCFD review or executive remuneration KPI calibration.
- In those earlier subsections, write only: "documented 2024 Board and committee decisions" or similar general wording.
- Put all decision dates and specific decision topics only in the final "Governance decisions, controls and evidence boundaries" subsection.
- Do not mention TCFD/IFRS S2 alignment indicators in the Overview. Mention them only in the final boundaries subsection.
- When mentioning alignment indicators, use this meaning: source data includes alignment indicators, but available documentation does not evidence a separate governance control process designed to verify or assure that alignment.
- Avoid "multi-layer governance structure", "risk and control processes", "governance and control processes", "focused review and challenge", and "strengthening competence".
- Use this safer Overview wording pattern: "Based on available evidence, climate-related governance is reflected in Board oversight, ESG & Sustainability Committee activity, executive management responsibility, semi-annual Board reporting, ERM integration indicators, climate checks for major transactions and a climate risk register."
Precision wording rules:
- Do not write that Board reporting is "supplemented by" net-zero, transition-plan or scenario-methodology decisions. State semi-annual Board reporting and separate 2024 decision evidence as two distinct channels.
- Do not use "maintains a dedicated climate risk register". Use: "uses a climate risk register that, in 2024, recorded 8 climate-related risks" when risk-register evidence is available.
- Do not use "supporting processes embedded". Use: "supporting processes reflected in ERM integration indicators, climate risk reporting and climate checks for major transactions".
- Do not state that decision items are part of a reporting pack unless the evidence explicitly says so.
- The section must open with an "Overview" explaining the governance model across Board, committees, management and controls.
- Do not overuse "available evidence" in every paragraph. Use it mainly when describing limitations.
- Use smooth banking-report language: "oversight is reflected in", "the governance structure includes", "source data indicates", "documentation reviewed does not specify".
- Avoid strong interpretive verbs such as "demonstrates", "ensures", "supports", "confirms" or "drives" unless the evidence directly proves the mechanism.
- Prefer "indicates", "is evidenced by", "is reflected in" or "source data shows".
- Do not claim that governance oversight supports or ensures TCFD/IFRS S2 alignment; state only that source data flags alignment if relevant.

Additional precision rules (avoid the recurring overclaims):
- When describing monitoring, do not imply a broader or fully designed control system. Refer only to the evidenced mechanisms: the climate risk register, recorded monitoring frequencies, the ERM-integration indicator and the major-transactions climate check. Avoid wording that suggests a comprehensive control framework beyond these.
- Do not state that mitigation actions are recorded "for each risk" or "for every risk". The evidence supports a register-level set of mitigation actions, not a per-risk mapping. Describe them as mitigation actions recorded in the register, without attributing a specific action to every risk.

- External assurance is limited to Scope 1 and Scope 2 greenhouse gas emissions only. Never describe assurance as covering "selected climate-related metrics", financed emissions or other Scope 3; state that assurance covers Scope 1 and Scope 2 emissions and that financed emissions and other Scope 3 are outside the assurance scope.

- Place specific climate-related decision topics and dates ONLY in the final "Governance decisions, controls and evidence boundaries" subsection. In the Board-role and committee subsections, refer generically to the fact that documented climate-related decisions were taken in 2024 — do not restate the individual decision topics or dates there.

- Do not assert that the Board, a committee, or a management committee "is responsible for" specific duties, "monitors", or "oversees" particular activities as defined accountabilities when no charter, terms of reference or mandate is evidenced. Frame these as evidenced activities and reporting lines instead - for example "available documentation indicates the committee prepared/discussed/considered ..." and "the committee reports to the Board" - rather than "is responsible for" or "monitors ... within its remit".

REPORTING VOICE (this overrides any other wording guidance, including wording in the evidence): Write strictly as the Bank's own published disclosure, in the Bank's voice, and state facts directly. The following are BANNED from the final section: "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", "based on available evidence/documentation", and any phrasing that narrates what "the documentation" or "the evidence" does or does not show. Write "The Board met eight times in 2024", never "Available documentation indicates the Board met eight times". For genuine gaps, write the Bank's position: "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".

- CRITICAL - decision placement: outside the final 'Governance decisions, controls and evidence boundaries' subsection, do NOT name any specific 2024 decision or decision topic - including the transition plan, (interim) net-zero targets, the scenario-analysis methodology, the carbon-credit or procurement budget, the ESG report/disclosures approval, remuneration KPI calibration or capital allocation - and do NOT use forward-references that name them (for example, never write 'decisions on the transition plan and interim net-zero targets described in the final subsection'). Where you need to evidence Board oversight of targets, write generically, for example: 'The Board oversees the setting and monitoring of climate-related targets through its consideration of management reports and its documented climate-related decisions in 2024, which are set out in the Governance decisions subsection.' The specific decisions, their topics and their dates appear only in that final subsection.
- Describe the semi-annual climate risk reporting to the Board only as providing an overview of material climate-related risks drawn from the climate risk register. Do NOT enumerate the contents of the Board reporting pack (such as categorisation, risk ratings, time horizons, monitoring frequencies, links to scenario analysis or mitigation actions) as the specific items the Board is informed about, and do NOT state that climate-related items are reflected in capital planning. The only capital-related linkage that may be mentioned is the recorded mitigation action applying climate scenario stress testing within the ICAAP.
- When referring to integration into the enterprise risk management framework, qualify it with the evidenced basis (the integration flags recorded in the climate risk register, for example that six of the eight recorded risks are integrated into enterprise risk management). Do NOT present this as a comprehensive or bank-wide integration of climate risk across the full ERM framework, and do NOT list 'integration of climate-related risks into the enterprise risk management framework' among formalised controls and procedures.
- Do NOT state or imply that the Bank integrates climate-related governance into its overall, broader or enterprise corporate governance framework, or that climate governance is embedded enterprise-wide. Describe only the specific evidenced mechanisms (Board oversight and agenda coverage, committee existence and activity, management responsibility, semi-annual reporting, the climate risk register, the per-risk ERM integration flags, major-transaction climate checks and remuneration linkages) without asserting an overarching or enterprise-wide integration.
- Do NOT attribute to the ESG & Sustainability Committee a standing or ongoing pre-submission review role over targets, methodologies or plans. State only what the committee considered or reviewed in 2024 as recorded in the documented decisions (for example "the committee considered ... in 2024"); do not describe a defined recurring process of reviewing items before they are submitted to the Board.
- When presenting multi-year metric trends, state the recorded values for each year and do NOT assert a consistent or sustained direction (increase, growth, upward trend, recurring presence) unless every year-on-year step moves the same way. For non-monotonic series (for example all-executive climate-linked remuneration of 8.7% in 2022, 15.3% in 2023 and 15.1% in 2024), present the figures without claiming a sustained increase, and avoid causal or qualitative conclusions such as "this trend indicates ...".
- Describe climate checks for major transactions only as applied to major transactions, as recorded. Do not say they 'provide for' climate considerations being taken into account across significant credit or investment decisions, and do not assert breadth across both credit and investment decision-making beyond what is recorded.
- Attribute semi-annual climate risk reporting to the Board to management generally (for example 'Climate risk reporting is provided to the Board on a semi-annual basis'); do not attribute it specifically to the Climate Risk Management Committee unless that committee is identified as the provider.
- Do not assert a causal link between the reporting cadence and specific decisions such as major transactions (avoid wording like 'the reporting cadence enables the Board to take climate-related factors into account in decisions on major transactions').
- Where the climate risk register evidence includes the number of risks that changed since the prior period (changed_since_prior_period_count), state it (for example 'four of the eight climate-related risks changed since the prior period').

Output rules:
- Return only the complete Governance section.
- Keep exactly the six required subsections.
""".strip()


def build_writer_prompt(evidence: dict, judge_feedback: str = None) -> str:
    profile = evidence.get("availability_profile", {})
    gov = evidence.get("governance_2024", {})
    management = evidence.get("management_process_evidence", {})
    strict = evidence.get("strict_governance_evidence", {})

    instructions = []

    instructions.append(
        "- Use board size, independent-director percentage, full-board meeting count, climate agenda percentage, committee meeting count and board reporting frequency."
    )
    instructions.append(
        "- Include the available three-year trend narrative for ESG committee meetings, board climate expertise, CEO ESG-linked pay, all-executive climate-linked pay, climate agenda frequency and full Board meeting frequency."
    )
    instructions.append(
        "- Use the Emirates-style adapted structure: Overview; role of the Board; Board committees; Management responsibility; Skills/competencies/remuneration; Decisions/controls/boundaries."
    )
    instructions.append(
        "- Keep the detailed dated decisions in the final Governance decisions, controls and evidence boundaries subsection. Do not duplicate the full list earlier."
    )
    instructions.append(
        "- Do not name specific 2024 decision topics in the Overview, Board, or Committee subsections. Refer only generally to documented 2024 Board and committee decisions there."
    )
    instructions.append(
        "- Put all decision dates and all specific decision topics only in the final Governance decisions, controls and evidence boundaries subsection."
    )
    instructions.append(
        "- Mention TCFD/IFRS S2 alignment indicators only in the final boundaries subsection, together with the boundary that no separate governance control process designed to verify or assure alignment is evidenced."
    )
    instructions.append(
        "- Avoid broad phrases such as multi-layer governance structure, risk and control processes, governance and control processes, focused review and challenge, and strengthening competence."
    )
    instructions.append(
        "- Keep semi-annual Board reporting separate from 2024 Board/committee decisions. Do not imply that decision items formed part of the semi-annual reporting pack."
    )
    instructions.append(
        "- Use evidence-anchored wording for the risk register: 'uses a climate risk register that, in 2024, recorded 8 climate-related risks'. Avoid 'maintains a dedicated climate risk register'."
    )
    instructions.append(
        "- Replace broad process language with evidenced mechanisms: ERM integration indicators, climate risk reporting and climate checks for major transactions."
    )

    if profile.get("formal_governance_mandate_available"):
        instructions.append("- Describe the formal governance mandate using the supplied direct evidence.")
    else:
        instructions.append(
            "- Committee and Board activity are evidenced, but no separate climate-specific committee charter/terms of reference/formal mandate is available. State this boundary once, preferably in the final boundaries subsection."
        )

    if profile.get("board_tradeoff_evidence_available"):
        instructions.append("- Describe only the documented Board trade-offs included in evidence.")
    else:
        instructions.append(
            "- Board decisions are evidenced, but quantified or specific Board-level trade-offs are not documented. State this boundary once without inventing trade-offs."
        )

    if profile.get("management_risk_register_available"):
        instructions.append(
            "- Management responsibility must describe the available risk-register process: eight 2024 risks, risk categories, risk ratings, time horizons, monitoring frequencies, scenario links, mitigation actions and ERM integration count."
        )
    elif profile.get("management_governance_controls_available"):
        instructions.append(
            "- Management responsibility must NOT describe a risk-register workflow. Use only the Climate Risk Management Committee, semi-annual Board reporting, ERM integration indicator and major-transaction climate check."
        )
    else:
        instructions.append(
            "- Management process evidence is not available. State the boundary without inventing a process."
        )

    if profile.get("formal_escalation_thresholds_available"):
        instructions.append("- Describe formal escalation thresholds/routes exactly as evidenced.")
    else:
        instructions.append(
            "- Formal escalation thresholds are not evidenced. Use boundary wording such as: documentation reviewed does not evidence formal escalation thresholds or trigger-based escalation mechanics."
        )

    if profile.get("skills_adequacy_process_available"):
        instructions.append("- Describe the formal Board skills adequacy assessment process from evidence.")
    else:
        instructions.append(
            "- Use Board climate expertise percentage and skills development programme only. State that no formal Board skills adequacy assessment process is documented. Do not invent detailed training topics."
        )

    if profile.get("remuneration_evidence_available"):
        instructions.append(
            "- Use both CEO ESG-linked remuneration percentage and all-executive climate-linked remuneration percentage, including available comparative trend values."
        )

    if profile.get("assurance_evidence_available"):
        instructions.append(
            "- Describe assurance using exact provider, level, standard and scope."
        )
        if profile.get("assurance_scope_is_scope1_scope2_only"):
            instructions.append(
                "- Clarify that assurance covers Scope 1 and Scope 2 emissions only, and financed emissions / Scope 3 are outside the stated assurance scope."
            )

    section_blueprint = """
SECTION BLUEPRINT:
### Governance

#### Overview
Summarise the climate governance architecture: Board oversight, Board/committee activity, management responsibility, risk-register/ERM linkage if available, skills/remuneration and controls. Keep it concise.
Do NOT assert that climate governance is integrated into the Bank's overall corporate governance framework or that climate risk is integrated across the full enterprise risk management framework; list only the evidenced mechanisms, and where ERM is mentioned carry the per-risk integration-flag qualifier.

#### The role of the Board of Directors
Use Board size, independence, Board meeting frequency, climate agenda frequency, Board reporting frequency and Board oversight facts. Include relevant trend values naturally. Refer only generally to documented 2024 Board decisions without naming decision topics.

#### Board committees and climate-related oversight
Use ESG Committee meeting trend and committee oversight facts. Refer only generally to documented committee decisions without naming decision topics. If committee charters or terms of reference are unavailable, omit them entirely (do not state a boundary).

#### Management responsibility for climate-related risks and opportunities
Use management committee name, reporting frequency to the Board, ERM integration, major-transaction climate check and climate risk register process if available. Use 'uses a climate risk register that, in 2024, recorded 8 climate-related risks' rather than 'maintains a dedicated climate risk register'.

#### Skills, competencies and remuneration
Use Board climate expertise trend, skills development evidence, CEO ESG-linked remuneration and all-executive climate-linked remuneration. If a formal skills-adequacy process is unavailable, omit it (do not state a boundary).

#### Governance decisions, controls and evidence boundaries
Include all dated/grouped 2024 decisions and all specific decision topics here only, plus assurance scope and governance controls. Do NOT include data-availability boundary statements for missing formal mandate, trade-offs, skills-adequacy assessment, escalation thresholds or alignment-control process; these genuinely-missing items are recorded in metadata, not in the body.
""".strip()

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION RULES:
- Fix the judge's valid issues using available evidence.
- Preserve correct evidence-boundary statements.
- Never invent unavailable information.
""".strip()

    return f"""
{IFRS_GOVERNANCE_REQUIREMENTS}

WRITING STYLE GUIDE — STYLE ONLY, NOT EVIDENCE:
{build_style_context("governance")}

STYLE APPLICATION RULES:
- Use the style guide only for tone, flow, structure and formatting.
- Use compact Governance evidence for facts.
- Use IFRS requirements only to decide what topics to cover.
- Do not copy reference-report wording or mention forbidden reference terms.

BANK:
{evidence['bank']['name']} ({evidence['bank']['country']})
REPORTING YEAR: {evidence['reporting_year']}
COMPARATIVE YEARS: {evidence['comparative_years']}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

{section_blueprint}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("governance")}

REQUIREMENT-TO-EVIDENCE POLICY:
- Use IFRS requirements to decide which disclosure topics must be addressed.
- Use compact bank evidence to write bank-specific facts.
- If an IFRS requirement is relevant but the bank evidence is unavailable, OMIT that topic from the section entirely. Do NOT write an evidence-boundary statement, a "the Bank does not disclose..." sentence, or any mention of it. These genuinely-missing items are tracked separately in metadata, not in the report body.
- Stay strictly at the level of detail the evidence provides. When the evidence only names, flags or counts something (a reporting cadence, a committee, a control, a register, an integration flag), do NOT describe its contents, its internal process, what it covers, or any surrounding framework. For example: do not describe what a Board reporting pack contains, do not describe a committee's review/approval process, and do not assert control functions or a governance KPI framework, unless the evidence states those specifics.
- SPECIFIC AND IMPORTANT: Do NOT attribute the contents of the climate risk register (quantified financial impacts, mitigation/management actions, ratings) to the semi-annual Board climate risk reporting, and do NOT state that the Board is informed of those specific elements. The evidence supports the reporting CADENCE and that the register CONTAINS those fields, but NOT that the Board reporting pack includes them. Describe Board reporting only as: the Board receives climate risk reporting at the evidenced cadence, drawn from the climate risk register - without describing what that reporting contains.
- LOW-HALLUCINATION PHRASING: Use the exact wording provided in the evidence "approved_phrasing" object for ERM integration, governance metrics and transaction checks. Do NOT paraphrase these into stronger claims. Specifically: do NOT write "integration of climate indicators into enterprise risk management" (ERM integration is a per-risk flag for N of the recorded risks, not enterprise-wide and not "indicators"); do NOT write "governance key performance indicators" or "governance KPIs" (these are governance indicators the Bank reports, not a formal KPI framework); do NOT write that the Bank "applies" or "maintains a policy" of climate checks (it performed climate checks in 2024).
- Do not enumerate absent governance mechanisms anywhere in the section. Boundary language in the body is limited to qualifying disclosed data (for example the scope of assurance, or that committee naming differs across years).
- Treat core_standard items as mandatory disclosure guidance.
- Treat industry_guidance items as banking-specific application guidance.
- Do not display IFRS paragraph references or requirement IDs in the final report.

COMPACT GOVERNANCE EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Governance evidence tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Climate risk register present in Governance evidence: {evidence.get('payload_profile', {}).get('climate_risk_register_in_payload')}
- Management process instruction: {management.get('process_flow_instruction')}
- Formal mandate instruction: {strict.get('formal_governance_mandate_instruction')}
- Board trade-off instruction: {strict.get('board_tradeoff_instruction')}
- Skills instruction: {strict.get('skills_adequacy_instruction')}
- Assurance instruction: {strict.get('assurance_scope_limitation', {}).get('instruction')}

GENERAL WRITING RULES:

SENIOR GOVERNANCE FIXES:
- Keep specific 2024 decision topics and dates only in "#### Governance decisions, controls and evidence boundaries".
- In Board and committee subsections, refer only to "documented climate-related decisions in 2024"; do not name net-zero revisions, transition plan endorsements, scenario methodology, carbon credit budgets, TCFD reviews or remuneration KPI calibration outside the final subsection.
- For the management committee, state only that management responsibility is assigned to the Climate Risk Management Committee and supported by the risk register and semi-annual Board reporting. Do not assert broader committee activity unless evidenced.
- Scenario links should be described as "scenario analysis identifiers", not as proof of internal assessments unless evidenced.
- Add a neutral boundary that source data records different management committee names by year and does not evidence continuity or formal renaming.

- Use public wording for ERM integration: write 'ERM integration indicators' or 'source data indicates ERM integration', not 'ERM integration flags'.
- Do not duplicate the detailed Board-decision list before the final decisions/boundaries subsection.
- Do not name specific 2024 decision topics before the final decisions/boundaries subsection.
- Move TCFD/IFRS S2 alignment indicators to the final boundaries subsection only.
- Do not use broad phrases such as 'multi-layer governance structure', 'risk and control processes', 'governance and control processes', 'focused review and challenge', or 'strengthening competence'.
- Keep semi-annual climate risk reporting and separate Board/committee decision evidence distinct.
- Do not imply that net-zero target revisions, transition-plan updates or climate scenario methodology approvals were part of the semi-annual Board reporting pack unless explicitly evidenced.
- Do not use the phrase 'maintains a dedicated climate risk register'.
- Do not use the phrase 'supporting processes embedded'.
- Use dates and decision descriptions mainly in the final Governance decisions, controls and evidence boundaries subsection.
- Interpret climate_on_board_agenda_pct as the percentage of Board meetings where climate appeared on the agenda.
- A true major_transactions_climate_check flag indicates evidence of climate checks; it does not prove a formal mandatory policy.
- Do not infer that committee names from 2022, 2023 and 2024 represent the same renamed committee.
- Do not describe carbon credit procurement as recurring governance unless topic evidence explicitly supports that. If it appears only as a decision title, mention it only as a 2024 decision.
- When alignment indicators such as TCFD-aligned or IFRS S2-aligned are present, state only that source data flags the disclosure as aligned; do not claim governance oversight supports or ensures alignment.

{feedback_block}

Write the complete Governance section now.
Return only the final Governance section.
""".strip()

## 8. Governance writer prompt


In [14]:
# ── GOVERNANCE JUDGE PROMPT ─────────────────────────────────
JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate disclosure.

You evaluate whether the Governance section is:
- supported by the compact evidence;
- aligned with the Governance disclosure checklist;
- structured like a dedicated bank IFRS S1/S2 report section rather than a checklist answer;
- transparent about missing evidence;
- free from hallucinated policies, processes, trade-offs, assurance coverage and unsupported governance claims.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Governance draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

REPORTING VOICE: The section must read as the Bank's own published disclosure. Treat as a material wording defect any report-machinery language - "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", or phrasing that narrates what "the documentation"/"the evidence" does or does not show. Facts must be stated directly; genuine gaps framed as the Bank's position ("the Bank does not currently disclose ...", "... is outside the scope of this report", "... has not yet been quantified"). List any such phrasing in material_unsupported_claims so it is corrected.

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

DECISION-PLACEMENT GROUND TRUTH: The pre-checks include "decision_topic_placement". If its status is "pass", decision-topic placement is correct - you MUST set checklist "no_duplicate_decisions" and "board_decisions_used_correctly" to true with respect to placement/duplication, and you MUST NOT report decision-topic duplication or leakage as an issue, unsupported claim, or material claim. Generic references to oversight areas (e.g. scenario analysis, transition planning, remuneration) or to register mitigation labels in narrative subsections are permitted and are not duplication. Defer to this deterministic result over your own reading of where topics appear.\n\nSECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("governance")}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
0. Use the IFRS requirements pack and the ifrs_requirement_coverage pre-check ONLY to confirm that topics BACKED BY THE COMPACT EVIDENCE are addressed. Do NOT penalise a core IFRS topic that is genuinely unevidenced in the compact evidence (for example where an availability flag is false, or the data simply does not contain it). Under the Bank's reporting policy such genuinely-missing items are OMITTED from the section and recorded in metadata; their omission is correct, must NOT reduce the score, must NOT be treated as omitted available evidence, and must NOT require an in-body boundary statement. For every "*_handled_according_to_availability" checklist item, mark true if the relevant evidence is present and used correctly OR if it is absent and simply omitted; do not mark it false merely because an absent item was omitted instead of boundary-disclosed.
1. Penalise any claim that contradicts evidence or invents unavailable governance information.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed evidence boundaries are good practice and do not by themselves lower the score.
4. Verify that the section follows the adapted bank IFRS S1/S2 style structure:
   Overview; role of the Board; Board committees; management responsibility; skills/competencies/remuneration; decisions/controls/boundaries.
5. Distinguish committee activity evidence from formal charter/mandate evidence.
6. Distinguish governance-level management controls from risk-register process evidence.
7. If climate_risk_register is available, the draft should describe the management risk-register process at a reasonable summary level: risk identification/register, risk categories, time horizons, ratings, monitoring frequencies, scenario links, mitigation actions and ERM integration.
8. If climate_risk_register is absent, the draft must not describe risk counts, risk categories, scenario links, monitoring frequencies by risk, mitigation actions or a risk-register workflow.
9. If formal mandate, Board trade-offs, formal escalation thresholds or skills adequacy assessment are unavailable, the draft must not invent them.
10. Verify exact figures and trends: confirm that every current-year figure and multi-year trend shown in the evidence (board size, independent-director %, climate agenda %, ESG committee meetings, board climate expertise %, CEO and all-executive climate-linked remuneration %, board meeting frequency) is used in the draft exactly as given in the evidence and not altered, rounded away, or invented.
11. Verify that assurance is limited to Scope 1 and Scope 2 emissions when that is the stated scope and does not imply financed emissions assurance.
12. Verify no visible IFRS paragraph references, meeting IDs or internal refs appear in the final section.
13. Verify decisions are not unnecessarily duplicated across multiple subsections.
14. Do not penalise accurate limitation statements such as "the Bank does not currently disclose formal escalation thresholds".
15. Alignment flags should be described only as source-data flags, not as proof that governance oversight ensures alignment.
16. Penalise duplicated decision content: specific 2024 decision topics and dates should appear only in the final Governance decisions, controls and evidence boundaries subsection.
17. Earlier Board and Committee subsections may refer generally to documented 2024 decisions, but should not name specific topics such as net-zero target revision, transition plan endorsement, climate scenario methodology, carbon credit procurement budget, ESG report approval, TCFD review or executive remuneration KPI calibration.
18. Alignment flags should not appear in the Overview. They should be disclosed only as source-data indicators in the final boundaries subsection, with a clear statement that no separate governance control process designed to verify or assure alignment is evidenced.
19. Penalise broad governance/control wording that implies processes beyond the evidence, such as "multi-layer governance structure", "risk and control processes", "governance and control processes", "focused review and challenge" or "strengthening competence", unless clearly anchored to evidenced mechanisms.
16. Do not link semi-annual reporting to specific decision items unless the evidence explicitly states that the decisions were part of the reporting pack.
17. Prefer evidence-anchored risk-register wording such as 'uses a climate risk register'; penalise stronger operational-control wording such as 'maintains a dedicated climate risk register' when ownership/control maintenance is not evidenced.

SCORING:
Score on completeness RELATIVE TO AVAILABLE EVIDENCE, not on how much data the bank happens to hold. Correctly disclosed evidence boundaries (genuine unavailability, accurately stated) are good IFRS practice and MUST NOT reduce the score; the number of disclosed boundaries is irrelevant to the score. Penalise ONLY: unsupported or overclaimed statements, omitted AVAILABLE evidence, factual inconsistencies with the evidence, hallucinations, and structural or deterministic failures.
- 9-10: Materially complete relative to available evidence, report-ready structure, no unsupported or overclaimed statements, no omitted available evidence, and all genuine data gaps correctly disclosed as boundaries.
- 8: As 9-10 but with one minor evidence-use issue or one slightly loose statement to tighten.
- 7: Usable but contains an unsupported/overclaimed statement, omits available evidence, or has a structural/keyword gap.
- 6: Revision required: available evidence omitted, unsupported wording remains, factual inconsistency with evidence, or required structure/boundary keywords missing.
- 5 or below: Major evidence failure, hallucination, contradiction, or missing core subsection.

When listing omitted evidence, separate MATERIAL omissions (which must be incorporated) from OPTIONAL ones (acceptable to leave out); never mark internal record IDs or optional traceability codes as material. Likewise, separate unsupported claims into material_unsupported_claims (misrepresent the evidence or overstate scope/assurance/compliance — must fix) and minor_wording_to_tighten (defensible phrasing such as 'covers' vs 'includes' or singular/plural that does not misrepresent the evidence — an acceptable minor limitation). Do not list trivial wording as material. Return valid JSON only. All checklist values must be booleans true/false, not strings.
Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "report_style_structure_present": <true/false>,
    "board_metrics_used_correctly": <true/false>,
    "trend_metrics_used_correctly": <true/false>,
    "board_decisions_used_correctly": <true/false>,
    "formal_mandate_handled_according_to_availability": <true/false>,
    "board_tradeoffs_handled_according_to_availability": <true/false>,
    "management_evidence_handled_according_to_availability": <true/false>,
    "risk_register_used_if_available_or_not_invented_if_absent": <true/false>,
    "escalation_handled_according_to_availability": <true/false>,
    "skills_handled_according_to_availability": <true/false>,
    "remuneration_evidence_used_correctly": <true/false>,
    "assurance_scope_used_correctly": <true/false>,
    "no_unsupported_committee_evolution": <true/false>,
    "no_duplicate_decisions": <true/false>,
    "no_visible_ifrs_refs_or_internal_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "material_evidence_omitted": [<omitted available evidence whose absence materially weakens the disclosure; MUST be incorporated as report prose>],
  "optional_evidence_omitted": [<omitted available evidence that is acceptable to leave out; do not force inclusion>],
  "unsupported_claims": [<specific unsupported claims>],
  "material_unsupported_claims": [<claims that misrepresent the evidence, assert unevidenced facts, or overstate scope/assurance/compliance; these MUST be removed or reframed>],
  "minor_wording_to_tighten": [<defensible phrasing that could be tightened but does NOT misrepresent the evidence; acceptable as a minor limitation>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()

## 9. Governance judge prompt


In [15]:
# ── GOVERNANCE EVALUATION MODE ───────────────────────────────
# Data-aware deterministic pre-checks are now active.
# They do not directly approve/reject the section or cap the score.
# Instead, their findings are passed to the GPT-5.2 Judge so the Judge can evaluate
# the draft against the actual available Governance data.

print("Governance evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

Governance evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks


## 10. Governance evaluation mode


In [16]:
# ── GOVERNANCE LANGGRAPH NODES ───────────────────────────────

def normalize_json_booleans(obj):
    """Normalize LLM JSON outputs where booleans may be returned as strings."""
    if isinstance(obj, dict):
        return {k: normalize_json_booleans(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [normalize_json_booleans(v) for v in obj]
    if isinstance(obj, str):
        if obj.strip().lower() == "true":
            return True
        if obj.strip().lower() == "false":
            return False
    return obj

def _contains_any(text: str, phrases: list[str]) -> bool:
    text_l = text.lower()
    return any(p.lower() in text_l for p in phrases)


def run_governance_deterministic_checks(draft: str, evidence: dict) -> dict:
    """Data-aware deterministic checks passed to the Judge as evidence-aware signals."""
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})
    gov = evidence.get("governance_2024", {})

    required_headings = [
        "#### Overview",
        "#### The role of the Board of Directors",
        "#### Board committees and climate-related oversight",
        "#### Management responsibility for climate-related risks and opportunities",
        "#### Skills, competencies and remuneration",
        "#### Governance decisions, controls and evidence boundaries",
    ]

    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})


    # Decision duplication and placement checks.
    final_heading = "#### Governance decisions, controls and evidence boundaries"
    pre_final_text = text_l.split(final_heading.lower(), 1)[0] if final_heading.lower() in text_l else text_l

    decision_topic_patterns = [
        "net-zero",
        "net zero",
        "interim target",
        "transition plan",
        "scenario analysis methodology",
        "climate scenario analysis methodology",
        "carbon credit",
        "procurement budget",
        "esg report",
        "tcfd review",
        "executive remuneration esg kpi",
        "remuneration esg kpi",
    ]
    found_pre_final_decisions = [p for p in decision_topic_patterns if p in pre_final_text]
    if found_pre_final_decisions:
        warnings.append({
            "check": "specific_decision_topics_before_final_section",
            "details": found_pre_final_decisions
        })

    # Broad/overstated wording checks.
    broad_phrases = [
        "multi-layer climate governance structure",
        "multi-layer governance structure",
        "risk and control processes",
        "governance and control processes",
        "focused review and challenge",
        "strengthening competence",
    ]
    found_broad = [p for p in broad_phrases if p in text_l]
    if found_broad:
        warnings.append({
            "check": "broad_governance_or_control_wording",
            "details": found_broad
        })

    # Alignment flag placement and boundary check.
    overview_heading = "#### overview"
    board_heading = "#### the role of the board of directors"
    overview_section_l = ""
    if overview_heading in text_l:
        overview_section_l = text_l.split(overview_heading, 1)[1]
        if board_heading in overview_section_l:
            overview_section_l = overview_section_l.split(board_heading, 1)[0]

    if ("tcfd" in overview_section_l) or ("ifrs s2" in overview_section_l):
        warnings.append({
            "check": "alignment_flags_in_overview",
            "details": "Move TCFD/IFRS S2 alignment flags to the final evidence-boundaries subsection."
        })

    if ("tcfd" in text_l or "ifrs s2" in text_l) and "separate governance control process" not in text_l:
        warnings.append({
            "check": "alignment_boundary_missing",
            "details": "When mentioning alignment flags, state that no separate governance control process designed to verify or assure alignment is evidenced."
        })


    # Specific wording checks to prevent minor overreach found during evaluation.
    if "supplemented by specific decision items" in text_l:
        warnings.append({
            "check": "semi_annual_reporting_decision_linkage_overstated",
            "details": "Keep semi-annual Board reporting separate from individual Board/committee decision evidence."
        })

    if "maintains a dedicated climate risk register" in text_l:
        warnings.append({
            "check": "risk_register_control_wording_overstated",
            "details": "Use 'uses a climate risk register' or 'documents climate-related risks in a climate risk register' instead."
        })

    if "supporting processes embedded" in text_l:
        warnings.append({
            "check": "supporting_processes_embedding_overstated",
            "details": "Use specific evidenced mechanisms instead of broad embedded-process language."
        })

    # Report-style structure checks.
    if "#### Overview" in text:
        overview_text = text.split("#### Overview", 1)[1].split("#### The role of the Board of Directors", 1)[0].lower()
        if len(overview_text.split()) < 50:
            warnings.append({
                "check": "overview_may_be_too_short",
                "details": "Overview is present but may be too short to explain the governance model."
            })

    boundary_heading = "#### Governance decisions, controls and evidence boundaries"
    if boundary_heading in text:
        boundary_text = text.split(boundary_heading, 1)[1].lower()
        boundary_terms = ["mandate", "trade-off", "tradeoff", "skills", "escalation", "assurance"]
        if not any(term in boundary_text for term in boundary_terms):
            warnings.append({
                "check": "governance_style_boundary_section_may_be_weak",
                "details": "Final Governance decisions/controls/boundaries subsection may not clearly include key evidence boundaries."
            })

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS paragraph references/tags found."})

    if "[REF:" in text or re.search(r"MTG-BANK\d+", text):
        failures.append({"check": "internal_refs_visible", "details": "Internal meeting references should not appear in final text."})

    # ── Data-derived, availability-gated current-year figures ──
    # Values are read from the compact evidence (not hard-coded) and are required
    # only when the availability profile confirms the metric exists for this bank.
    def _fmt_num(v):
        if isinstance(v, bool):
            return None
        if isinstance(v, float) and v.is_integer():
            return str(int(v))
        if isinstance(v, (int, float)):
            return str(v)
        sv = str(v).strip()
        return sv or None

    trend_avail = profile.get("trend_metrics_available", {}) or {}

    current_year_required = [
        ("board_core_metrics_available", "climate_on_board_agenda_pct", "climate agenda frequency"),
        ("board_core_metrics_available", "board_climate_expertise_pct", "board climate expertise"),
        ("remuneration_evidence_available", "ceo_esg_compensation_pct", "CEO ESG-linked remuneration"),
        ("remuneration_evidence_available", "all_exec_climate_remuneration_pct", "all-executive climate-linked remuneration"),
    ]
    for gate, field, label in current_year_required:
        if not profile.get(gate, False):
            continue  # metric not available for this bank -> do not demand it
        needle = _fmt_num(gov.get(field))
        if needle is None:
            continue  # value absent -> correctly disclosed as a boundary, not a failure
        if needle not in text:
            failures.append({
                "check": f"missing_{field}",
                "details": f"{needle}% {label} is available in the evidence and must be used.",
            })

    # ── Data-derived, availability-gated trend sequences ──
    # Each [comparative_years..., reporting_year] sequence is built from the compact
    # evidence and required only when that trend metric is flagged available.
    trend_rows = {r.get("year"): r for r in (evidence.get("governance_trend", []) or []) if isinstance(r, dict)}
    gov2024_field = {
        "esg_committee_meetings": "esg_committee_meetings_per_year",
        "board_climate_expertise_pct": "board_climate_expertise_pct",
        "ceo_esg_compensation_pct": "ceo_esg_compensation_pct",
        "all_exec_climate_remuneration_pct": "all_exec_climate_remuneration_pct",
        "climate_on_board_agenda_pct": "climate_on_board_agenda_pct",
        "board_full_meeting_frequency": "board_full_meeting_frequency",
    }
    for metric, cur_field in gov2024_field.items():
        if not trend_avail.get(metric, False):
            continue  # trend not available -> do not demand it
        seq = []
        for yr in (evidence.get("comparative_years", []) or []):
            val = _fmt_num(trend_rows.get(yr, {}).get(metric))
            if val is not None:
                seq.append(val)
        cur = _fmt_num(gov.get(cur_field))
        if cur is not None:
            seq.append(cur)
        if len(seq) < 2:
            continue  # not enough points to assert a trend
        missing = [v for v in seq if v not in text]
        if missing:
            failures.append({
                "check": f"missing_{metric}_trend",
                "details": f"Trend values available in evidence but missing from draft: {missing} (full sequence {seq}).",
            })

    if not profile.get("formal_governance_mandate_available"):
        if _contains_any(text, ["formal mandate", "committee charter", "terms of reference"]) and not _contains_any(text, ["does not include", "not documented", "not available", "available documentation does not", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "formal_mandate_overclaimed", "details": "Formal mandate/charter language used without a limitation."})

    if not profile.get("board_tradeoff_evidence_available"):
        if _contains_any(text, ["trade-off", "tradeoff"]) and not _contains_any(text, ["not describe", "not documented", "not available", "does not provide", "does not evidence", "documentation reviewed does not", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "board_tradeoffs_overclaimed", "details": "Trade-off language used without direct evidence or limitation."})

    management_heading = "#### Management responsibility for climate-related risks and opportunities"
    next_heading = "#### Skills, competencies and remuneration"
    management_section = ""
    if management_heading in text:
        management_section = text.split(management_heading, 1)[1]
        if next_heading in management_section:
            management_section = management_section.split(next_heading, 1)[0]
    management_section_l = management_section.lower()

    if profile.get("management_risk_register_available"):
        required_groups = {
            "risk_register": ["risk register"],
            "risk_count": ["8 climate-related risks", "8 climate risks", "8 risks", "eight climate-related risks"],
            "categories": ["risk categor", "classified into categories", "physical", "transition"],
            "monitoring": ["monitoring", "quarterly", "semi-annual", "semi annual"],
            "scenario_links": ["scenario"],
            "mitigation": ["mitigation", "mitigat"],
            "erm": ["erm", "enterprise risk management"],
        }
        missing_groups = [
            group for group, terms in required_groups.items()
            if not any(term in management_section_l for term in terms)
        ]
        if missing_groups:
            failures.append({
                "check": "risk_register_process_omitted",
                "details": f"Risk-register evidence is available but the Management responsibility subsection is missing: {missing_groups}"
            })
    else:
        if _contains_any(text, ["risk register", "risk categories", "scenario links", "mitigation actions", "monitoring frequencies"]) and not _contains_any(text, ["does not contain", "not available", "not documented", "available evidence does not", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "risk_register_process_invented", "details": "Risk-register process language appears although risk register is absent from Governance evidence."})

    if not profile.get("formal_escalation_thresholds_available"):
        escalation_terms = [
            "escalation threshold", "formal escalation", "escalation trigger",
            "trigger-based escalation", "defined escalation pathway", "escalation route"
        ]
        allowed_boundary = _contains_any(text, [
            "not documented", "not available", "does not specify", "does not evidence",
            "not evidenced", "no formal escalation",
            "available documentation does not evidence",
            "no such structures are therefore described", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"])
        if any(term in text_l for term in escalation_terms) and not allowed_boundary:
            failures.append({"check": "escalation_threshold_overclaimed", "details": "Formal escalation language appears without evidence."})

    if not profile.get("skills_adequacy_process_available"):
        if _contains_any(text, ["skills adequacy assessment", "skills matrix", "skills gap analysis", "formal skills assessment"]) and not _contains_any(text, ["not documented", "not available", "does not describe", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "skills_process_overclaimed", "details": "Formal skills adequacy process appears without evidence."})

    # Board reporting must not be described as containing risk-register CONTENTS
    # (quantified financial impacts, management/mitigation actions). Only cadence + that the
    # reporting draws on the register is evidenced.
    _report_cues = ["board is informed", "board is updated", "board receives",
                    "board pack", "reporting pack", "reporting to the board",
                    "report to the board", "this reporting, the board"]
    _content_cues = ["financial impact", "management action", "mitigation action"]
    _br_overclaim = False
    for _rc in _report_cues:
        _p = text_l.find(_rc)
        while _p != -1:
            if any(_cc in text_l[_p:_p + 220] for _cc in _content_cues):
                _br_overclaim = True
                break
            _p = text_l.find(_rc, _p + 1)
        if _br_overclaim:
            break
    if _br_overclaim:
        failures.append({"check": "board_reporting_content_overclaimed",
                         "details": "Board reporting is described as including register contents (financial impacts or management/mitigation actions); evidence supports only the reporting cadence and that it draws on the register."})

    # ERM integration must be framed as a per-risk flag, not enterprise-wide / "indicator" integration.
    if "integration of climate indicators" in text_l or "integration of climate-related indicators" in text_l:
        failures.append({"check": "erm_integration_overclaimed",
                         "details": "ERM integration framed as 'integration of climate indicators'; evidence supports only a per-risk ERM-integration flag."})
    elif ("enterprise risk management" in text_l) and ("integrat" in text_l):
        _erm_ok = any(q in text_l for q in ["of the eight", "of the 8", "6 of", "six of",
                                            "per-risk", "per risk", "flag", "selected climate",
                                            "recorded climate-related risks"])
        if not _erm_ok:
            failures.append({"check": "erm_integration_overclaimed",
                             "details": "ERM integration mentioned without a per-risk qualifier (e.g. 'N of 8 risks flagged'); reads as enterprise-wide integration."})

    # Governance metrics must not be framed as a formal KPI framework.
    if "governance key performance indicator" in text_l or "governance kpi" in text_l:
        failures.append({"check": "governance_kpi_overclaimed",
                         "details": "Governance metrics framed as 'governance KPIs/key performance indicators'; evidence supports reported indicators, not a KPI framework."})

    if profile.get("assurance_scope_is_scope1_scope2_only"):
        if "assurance" in text_l and "financed emissions" not in text_l:
            warnings.append({"check": "financed_emissions_scope_boundary_missing", "details": "Assurance section may not clearly state financed emissions are outside assurance scope."})
        if _contains_any(text, ["financed emissions are assured", "scope 3 emissions are assured", "assurance over financed emissions"]):
            failures.append({"check": "assurance_scope_overclaimed", "details": "Draft implies financed emissions / Scope 3 are assured."})

    forbidden_strong_phrases = [
        "governance oversight supports the bank's tcfd",
        "governance oversight supports the bank’s tcfd",
        "governance oversight ensures",
        "ensures alignment",
        "supports alignment",
        "fully aligned",
    ]
    found_strong = [p for p in forbidden_strong_phrases if p in text_l]
    if found_strong:
        failures.append({"check": "unsupported_alignment_or_strong_claim", "details": found_strong})


    if "climate risk management committee activity" in text_l:
        warnings.append({
            "check": "governance_management_committee_activity_overclaim",
            "details": "Use responsibility/process wording; management committee activity is not separately evidenced."
        })

    if "used in internal assessments" in text_l:
        warnings.append({
            "check": "governance_scenario_link_use_context_overclaim",
            "details": "Use 'linked to scenario analysis identifiers' unless internal assessment use is evidenced."
        })

    style_leak = check_reference_identity_leak(text)
    if not style_leak.get("passed", True):
        failures.append({"check": "reference_identity_leak", "details": style_leak.get("matches", [])})

    completeness = check_section_completeness(text, "governance")
    failures.extend(completeness.get("failures", []))
    warnings.extend(completeness.get("warnings", []))

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
        "decision_topic_placement": {
            "status": "pass" if not found_pre_final_decisions else "warn",
            "specific_decision_topics_before_final_section": found_pre_final_decisions,
            "note": "PASS means no specific dated decision topics appear before the final 'Governance decisions' subsection; decision placement/duplication is correct.",
        },
    }


def writer_node(state: GovernanceState) -> GovernanceState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_writer_prompt(state["evidence"], judge_feedback=feedback)

    draft = call_writer_llm(
        system_prompt=WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {
        **state,
        "draft": draft.strip(),
        "status": "judging"
    }


def judge_node(state: GovernanceState) -> GovernanceState:
    draft = state["draft"]
    deterministic_checks = run_governance_deterministic_checks(draft, state["evidence"])
    deterministic_checks["ifrs_requirement_coverage"] = check_ifrs_requirement_coverage(draft, "governance")

    judge_prompt = build_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )
    judge_result = call_judge_llm_json(
        system_prompt=JUDGE_SYSTEM,
        user_prompt=judge_prompt,
    )
    judge_result = normalize_json_booleans(judge_result)

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    judge_result["_best_so_far"] = track_best(state, judge_result)  # non-regression: keep best draft across iterations

    print("\nGOVERNANCE JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    return {
        **state,
        "judge_result": judge_result,
        "status": "judging",
    }


GOVERNANCE_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Governance section using only:
- the supplied compact Governance evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- If risk-register evidence is unavailable, do not describe a risk-register workflow.
- Keep the exact six-subsection Emirates-style Governance structure. Place all specific 2024 decision dates/topics only in the final Governance decisions, controls and evidence boundaries subsection.
- Do not add visible IFRS paragraph references, meeting IDs or internal refs.
- Return only the complete revised Governance section using the required six headings.
""".strip()


def reviser_node(state: GovernanceState) -> GovernanceState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT GOVERNANCE REQUIREMENTS:
{IFRS_GOVERNANCE_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("governance")}

COMPACT GOVERNANCE EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- When evidence is unavailable, preserve or improve the accurate boundary statement.
- Never invent a missing process, policy, threshold, trade-off, risk-register workflow, or assurance scope.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Governance section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=GOVERNANCE_REVISER_SYSTEM,
        user_prompt=(revision_prompt + "\n\nPRIORITISED REVISION BRIEF (address every item using available evidence only):\n" + build_reviser_focus(judge)),
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nGovernance revised with GPT-5.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def finalize_node(state: GovernanceState) -> GovernanceState:
    new_state = finalize_state(state, target_score=TARGET_SCORE)
    gate = new_state["judge_result"]["finalize_gate"]
    print(f"\n{'='*50}")
    print("FINALIZED GOVERNANCE")
    print(f"  Score: {gate['score']}/{gate['target_score']} | publishable={gate['publishable']}")
    print(f"  Status: {new_state['judge_result']['approval_status']} | revisions: {state['revision_count']}")
    if not gate["publishable"]:
        print("  Blocking reasons:", gate["blocking_reasons"])
    print(f"{'='*50}")
    return new_state


def route_after_judge(state: GovernanceState) -> str:
    return route_after_judge_generic(state, target_score=TARGET_SCORE, revise_label="revise")




## 11. Governance LangGraph nodes and deterministic checks


In [17]:
# ── BUILD AND COMPILE GRAPH ───────────────────────────────────
builder = StateGraph(GovernanceState)

builder.add_node("writer",   writer_node)
builder.add_node("judge",    judge_node)
builder.add_node("reviser",  reviser_node)
builder.add_node("finalize", finalize_node)

builder.add_edge(START,      "writer")
builder.add_edge("writer",   "judge")
builder.add_edge("reviser",  "judge")
builder.add_edge("finalize", END)

builder.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "revise":   "reviser",
        "finalize": "finalize",
    }
)

graph = builder.compile()
print("Graph compiled")

Graph compiled


## 12. Build and compile Governance graph


In [18]:
# ── RUN ──────────────────────────────────────────────────────
initial_state: GovernanceState = {
    "bank_name":      bank_name,
    "evidence":       evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions": 3,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {}
}

print(f"Starting governance generation for: {bank_name}\n")
result = run_section_with_floor(graph, initial_state, label="governance")
governance_result = result  # conventional name so downstream cells (e.g. General Requirements) find it

Starting governance generation for: Eurolux Universal Bank AG




WRITER (initial draft)
Draft length: 1290 words

GOVERNANCE JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS
{
  "overall_score": 7,
  "evidence_support_score": 7,
  "ifrs_alignment_score": 8,
  "specificity_score": 8,
  "hallucination_risk": "medium",
  "approval_status": "revision_required",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "report_style_structure_present": true,
    "board_metrics_used_correctly": true,
    "trend_metrics_used_correctly": true,
    "board_decisions_used_correctly": true,
    "formal_mandate_handled_according_to_availability": false,
    "board_tradeoffs_handled_according_to_availability": true,
    "management_evidence_handled_according_to_availability": true,
    "risk_register_used_if_available_or_not_invented_if_absent": true,
    "escalation_handled_according_to_availability": true,
    "skills_handled_according_to_availability": false,
    "remuneration_evidence_used_correctly": true,
    "assurance_scope_used_corr

## 13. Run Governance generation


In [19]:
# ── OUTPUT ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL JUDGE RESULT")
print("="*60)
print(json.dumps(result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GOVERNANCE SECTION")
print("="*60)
print(result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "governance_BANK01.md", "w", encoding="utf-8") as f:
    f.write(result["final_section"])

with open(output_dir / "governance_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "governance",
        "status":          result["status"],
        "approval_status": result["judge_result"].get("approval_status"),
        "raw_score":       result["judge_result"].get("raw_score_before_caps"),
        "final_score":     result["judge_result"].get("overall_score"),
        "score_cap_reason": result["judge_result"].get("score_cap_reason"),
        "revisions":       result["revision_count"],
        "approved":        result["judge_result"].get("approved"),
        "checklist":       result["judge_result"].get("checklist"),
        "issues":         result["judge_result"].get("main_issues"),
        "available_evidence_omitted": result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": result["judge_result"].get("unsupported_claims"),
        "material_unsupported_claims": result["judge_result"].get("material_unsupported_claims"),
        "ifrs_required_but_missing_in_data": governance_missing_ifrs,
        "minor_wording_to_tighten": result["judge_result"].get("minor_wording_to_tighten"),
        "correctly_disclosed_evidence_boundaries": result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_governance_payload_path": str(RAW_GOVERNANCE_PATH),
        "compact_governance_evidence_path": str(COMPACT_GOVERNANCE_PATH),
    }, f, indent=2, ensure_ascii=False)

print(f"\nSaved to outputs/governance_BANK01.md")
print(f"Raw Governance payload saved to: {RAW_GOVERNANCE_PATH}")
print(f"Compact Governance evidence saved to: {COMPACT_GOVERNANCE_PATH}")



FINAL JUDGE RESULT
{
  "overall_score": 8,
  "evidence_support_score": 8,
  "ifrs_alignment_score": 8,
  "specificity_score": 9,
  "hallucination_risk": "medium",
  "approval_status": "needs_human_review",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "report_style_structure_present": true,
    "board_metrics_used_correctly": true,
    "trend_metrics_used_correctly": true,
    "board_decisions_used_correctly": true,
    "formal_mandate_handled_according_to_availability": true,
    "board_tradeoffs_handled_according_to_availability": true,
    "management_evidence_handled_according_to_availability": false,
    "risk_register_used_if_available_or_not_invented_if_absent": true,
    "escalation_handled_according_to_availability": true,
    "skills_handled_according_to_availability": true,
    "remuneration_evidence_used_correctly": true,
    "assurance_scope_used_correctly": true,
    "no_unsupported_committee_evolution": true,
    "no_duplicate_decisio

# Strategy section

This section is added after Governance and follows an Emirates NBD-style IFRS S1/S2 banking disclosure structure.


## Strategy setup helpers


In [20]:

# ============================================================
# STRATEGY SECTION GENERATOR
# IFRS S1/S2 strategy | uses same Azure REST helper functions
# ============================================================
# This section is added after Governance and reuses:
# - payload
# - bank_name
# - call_writer_llm()
# - call_judge_llm_json()
# - call_reviser_llm()
# - _is_present(), _safe_int(), _normalise_text()


def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        # handle NaN from JSON payloads
        if isinstance(value, float) and value != value:
            return default
        return float(value)
    except Exception:
        return default


def _json_clean(obj):
    """Make dictionaries/lists safe for JSON prompt dumps by replacing NaN with None."""
    if isinstance(obj, dict):
        return {k: _json_clean(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_json_clean(v) for v in obj]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _pick_latest(records: list[dict], year: int = 2024) -> dict:
    for r in records:
        if isinstance(r, dict) and _safe_int(r.get("reporting_year")) == year:
            return r
    return records[-1] if records else {}


print("Strategy helper functions ready")


Strategy helper functions ready


## Strategy evidence extractor


In [21]:
# ── FAST STRATEGY EVIDENCE EXTRACTOR ────────────────────────
# This version summarizes the Strategy payload before sending it to the LLM.
# It reflects the actual section-specific strategy payload:
# - climate_scenarios are present;
# - climate_risk_register is present;
# - value_chain_map is present;
# - climate_opportunities are present;
# - full top-level targets may be absent, but reporting_kpis.target_summary is available.

def _present(v) -> bool:
    if v is None:
        return False
    if isinstance(v, str):
        return v.strip() != "" and v.strip().lower() not in {"nan", "none", "null"}
    try:
        return not (isinstance(v, float) and v != v)
    except Exception:
        return True


def _num(v):
    try:
        if not _present(v):
            return None
        return float(v)
    except Exception:
        return None


def _compact_clean(obj):
    if isinstance(obj, dict):
        cleaned = {k: _compact_clean(v) for k, v in obj.items() if _present(v)}
        return {k: v for k, v in cleaned.items() if v not in ({}, [], None)}
    if isinstance(obj, list):
        cleaned = [_compact_clean(x) for x in obj if _present(x)]
        return [x for x in cleaned if x not in ({}, [], None)]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _top_n(rows: list[dict], key: str, n: int = 5) -> list[dict]:
    return sorted(rows, key=lambda r: float(r.get(key) or 0), reverse=True)[:n]


def summarize_strategy_evidence(payload: dict) -> dict:
    metadata = payload.get("metadata", {})
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    scenarios = [s for s in payload.get("climate_scenarios", []) if isinstance(s, dict)]
    risks_all = [r for r in payload.get("climate_risk_register", []) if isinstance(r, dict)]
    risks_2024 = [r for r in risks_all if r.get("reporting_year") == 2024]
    value_chain = [v for v in payload.get("value_chain_map", []) if isinstance(v, dict)]
    opportunities = [o for o in payload.get("climate_opportunities", []) if isinstance(o, dict) and o.get("reporting_year") == 2024]

    full_targets = [t for t in payload.get("targets", []) if isinstance(t, dict)]
    target_summary = reporting_kpis.get("target_summary", [])
    targets = full_targets if full_targets else target_summary

    physical_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("physical")]
    transition_risks = [r for r in risks_2024 if str(r.get("risk_category", "")).startswith("transition")]

    risk_summary = {
        "risk_count_2024": len(risks_2024),
        "physical_risk_count": len(physical_risks),
        "transition_risk_count": len(transition_risks),
        "time_horizons": sorted({r.get("time_horizon") for r in risks_2024 if _present(r.get("time_horizon"))}),
        "risk_categories": sorted({r.get("risk_category") for r in risks_2024 if _present(r.get("risk_category"))}),
        "risk_ratings": sorted({r.get("risk_rating") for r in risks_2024 if _present(r.get("risk_rating"))}),
        "top_risks_by_financial_impact": [
            {
                "risk_id": r.get("risk_id"),
                "risk_name": r.get("risk_name"),
                "risk_category": r.get("risk_category"),
                "risk_rating": r.get("risk_rating"),
                "time_horizon": r.get("time_horizon"),
                "financial_impact_meur": r.get("financial_impact_meur"),
                "mitigation_actions": r.get("mitigation_actions"),
                "scenario_analysis_link": r.get("scenario_analysis_link"),
            }
            for r in _top_n(risks_2024, "financial_impact_meur", 6)
        ],
    }

    scenario_types = sorted({s.get("scenario_type") for s in scenarios if _present(s.get("scenario_type"))})
    scenario_names = sorted({s.get("scenario_name") for s in scenarios if _present(s.get("scenario_name"))})
    frameworks = sorted({s.get("framework") for s in scenarios if _present(s.get("framework"))})
    horizon_years = sorted({s.get("horizon_year") for s in scenarios if _present(s.get("horizon_year"))})

    def max_record(field: str):
        rows = [s for s in scenarios if _present(s.get(field))]
        if not rows:
            return None
        s = max(rows, key=lambda x: float(x.get(field) or 0))
        return {
            "scenario_id": s.get("scenario_id"),
            "scenario_name": s.get("scenario_name"),
            "scenario_type": s.get("scenario_type"),
            "horizon": s.get("horizon"),
            "horizon_year": s.get("horizon_year"),
            field: s.get(field),
        }

    resilience_by_type = {}
    methodology_by_type = {}
    for s in scenarios:
        stype = s.get("scenario_type")
        if not _present(stype):
            continue
        if _present(s.get("resilience_assessment")):
            resilience_by_type.setdefault(stype, s.get("resilience_assessment"))
        if _present(s.get("methodology_notes")):
            methodology_by_type.setdefault(stype, s.get("methodology_notes"))

    scenario_summary = {
        "available": bool(scenarios),
        "scenario_count": len(scenarios),
        "frameworks": frameworks,
        "scenario_types": scenario_types,
        "scenario_names": scenario_names,
        "horizon_years": horizon_years,
        "methodology_summary_by_scenario_type": methodology_by_type,
        "resilience_assessment_by_scenario_type": resilience_by_type,
        "max_physical_risk_loss_pct_capital": max_record("physical_risk_loss_pct_capital"),
        "max_transition_risk_loss_pct_capital": max_record("transition_risk_loss_pct_capital"),
        "max_stranded_assets_estimate_meur": max_record("stranded_assets_estimate_meur"),
        "max_revenue_at_risk_meur": max_record("revenue_at_risk_meur"),
        "key_assumption_ranges": {
            "carbon_price_eur_per_tco2e": {
                "min": min([s.get("carbon_price_assumption_eur_per_tco2e") for s in scenarios if _present(s.get("carbon_price_assumption_eur_per_tco2e"))], default=None),
                "max": max([s.get("carbon_price_assumption_eur_per_tco2e") for s in scenarios if _present(s.get("carbon_price_assumption_eur_per_tco2e"))], default=None),
            },
            "technology_readiness": sorted({s.get("technology_readiness") for s in scenarios if _present(s.get("technology_readiness"))}),
            "temperature_outcome_c": sorted({s.get("temperature_outcome_c") for s in scenarios if _present(s.get("temperature_outcome_c"))}),
        },
    }

    material_vc = [v for v in value_chain if v.get("materiality_flag") is True]
    quantified_vc = [v for v in material_vc if _present(v.get("financial_exposure_meur"))]

    value_chain_summary = {
        "available": bool(value_chain),
        "node_count": len(value_chain),
        "material_node_count": len(material_vc),
        "qualitative_nodes_without_financial_exposure": len([v for v in material_vc if not _present(v.get("financial_exposure_meur"))]),
        "node_types": sorted({v.get("node_type") for v in value_chain if _present(v.get("node_type"))}),
        "largest_quantified_nodes": [
            {
                "node_name": v.get("node_name"),
                "node_type": v.get("node_type"),
                "upstream_downstream": v.get("upstream_downstream"),
                "climate_exposure_type": v.get("climate_exposure_type"),
                "financial_exposure_meur": v.get("financial_exposure_meur"),
                "climate_risk_description": v.get("climate_risk_description"),
            }
            for v in _top_n(quantified_vc, "financial_exposure_meur", 6)
        ],
        "own_operations_examples": [
            {
                "node_name": v.get("node_name"),
                "climate_exposure_type": v.get("climate_exposure_type"),
                "scope3_category": v.get("scope3_category"),
                "climate_risk_description": v.get("climate_risk_description"),
            }
            for v in material_vc
            if v.get("node_type") == "own_operations"
        ][:3],
    }

    opportunity_summary = {
        "available": bool(opportunities),
        "opportunity_count": len(opportunities),
        "total_estimated_revenue_impact_meur": round(sum(float(o.get("estimated_revenue_impact_meur") or 0) for o in opportunities), 2) if opportunities else None,
        "examples": [
            {
                "opportunity_id": o.get("opportunity_id"),
                "opportunity_type": o.get("opportunity_type"),
                "category": o.get("category"),
                "description": o.get("description"),
                "estimated_revenue_impact_meur": o.get("estimated_revenue_impact_meur"),
                "time_horizon": o.get("time_horizon"),
                "confidence_level": o.get("confidence_level"),
                "linked_risk_category": o.get("linked_risk_category"),
            }
            for o in opportunities
        ],
    }

    financial_2024 = next(
        (f for f in payload.get("financial_summary", []) if isinstance(f, dict) and f.get("reporting_year") == 2024),
        {},
    )

    portfolio_summary = {
        "total_assets_meur": bank.get("total_assets_meur") or financial_2024.get("total_assets_meur"),
        "total_loans_meur": bank.get("total_loans_meur") or financial_2024.get("total_loans_meur"),
        "green_loans_meur": financial_2024.get("green_loans_meur"),
        "green_loans_pct": financial_2024.get("green_loans_pct") or reporting_kpis.get("green_loans_pct_2024"),
        "financed_emissions_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
        "climate_capex_meur": reporting_kpis.get("climate_capex_2024_meur") or financial_2024.get("climate_capex_meur"),
        "climate_opex_meur": reporting_kpis.get("climate_opex_2024_meur") or financial_2024.get("climate_opex_meur"),
        "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
        "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
        "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
        "fossil_fuel_exposure_meur": reporting_kpis.get("fossil_fuel_exposure_meur"),
    }

    compact = {
        "bank": {
            "bank_id": bank.get("bank_id"),
            "bank_name": bank.get("bank_name"),
            "country": bank.get("country"),
            "reporting_year": metadata.get("reporting_year", 2024),
            "reporting_currency": bank.get("reporting_currency"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur": bank.get("total_loans_meur"),
        },
        "payload_profile": {
            "source_payload": "strategy",
            "top_level_keys": list(payload.keys()),
            "full_targets_table_available": bool(full_targets),
            "target_summary_available": bool(target_summary),
            "climate_scenarios_count": len(scenarios),
            "risk_register_count": len(risks_all),
            "value_chain_node_count": len(value_chain),
            "opportunity_count": len(opportunities),
        },
        "risk_summary": risk_summary,
        "scenario_summary": scenario_summary,
        "value_chain_summary": value_chain_summary,
        "opportunity_summary": opportunity_summary,
        "portfolio_summary": portfolio_summary,
        "targets": targets,
        "business_model_impacts": payload.get("business_model_impacts", []),
        "strategy_tradeoff_decisions": payload.get("strategy_tradeoff_decisions", []),
        "evidence_boundaries": {
            "scenario_outputs_are_modelled_estimates": True,
            "opportunity_impacts_are_estimates": True,
            "do_not_overclaim_resilience": True,
            "do_not_overclaim_paris_alignment": True,
            "business_model_detail_available": bool(payload.get("business_model_impacts")),
            "value_chain_detail_available": bool(value_chain_summary),
            "tradeoff_evidence_available": bool(payload.get("strategy_tradeoff_decisions")),
            "high_carbon_and_fossil_exposure_available": _present(portfolio_summary.get("high_carbon_sector_exposure_pct")) and _present(portfolio_summary.get("fossil_fuel_exposure_pct")),
            "full_targets_table_available": bool(full_targets),
            "target_summary_available": bool(target_summary),
            "target_instruction": (
                "Only reporting_kpis.target_summary is available; do not invent target baselines, validation bodies, milestones, gross/net status or planned credit details."
                if not full_targets and target_summary else
                "Full target records are available; use only fields explicitly provided."
            ),
        },
    }
    compact["transition_plan"] = payload.get("transition_plan", [])
    compact["climate_financial_effects"] = payload.get("climate_financial_effects", [])
    compact["resilience_capacity"] = payload.get("resilience_assessment", [])
    compact["evidence_boundaries"]["transition_plan_available"] = bool(payload.get("transition_plan"))
    compact["evidence_boundaries"]["financial_effects_available"] = bool(payload.get("climate_financial_effects"))
    compact["evidence_boundaries"]["resilience_capacity_available"] = bool(payload.get("resilience_assessment"))
    return _compact_clean(compact)


strategy_evidence = summarize_strategy_evidence(strategy_payload)

raw_size = len(json.dumps(strategy_payload, ensure_ascii=False))
compact_size = len(json.dumps(strategy_evidence, ensure_ascii=False))
print("Compact Strategy evidence ready")
print(f"Raw strategy payload size: {raw_size:,} chars")
print(f"Compact evidence size: {compact_size:,} chars")
print(f"Reduction: {round((1 - compact_size / max(raw_size, 1)) * 100, 1)}%")
print(json.dumps({
    "bank": strategy_evidence["bank"],
    "scenario_count": strategy_evidence["scenario_summary"].get("scenario_count"),
    "risk_count_2024": strategy_evidence["risk_summary"].get("risk_count_2024"),
    "value_chain_available": strategy_evidence["value_chain_summary"].get("available", True),
    "opportunities_available": strategy_evidence["opportunity_summary"].get("available", True),
    "full_targets_table_available": strategy_evidence["payload_profile"].get("full_targets_table_available"),
    "target_summary_available": strategy_evidence["payload_profile"].get("target_summary_available"),
    "business_model_impacts": len(strategy_evidence.get("business_model_impacts", [])),
    "strategy_tradeoffs": len(strategy_evidence.get("strategy_tradeoff_decisions", [])),
    "high_carbon_pct": strategy_evidence["portfolio_summary"].get("high_carbon_sector_exposure_pct"),
    "fossil_fuel_pct": strategy_evidence["portfolio_summary"].get("fossil_fuel_exposure_pct"),
}, indent=2, ensure_ascii=False))

Compact Strategy evidence ready
Raw strategy payload size: 56,010 chars
Compact evidence size: 13,575 chars
Reduction: 75.8%
{
  "bank": {
    "bank_id": "BANK01",
    "bank_name": "Eurolux Universal Bank AG",
    "country": "DE",
    "reporting_year": 2024,
    "reporting_currency": "EUR",
    "total_assets_meur": 850000,
    "total_loans_meur": 31150.64
  },
  "scenario_count": 18,
  "risk_count_2024": 8,
  "value_chain_available": true,
  "opportunities_available": true,
  "full_targets_table_available": false,
  "target_summary_available": true,
  "business_model_impacts": 0,
  "strategy_tradeoffs": 0,
  "high_carbon_pct": 35.5,
  "fossil_fuel_pct": 22.15
}


## Strategy evidence availability, traceability and saving


In [22]:
# ── STRATEGY EVIDENCE AVAILABILITY + SAVING ──────────────────
# The raw Strategy payload remains the source of truth.
# The compact Strategy evidence is the agent-ready input used by Writer/Judge/Reviser.

def build_strategy_availability_profile(evidence: dict) -> dict:
    risk = evidence.get("risk_summary", {})
    scenarios = evidence.get("scenario_summary", {})
    value_chain = evidence.get("value_chain_summary", {})
    opportunities = evidence.get("opportunity_summary", {})
    portfolio = evidence.get("portfolio_summary", {})
    boundaries = evidence.get("evidence_boundaries", {})
    targets = evidence.get("targets", [])
    business_model_impacts = evidence.get("business_model_impacts", [])
    tradeoffs = evidence.get("strategy_tradeoff_decisions", [])

    def present(value) -> bool:
        return value is not None and str(value).strip().lower() not in {
            "", "none", "null", "nan"
        }

    scenario_assumptions_available = any(
        present(scenarios.get(field))
        for field in [
            "frameworks",
            "scenario_types",
            "scenario_names",
            "horizon_years",
            "methodology_summary_by_scenario_type",
            "key_assumption_ranges",
        ]
    )

    quantified_value_chain_available = bool(value_chain.get("largest_quantified_nodes"))
    opportunity_estimates_available = any(
        present(item.get("estimated_revenue_impact_meur"))
        for item in opportunities.get("examples", [])
        if isinstance(item, dict)
    ) or present(opportunities.get("total_estimated_revenue_impact_meur"))

    return {
        "physical_risk_evidence_available": bool(risk.get("physical_risk_count")),
        "transition_risk_evidence_available": bool(risk.get("transition_risk_count")),
        "time_horizons_available": bool(risk.get("time_horizons"))
        or bool(scenarios.get("horizon_years")),
        "business_model_detail_available": bool(business_model_impacts),
        "value_chain_detail_available": bool(value_chain),
        "quantified_value_chain_exposure_available": quantified_value_chain_available,
        "strategy_tradeoff_evidence_available": bool(tradeoffs),
        "portfolio_financial_metrics_available": any(
            present(portfolio.get(field))
            for field in [
                "green_loans_meur",
                "green_loans_pct",
                "financed_emissions_tco2e",
                "carbon_intensity_tco2e_per_meur",
            ]
        ),
        "resource_allocation_evidence_available": (
            present(portfolio.get("climate_capex_meur"))
            or present(portfolio.get("climate_opex_meur"))
        ),
        "high_carbon_exposure_available": (
            present(portfolio.get("high_carbon_sector_exposure_pct"))
            or present(portfolio.get("high_carbon_sector_exposure_meur"))
        ),
        "fossil_fuel_exposure_available": (
            present(portfolio.get("fossil_fuel_exposure_pct"))
            or present(portfolio.get("fossil_fuel_exposure_meur"))
        ),
        "climate_opportunities_available": bool(opportunities.get("examples")),
        "quantified_opportunity_estimates_available": opportunity_estimates_available,
        "scenario_analysis_available": bool(scenarios.get("available")),
        "scenario_assumptions_available": scenario_assumptions_available,
        "scenario_financial_outputs_available": any(
            scenarios.get(field)
            for field in [
                "max_physical_risk_loss_pct_capital",
                "max_transition_risk_loss_pct_capital",
                "max_stranded_assets_estimate_meur",
                "max_revenue_at_risk_meur",
            ]
        ),
        "resilience_evidence_available": bool(
            scenarios.get("resilience_assessment_by_scenario_type")
        ),
        "full_targets_table_available": bool(
            evidence.get("payload_profile", {}).get("full_targets_table_available")
        ),
        "target_summary_available": bool(
            evidence.get("payload_profile", {}).get("target_summary_available")
        ),
        "targets_available": bool(targets),
        "scenario_outputs_are_modelled_estimates": bool(
            boundaries.get("scenario_outputs_are_modelled_estimates", True)
        ),
        "opportunity_impacts_are_estimates": bool(
            boundaries.get("opportunity_impacts_are_estimates", True)
        ),
        "writer_policy": {
            "use_available_evidence": (
                "Use every material Strategy evidence item that is available and relevant."
            ),
            "handle_unavailable_evidence": (
                "When a material Strategy disclosure is unsupported by available evidence, state the boundary once in the relevant subsection and do not invent it."
            ),
            "target_boundary": boundaries.get("target_instruction"),
            "scenario_wording": (
                "Describe scenario financial outputs as modelled estimates, not actual losses."
            ),
            "opportunity_wording": (
                "Describe opportunity revenue impacts as estimates with confidence level, not guaranteed future revenue."
            ),
            "resilience_wording": (
                "Confine resilience statements to the relevant scenario assumptions and boundaries."
            ),
            "final_language": (
                "Write in the Bank's own voice and state facts directly. Do not refer to the report's evidence base: never use 'available evidence', 'available documentation', 'documentation reviewed', 'source data', 'internally generated analytical outputs' or 'payload' in the final disclosure. Frame genuine gaps as the Bank's position - 'the Bank does not currently disclose X', 'X is outside the scope of this report', or 'X has not yet been quantified'."
            ),
        },
    }


def add_strategy_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_strategy_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": [
            "bank",
            "financial_summary",
            "climate_scenarios",
            "climate_risk_register",
            "value_chain_map",
            "climate_opportunities",
            "reporting_kpis",
        ],
        "top_risk_refs": [
            item.get("risk_id") or item.get("risk_name")
            for item in evidence.get("risk_summary", {}).get("top_risks_by_financial_impact", [])
            if item.get("risk_id") or item.get("risk_name")
        ],
        "scenario_refs": [
            item.get("scenario_id")
            for item in [
                evidence.get("scenario_summary", {}).get("max_physical_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_transition_risk_loss_pct_capital"),
                evidence.get("scenario_summary", {}).get("max_stranded_assets_estimate_meur"),
                evidence.get("scenario_summary", {}).get("max_revenue_at_risk_meur"),
            ]
            if isinstance(item, dict) and item.get("scenario_id")
        ],
        "value_chain_refs": [
            item.get("node_name")
            for item in evidence.get("value_chain_summary", {}).get("largest_quantified_nodes", [])
            if item.get("node_name")
        ],
        "opportunity_refs": [
            item.get("opportunity_id") or item.get("opportunity_type")
            for item in evidence.get("opportunity_summary", {}).get("examples", [])
            if item.get("opportunity_id") or item.get("opportunity_type")
        ],
    }
    return evidence


strategy_evidence = add_strategy_traceability(
    strategy_evidence,
    STRATEGY_PAYLOAD_PATH,
)

raw_strategy_payload = {
    key: strategy_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "financial_summary",
        "climate_scenarios",
        "climate_risk_register",
        "value_chain_map",
        "climate_opportunities",
        "targets",
        "reporting_kpis",
        "business_model_impacts",
        "strategy_tradeoff_decisions",
    ]
    if key in strategy_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_STRATEGY_PATH = output_dir / "payload_BANK01_strategy_raw.json"
COMPACT_STRATEGY_PATH = output_dir / "compact_strategy_evidence_BANK01.json"

with open(RAW_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_strategy_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_STRATEGY_PATH, "w", encoding="utf-8") as f:
    json.dump(strategy_evidence, f, indent=2, ensure_ascii=False)

print("Strategy evidence prepared")
print(f"- Raw Strategy payload: {RAW_STRATEGY_PATH}")
print(f"- Compact Strategy evidence: {COMPACT_STRATEGY_PATH}")
print("- Availability profile:")
print(json.dumps(strategy_evidence["availability_profile"], indent=2, ensure_ascii=False))


# ── TARGET AVAILABILITY DIAGNOSTIC ───────────────────────────
print("full_targets_table_available:", strategy_evidence["payload_profile"].get("full_targets_table_available"))
print("target_summary_available:", strategy_evidence["payload_profile"].get("target_summary_available"))
print("targets count:", len(strategy_evidence.get("targets", [])))
print(
    "TGT003 planned_carbon_credits_pct:",
    next(
        (
            t.get("planned_carbon_credits_pct")
            for t in strategy_evidence.get("targets", [])
            if isinstance(t, dict) and t.get("target_id") == "TGT003"
        ),
        "NOT FOUND"
    )
)


Strategy evidence prepared
- Raw Strategy payload: outputs\payload_BANK01_strategy_raw.json
- Compact Strategy evidence: outputs\compact_strategy_evidence_BANK01.json
- Availability profile:
{
  "physical_risk_evidence_available": true,
  "transition_risk_evidence_available": true,
  "time_horizons_available": true,
  "business_model_detail_available": false,
  "value_chain_detail_available": true,
  "quantified_value_chain_exposure_available": true,
  "strategy_tradeoff_evidence_available": false,
  "portfolio_financial_metrics_available": true,
  "resource_allocation_evidence_available": true,
  "high_carbon_exposure_available": true,
  "fossil_fuel_exposure_available": true,
  "climate_opportunities_available": true,
  "quantified_opportunity_estimates_available": true,
  "scenario_analysis_available": true,
  "scenario_assumptions_available": true,
  "scenario_financial_outputs_available": true,
  "resilience_evidence_available": true,
  "full_targets_table_available": false,
  "ta

## Strategy state definition


In [23]:

# ── STRATEGY STATE DEFINITION ───────────────────────────────
class StrategyState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict


## Strategy requirements — Emirates-style structure


In [24]:
# ── STRATEGY REQUIREMENTS ───────────────────────────────────
# IFRS references are used internally only. Final text must not show paragraph references.

STRATEGY_REQUIREMENTS = """
STRICT STRATEGY DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Presentation style:
- Write like a dedicated bank IFRS S1/S2 Strategy section, not like a checklist answer.
- Follow an Emirates NBD-style flow: overview, risks/opportunities across value chain, strategic management, effects on business model/value chain/decision-making, financial effects, resilience, and evidence boundaries.
- Do not overuse the phrase "available evidence" in the main narrative. Use it mainly in the limitations subsection.

Final section title and headings must be exactly:
### Strategy
#### Overview
#### Sustainability risks and opportunities across the value chain
#### Strategic management of climate-related risks and opportunities
#### Climate-related effects on business model, value chain and decision-making
#### Financial effects and resource allocation
#### Climate resilience and scenario analysis
#### Strategy limitations and evidence boundaries

CRITICAL COMPLETION RULE:
The final output must contain complete text under all seven headings. The final subsection must not end mid-sentence. If space is limited, shorten the earlier subsections rather than truncating the final two subsections.

MANDATORY CONTENT FOR #### Overview:
This subsection must exist and must:
- briefly introduce the Bank's climate-related strategy architecture;
- link sustainability strategy to value chain, risk management, transition planning, climate opportunities, resource allocation and scenario analysis;
- avoid a generic sustainability marketing tone and stay evidence-based.

MANDATORY CONTENT FOR #### Sustainability risks and opportunities across the value chain:
This subsection must exist and must:
- describe physical and transition risks using supplied risk evidence;
- explain where risks and opportunities arise across own operations, suppliers and financing counterparties when value-chain evidence is available;
- use high-carbon exposure, fossil-fuel exposure and green-loan indicators when available;
- describe opportunities as estimates or identified opportunities, not guaranteed future benefits.

MANDATORY CONTENT FOR #### Strategic management of climate-related risks and opportunities:
This subsection must exist and must:
- describe how the Bank manages identified climate-related risks and opportunities through transition planning, portfolio steering, client engagement, sector exposure limits, green finance or other evidenced mechanisms;
- use target summary or full target records only to the extent evidenced;
- state missing transition-plan assumptions or dependencies where unavailable.

MANDATORY CONTENT FOR #### Climate-related effects on business model, value chain and decision-making:
This subsection must exist and must:
- describe business model, value chain and decision-making effects using only available evidence;
- if detailed business_model_impacts or trade-off decisions are unavailable, state that boundary without inventing detailed business-model impact analysis or quantified trade-offs.

MANDATORY CONTENT FOR #### Financial effects and resource allocation:
This subsection must exist and must include:
- Climate capex: EUR 476.95 million when available in evidence.
- Climate opex: EUR 219.32 million when available in evidence.
- If activity-level breakdown is not available, state that boundary but still disclose the totals.
- Scenario financial effects must be labelled as modelled estimates, not actual realised losses.
- Opportunity revenue impacts must be labelled as estimates, not guaranteed or assured revenue.

MANDATORY CONTENT FOR #### Climate resilience and scenario analysis:
This subsection must exist and must cover ALL of the following when available in evidence:
a) Scenario framework: NGFS v4.
b) Scenario types: orderly, disorderly and hot_house / hot house.
c) Named scenarios, including Net Zero 2050, Below 2°C, Divergent Net Zero, Delayed Transition, NDC and Current Policies when present.
d) Time horizons and horizon years: short-term 2025, medium-term 2030, long-term 2050 when present.
e) Scope of analysis: lending book exposures and covered geographies when present.
f) Key assumptions:
   - carbon price assumptions and ranges;
   - temperature outcomes;
   - technology readiness;
   - macroeconomic assumptions where available;
   - energy assumptions such as renewable energy share where available;
   - methodology notes and scope of analysis.
g) Key modelled financial outputs, clearly labelled as modelled estimates:
   - maximum physical risk loss;
   - maximum transition risk loss;
   - maximum stranded assets estimate;
   - maximum revenue at risk.
h) Scenario-specific resilience:
   - orderly scenario resilience;
   - disorderly scenario resilience;
   - hot house scenario resilience.
   Do NOT collapse these into a single general bank-wide resilience claim.
i) Adaptation capacity from available evidence only:
   - financial resources: climate capex, climate opex, capital buffer or resource-allocation evidence;
   - portfolio flexibility: sector exposure limits, green loan growth, decarbonisation glide-path monitoring, client engagement, collateral overlays or similar evidenced mechanisms;
   - current/planned investment effects: use climate capex/opex and transition finance indicators.
   Do NOT invent asset redeployment, decommissioning or operational adaptation capacity if not evidenced.

MANDATORY CONTENT FOR #### Strategy limitations and evidence boundaries:
This subsection must exist, must be complete, and must include the relevant boundary statements below:
- Business model: If business_model_impacts are unavailable, state that the available evidence describes risk, opportunity, portfolio and value-chain effects, but does not include a separate detailed business-model impact assessment.
- Trade-offs: If strategy_tradeoff_decisions are unavailable, state that the available evidence does not provide quantified or documented trade-off analysis for strategy decisions.
- Targets: If full target records are available, use only the fields explicitly provided. If only target_summary is available, state that only target summary evidence is available and therefore baseline details, milestones, validation bodies, gross/net status and planned carbon-credit use are not described.
- Transition plan: If transition-plan assumptions and dependencies are unavailable, state that available evidence identifies transition-plan and net-zero target governance decisions but does not specify the key assumptions or dependencies underlying the transition plan.
- Scenarios: State that scenario financial outputs are modelled estimates and should not be interpreted as realised losses.
- Opportunities: State that opportunity revenue impacts are estimates and should not be interpreted as assured future revenue.
- Resilience: State that resilience observations are scenario-specific and do not constitute a general statement of overall bank-wide resilience.

Data-aware coverage requirements:
1. Use the Strategy evidence as the source of Strategy facts.
2. Use risk register, climate scenarios, value-chain map, climate opportunities, financial summary and reporting KPIs when present.
3. Distinguish physical risks from transition risks.
4. Cover time horizons only where risk-register or scenario evidence supports them.
5. Use value-chain evidence when available. If some material nodes lack financial exposure, state that boundary.
6. Use climate-opportunity evidence when available. Opportunity revenue impacts must be described as estimates, not guaranteed revenue.
7. Use high-carbon and fossil-fuel exposure metrics when present.
8. Use climate capex and opex as resource-allocation evidence when present.
9. Use target evidence carefully:
   - if full target records are available, use only fields explicitly provided;
   - if only reporting_kpis.target_summary is available, use only target type, scope, status, target year, framework and progress;
   - do not invent baselines, validation bodies, gross/net status, milestones, planned credits or full target mechanics unless full target records are present.

Transition plan disclosure:
- Use updated transition plan and net-zero interim target evidence only if evidenced.
- Use target frameworks and progress from target_summary or full targets.
- If key transition-plan assumptions or dependencies are not available, state this boundary.
- Do not invent dependencies, policy assumptions, customer behaviour assumptions or financing dependencies.

Other boundaries:
- Scenario financial outputs are modelled estimates, not actual losses.
- Resilience statements must remain scenario-specific and must not claim overall bank resilience.
- Do not claim overall Paris alignment. If target/scenario flags indicate alignment, use cautious wording and keep it tied to the specific target or scenario evidence.
- If business_model_impacts or strategy_tradeoff_decisions are absent, state the evidence boundary once; do not invent detailed business-model impacts or trade-off analysis.
- Do not include visible IFRS paragraph references or bracketed IFRS tags in the final output.
- Use "available evidence", "available documentation" or "source data" in limitation wording. Do not use the word "payload" in the final report.
- Keep the Strategy section report-style and avoid generic promotional language such as "leader", "fully resilient", "guaranteed", "ensures", or "Paris-aligned" unless explicitly and narrowly supported.
""".strip()

print("Strategy requirements ready — Emirates-style structure")

Strategy requirements ready — Emirates-style structure


## Strategy writer and judge prompts


In [25]:
# ── STRATEGY WRITER / JUDGE PROMPTS ─────────────────────────
STRATEGY_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Strategy section of a dedicated IFRS S1/S2-style climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style similar to a dedicated bank IFRS S1/S2 report.

Evidence rules:
- Translate raw data field names into public report wording; avoid snake_case and key=value syntax in the final section.
- Never end the section mid-sentence; complete every required subsection, especially the final evidence-boundary subsection.
- Use the clean style guide only for writing style, tone, structure and formatting; never use it as factual evidence.
- Use only the compact Strategy evidence supplied by the user prompt.
- Do not invent strategy decisions, business-model impacts, trade-offs, value-chain nodes, opportunities, target details, scenario assumptions, financial effects or resilience conclusions.
- If evidence is missing, state any genuine gap once, in the Bank's own voice as the Bank's position - for example "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references or bracketed evidence tags.
- Do not use markdown tables.

Length and completion rules:
- Write all seven subsections completely.
- Do not over-expand the earlier subsections.
- The Overview should be concise. Each middle subsection should be around 120–200 words unless more evidence is needed.
- The Climate resilience and Strategy limitations subsections are mandatory and must be completed.
- If there is limited space, shorten earlier sections instead of omitting or truncating the final two sections.
- Do not end the output with an unfinished sentence.

Cautious wording rules:
- Use cautious wording for adaptation capacity.
- Prefer "available evidence indicates mechanisms such as", "available evidence identifies", and "the evidence does not provide enough detail to assess full operational adaptation capacity".
- Avoid "supported by", "demonstrates resilience", "proves flexibility", and "current and planned investments" unless planned investment evidence is explicit.


Emirates-style structure rules:
- Begin with an Overview subsection.
- Present risks and opportunities across the value chain before discussing management actions.
- Keep resilience and scenario analysis as a dedicated subsection.
- Keep limitations and evidence boundaries as the final subsection.
- Avoid generic sustainability marketing language. Use bank-report wording and evidence-based claims.

Additional precision rules (avoid the recurring overclaims):
- Keep transition-risk and physical-risk mitigation tools separate. Collateral revaluation relates to stranded-asset / transition exposure; flood or hazard mapping overlays relate to physical risk. Never present hazard or flood mapping as a stranded-asset (transition) mitigation mechanism.
- In the resilience and scenario subsection, state stranded assets explicitly as a scenario output: name the concept, give the modelled estimate, and tie it to the cited scenario and horizon, so the stranded-assets treatment is unmistakable.
- If planned sources of funding to implement the strategy are not in the evidence, state that as a boundary rather than implying funding is in place.

- When citing a mitigation action against a specific named risk, use ONLY the action the evidence records for that exact risk. Do not attach "collateral revaluation and flood mapping overlay" (or any other action) to the stranded-assets / fossil-fuel exposure risk unless the per-risk evidence links them. If the per-risk mapping is unclear, describe mitigation actions as a register-level set without attributing a specific action to a specific named risk.

REPORTING VOICE (this overrides any other wording guidance, including wording in the evidence): Write strictly as the Bank's own published disclosure, in the Bank's voice, and state facts directly. The following are BANNED from the final section: "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", "based on available evidence/documentation", and any phrasing that narrates what "the documentation" or "the evidence" does or does not show. Write "The Board met eight times in 2024", never "Available documentation indicates the Board met eight times". For genuine gaps, write the Bank's position: "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".

- Do not state that target validation bodies, baselines or milestones are unavailable. The Bank's climate targets and their recorded attributes (including framework and any validation references) are set out in the Metrics and targets section; refer the reader there rather than describing target detail as unavailable in this section.

Output rules:
- Return only the complete Strategy section.
- Organise the section into clear report-style subsections covering the required topics (overview; value-chain risks and opportunities; strategic management; effects on business model and decision-making; financial effects and resource allocation; resilience and scenario analysis; limitations and boundaries). Heading wording may vary.
""".strip()


def build_strategy_writer_prompt(
    evidence: dict,
    judge_feedback: str | None = None,
) -> str:
    profile = evidence.get("availability_profile", {})
    boundaries = evidence.get("evidence_boundaries", {})
    instructions = []
    instructions.append("- Frame portfolio steering and transition/net-zero ambitions as the Bank's stated intent and direction of travel, not as demonstrated or assured alignment outcomes; do not imply alignment has been achieved.")

    if profile.get("physical_risk_evidence_available") and profile.get("transition_risk_evidence_available"):
        instructions.append("- Distinguish and describe both physical and transition risks using the supplied risk examples.")
    else:
        instructions.append("- Describe only the risk types evidenced; do not invent missing risk categories.")

    if profile.get("time_horizons_available"):
        instructions.append("- Use available short-, medium- and long-term horizons where supported by risk/scenario evidence.")

    if profile.get("business_model_detail_available"):
        instructions.append("- Explain supplied business-model impacts and transmission channels.")
    else:
        instructions.append("- Detailed business_model_impacts evidence is absent. Use lending, portfolio, value-chain and financial evidence, and state that detailed business-model impact evidence is limited.")

    if profile.get("value_chain_detail_available"):
        instructions.append("- Use the value-chain map, including own operations, suppliers and financing counterparties.")
        if profile.get("quantified_value_chain_exposure_available"):
            instructions.append("- Include quantified value-chain exposures and state that some material nodes are qualitative where financial exposure is missing.")
    else:
        instructions.append("- Value-chain evidence is unavailable. Do not invent value-chain nodes or exposures.")

    if profile.get("strategy_tradeoff_evidence_available"):
        instructions.append("- Describe only documented Strategy trade-offs provided in evidence.")
    else:
        instructions.append("- No strategy_tradeoff_decisions evidence is available. State that quantified/documented trade-off analysis is not available.")

    if profile.get("resource_allocation_evidence_available"):
        instructions.append("- Use climate capex and climate opex as resource-allocation evidence.")

    if profile.get("high_carbon_exposure_available"):
        instructions.append("- Use high-carbon sector exposure metrics.")
    if profile.get("fossil_fuel_exposure_available"):
        instructions.append("- Use fossil-fuel exposure metrics.")

    if profile.get("climate_opportunities_available"):
        instructions.append("- Describe the concrete climate opportunities supplied in evidence.")
        if profile.get("quantified_opportunity_estimates_available"):
            instructions.append("- Use opportunity revenue estimates cautiously and identify them as estimates with confidence levels.")
    else:
        instructions.append("- No climate-opportunity evidence is available. Do not invent opportunities.")

    if profile.get("scenario_analysis_available"):
        instructions.append("- Explain scenario families, framework, horizons, assumptions and key modelled outputs.")
        instructions.append("- Explicitly cover scenario inputs/assumptions: carbon price, temperature outcomes, technology readiness, macroeconomic assumptions where available, renewable energy share where available, methodology notes and scope of analysis.")
    else:
        instructions.append("- Scenario analysis is unavailable. Do not invent scenario results.")

    if profile.get("resilience_evidence_available"):
        instructions.append("- Describe resilience only within supplied scenario-specific assumptions and boundaries.")
        instructions.append("- Cover adaptation-capacity evidence using available financial resources, portfolio flexibility mechanisms and current/planned investment evidence. Do not invent asset redeployment/decommissioning capacity.")

    if profile.get("targets_available"):
        if profile.get("full_targets_table_available"):
            instructions.append("- Full target records are available; use only fields explicitly provided.")
        elif profile.get("target_summary_available"):
            instructions.append("- Only target_summary is available. Use target type, scope, status, year, framework and progress only; do not invent baselines, validation bodies, milestones, gross/net status or planned credits.")
    else:
        instructions.append("- Target evidence is unavailable. Do not invent target information.")

    instructions.append("- For transition-plan disclosure, use available updated transition plan/net-zero target decision evidence and target frameworks/progress only. If key assumptions or dependencies are not available, state that boundary.")

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION POLICY:
- Fix omissions using available evidence.
- Preserve accurate evidence-boundary statements where evidence is unavailable.
- Never invent evidence to satisfy the judge.
""".strip()

    return f"""
{STRATEGY_REQUIREMENTS}

WRITING STYLE GUIDE — STYLE ONLY, NOT EVIDENCE:
{build_style_context("strategy")}

STYLE APPLICATION RULES:
- Use the style guide only for tone, flow, structure and formatting.
- Use compact Strategy evidence for facts.
- Use IFRS requirements only to decide what topics to cover.
- Do not copy reference-report wording or mention forbidden reference terms.

BANK:
{evidence['bank']['bank_name']} ({evidence['bank']['bank_id']})

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:

SENIOR STRATEGY FIXES:
- Never use raw tokens such as SBTi_1.5°C; write "SBTi 1.5°C pathway".
- Do not state that baselines, milestones, validation bodies, gross-versus-net treatment or carbon-credit use are not described in the source data. If needed, say detailed assumptions and validation evidence are described only where available in Metrics and targets.
- Technology readiness must be described as varying by scenario across low, medium and high levels if mentioned.
- Avoid overstrong wording such as "ensuring that climate-related considerations are embedded"; use "supports incorporation" or "is intended to support".
- In Financial effects, clarify that climate capex/opex are resource allocation indicators, not recognised P&L impacts, and name possible affected line items qualitatively only where mapping is not available.
- In Scenario analysis, state that stranded asset estimates are derived from sector-level fossil fuel capex exposure when discussing the Net Zero 2050 estimate.
- Constrain resilience conclusions to named scenario assumptions and internal modelled outputs.

- Do not say baselines, milestones, validation bodies, gross/net status or carbon-credit use are absent from all source data, because these may be available in the Metrics and targets evidence. In Strategy, state only that detailed target assumptions and dependencies are not fully specified in Strategy evidence and are described only where available in Metrics and targets.
- Do not expose raw target field names or underscore-form framework names. Translate values into public report wording.
{chr(10).join(instructions)}

SECTION BLUEPRINT:
### Strategy

#### Overview
Summarise how the Bank's climate strategy is organised around value-chain risks and opportunities, transition planning, resource allocation and scenario analysis. Keep this concise.

#### Sustainability risks and opportunities across the value chain
Use physical and transition risk evidence, value-chain map evidence, high-carbon/fossil-fuel exposure and climate opportunities where available.

#### Strategic management of climate-related risks and opportunities
Describe evidenced management actions such as transition plan updates, target governance, green finance, exposure limits, client engagement, scenario stress testing, collateral overlays or other mechanisms present in the evidence. If transition_plan evidence is present, describe the transition plan's key assumptions, the dependencies it relies on, how it is resourced, and progress since prior periods.

#### Climate-related effects on business model, value chain and decision-making
Describe impacts on lending, portfolio composition, value chain and decision-making only where evidenced. If detailed business-model impact or trade-off evidence is unavailable, state that boundary.

#### Financial effects and resource allocation
Use climate capex, climate opex, scenario financial outputs and opportunity estimates cautiously. Label modelled outputs as estimates. If climate_financial_effects evidence is present, describe the current and anticipated effects of climate-related risks and opportunities on specific financial-statement line items (for example the expected credit loss allowance on loans, or net interest income), distinguishing current from anticipated effects and flagging any with a significant risk of a material adjustment in the next annual period.

#### Climate resilience and scenario analysis
Describe NGFS v4 scenario families, scenario names, horizons, assumptions, methodology, scope and scenario-specific resilience observations. Do not claim overall bank-wide resilience. If resilience_capacity evidence is present, describe the Bank's capacity to adjust its strategy and business model (including the flexibility of its financial resources and its ability to redeploy or repurpose assets) and the significant areas of uncertainty, by scenario type.

#### Strategy limitations and evidence boundaries
State evidence boundaries for business-model impacts, trade-offs, targets, transition-plan assumptions, scenario estimates, opportunity estimates and resilience.

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("strategy")}

REQUIREMENT-TO-EVIDENCE POLICY:
- Use IFRS requirements to decide which strategy topics must be addressed.
- Use compact bank evidence to write bank-specific facts.
- If an IFRS requirement is relevant but evidence is unavailable, disclose the evidence boundary instead of inventing strategy, transition-plan, resilience or financial-effect details.
- Treat core_standard items as mandatory disclosure guidance.
- Treat industry_guidance items as banking-specific application guidance.
- Do not display IFRS paragraph references or requirement IDs in the final report.

COMPACT STRATEGY EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Strategy payload tables available: {evidence.get('payload_profile', {}).get('top_level_keys')}
- Full targets table available: {evidence.get('payload_profile', {}).get('full_targets_table_available')}
- Target summary available: {evidence.get('payload_profile', {}).get('target_summary_available')}
- Target instruction: {boundaries.get('target_instruction')}
- Business model impacts available: {boundaries.get('business_model_detail_available')}
- Trade-off evidence available: {boundaries.get('tradeoff_evidence_available')}
- Scenario outputs are modelled estimates: {boundaries.get('scenario_outputs_are_modelled_estimates')}
- Opportunity impacts are estimates: {boundaries.get('opportunity_impacts_are_estimates')}
- Transition plan evidence available: {boundaries.get('transition_plan_available')}
- Financial effects evidence available: {boundaries.get('financial_effects_available')}
- Resilience capacity evidence available: {boundaries.get('resilience_capacity_available')}
- Only state a boundary for transition-plan assumptions, financial effects or resilience capacity where that evidence is ABSENT; where present, disclose it in the relevant subsection instead.

GENERAL WRITING RULES:
- Do not claim that opportunities are guaranteed revenue.
- Do not claim that scenario losses are actual losses.
- Do not claim that the bank is generally resilient.
- Do not claim overall Paris alignment.
- Use exact figures from compact evidence.
- Mention evidence boundaries only once in the relevant subsection.

{feedback_block}

COMPLETION PRIORITY:
You must finish all seven subsections. The final subsection "Strategy limitations and evidence boundaries" must contain complete boundary statements. Do not end the output with an unfinished sentence. If necessary, shorten earlier subsections to ensure the final two subsections are complete.

Write the complete Strategy section only.
""".strip()


STRATEGY_JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate Strategy disclosure.

You evaluate whether the Strategy section is:
- supported by compact evidence;
- aligned with the Strategy disclosure checklist;
- transparent about missing evidence;
- free from unsupported claims about targets, opportunities, scenarios, business model, value chain, Paris alignment and resilience.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_strategy_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Strategy draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

REPORTING VOICE: The section must read as the Bank's own published disclosure. Treat as a material wording defect any report-machinery language - "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", or phrasing that narrates what "the documentation"/"the evidence" does or does not show. Facts must be stated directly; genuine gaps framed as the Bank's position ("the Bank does not currently disclose ...", "... is outside the scope of this report", "... has not yet been quantified"). List any such phrasing in material_unsupported_claims so it is corrected.

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("strategy")}

COMPACT STRATEGY EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
0. Check the IFRS requirements pack and the ifrs_requirement_coverage pre-check. Penalise missing mandatory core IFRS topics unless the draft clearly discloses an evidence boundary.
1. Penalise claims that contradict evidence, invent unsupported information, or overstate estimates.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed boundaries are acceptable but lower completeness.
4. Verify that the section is organised into clear report-style subsections covering the required topics (overview; value-chain risks and opportunities; strategic management; effects on business model and decision-making; financial effects and resource allocation; resilience and scenario analysis; limitations and boundaries) and reads like a report section rather than a checklist. The exact heading wording does not matter.
4A. Verify use of physical and transition risks, time horizons, scenario evidence, value-chain map, opportunities, portfolio metrics and resource allocation.
5. If full targets table is unavailable, the draft must not invent baselines, validation bodies, milestones, gross/net status or planned credits.
6. If business_model_impacts are unavailable, the draft must not claim detailed business-model impact analysis beyond available lending/value-chain/financial channels.
7. If strategy_tradeoff_decisions are unavailable, the draft must not invent trade-off analysis.
8. Opportunity revenue impacts must be described as estimates, not guaranteed future revenue.
9. Scenario financial outputs must be described as modelled estimates, not actual losses.
10. Scenario assumptions should cover framework, scenario types/names, horizons, carbon price, temperature outcomes, technology readiness, macroeconomic/energy assumptions where available, methodology notes and scope.
11. Resilience claims must remain scenario-specific and bounded and should discuss adaptation capacity only from available evidence.
12. Transition-plan discussion must use available transition-plan/target evidence and state boundaries for missing assumptions/dependencies.
13. Overall Paris-alignment claims are not allowed unless directly supported and carefully bounded.
14. No visible IFRS paragraph references or bracketed tags should appear.
15. All checklist values must be JSON booleans true/false, not strings.

SCORING:
Score on completeness RELATIVE TO AVAILABLE EVIDENCE, not on how much data the bank happens to hold. Correctly disclosed evidence boundaries (genuine unavailability, accurately stated) are good IFRS practice and MUST NOT reduce the score; the number of disclosed boundaries is irrelevant to the score. Penalise ONLY: unsupported or overclaimed statements, omitted AVAILABLE evidence, factual inconsistencies with the evidence, hallucinations, and structural or deterministic failures.
- 9-10: Materially complete relative to available evidence, report-ready structure, no unsupported or overclaimed statements, no omitted available evidence, and all genuine data gaps correctly disclosed as boundaries.
- 8: As 9-10 but with one minor evidence-use issue or one slightly loose statement to tighten.
- 7: Usable but contains an unsupported/overclaimed statement, omits available evidence, or has a structural/keyword gap.
- 6: Revision required: available evidence omitted, unsupported wording remains, factual inconsistency with evidence, or required structure/boundary keywords missing.
- 5 or below: Major evidence failure, hallucination, contradiction, or missing core subsection.

When listing omitted evidence, separate MATERIAL omissions (which must be incorporated) from OPTIONAL ones (acceptable to leave out); never mark internal record IDs or optional traceability codes as material. Likewise, separate unsupported claims into material_unsupported_claims (misrepresent the evidence or overstate scope/assurance/compliance — must fix) and minor_wording_to_tighten (defensible phrasing such as 'covers' vs 'includes' or singular/plural that does not misrepresent the evidence — an acceptable minor limitation). Do not list trivial wording as material. Return valid JSON only. Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "overview_present": <true/false>,
    "report_style_structure_present": <true/false>,
    "physical_and_transition_risks_used_correctly": <true/false>,
    "time_horizons_used_correctly": <true/false>,
    "business_model_handled_according_to_availability": <true/false>,
    "value_chain_used_correctly": <true/false>,
    "strategy_tradeoffs_handled_according_to_availability": <true/false>,
    "financial_effects_used_correctly": <true/false>,
    "resource_allocation_used_correctly": <true/false>,
    "high_carbon_and_fossil_exposure_used_if_available": <true/false>,
    "opportunities_used_correctly": <true/false>,
    "targets_handled_according_to_availability": <true/false>,
    "scenario_analysis_used_correctly": <true/false>,
    "scenario_assumptions_used_correctly": <true/false>,
    "transition_plan_handled_according_to_availability": <true/false>,
    "adaptation_capacity_handled_according_to_availability": <true/false>,
    "resilience_not_overclaimed": <true/false>,
    "paris_alignment_not_overclaimed": <true/false>,
    "modelled_estimates_not_overstated": <true/false>,
    "no_visible_ifrs_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "material_evidence_omitted": [<omitted available evidence whose absence materially weakens the disclosure; MUST be incorporated as report prose>],
  "optional_evidence_omitted": [<omitted available evidence that is acceptable to leave out; do not force inclusion>],
  "unsupported_claims": [<specific unsupported claims>],
  "material_unsupported_claims": [<claims that misrepresent the evidence, assert unevidenced facts, or overstate scope/assurance/compliance; these MUST be removed or reframed>],
  "minor_wording_to_tighten": [<defensible phrasing that could be tightened but does NOT misrepresent the evidence; acceptable as a minor limitation>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()


print("Strategy prompts ready — Emirates-style structure")

Strategy prompts ready — Emirates-style structure


## Strategy evaluation mode


In [26]:
# ── STRATEGY EVALUATION MODE ───────────────────────────────
# Data-aware deterministic pre-checks are now active.
# They do not directly approve/reject the section or cap the score.
# Instead, their findings are passed to the GPT-5.2 Judge.

print("Strategy evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

Strategy evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks


## Strategy LangGraph nodes and deterministic checks


In [27]:
# ── STRATEGY LANGGRAPH NODES ────────────────────────────────

def extract_numbers(text: str) -> list[float]:
    nums = re.findall(r"\b\d+(?:\.\d+)?\b", text.replace(",", ""))
    return [float(n) for n in nums]

def number_present(text: str, expected: float, tolerance: float = 0.2) -> bool:
    nums = extract_numbers(text)
    return any(abs(n - expected) <= tolerance for n in nums)

def run_strategy_deterministic_checks(draft: str, evidence: dict) -> dict:
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})

    required_headings = [
        "#### Overview",
        "#### Sustainability risks and opportunities across the value chain",
        "#### Strategic management of climate-related risks and opportunities",
        "#### Climate-related effects on business model, value chain and decision-making",
        "#### Financial effects and resource allocation",
        "#### Climate resilience and scenario analysis",
        "#### Strategy limitations and evidence boundaries",
    ]

    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    # Relaxed structure rule: accept report-style structure with the writer's own
    # headings. Only fail if the section is not genuinely structured; treat a
    # different-but-valid heading set as a warning, not a failure.
    heading_count = text.count("#### ")
    if heading_count < 4:
        failures.append({"check": "missing_report_structure",
                         "details": f"Section is not in report-style structure (only {heading_count} headings found)."})
    elif missing_headings:
        warnings.append({"check": "non_standard_headings",
                         "details": f"Report-style headings differ from the standard set (acceptable): {missing_headings}"})

    # Report-style structure checks.
    if "#### Overview" in text:
        overview_text = text.split("#### Overview", 1)[1].split("#### Sustainability risks and opportunities across the value chain", 1)[0].lower()
        if len(overview_text.split()) < 45:
            warnings.append({
                "check": "strategy_overview_may_be_too_short",
                "details": "Overview is present but may be too short to introduce the strategy architecture."
            })

    if "checklist" in text_l or "ifrs s2 requires" in text_l:
        warnings.append({
            "check": "strategy_may_read_like_checklist",
            "details": "Strategy section should read like a report section, not a checklist response."
        })

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS references/tags found."})

    unsupported_strong_phrases = [
        "fully resilient",
        "guarantees",
        "guaranteed",
        "proves resilience",
        "fully paris aligned",
        "overall paris aligned",
        "no material risk",
    ]
    found_strong = [p for p in unsupported_strong_phrases if p in text_l]
    if found_strong:
        failures.append({"check": "unsupported_strong_strategy_language", "details": found_strong})

    if profile.get("climate_opportunities_available") and "opportun" not in text_l:
        warnings.append({"check": "opportunities_may_be_omitted", "details": "Climate opportunity evidence exists but opportunity wording is not detected."})

    if profile.get("quantified_opportunity_estimates_available"):
        if "estimate" not in text_l and "estimated" not in text_l:
            failures.append({"check": "opportunity_estimates_not_labelled", "details": "Opportunity revenue impacts should be described as estimates."})

    if profile.get("scenario_financial_outputs_available"):
        estimate_terms = ["modelled", "modeled", "scenario", "estimate", "projected"]
        if not any(term in text_l for term in estimate_terms):
            failures.append({"check": "scenario_outputs_not_labelled_as_estimates", "details": "Scenario financial outputs should be labelled as modelled estimates."})

    if profile.get("value_chain_detail_available") and "value chain" not in text_l:
        warnings.append({"check": "value_chain_may_be_omitted", "details": "Value-chain evidence exists but value-chain wording is not detected."})

    if profile.get("high_carbon_exposure_available") and "high-carbon" not in text_l and "high carbon" not in text_l:
        warnings.append({"check": "high_carbon_exposure_may_be_omitted", "details": "High-carbon exposure metrics exist but may be omitted."})

    if profile.get("fossil_fuel_exposure_available") and "fossil" not in text_l:
        warnings.append({"check": "fossil_fuel_exposure_may_be_omitted", "details": "Fossil-fuel exposure metrics exist but may be omitted."})

    if profile.get("resource_allocation_evidence_available"):
        _capex = (evidence.get("portfolio_summary", {}) or {}).get("climate_capex_meur")
        if profile.get("resource_allocation_evidence_available", False) and _capex is not None:
            if not number_present(text, float(_capex), tolerance=0.2):
                failures.append({"check": "missing_climate_capex", "details": f"Climate capex of EUR {_capex} million is available and should be used."})
        _opex = (evidence.get("portfolio_summary", {}) or {}).get("climate_opex_meur")
        if profile.get("resource_allocation_evidence_available", False) and _opex is not None:
            if not number_present(text, float(_opex), tolerance=0.2):
                failures.append({"check": "missing_climate_opex", "details": f"Climate opex of EUR {_opex} million is available and should be used."})

    if profile.get("target_summary_available") and not profile.get("full_targets_table_available"):
        unsupported_target_terms = [
            "baseline year",
            "baseline value",
            "validation body",
            "third-party validated",
            "gross target",
            "net target",
            "interim milestone",
            "planned carbon credits",
        ]
        found_target_terms = [p for p in unsupported_target_terms if term_in_claiming_context(text_l, p)]
        if found_target_terms:
            failures.append({"check": "full_target_details_invented", "details": found_target_terms})

    if not profile.get("strategy_tradeoff_evidence_available"):
        if ("trade-off" in text_l or "tradeoff" in text_l) and not any(term in text_l for term in ["not available", "not documented", "does not provide", "no quantified"]):
            failures.append({"check": "strategy_tradeoff_overclaimed", "details": "Trade-off language appears without available trade-off evidence or limitation."})

    if not profile.get("business_model_detail_available"):
        if "detailed business model" in text_l and not any(term in text_l for term in ["limited", "not available", "not detailed"]):
            failures.append({"check": "business_model_detail_overclaimed", "details": "Detailed business-model impact evidence is absent."})

    if profile.get("scenario_assumptions_available"):
        assumption_terms = ["carbon price", "temperature", "technology readiness", "methodology", "scope of analysis"]
        missing_terms = [term for term in assumption_terms if term not in text_l]
        if missing_terms:
            warnings.append({"check": "scenario_assumption_terms_may_be_missing", "details": missing_terms})

    if profile.get("resilience_evidence_available"):
        if "resilience" in text_l and not any(term in text_l for term in ["scenario", "under", "within", "assumption"]):
            failures.append({"check": "resilience_not_scenario_bounded", "details": "Resilience wording should be scenario-specific and bounded."})


    # The final limitations subsection must be complete, not just present.
    limitations_heading = "#### Strategy limitations and evidence boundaries"

    if limitations_heading in text:
        limitations_text = text.split(limitations_heading, 1)[1].strip()
        limitations_l = limitations_text.lower()

        required_boundary_terms = [
            "business-model",
            "business model",
            "trade-off",
            "tradeoff",
            "transition-plan",
            "transition plan",
            "modelled estimates",
            "modeled estimates",
            "opportunity revenue impacts",
            "scenario-specific",
            "scenario specific",
        ]

        if len(limitations_text.split()) < 90:
            failures.append({
                "check": "limitations_section_incomplete",
                "details": "Strategy limitations section is present but too short or truncated."
            })

        missing_boundary_terms = [
            term for term in required_boundary_terms
            if term not in limitations_l
        ]

        # Treat paired alternatives as satisfied if either spelling exists.
        if ("business-model" in missing_boundary_terms and "business model" not in missing_boundary_terms):
            missing_boundary_terms.remove("business-model")
        if ("business model" in missing_boundary_terms and "business-model" not in missing_boundary_terms):
            missing_boundary_terms.remove("business model")
        if ("trade-off" in missing_boundary_terms and "tradeoff" not in missing_boundary_terms):
            missing_boundary_terms.remove("trade-off")
        if ("tradeoff" in missing_boundary_terms and "trade-off" not in missing_boundary_terms):
            missing_boundary_terms.remove("tradeoff")
        if ("transition-plan" in missing_boundary_terms and "transition plan" not in missing_boundary_terms):
            missing_boundary_terms.remove("transition-plan")
        if ("transition plan" in missing_boundary_terms and "transition-plan" not in missing_boundary_terms):
            missing_boundary_terms.remove("transition plan")
        if ("modelled estimates" in missing_boundary_terms and "modeled estimates" not in missing_boundary_terms):
            missing_boundary_terms.remove("modelled estimates")
        if ("modeled estimates" in missing_boundary_terms and "modelled estimates" not in missing_boundary_terms):
            missing_boundary_terms.remove("modeled estimates")
        if ("scenario-specific" in missing_boundary_terms and "scenario specific" not in missing_boundary_terms):
            missing_boundary_terms.remove("scenario-specific")
        if ("scenario specific" in missing_boundary_terms and "scenario-specific" not in missing_boundary_terms):
            missing_boundary_terms.remove("scenario specific")

        if missing_boundary_terms:
            warnings.append({
                "check": "limitations_boundaries_incomplete",
                "details": f"Boundary concepts not stated with the standard keywords (acceptable if covered in other wording): {missing_boundary_terms}"
            })
    else:
        failures.append({
            "check": "limitations_section_missing",
            "details": "Strategy limitations and evidence boundaries subsection is missing."
        })

    # Scenario/resilience subsection must contain key concepts when scenario evidence is available.
    if "#### Climate resilience and scenario analysis" in text:
        scenario_text = text.split(
            "#### Climate resilience and scenario analysis", 1
        )[1].split("#### Strategy limitations and evidence boundaries", 1)[0].lower()

        required_scenario_terms = [
            "ngfs",
            "orderly",
            "disorderly",
            "hot",
            "carbon price",
            "temperature",
            "technology readiness",
            "modelled",
            "physical risk loss",
            "transition risk loss",
            "stranded assets",
            "revenue at risk",
        ]

        missing_scenario_terms = [
            term for term in required_scenario_terms
            if term not in scenario_text
        ]

        if missing_scenario_terms:
            failures.append({
                "check": "scenario_analysis_incomplete",
                "details": f"Missing scenario concepts: {missing_scenario_terms}"
            })

        if "overall resilience" in scenario_text or "fully resilient" in scenario_text:
            failures.append({
                "check": "resilience_overclaimed",
                "details": "Resilience wording implies overall resilience instead of scenario-specific resilience."
            })
    else:
        failures.append({
            "check": "scenario_resilience_section_missing",
            "details": "Climate resilience and scenario analysis subsection is missing."
        })



    if "sbti_1.5" in text_l:
        failures.append({"check": "raw_strategy_framework_token", "details": "Use 'SBTi 1.5°C pathway', not raw SBTi_1.5 token."})

    if "baselines, milestones, validation bodies" in text_l and "are not described" in text_l:
        failures.append({
            "check": "strategy_metrics_targets_contradiction",
            "details": "Do not contradict Metrics and targets by saying target details are not described in source data."
        })

    if "technology readiness is assumed to be medium to high" in text_l:
        warnings.append({
            "check": "scenario_technology_readiness_range_too_narrow",
            "details": "Use low/medium/high range by scenario type."
        })

    if "ensuring that climate-related considerations are embedded" in text_l:
        warnings.append({
            "check": "strategy_overstrong_embedding_language",
            "details": "Use softer evidence-bound wording."
        })

    style_leak = check_reference_identity_leak(text)
    if not style_leak.get("passed", True):
        failures.append({"check": "reference_identity_leak", "details": style_leak.get("matches", [])})

    completeness = check_section_completeness(text, "strategy")
    failures.extend(completeness.get("failures", []))
    warnings.extend(completeness.get("warnings", []))

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def strategy_writer_node(state: StrategyState) -> StrategyState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_strategy_writer_prompt(state["evidence"], judge_feedback=feedback)
    draft = call_strategy_writer_llm(
        system_prompt=STRATEGY_WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"STRATEGY WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {**state, "draft": draft.strip(), "status": "judging"}


def strategy_judge_node(state: StrategyState) -> StrategyState:
    draft = state["draft"]
    deterministic_checks = run_strategy_deterministic_checks(draft, state["evidence"])
    deterministic_checks["ifrs_requirement_coverage"] = check_ifrs_requirement_coverage(draft, "strategy")

    prompt = build_strategy_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )
    judge_result = call_judge_llm_json(
        system_prompt=STRATEGY_JUDGE_SYSTEM,
        user_prompt=prompt,
    )
    judge_result = normalize_json_booleans(judge_result)

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    judge_result["_best_so_far"] = track_best(state, judge_result)  # non-regression: keep best draft across iterations

    print("\nSTRATEGY JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    approved = bool(judge_result.get("approved"))
    status = "approved" if approved else "revising"

    return {
        **state,
        "judge_result": judge_result,
        "status": status,
    }


STRATEGY_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Strategy section using only:
- the supplied compact Strategy evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Clearly distinguish estimates, scenarios and actual financial effects.
- Keep target details within available evidence.
- Keep the exact six-subsection structure.
- Do not add visible IFRS paragraph references.
- Return only the complete revised Strategy section using the required seven headings.
""".strip()


def strategy_reviser_node(state: StrategyState) -> StrategyState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT STRATEGY REQUIREMENTS:
{STRATEGY_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("strategy")}

COMPACT STRATEGY EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- Preserve or improve accurate boundary statements where evidence is unavailable.
- Never invent missing opportunities, trade-offs, business-model impacts, value-chain effects, target details, scenario assumptions, financial impacts, or resilience conclusions.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Strategy section.
""".strip()

    revised_draft = call_strategy_reviser_llm(
        system_prompt=STRATEGY_REVISER_SYSTEM,
        user_prompt=(revision_prompt + "\n\nPRIORITISED REVISION BRIEF (address every item using available evidence only):\n" + build_reviser_focus(judge)),
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nStrategy revised with GPT-5.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def strategy_finalize_node(state: StrategyState) -> StrategyState:
    new_state = finalize_state(state, target_score=TARGET_SCORE)
    gate = new_state["judge_result"]["finalize_gate"]
    print(f"\n{'='*50}")
    print("FINALIZED STRATEGY")
    print(f"  Score: {gate['score']}/{gate['target_score']} | publishable={gate['publishable']}")
    print(f"  Status: {new_state['judge_result']['approval_status']} | revisions: {state['revision_count']}")
    if not gate["publishable"]:
        print("  Blocking reasons:", gate["blocking_reasons"])
    print(f"{'='*50}")
    return new_state


def strategy_should_continue(state: StrategyState) -> str:
    return route_after_judge_generic(state, target_score=TARGET_SCORE, revise_label="reviser")




## Build and compile Strategy graph


In [28]:

# ── BUILD AND COMPILE STRATEGY GRAPH ────────────────────────
strategy_builder = StateGraph(StrategyState)

strategy_builder.add_node("writer",   strategy_writer_node)
strategy_builder.add_node("judge",    strategy_judge_node)
strategy_builder.add_node("reviser",  strategy_reviser_node)
strategy_builder.add_node("finalize", strategy_finalize_node)

strategy_builder.add_edge(START, "writer")
strategy_builder.add_edge("writer", "judge")
strategy_builder.add_conditional_edges(
    "judge",
    strategy_should_continue,
    {
        "finalize": "finalize",
        "reviser": "reviser",
    }
)
strategy_builder.add_edge("reviser", "judge")
strategy_builder.add_edge("finalize", END)

strategy_graph = strategy_builder.compile()
print("Strategy graph compiled")


Strategy graph compiled


## Run Strategy generation


In [29]:

# ── RUN STRATEGY SECTION ────────────────────────────────────
strategy_initial_state: StrategyState = {
    "bank_name":      bank_name,
    "evidence":       strategy_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions": 3,  # one GPT-5.1 revision pass if the GPT-5.2 judge rejects the draft
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}

print(f"Starting strategy generation for: {bank_name}\n")
strategy_result = run_section_with_floor(strategy_graph, strategy_initial_state, label="strategy")


Starting strategy generation for: Eurolux Universal Bank AG


STRATEGY WRITER (initial draft)
Draft length: 2413 words

STRATEGY JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS
{
  "overall_score": 7,
  "evidence_support_score": 8,
  "ifrs_alignment_score": 7,
  "specificity_score": 8,
  "hallucination_risk": "medium",
  "approval_status": "revision_required",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "overview_present": true,
    "report_style_structure_present": true,
    "physical_and_transition_risks_used_correctly": true,
    "time_horizons_used_correctly": true,
    "business_model_handled_according_to_availability": true,
    "value_chain_used_correctly": true,
    "strategy_tradeoffs_handled_according_to_availability": true,
    "financial_effects_used_correctly": false,
    "resource_allocation_used_correctly": true,
    "high_carbon_and_fossil_exposure_used_if_available": true,
    "opportunities_used_correctly": true,
    "targets_handle

## Print and save Strategy output


In [30]:

# ── STRATEGY OUTPUT ─────────────────────────────────────────
print("\n" + "="*60)
print("FINAL STRATEGY JUDGE RESULT")
print("="*60)
print(json.dumps(strategy_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("STRATEGY SECTION")
print("="*60)
print(strategy_result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "strategy_BANK01.md", "w", encoding="utf-8") as f:
    f.write(strategy_result["final_section"])

with open(output_dir / "strategy_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "strategy",
        "status":         strategy_result["status"],
        "approval_status": strategy_result["judge_result"].get("approval_status"),
        "raw_score":      strategy_result["judge_result"].get("raw_score_before_caps"),
        "final_score":    strategy_result["judge_result"].get("overall_score"),
        "score_cap_reason": strategy_result["judge_result"].get("score_cap_reason"),
        "revisions":      strategy_result["revision_count"],
        "approved":       strategy_result["judge_result"].get("approved"),
        "checklist":      strategy_result["judge_result"].get("checklist"),
        "issues":         strategy_result["judge_result"].get("main_issues"),
        "available_evidence_omitted": strategy_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": strategy_result["judge_result"].get("unsupported_claims"),
        "material_unsupported_claims": strategy_result["judge_result"].get("material_unsupported_claims"),
        "minor_wording_to_tighten": strategy_result["judge_result"].get("minor_wording_to_tighten"),
        "correctly_disclosed_evidence_boundaries": strategy_result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_strategy_payload_path": str(RAW_STRATEGY_PATH),
        "compact_strategy_evidence_path": str(COMPACT_STRATEGY_PATH),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved to outputs/strategy_BANK01.md")

print(f"Raw Strategy payload saved to: {RAW_STRATEGY_PATH}")
print(f"Compact Strategy evidence saved to: {COMPACT_STRATEGY_PATH}")



FINAL STRATEGY JUDGE RESULT
{
  "overall_score": 8,
  "evidence_support_score": 8,
  "ifrs_alignment_score": 8,
  "specificity_score": 9,
  "hallucination_risk": "medium",
  "approval_status": "needs_human_review",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "overview_present": true,
    "report_style_structure_present": true,
    "physical_and_transition_risks_used_correctly": true,
    "time_horizons_used_correctly": true,
    "business_model_handled_according_to_availability": true,
    "value_chain_used_correctly": true,
    "strategy_tradeoffs_handled_according_to_availability": true,
    "financial_effects_used_correctly": false,
    "resource_allocation_used_correctly": true,
    "high_carbon_and_fossil_exposure_used_if_available": true,
    "opportunities_used_correctly": true,
    "targets_handled_according_to_availability": false,
    "scenario_analysis_used_correctly": true,
    "scenario_assumptions_used_correctly": true,
    "transiti

# Risk management section

This section is added after Strategy and follows an Emirates NBD-style IFRS S1/S2 banking disclosure structure.


## Risk Management setup


In [31]:
# ============================================================
# RISK MANAGEMENT SECTION GENERATOR
# ============================================================

print("Starting Risk Management section setup...")

Starting Risk Management section setup...


## Risk Management evidence extractor


In [32]:
# ── RISK MANAGEMENT EVIDENCE EXTRACTOR ──────────────────────
# This extractor uses the section-specific Risk Management payload.
# Expected useful tables:
# - climate_risk_register
# - climate_scenarios
# - risk_management_process / risk_controls / risk_appetite if present
# - reporting_kpis, bank, metadata
#
# It is intentionally defensive because section payloads may vary.

def _rm_present(value) -> bool:
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip().lower() not in {"", "none", "null", "nan"}
    try:
        return not (isinstance(value, float) and value != value)
    except Exception:
        return True


def _rm_num(value):
    try:
        if not _rm_present(value):
            return None
        return float(value)
    except Exception:
        return None


def _rm_safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _rm_clean(obj):
    if isinstance(obj, dict):
        cleaned = {k: _rm_clean(v) for k, v in obj.items() if _rm_present(v)}
        return {k: v for k, v in cleaned.items() if v not in ({}, [], None)}
    if isinstance(obj, list):
        cleaned = [_rm_clean(x) for x in obj if _rm_present(x)]
        return [x for x in cleaned if x not in ({}, [], None)]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _rm_top_n(rows: list[dict], key: str, n: int = 5) -> list[dict]:
    return sorted(rows, key=lambda r: float(r.get(key) or 0), reverse=True)[:n]


def _rm_as_list(value) -> list:
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    return []


def summarize_risk_management_evidence(payload: dict) -> dict:
    metadata = payload.get("metadata", {})
    reporting_year = int(metadata.get("reporting_year", 2024))
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    risks_all = [
        r for r in payload.get("climate_risk_register", [])
        if isinstance(r, dict)
    ]
    risks_2024 = [
        r for r in risks_all
        if _rm_safe_int(r.get("reporting_year"), reporting_year) == reporting_year
    ]

    scenarios = [
        s for s in payload.get("climate_scenarios", [])
        if isinstance(s, dict)
    ]

    risk_management_process = _rm_as_list(payload.get("risk_management_process"))
    risk_controls = _rm_as_list(payload.get("risk_controls"))
    risk_appetite = _rm_as_list(payload.get("risk_appetite"))
    risk_policies = _rm_as_list(payload.get("risk_policies"))

    physical_risks = [
        r for r in risks_2024
        if str(r.get("risk_category", "")).lower().startswith("physical")
    ]
    transition_risks = [
        r for r in risks_2024
        if str(r.get("risk_category", "")).lower().startswith("transition")
    ]

    integrated_count = sum(1 for r in risks_2024 if r.get("erm_integrated_flag") is True)
    changed_count = sum(1 for r in risks_2024 if r.get("changed_since_prior_period") is True)

    risk_categories = sorted({
        str(r.get("risk_category"))
        for r in risks_2024
        if _rm_present(r.get("risk_category"))
    })
    risk_ratings = sorted({
        str(r.get("risk_rating"))
        for r in risks_2024
        if _rm_present(r.get("risk_rating"))
    })
    time_horizons = sorted({
        str(r.get("time_horizon"))
        for r in risks_2024
        if _rm_present(r.get("time_horizon"))
    })
    monitoring_frequencies = sorted({
        str(r.get("monitoring_frequency"))
        for r in risks_2024
        if _rm_present(r.get("monitoring_frequency"))
    })
    scenario_links = sorted({
        str(r.get("scenario_analysis_link"))
        for r in risks_2024
        if _rm_present(r.get("scenario_analysis_link"))
    })
    scenario_linked_risk_count = sum(
        1 for r in risks_2024 if _rm_present(r.get("scenario_analysis_link"))
    )
    mitigation_actions = sorted({
        str(r.get("mitigation_actions"))
        for r in risks_2024
        if _rm_present(r.get("mitigation_actions"))
    })

    top_risks = []
    for r in _rm_top_n(risks_2024, "financial_impact_meur", 8):
        top_risks.append({
            "risk_id": r.get("risk_id"),
            "risk_name": r.get("risk_name"),
            "risk_category": r.get("risk_category"),
            "risk_rating": r.get("risk_rating"),
            "time_horizon": r.get("time_horizon"),
            "financial_impact_meur": r.get("financial_impact_meur"),
            "monitoring_frequency": r.get("monitoring_frequency"),
            "erm_integrated_flag": r.get("erm_integrated_flag"),
            "scenario_analysis_link": r.get("scenario_analysis_link"),
            "mitigation_actions": r.get("mitigation_actions"),
            "changed_since_prior_period": r.get("changed_since_prior_period"),
        })

    scenario_summary = {
        "available": bool(scenarios),
        "scenario_count": len(scenarios),
        "frameworks": sorted({s.get("framework") for s in scenarios if _rm_present(s.get("framework"))}),
        "scenario_types": sorted({s.get("scenario_type") for s in scenarios if _rm_present(s.get("scenario_type"))}),
        "scenario_names": sorted({s.get("scenario_name") for s in scenarios if _rm_present(s.get("scenario_name"))}),
        "horizon_years": sorted({s.get("horizon_year") for s in scenarios if _rm_present(s.get("horizon_year"))}),
        "used_by_risks": scenario_links,
    }

    controls_summary = {
        "risk_management_process_records": risk_management_process[:5],
        "risk_controls_records": risk_controls[:8],
        "risk_appetite_records": risk_appetite[:5],
        "risk_policies_records": risk_policies[:5],
        "process_records_available": bool(risk_management_process),
        "controls_available": bool(risk_controls),
        "risk_appetite_available": bool(risk_appetite),
        "policies_available": bool(risk_policies),
    }

    compact = {
        "bank": {
            "bank_id": bank.get("bank_id"),
            "bank_name": bank.get("bank_name"),
            "country": bank.get("country"),
            "reporting_year": reporting_year,
            "reporting_currency": bank.get("reporting_currency"),
            "regulatory_regime": bank.get("regulatory_regime"),
        },
        "payload_profile": {
            "source_payload": "risk_management",
            "top_level_keys": list(payload.keys()),
            "climate_risk_register_available": bool(risks_2024),
            "climate_risk_register_count_2024": len(risks_2024),
            "climate_scenarios_available": bool(scenarios),
            "climate_scenarios_count": len(scenarios),
            "risk_management_process_available": bool(risk_management_process),
            "risk_controls_available": bool(risk_controls),
            "risk_appetite_available": bool(risk_appetite),
            "risk_policies_available": bool(risk_policies),
        },
        "risk_identification_and_assessment": {
            "risk_count_2024": len(risks_2024),
            "physical_risk_count": len(physical_risks),
            "transition_risk_count": len(transition_risks),
            "risk_categories": risk_categories,
            "risk_ratings": risk_ratings,
            "time_horizons": time_horizons,
            "changed_since_prior_period_count": changed_count,
            "top_risks_by_financial_impact": top_risks,
        },
        "risk_prioritisation_and_monitoring": {
            "monitoring_frequencies": monitoring_frequencies,
            "risk_ratings": risk_ratings,
            "scenario_analysis_links": scenario_links,
            "scenario_linked_risk_count": scenario_linked_risk_count,
            "top_monitored_risk_examples": top_risks[:5],
        },
        "risk_mitigation_and_controls": {
            "erm_integrated_count": integrated_count,
            "risk_count": len(risks_2024),
            "erm_integration_pct": round((integrated_count / len(risks_2024)) * 100, 1) if risks_2024 else None,
            "mitigation_actions": mitigation_actions[:10],
            "material_risk_examples": top_risks[:5],
            "controls_summary": controls_summary,
        },
        "scenario_analysis_in_risk_management": scenario_summary,
        "reporting_kpis": {
            "financed_emissions_2024_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
            "carbon_intensity_2024_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
            "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
            "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
        },
        "evidence_boundaries": {
            "formal_risk_appetite_thresholds_available": bool(risk_appetite),
            "formal_risk_policy_documents_available": bool(risk_policies),
            "formal_controls_procedures_available": bool(risk_controls or risk_management_process),
            "formal_escalation_thresholds_available": any(
                "escalat" in json.dumps(item, ensure_ascii=False).lower()
                for item in (risk_controls + risk_management_process + risk_policies)
            ),
            "risk_register_available": bool(risks_2024),
            "scenario_outputs_are_modelled_estimates": bool(scenarios),
            "do_not_invent_missing_controls": True,
            "boundary_instruction": (
                "Use the risk register and scenario links as the main risk-management evidence. "
                "If formal risk appetite thresholds, formal policies, formal escalation thresholds or detailed controls/procedures are not available, state the boundary once and do not invent them."
            ),
        },
    }

    compact["climate_financial_effects"] = payload.get("climate_financial_effects", [])
    compact["payload_profile"]["financial_effects_available"] = bool(payload.get("climate_financial_effects"))
    return _rm_clean(compact)


risk_management_evidence = summarize_risk_management_evidence(risk_management_payload)

raw_size = len(json.dumps(risk_management_payload, ensure_ascii=False))
compact_size = len(json.dumps(risk_management_evidence, ensure_ascii=False))
print("Compact Risk Management evidence ready")
print(f"Raw Risk Management payload size: {raw_size:,} chars")
print(f"Compact evidence size: {compact_size:,} chars")
print(f"Reduction: {round((1 - compact_size / max(raw_size, 1)) * 100, 1)}%")
print(json.dumps({
    "bank": risk_management_evidence["bank"],
    "risk_count_2024": risk_management_evidence["risk_identification_and_assessment"].get("risk_count_2024"),
    "physical_risk_count": risk_management_evidence["risk_identification_and_assessment"].get("physical_risk_count"),
    "transition_risk_count": risk_management_evidence["risk_identification_and_assessment"].get("transition_risk_count"),
    "erm_integrated_count": risk_management_evidence["risk_mitigation_and_controls"].get("erm_integrated_count"),
    "scenario_count": risk_management_evidence["scenario_analysis_in_risk_management"].get("scenario_count"),
    "formal_risk_appetite_thresholds_available": risk_management_evidence["evidence_boundaries"].get("formal_risk_appetite_thresholds_available"),
    "formal_escalation_thresholds_available": risk_management_evidence["evidence_boundaries"].get("formal_escalation_thresholds_available"),
}, indent=2, ensure_ascii=False))

Compact Risk Management evidence ready
Raw Risk Management payload size: 124,920 chars
Compact evidence size: 11,051 chars
Reduction: 91.2%
{
  "bank": {
    "bank_id": "BANK01",
    "bank_name": "Eurolux Universal Bank AG",
    "country": "DE",
    "reporting_year": 2024,
    "reporting_currency": "EUR",
    "regulatory_regime": "CSRD"
  },
  "risk_count_2024": 8,
  "physical_risk_count": 3,
  "transition_risk_count": 5,
  "erm_integrated_count": 6,
  "scenario_count": 18,
  "formal_risk_appetite_thresholds_available": false,
  "formal_escalation_thresholds_available": false
}


## Risk Management evidence availability, traceability and saving


In [33]:
# ── RISK MANAGEMENT EVIDENCE AVAILABILITY + SAVING ──────────

def build_risk_management_availability_profile(evidence: dict) -> dict:
    risk = evidence.get("risk_identification_and_assessment", {})
    monitoring = evidence.get("risk_prioritisation_and_monitoring", {})
    mitigation = evidence.get("risk_mitigation_and_controls", {})
    scenarios = evidence.get("scenario_analysis_in_risk_management", {})
    boundaries = evidence.get("evidence_boundaries", {})

    def present(v) -> bool:
        return v is not None and str(v).strip().lower() not in {"", "none", "null", "nan"}

    return {
        "risk_register_available": bool(boundaries.get("risk_register_available")),
        "physical_risk_evidence_available": bool(risk.get("physical_risk_count")),
        "transition_risk_evidence_available": bool(risk.get("transition_risk_count")),
        "risk_categories_available": bool(risk.get("risk_categories")),
        "risk_ratings_available": bool(risk.get("risk_ratings")),
        "time_horizons_available": bool(risk.get("time_horizons")),
        "monitoring_frequency_available": bool(monitoring.get("monitoring_frequencies")),
        "scenario_links_available": bool(monitoring.get("scenario_analysis_links")),
        "mitigation_actions_available": bool(mitigation.get("mitigation_actions")),
        "erm_integration_available": present(mitigation.get("erm_integrated_count")),
        "scenario_analysis_available": bool(scenarios.get("available")),
        "formal_controls_procedures_available": bool(boundaries.get("formal_controls_procedures_available")),
        "formal_risk_appetite_thresholds_available": bool(boundaries.get("formal_risk_appetite_thresholds_available")),
        "formal_risk_policy_documents_available": bool(boundaries.get("formal_risk_policy_documents_available")),
        "formal_escalation_thresholds_available": bool(boundaries.get("formal_escalation_thresholds_available")),
        "writer_policy": {
            "use_available_evidence": "Use all material Risk Management evidence that is available and relevant.",
            "handle_unavailable_evidence": "When formal controls, policies, risk appetite thresholds or escalation thresholds are not evidenced, state the boundary once and do not invent them.",
            "scenario_wording": "Scenario analysis links should be described as forward-looking risk assessment inputs. Do not treat scenario outputs as actual losses.",
            "final_language": "Write in the Bank's own voice and state facts directly. Do not refer to the report's evidence base: never use 'available evidence', 'available documentation', 'documentation reviewed', 'source data', 'internally generated analytical outputs' or 'payload' in the final disclosure. Frame genuine gaps as the Bank's position - 'the Bank does not currently disclose X', 'X is outside the scope of this report', or 'X has not yet been quantified'.",
        },
    }


def add_risk_management_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_risk_management_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": [
            key for key in [
                "bank",
                "metadata",
                "climate_risk_register",
                "climate_scenarios",
                "risk_management_process",
                "risk_controls",
                "risk_appetite",
                "risk_policies",
                "reporting_kpis",
            ]
            if key in risk_management_payload
        ],
        "top_risk_refs": [
            item.get("risk_id") or item.get("risk_name")
            for item in evidence.get("risk_identification_and_assessment", {}).get("top_risks_by_financial_impact", [])
            if item.get("risk_id") or item.get("risk_name")
        ],
        "scenario_links": evidence.get("risk_prioritisation_and_monitoring", {}).get("scenario_analysis_links", []),
    }
    return evidence


risk_management_evidence = add_risk_management_traceability(
    risk_management_evidence,
    RISK_MANAGEMENT_PAYLOAD_PATH,
)

raw_risk_management_payload = {
    key: risk_management_payload.get(key)
    for key in [
        "metadata",
        "bank",
        "climate_risk_register",
        "climate_scenarios",
        "risk_management_process",
        "risk_controls",
        "risk_appetite",
        "risk_policies",
        "reporting_kpis",
    ]
    if key in risk_management_payload
}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

RAW_RISK_MANAGEMENT_PATH = output_dir / "payload_BANK01_risk_management_raw.json"
COMPACT_RISK_MANAGEMENT_PATH = output_dir / "compact_risk_management_evidence_BANK01.json"

with open(RAW_RISK_MANAGEMENT_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_risk_management_payload, f, indent=2, ensure_ascii=False)

with open(COMPACT_RISK_MANAGEMENT_PATH, "w", encoding="utf-8") as f:
    json.dump(risk_management_evidence, f, indent=2, ensure_ascii=False)

print("Risk Management evidence prepared")
print(f"- Raw Risk Management payload: {RAW_RISK_MANAGEMENT_PATH}")
print(f"- Compact Risk Management evidence: {COMPACT_RISK_MANAGEMENT_PATH}")
print("- Availability profile:")
print(json.dumps(risk_management_evidence["availability_profile"], indent=2, ensure_ascii=False))

Risk Management evidence prepared
- Raw Risk Management payload: outputs\payload_BANK01_risk_management_raw.json
- Compact Risk Management evidence: outputs\compact_risk_management_evidence_BANK01.json
- Availability profile:
{
  "risk_register_available": true,
  "physical_risk_evidence_available": true,
  "transition_risk_evidence_available": true,
  "risk_categories_available": true,
  "risk_ratings_available": true,
  "time_horizons_available": true,
  "monitoring_frequency_available": true,
  "scenario_links_available": true,
  "mitigation_actions_available": true,
  "erm_integration_available": true,
  "scenario_analysis_available": true,
  "formal_controls_procedures_available": false,
  "formal_risk_appetite_thresholds_available": false,
  "formal_risk_policy_documents_available": false,
  "formal_escalation_thresholds_available": false,
  "writer_policy": {
    "use_available_evidence": "Use all material Risk Management evidence that is available and relevant.",
    "handle_un

## Risk Management state definition


In [34]:
# ── RISK MANAGEMENT STATE DEFINITION ────────────────────────
class RiskManagementState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

## Risk Management requirements — Emirates-style structure


In [35]:
# ── RISK MANAGEMENT REQUIREMENTS ────────────────────────────
# IFRS references are used internally only. Final text must not show paragraph references.

RISK_MANAGEMENT_REQUIREMENTS = """
STRICT RISK MANAGEMENT DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Presentation style:
- Write like a dedicated bank IFRS S1/S2 Risk management section, not like a checklist answer.
- Follow an Emirates NBD-style flow: sustainability-related risk management overview, upstream risks, internal operations risks, downstream risks, climate risk management, then limitations.
- Keep the upstream/internal/downstream subsections evidence-based. If the evidence is climate-risk-register-based rather than a full value-chain risk taxonomy, state the boundary instead of inventing a full value-chain risk process.
- Do not overuse the phrase "available evidence" in the main narrative. Use it mainly in the limitations subsection.
- Do not use the word "payload" in the final report.

Final section title and headings must be exactly:
### Risk management
#### Sustainability-related risk management overview
#### Upstream sustainability risks
#### Risk management across internal operations
#### Downstream sustainability risks
#### Climate risk management
#### Risk management limitations and evidence boundaries

Emirates-style subsection mapping:
1. Sustainability-related risk management overview:
   - Summarise the risk management architecture, risk register, ERM integration, monitoring approach and scenario-analysis links.
2. Upstream sustainability risks:
   - Discuss risks related to funding, suppliers, external dependencies or capital access only when evidenced.
   - If only climate risk register evidence is available, state that upstream-specific risk-process evidence is limited and avoid inventing a separate upstream process.
3. Risk management across internal operations:
   - Discuss internal operational risk management only where evidenced.
   - Use operational climate risks, controls, major-transaction checks, data/reporting processes or formal controls only when available.
4. Downstream sustainability risks:
   - For a bank, downstream risks mainly arise from lending, financed emissions, counterparties, collateral, high-carbon exposures and sector exposures where evidenced.
   - Use the climate risk register, exposure metrics and mitigation actions to explain downstream risk management.
5. Climate risk management:
   - Give the detailed climate-risk-register process: physical and transition categories, ratings, time horizons, monitoring frequencies, scenario links, mitigation actions and ERM integration.
   - Scenario links must be described as forward-looking assessment inputs, not actual losses.
6. Limitations:
   - State boundaries for absent formal policies, controls, risk appetite thresholds, escalation thresholds and upstream/internal/downstream process specificity.

Internal IFRS S1/S2 Risk Management coverage map:
1. Processes to identify climate-related risks and opportunities:
   - Use the climate risk register where available.
   - Distinguish physical and transition risks.
   - Explain the risk categories, time horizons and ratings used.
2. Processes to assess and prioritise risks:
   - Use financial impact, risk ratings, time horizons and monitoring frequency where evidenced.
   - Do not invent probability scoring, risk appetite thresholds or prioritisation methodology if not evidenced.
3. Processes to monitor risks:
   - Use monitoring frequencies and scenario links where available.
   - Use changed_since_prior_period evidence if available.
4. Processes to manage and mitigate risks:
   - Use mitigation actions and ERM integration evidence.
   - If formal controls/procedures are unavailable, state the boundary.
5. Integration into overall risk management:
   - Use ERM integration counts/flags.
   - Do not invent enterprise-wide control design beyond available evidence.
6. Scenario analysis:
   - Use scenario-analysis links and scenario frameworks/types when available.
   - Treat scenario analysis as a forward-looking risk assessment input, not actual loss evidence.
7. Limitations:
   - State boundaries for missing formal risk appetite thresholds, formal escalation thresholds, formal policies or detailed controls/procedures.
   - Do not use the technical word "payload" in the final report.
   - Do not include visible IFRS paragraph references, internal IDs or bracketed evidence tags.

Evidence-tightening rules:
- Scenario analysis must be described as linked to selected risks, not as a portfolio-wide application, unless portfolio-wide scenario use is explicitly evidenced.
- Do not use phrases such as "calibration of mitigation responses" unless the evidence explicitly shows scenario outputs calibrated mitigation actions.
- Borrower engagement and transition-plan covenants may be described only as recorded mitigation actions. Do not add resilience/adaptive-measure effects unless directly evidenced.
- If reported KPIs are available, they may be used as monitoring context for reputational and transition risks, but must not be described as risk appetite limits or thresholds.

Data-aware requirements:
- If risk register evidence is available, the section must describe risk identification, assessment, prioritisation, monitoring and mitigation using that evidence.
- If risk register evidence is unavailable, state the boundary and do not invent a climate risk-management process.
- Use available exact counts and categories where possible.
- Use cautious wording: "available evidence indicates", "is evidenced by", "source data shows".
- Avoid unsupported strong claims such as "fully integrated", "comprehensive", "robust", "ensures" unless the evidence directly supports them.
""".strip()

print("Risk Management requirements ready — Emirates-style structure")

Risk Management requirements ready — Emirates-style structure


## Risk Management writer and judge prompts


In [36]:
# ── RISK MANAGEMENT WRITER / JUDGE PROMPTS ──────────────────

RISK_MANAGEMENT_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Risk Management section of a dedicated IFRS S1/S2-style climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style similar to a dedicated bank IFRS S1/S2 report.

Evidence rules:
- Translate raw data field names into public report wording; avoid snake_case and key=value syntax in the final section.
- Never end the section mid-sentence; complete every required subsection, especially the final evidence-boundary subsection.
- Use the clean style guide only for writing style, tone, structure and formatting; never use it as factual evidence.
- Use only the compact Risk Management evidence supplied by the user prompt.
- Do not invent risk appetite thresholds, formal policies, escalation triggers, control procedures, probability models, governance mechanisms, scenario outputs or mitigation actions.
- When evidence is missing, state any genuine gap once, in the Bank's own voice as the Bank's position - for example "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references, internal references or bracketed evidence tags.
- Do not use markdown tables.

Wording rules:
- Prefer "selected risks are linked to scenario references" instead of broad portfolio-wide scenario claims.
- Prefer "mitigation actions are recorded in the risk register" instead of "scenario analysis calibrates mitigation responses".
- Do not state that borrower covenants support resilience or adaptive measures unless this is explicitly evidenced.
- Avoid "fully integrated", "comprehensive", "robust", "ensures", "calibration", "portfolio-wide" and "bank-wide" unless directly evidenced.


Emirates-style structure rules:
- Use the exact six required subsections.
- Start with a sustainability-related risk management overview.
- Organise the middle of the section around upstream, internal operations and downstream sustainability risks.
- Put the detailed climate risk register process under "Climate risk management".
- Keep limitations and evidence boundaries as the final subsection.
- Do not invent upstream, internal operations or downstream risk processes if the evidence is not available; use boundary wording.

Additional precision rules (avoid the recurring overclaims):
- When stating how many risks are linked to scenario analysis, use the scenario-linked-risk count from the evidence and clarify that those risks reference a smaller number of distinct scenarios (several risks may reference the same scenario). Do not imply that each linked risk corresponds to a separate or distinct scenario analysis.
- Keep wording anchored to the register evidence. Avoid phrases that imply a defined process framework beyond the recorded register, monitoring frequencies and mitigation actions.

- Use the exact mitigation-action wording from the register. Do not broaden it (for example, do not rewrite "flood mapping overlay" as "geospatial hazard overlays").
- Attribute each mitigation action only to the specific risk the register links it to. Do not present a risk-specific action (such as transition-plan covenants, sector exposure limits, or collateral revaluation) as a bank-wide standard control or process.

- Enterprise risk management (ERM) integration is evidenced ONLY by the register's integration flag (six of the eight 2024 risks). "Climate scenario stress testing integrated into ICAAP" is a recorded mitigation-action label for specific risks; it does NOT, by itself, evidence ERM integration. Do not present ICAAP stress testing as proof of, or a causal driver of, ERM integration — keep the two separate and anchored to their own evidence.

REPORTING VOICE (this overrides any other wording guidance, including wording in the evidence): Write strictly as the Bank's own published disclosure, in the Bank's voice, and state facts directly. The following are BANNED from the final section: "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", "based on available evidence/documentation", and any phrasing that narrates what "the documentation" or "the evidence" does or does not show. Write "The Board met eight times in 2024", never "Available documentation indicates the Board met eight times". For genuine gaps, write the Bank's position: "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".

- Present mitigation actions at the register level. Where a recorded mitigation action would read as mismatched to a specific named risk (for example a flood or hazard mapping overlay against a stranded-asset risk), describe the mitigation actions the register records collectively, rather than asserting that a particular action mitigates a particular named risk.

Output rules:
- Return only the complete Risk Management section.
- Keep exactly the six required subsections.
""".strip()


def build_risk_management_writer_prompt(
    evidence: dict,
    judge_feedback: str | None = None,
) -> str:
    profile = evidence.get("availability_profile", {})
    risk = evidence.get("risk_identification_and_assessment", {})
    monitoring = evidence.get("risk_prioritisation_and_monitoring", {})
    mitigation = evidence.get("risk_mitigation_and_controls", {})
    scenarios = evidence.get("scenario_analysis_in_risk_management", {})
    boundaries = evidence.get("evidence_boundaries", {})
    kpis = evidence.get("reporting_kpis", {})
    kpi_values_available = any(
        value is not None and str(value).strip().lower() not in {"", "none", "null", "nan"}
        for value in kpis.values()
    )


    instructions = []

    if profile.get("risk_register_available"):
        instructions.append("- Use the climate risk register as the main source for risk identification, assessment, monitoring and mitigation.")
        instructions.append(f"- State that the 2024 register contains {risk.get('risk_count_2024')} risks, including {risk.get('physical_risk_count')} physical risks and {risk.get('transition_risk_count')} transition risks, if these counts are available.")
        instructions.append("- Use risk categories, ratings, time horizons and top material risks by financial impact.")
    else:
        instructions.append("- Risk register evidence is unavailable. State this boundary and do not invent a risk-register process.")

    if profile.get("monitoring_frequency_available"):
        instructions.append("- Describe monitoring frequencies using the exact values in evidence.")

    if profile.get("scenario_links_available") or profile.get("scenario_analysis_available"):
        instructions.append("- Describe scenario analysis as a forward-looking input for selected risks that reference scenario IDs. Do not imply portfolio-wide application unless evidenced, and do not describe scenario outputs as actual losses.")

    if profile.get("mitigation_actions_available"):
        instructions.append("- Use the evidenced mitigation actions and link them to risk management without overclaiming effectiveness.")

    if profile.get("erm_integration_available"):
        instructions.append("- Describe ERM integration using the evidenced integration count/percentage.")

    if kpi_values_available:
        instructions.append("- Add one cautious sentence linking reported climate-related KPIs, such as financed emissions, carbon intensity, high-carbon sector exposure and fossil-fuel exposure, to monitoring context for transition/reputational risks. Do not describe these KPIs as risk appetite limits.")

    if not profile.get("formal_risk_appetite_thresholds_available"):
        instructions.append("- State that formal climate risk appetite thresholds are not evidenced if risk appetite details are required.")

    if not profile.get("formal_escalation_thresholds_available"):
        instructions.append("- State that formal escalation thresholds/triggers are not evidenced. Do not invent escalation routes.")

    if not profile.get("formal_controls_procedures_available"):
        instructions.append("- State that detailed formal controls/procedures are not evidenced. Do not invent them.")

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION POLICY:
- Fix omissions using available evidence.
- Preserve accurate evidence-boundary statements where evidence is unavailable.
- Never invent evidence to satisfy the judge.
""".strip()

    return f"""
{RISK_MANAGEMENT_REQUIREMENTS}

WRITING STYLE GUIDE — STYLE ONLY, NOT EVIDENCE:
{build_style_context("risk_management")}

STYLE APPLICATION RULES:
- Use the style guide only for tone, flow, structure and formatting.
- Use compact Risk Management evidence for facts.
- Use IFRS requirements only to decide what topics to cover.
- Do not copy reference-report wording or mention forbidden reference terms.

BANK:
{evidence['bank']['bank_name']} ({evidence['bank'].get('bank_id')})
REPORTING YEAR: {evidence['bank'].get('reporting_year')}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:

SENIOR RISK MANAGEMENT FIXES:
- Do not claim integration of "other sustainability-related risks" unless separately evidenced; focus on climate-related risks.
- Treat ERM integration as evidenced by register indicators/flags for 6 of 8 climate risks, not as proof of the broader ERM process design.
- Do not imply portfolio-wide ICAAP integration. Say specific risk-register mitigation actions include climate scenario stress testing integrated into ICAAP where recorded.
- Use the exact mitigation wording "Collateral revaluation and flood mapping overlay"; do not expand to hazard mapping tools/overlays unless evidenced.
- Explicitly boundary that broader ERM policies, formal risk appetite thresholds, formal controls and escalation triggers are not evidenced.

{chr(10).join(instructions)}

SECTION BLUEPRINT:
### Risk management

#### Sustainability-related risk management overview
Summarise the Bank's risk management architecture using the risk register, ERM integration, risk monitoring, scenario-analysis links and management responsibilities where available. If climate_financial_effects evidence is present, reference how identified climate-related risks are expected to affect credit-risk measures such as expected credit loss allowances and collateral values, distinguishing current from anticipated effects without overstating precision.

#### Upstream sustainability risks
Describe upstream risks only where evidenced. If there is no distinct upstream risk process evidence, state that source data is centred on climate risks in financing and risk management rather than a separate upstream risk taxonomy.

#### Risk management across internal operations
Describe internal operations risk management only where evidenced, such as operational climate risk entries, formal controls, data/reporting controls or major-transaction climate checks. Do not invent formal control procedures.

#### Downstream sustainability risks
Explain that, for a bank, material downstream climate risks arise mainly through lending, collateral, sector exposures, financed emissions and counterparties where evidenced. Use physical and transition risks from the register.

#### Climate risk management
Give the detailed climate risk register process: risk categories, physical/transition split, ratings, time horizons, monitoring frequencies, scenario references, mitigation actions and ERM integration.

#### Risk management limitations and evidence boundaries
State boundaries for formal policies, formal controls/procedures, risk appetite thresholds, escalation thresholds and limited upstream/internal/downstream process detail.

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("risk_management")}

REQUIREMENT-TO-EVIDENCE POLICY:
- Use IFRS requirements to decide which risk-management topics must be addressed.
- Use compact bank evidence to write bank-specific facts.
- If an IFRS requirement is relevant but evidence is unavailable, disclose the evidence boundary instead of inventing policies, risk appetite thresholds, escalation mechanisms, controls or procedures.
- Treat core_standard items as mandatory disclosure guidance.
- Treat industry_guidance items as banking-specific application guidance.
- Do not display IFRS paragraph references or requirement IDs in the final report.

COMPACT RISK MANAGEMENT EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Risk register available: {boundaries.get('risk_register_available')}
- Formal risk appetite thresholds available: {boundaries.get('formal_risk_appetite_thresholds_available')}
- Formal risk policy documents available: {boundaries.get('formal_risk_policy_documents_available')}
- Formal controls/procedures available: {boundaries.get('formal_controls_procedures_available')}
- Formal escalation thresholds available: {boundaries.get('formal_escalation_thresholds_available')}
- Scenario outputs are modelled estimates: {boundaries.get('scenario_outputs_are_modelled_estimates')}
- Boundary instruction: {boundaries.get('boundary_instruction')}

GENERAL WRITING RULES:
- Do not claim comprehensive or fully integrated risk management unless directly evidenced.
- Do not invent risk appetite limits, escalation thresholds, probability scoring, or detailed controls.
- Do not print internal risk IDs unless needed for traceability; prefer risk names/categories in final disclosure.
- Mention limitations only in the relevant subsection and in the final limitations subsection.

{feedback_block}

Write the complete Risk Management section only.
""".strip()


RISK_MANAGEMENT_JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate Risk Management disclosure.

You evaluate whether the Risk Management section is:
- supported by compact evidence;
- aligned with the Risk Management disclosure checklist;
- transparent about missing evidence;
- free from unsupported claims about risk appetite, controls, escalation, scenario analysis, mitigation and ERM integration.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_risk_management_judge_prompt(
    draft: str,
    evidence: dict,
    deterministic_checks: dict | None = None,
) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the Risk Management draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

REPORTING VOICE: The section must read as the Bank's own published disclosure. Treat as a material wording defect any report-machinery language - "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", or phrasing that narrates what "the documentation"/"the evidence" does or does not show. Facts must be stated directly; genuine gaps framed as the Bank's position ("the Bank does not currently disclose ...", "... is outside the scope of this report", "... has not yet been quantified"). List any such phrasing in material_unsupported_claims so it is corrected.

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("risk_management")}

COMPACT RISK MANAGEMENT EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
0A. Check the IFRS requirements pack and the ifrs_requirement_coverage pre-check. Penalise missing mandatory core IFRS topics unless the draft clearly discloses an evidence boundary.
0. Verify that the section follows the Emirates-style risk management structure: overview, upstream risks, internal operations risks, downstream risks, climate risk management and limitations.
1. Penalise any claim that contradicts evidence or invents unavailable risk-management information.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed evidence boundaries are acceptable but lower completeness.
4. If risk register evidence is available, verify use of risk count, physical/transition distinction, risk categories, ratings, time horizons, monitoring frequencies, mitigation actions and ERM integration.
5. If formal risk appetite thresholds, controls/procedures, policies or escalation thresholds are unavailable, the draft must not invent them.
6. Scenario analysis must be described as a forward-looking risk assessment input for selected linked risks, not actual loss evidence or portfolio-wide application unless evidenced.
7. Verify no visible IFRS paragraph references, internal refs or bracketed evidence tags appear.
8. Verify no unsupported strong claims such as fully integrated, comprehensive, robust, ensures, portfolio-wide scenario use, calibration of mitigation responses, or resilience/adaptation effects from borrower covenants.

SCORING:
Score on completeness RELATIVE TO AVAILABLE EVIDENCE, not on how much data the bank happens to hold. Correctly disclosed evidence boundaries (genuine unavailability, accurately stated) are good IFRS practice and MUST NOT reduce the score; the number of disclosed boundaries is irrelevant to the score. Penalise ONLY: unsupported or overclaimed statements, omitted AVAILABLE evidence, factual inconsistencies with the evidence, hallucinations, and structural or deterministic failures.
- 9-10: Materially complete relative to available evidence, report-ready structure, no unsupported or overclaimed statements, no omitted available evidence, and all genuine data gaps correctly disclosed as boundaries.
- 8: As 9-10 but with one minor evidence-use issue or one slightly loose statement to tighten.
- 7: Usable but contains an unsupported/overclaimed statement, omits available evidence, or has a structural/keyword gap.
- 6: Revision required: available evidence omitted, unsupported wording remains, factual inconsistency with evidence, or required structure/boundary keywords missing.
- 5 or below: Major evidence failure, hallucination, contradiction, or missing core subsection.

When listing omitted evidence, separate MATERIAL omissions (which must be incorporated) from OPTIONAL ones (acceptable to leave out); never mark internal record IDs or optional traceability codes as material. Likewise, separate unsupported claims into material_unsupported_claims (misrepresent the evidence or overstate scope/assurance/compliance — must fix) and minor_wording_to_tighten (defensible phrasing such as 'covers' vs 'includes' or singular/plural that does not misrepresent the evidence — an acceptable minor limitation). Do not list trivial wording as material. Return valid JSON only. Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "report_style_structure_present": <true/false>,
    "upstream_internal_downstream_structure_present": <true/false>,
    "risk_identification_used_correctly": <true/false>,
    "physical_transition_distinction_used": <true/false>,
    "risk_assessment_prioritisation_used_correctly": <true/false>,
    "monitoring_process_used_correctly": <true/false>,
    "mitigation_actions_used_correctly": <true/false>,
    "erm_integration_used_correctly": <true/false>,
    "scenario_analysis_used_correctly": <true/false>,
    "risk_appetite_handled_according_to_availability": <true/false>,
    "controls_procedures_handled_according_to_availability": <true/false>,
    "escalation_handled_according_to_availability": <true/false>,
    "evidence_boundaries_present": <true/false>,
    "no_visible_ifrs_refs_or_internal_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "material_evidence_omitted": [<omitted available evidence whose absence materially weakens the disclosure; MUST be incorporated as report prose>],
  "optional_evidence_omitted": [<omitted available evidence that is acceptable to leave out; do not force inclusion>],
  "unsupported_claims": [<specific unsupported claims>],
  "material_unsupported_claims": [<claims that misrepresent the evidence, assert unevidenced facts, or overstate scope/assurance/compliance; these MUST be removed or reframed>],
  "minor_wording_to_tighten": [<defensible phrasing that could be tightened but does NOT misrepresent the evidence; acceptable as a minor limitation>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()


print("Risk Management prompts ready — Emirates-style structure")

Risk Management prompts ready — Emirates-style structure


## Risk Management evaluation mode


In [37]:
# ── RISK MANAGEMENT EVALUATION MODE ─────────────────────────
# Data-aware deterministic pre-checks are active.
# They are passed to the GPT-5.2 Judge as signals, not used as standalone approval.

print("Risk Management evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

Risk Management evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks


## Risk Management LangGraph nodes and deterministic checks


In [38]:
# ── RISK MANAGEMENT LANGGRAPH NODES ─────────────────────────

def _rm_contains_any(text: str, phrases: list[str]) -> bool:
    text_l = text.lower()
    return any(p.lower() in text_l for p in phrases)


def run_risk_management_deterministic_checks(draft: str, evidence: dict) -> dict:
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})

    required_headings = [
        "#### Sustainability-related risk management overview",
        "#### Upstream sustainability risks",
        "#### Risk management across internal operations",
        "#### Downstream sustainability risks",
        "#### Climate risk management",
        "#### Risk management limitations and evidence boundaries",
    ]

    missing_headings = [h for h in required_headings if h not in text]

    warnings = []
    failures = []

    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})

    # Report-style structure checks.
    overview_heading = "#### Sustainability-related risk management overview"
    next_heading = "#### Upstream sustainability risks"
    if overview_heading in text:
        overview_text = text.split(overview_heading, 1)[1].split(next_heading, 1)[0].lower()
        if len(overview_text.split()) < 45:
            warnings.append({
                "check": "risk_management_overview_may_be_too_short",
                "details": "Overview is present but may be too short to explain the risk management architecture."
            })

    if "checklist" in text_l or "ifrs s2 requires" in text_l:
        warnings.append({
            "check": "risk_management_may_read_like_checklist",
            "details": "Risk Management should read like a report section, not a checklist response."
        })

    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS references/tags found."})

    if re.search(r"\[REF:|MTG-BANK\d+", text):
        failures.append({"check": "internal_refs_visible", "details": "Internal references should not appear in final text."})

    if profile.get("risk_register_available"):
        if "risk register" not in text_l:
            failures.append({"check": "risk_register_omitted", "details": "Risk register evidence is available and should be mentioned."})
        if "physical" not in text_l or "transition" not in text_l:
            failures.append({"check": "physical_transition_distinction_missing", "details": "Physical and transition risks should be distinguished."})
        if "monitor" not in text_l:
            failures.append({"check": "monitoring_process_missing", "details": "Monitoring frequencies/process should be discussed."})
        if "mitigation" not in text_l and "mitigat" not in text_l:
            failures.append({"check": "mitigation_actions_missing", "details": "Mitigation actions are available and should be discussed."})
        if "erm" not in text_l and "enterprise risk management" not in text_l:
            failures.append({"check": "erm_integration_missing", "details": "ERM integration evidence is available and should be discussed."})

    if profile.get("scenario_analysis_available") or profile.get("scenario_links_available"):
        if "scenario" not in text_l:
            failures.append({"check": "scenario_analysis_omitted", "details": "Scenario analysis evidence is available and should be discussed."})
        if "modelled" not in text_l and "modeled" not in text_l and "forward-looking" not in text_l:
            warnings.append({"check": "scenario_boundary_may_be_missing", "details": "Scenario analysis should be described as forward-looking/modelled."})

    if not profile.get("formal_risk_appetite_thresholds_available"):
        if _rm_contains_any(text, ["risk appetite threshold", "risk appetite limit", "formal risk appetite"]) and not _rm_contains_any(text, ["not available", "not evidenced", "not documented", "does not provide", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "risk_appetite_overclaimed", "details": "Formal risk appetite thresholds appear without evidence or boundary."})

    if not profile.get("formal_escalation_thresholds_available"):
        if _rm_contains_any(text, ["escalation threshold", "escalation trigger", "formal escalation"]) and not _rm_contains_any(text, ["not available", "not evidenced", "not documented", "does not provide", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "escalation_overclaimed", "details": "Formal escalation thresholds/triggers appear without evidence or boundary."})

    if not profile.get("formal_controls_procedures_available"):
        if _rm_contains_any(text, ["formal control procedure", "documented control procedure", "control framework"]) and not _rm_contains_any(text, ["not available", "not evidenced", "not documented", "does not provide", "does not currently disclose", "does not disclose", "not separately disclosed", "does not currently describe", "does not currently report", "not currently reported", "outside the scope", "not yet"]):
            failures.append({"check": "controls_overclaimed", "details": "Detailed formal controls/procedures appear without evidence or boundary."})

    unsupported_strong_phrases = [
        "fully integrated",
        "comprehensive climate risk management",
        "robust climate risk management",
        "ensures",
        "guarantees",
        "eliminates risk",
    ]
    found_strong = [p for p in unsupported_strong_phrases if p in text_l]
    if found_strong:
        warnings.append({"check": "possible_unsupported_strong_language", "details": found_strong})


    # Evidence-tightening checks based on common Risk Management overclaims.
    overclaim_patterns = {
        "portfolio_wide_scenario_overclaim": [
            "implications for the bank's portfolios",
            "implications for the bank’s portfolios",
            "portfolio-wide",
            "bank-wide scenario",
            "across the bank's portfolios",
            "across the bank’s portfolios",
        ],
        "mitigation_calibration_overclaim": [
            "calibration of mitigation",
            "calibrate mitigation",
            "calibrating mitigation",
        ],
        "borrower_resilience_overclaim": [
            "supporting discussions on resilience and adaptive measures",
            "resilience and adaptive measures",
            "adaptive measures with borrowers",
        ],
    }

    for check_name, phrases in overclaim_patterns.items():
        found = [phrase for phrase in phrases if phrase in text_l]
        if found:
            failures.append({
                "check": check_name,
                "details": f"Unsupported wording detected: {found}"
            })

    # Reported KPIs are optional but available monitoring context. Warn if omitted.
    kpis = evidence.get("reporting_kpis", {})
    kpi_values_available = any(
        value is not None and str(value).strip().lower() not in {"", "none", "null", "nan"}
        for value in kpis.values()
    )
    if kpi_values_available:
        kpi_terms = [
            "financed emissions",
            "carbon intensity",
            "high-carbon",
            "high carbon",
            "fossil fuel",
            "fossil-fuel",
        ]
        if not any(term in text_l for term in kpi_terms):
            warnings.append({
                "check": "reported_kpis_omitted_from_monitoring_context",
                "details": "Reported climate-related KPIs are available and can be used as monitoring context, without presenting them as risk appetite limits."
            })

    # Limitations subsection should be complete enough to explain unavailable formal structures.
    limitations_heading = "#### Risk management limitations and evidence boundaries"
    if limitations_heading in text:
        limitations_text = text.split(limitations_heading, 1)[1].lower()
        if len(limitations_text.split()) < 50:
            failures.append({
                "check": "limitations_section_too_short",
                "details": "Risk management limitations section is present but too short."
            })
    else:
        failures.append({
            "check": "limitations_section_missing",
            "details": "Risk management limitations and evidence boundaries subsection is missing."
        })


    if "climate-related and other sustainability-related risks" in text_l:
        failures.append({
            "check": "risk_management_overbroad_sustainability_risk_claim",
            "details": "Evidence supports climate-risk register mechanics, not other sustainability-related risk integration."
        })

    if "hazard mapping tools" in text_l or "hazard mapping overlays" in text_l:
        failures.append({
            "check": "risk_management_mitigation_wording_expanded",
            "details": "Use evidenced wording: collateral revaluation and flood mapping overlay."
        })

    if "bank’s established risk oversight processes" in text_l or "bank's established risk oversight processes" in text_l:
        warnings.append({
            "check": "risk_management_broader_erm_process_overclaim",
            "details": "Clarify ERM integration is evidenced by register indicators only."
        })

    style_leak = check_reference_identity_leak(text)
    if not style_leak.get("passed", True):
        failures.append({"check": "reference_identity_leak", "details": style_leak.get("matches", [])})

    completeness = check_section_completeness(text, "risk_management")
    failures.extend(completeness.get("failures", []))
    warnings.extend(completeness.get("warnings", []))

    # ── Derived factual-consistency check: scenario-linked-risk count ──
    # Verifies any "N of M risks ... scenario link" claim against the evidence
    # (computed, not hard-coded). Fires only for scenario-LINK claims whose
    # denominator matches the register total, to avoid false positives.
    rpm = evidence.get("risk_prioritisation_and_monitoring", {}) or {}
    ria = evidence.get("risk_identification_and_assessment", {}) or {}
    linked_count = rpm.get("scenario_linked_risk_count")
    if linked_count is None:
        linked_count = ria.get("scenario_linked_risk_count")
    register_total = ria.get("risk_count_2024")
    if isinstance(linked_count, int) and isinstance(register_total, int):
        _w2n = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
                "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12}
        _num = r"(\d+|one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve)"
        for sent in re.split(r"(?<=[.;])\s+", text):
            sl = sent.lower()
            if "scenario" not in sl:
                continue
            if not any(w in sl for w in ("link", "refer", "attach", "associat", "mapped")):
                continue
            for mm in re.finditer(_num + r"\s+of\s+(?:the\s+)?" + _num, sl):
                a, b = mm.group(1), mm.group(2)
                ai = int(a) if a.isdigit() else _w2n.get(a)
                bi = int(b) if b.isdigit() else _w2n.get(b)
                if ai is None or bi is None:
                    continue
                if bi == register_total and ai != linked_count:
                    failures.append({
                        "check": "scenario_link_count_mismatch",
                        "details": (f"Draft asserts {ai} of {bi} risks linked to scenario analysis, "
                                    f"but the evidence supports {linked_count} of {register_total}."),
                    })

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def risk_management_writer_node(state: RiskManagementState) -> RiskManagementState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_risk_management_writer_prompt(state["evidence"], judge_feedback=feedback)
    print("Risk Management writer prompt approx tokens:", round(len(prompt) / 4))

    draft = call_writer_llm(
        system_prompt=RISK_MANAGEMENT_WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"RISK MANAGEMENT WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {**state, "draft": draft.strip(), "status": "judging"}


def risk_management_judge_node(state: RiskManagementState) -> RiskManagementState:
    draft = state["draft"]
    deterministic_checks = run_risk_management_deterministic_checks(draft, state["evidence"])
    deterministic_checks["ifrs_requirement_coverage"] = check_ifrs_requirement_coverage(draft, "risk_management")

    prompt = build_risk_management_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )
    judge_result = call_judge_llm_json(
        system_prompt=RISK_MANAGEMENT_JUDGE_SYSTEM,
        user_prompt=prompt,
    )
    try:
        judge_result = normalize_json_booleans(judge_result)
    except NameError:
        pass

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    judge_result["_best_so_far"] = track_best(state, judge_result)  # non-regression: keep best draft across iterations

    print("\nRISK MANAGEMENT JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    approved = bool(judge_result.get("approved"))
    status = "approved" if approved else "revising"

    return {
        **state,
        "judge_result": judge_result,
        "status": status,
    }


RISK_MANAGEMENT_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Risk Management section using only:
- the supplied compact Risk Management evidence;
- the availability profile;
- the data-aware deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Keep the exact six-subsection Emirates-style Risk management structure.
- Do not add visible IFRS paragraph references or internal refs.
- Return only the complete revised Risk Management section using the required six headings.
""".strip()


def risk_management_reviser_node(state: RiskManagementState) -> RiskManagementState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]

    revision_prompt = f"""
STRICT RISK MANAGEMENT REQUIREMENTS:
{RISK_MANAGEMENT_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("risk_management")}

COMPACT RISK MANAGEMENT EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- Preserve or improve accurate boundary statements where evidence is unavailable.
- Never invent missing risk appetite thresholds, policies, control procedures, escalation thresholds, scenario results or mitigation actions.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete revised Risk Management section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=RISK_MANAGEMENT_REVISER_SYSTEM,
        user_prompt=(revision_prompt + "\n\nPRIORITISED REVISION BRIEF (address every item using available evidence only):\n" + build_reviser_focus(judge)),
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nRisk Management revised with GPT-5.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def risk_management_finalize_node(state: RiskManagementState) -> RiskManagementState:
    new_state = finalize_state(state, target_score=TARGET_SCORE)
    gate = new_state["judge_result"]["finalize_gate"]
    print(f"\n{'='*50}")
    print("FINALIZED RISK MANAGEMENT")
    print(f"  Score: {gate['score']}/{gate['target_score']} | publishable={gate['publishable']}")
    print(f"  Status: {new_state['judge_result']['approval_status']} | revisions: {state['revision_count']}")
    if not gate["publishable"]:
        print("  Blocking reasons:", gate["blocking_reasons"])
    print(f"{'='*50}")
    return new_state


def risk_management_should_continue(state: RiskManagementState) -> str:
    return route_after_judge_generic(state, target_score=TARGET_SCORE, revise_label="reviser")




## Build and compile Risk Management graph


In [39]:
# ── BUILD AND COMPILE RISK MANAGEMENT GRAPH ─────────────────
risk_management_builder = StateGraph(RiskManagementState)

risk_management_builder.add_node("writer",   risk_management_writer_node)
risk_management_builder.add_node("judge",    risk_management_judge_node)
risk_management_builder.add_node("reviser",  risk_management_reviser_node)
risk_management_builder.add_node("finalize", risk_management_finalize_node)

risk_management_builder.add_edge(START, "writer")
risk_management_builder.add_edge("writer", "judge")
risk_management_builder.add_conditional_edges(
    "judge",
    risk_management_should_continue,
    {
        "finalize": "finalize",
        "reviser": "reviser",
    }
)
risk_management_builder.add_edge("reviser", "judge")
risk_management_builder.add_edge("finalize", END)

risk_management_graph = risk_management_builder.compile()
print("Risk Management graph compiled")

Risk Management graph compiled


## Run Risk Management generation


In [40]:
# ── RUN RISK MANAGEMENT SECTION ─────────────────────────────
risk_management_initial_state: RiskManagementState = {
    "bank_name":      bank_name,
    "evidence":       risk_management_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions": 3,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}

print(f"Starting Risk Management generation for: {bank_name}\n")
risk_management_result = run_section_with_floor(risk_management_graph, risk_management_initial_state, label="risk_management")

Starting Risk Management generation for: Eurolux Universal Bank AG

Risk Management writer prompt approx tokens: 14338

RISK MANAGEMENT WRITER (initial draft)
Draft length: 2841 words
GPT-5.2 judge: connection reset/timeout (ConnectionResetError); retrying attempt 2/6 in 1.7s...

RISK MANAGEMENT JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS
{
  "overall_score": 7,
  "evidence_support_score": 7,
  "ifrs_alignment_score": 8,
  "specificity_score": 8,
  "hallucination_risk": "medium",
  "approval_status": "revision_required",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "report_style_structure_present": true,
    "upstream_internal_downstream_structure_present": true,
    "risk_identification_used_correctly": true,
    "physical_transition_distinction_used": true,
    "risk_assessment_prioritisation_used_correctly": true,
    "monitoring_process_used_correctly": true,
    "mitigation_actions_used_correctly": false,
    "erm_integration_used_correctly":

## Print and save Risk Management output


In [41]:
# ── RISK MANAGEMENT OUTPUT ──────────────────────────────────
print("\n" + "="*60)
print("FINAL RISK MANAGEMENT JUDGE RESULT")
print("="*60)
print(json.dumps(risk_management_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("RISK MANAGEMENT SECTION")
print("="*60)
print(risk_management_result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "risk_management_BANK01.md", "w", encoding="utf-8") as f:
    f.write(risk_management_result["final_section"])

with open(output_dir / "risk_management_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "risk_management",
        "status":         risk_management_result["status"],
        "approval_status": risk_management_result["judge_result"].get("approval_status"),
        "final_score":    risk_management_result["judge_result"].get("overall_score"),
        "revisions":      risk_management_result["revision_count"],
        "approved":       risk_management_result["judge_result"].get("approved"),
        "checklist":      risk_management_result["judge_result"].get("checklist"),
        "issues":         risk_management_result["judge_result"].get("main_issues"),
        "available_evidence_omitted": risk_management_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": risk_management_result["judge_result"].get("unsupported_claims"),
        "material_unsupported_claims": risk_management_result["judge_result"].get("material_unsupported_claims"),
        "minor_wording_to_tighten": risk_management_result["judge_result"].get("minor_wording_to_tighten"),
        "correctly_disclosed_evidence_boundaries": risk_management_result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_risk_management_payload_path": str(RAW_RISK_MANAGEMENT_PATH),
        "compact_risk_management_evidence_path": str(COMPACT_RISK_MANAGEMENT_PATH),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved to outputs/risk_management_BANK01.md")
print(f"Raw Risk Management payload saved to: {RAW_RISK_MANAGEMENT_PATH}")
print(f"Compact Risk Management evidence saved to: {COMPACT_RISK_MANAGEMENT_PATH}")


FINAL RISK MANAGEMENT JUDGE RESULT
{
  "overall_score": 8,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 8,
  "specificity_score": 9,
  "hallucination_risk": "low",
  "approval_status": "needs_human_review",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "report_style_structure_present": true,
    "upstream_internal_downstream_structure_present": true,
    "risk_identification_used_correctly": true,
    "physical_transition_distinction_used": true,
    "risk_assessment_prioritisation_used_correctly": true,
    "monitoring_process_used_correctly": true,
    "mitigation_actions_used_correctly": true,
    "erm_integration_used_correctly": true,
    "scenario_analysis_used_correctly": true,
    "risk_appetite_handled_according_to_availability": true,
    "controls_procedures_handled_according_to_availability": true,
    "escalation_handled_according_to_availability": true,
    "evidence_boundaries_present": true,
    "no_visible_ifrs_refs_or_i

# Metrics and targets section

This section is added after Risk Management and uses an Emirates NBD-style IFRS S1/S2 banking disclosure structure.


## Metrics and Targets setup


In [42]:
# ============================================================
# METRICS AND TARGETS SECTION GENERATOR
# ============================================================
print("Starting Metrics and Targets section setup...")

Starting Metrics and Targets section setup...


## Metrics and Targets evidence extractor


In [43]:
# ── METRICS AND TARGETS EVIDENCE EXTRACTOR ──────────────────

def _mt_present(value) -> bool:
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip().lower() not in {"", "none", "null", "nan"}
    try:
        return not (isinstance(value, float) and value != value)
    except Exception:
        return True


def _mt_safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def _mt_as_list(value) -> list:
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    return []


def _mt_clean(obj):
    if isinstance(obj, dict):
        cleaned = {k: _mt_clean(v) for k, v in obj.items() if _mt_present(v)}
        return {k: v for k, v in cleaned.items() if v not in ({}, [], None)}
    if isinstance(obj, list):
        cleaned = [_mt_clean(x) for x in obj if _mt_present(x)]
        return [x for x in cleaned if x not in ({}, [], None)]
    if isinstance(obj, float) and obj != obj:
        return None
    return obj


def _mt_find_records(payload: dict, candidate_keys: list[str]) -> list[dict]:
    rows = []
    for key in candidate_keys:
        value = payload.get(key)
        if isinstance(value, list):
            rows.extend([x for x in value if isinstance(x, dict)])
        elif isinstance(value, dict):
            rows.append(value)
    return rows


def _mt_latest_year_rows(rows: list[dict], year: int = 2024) -> list[dict]:
    filtered = []
    for row in rows:
        row_year = row.get("reporting_year") or row.get("year") or row.get("metric_year")
        if row_year is None or _mt_safe_int(row_year, year) == year:
            filtered.append(row)
    return filtered


def summarize_metrics_targets_evidence(payload: dict) -> dict:
    metadata = payload.get("metadata", {})
    reporting_year = int(metadata.get("reporting_year", 2024))
    bank = payload.get("bank", {})
    reporting_kpis = payload.get("reporting_kpis", {})

    financial_summary = [row for row in payload.get("financial_summary", []) if isinstance(row, dict)]
    financial_2024 = next((row for row in financial_summary if _mt_safe_int(row.get("reporting_year"), reporting_year) == reporting_year), {})

    targets = [row for row in payload.get("targets", []) if isinstance(row, dict)]
    target_summary = reporting_kpis.get("target_summary", [])
    if not isinstance(target_summary, list):
        target_summary = []

    ghg_records = _mt_find_records(payload, ["ghg_emissions", "emissions", "scope_emissions", "operational_emissions", "emissions_summary"])
    financed_records = _mt_find_records(payload, ["financed_emissions", "financed_emissions_summary", "portfolio_emissions", "pcaf_emissions"])
    portfolio_records = _mt_find_records(payload, ["portfolio_metrics", "climate_metrics", "environmental_kpis", "metrics", "reported_metrics"])
    methodology_records = _mt_find_records(payload, ["methodology", "metric_methodology", "calculation_methodology", "pcaf_methodology", "data_quality", "assumptions"])

    ghg_2024 = _mt_latest_year_rows(ghg_records, reporting_year)
    financed_2024 = _mt_latest_year_rows(financed_records, reporting_year)
    portfolio_2024 = _mt_latest_year_rows(portfolio_records, reporting_year)

    cross_industry_metrics = {
        "financed_emissions_2024_tco2e": reporting_kpis.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_2024_tco2e_per_meur": reporting_kpis.get("carbon_intensity_2024_tco2e_per_meur"),
        "high_carbon_sector_exposure_pct": reporting_kpis.get("high_carbon_sector_exposure_pct"),
        "high_carbon_sector_exposure_meur": reporting_kpis.get("high_carbon_sector_exposure_meur"),
        "fossil_fuel_exposure_pct": reporting_kpis.get("fossil_fuel_exposure_pct"),
        "fossil_fuel_exposure_meur": reporting_kpis.get("fossil_fuel_exposure_meur"),
        "green_loans_pct_2024": reporting_kpis.get("green_loans_pct_2024") or financial_2024.get("green_loans_pct"),
        "green_loans_meur_2024": reporting_kpis.get("green_loans_meur_2024") or financial_2024.get("green_loans_meur"),
        "climate_capex_2024_meur": reporting_kpis.get("climate_capex_2024_meur") or financial_2024.get("climate_capex_meur"),
        "climate_opex_2024_meur": reporting_kpis.get("climate_opex_2024_meur") or financial_2024.get("climate_opex_meur"),
        "total_loans_meur": bank.get("total_loans_meur") or financial_2024.get("total_loans_meur"),
        "total_assets_meur": bank.get("total_assets_meur") or financial_2024.get("total_assets_meur"),
    }

    target_rows = targets if targets else target_summary

    compact = {
        "bank": {
            "bank_id": bank.get("bank_id"),
            "bank_name": bank.get("bank_name"),
            "country": bank.get("country"),
            "reporting_year": reporting_year,
            "reporting_currency": bank.get("reporting_currency"),
            "regulatory_regime": bank.get("regulatory_regime"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur": bank.get("total_loans_meur"),
        },
        "payload_profile": {
            "source_payload": "metrics_targets",
            "top_level_keys": list(payload.keys()),
            "reporting_kpis_available": bool(reporting_kpis),
            "financial_summary_available": bool(financial_summary),
            "full_targets_available": bool(targets),
            "target_summary_available": bool(target_summary),
            "ghg_records_available": bool(ghg_records),
            "financed_emissions_records_available": bool(financed_records),
            "portfolio_metric_records_available": bool(portfolio_records),
            "methodology_records_available": bool(methodology_records),
        },
        "cross_industry_metrics": cross_industry_metrics,
        "ghg_and_financed_emissions": {
            "operational_ghg_records_available": bool(ghg_2024),
            "operational_ghg_records_2024": ghg_2024[:10],
            "financed_emissions_records_available": bool(financed_2024),
            "financed_emissions_records_2024": financed_2024[:10],
        },
        "climate_related_financial_metrics": {
            "green_loans_pct_2024": cross_industry_metrics.get("green_loans_pct_2024"),
            "green_loans_meur_2024": cross_industry_metrics.get("green_loans_meur_2024"),
            "climate_capex_2024_meur": cross_industry_metrics.get("climate_capex_2024_meur"),
            "climate_opex_2024_meur": cross_industry_metrics.get("climate_opex_2024_meur"),
            "high_carbon_sector_exposure_pct": cross_industry_metrics.get("high_carbon_sector_exposure_pct"),
            "high_carbon_sector_exposure_meur": cross_industry_metrics.get("high_carbon_sector_exposure_meur"),
            "fossil_fuel_exposure_pct": cross_industry_metrics.get("fossil_fuel_exposure_pct"),
            "fossil_fuel_exposure_meur": cross_industry_metrics.get("fossil_fuel_exposure_meur"),
            "portfolio_metric_records_2024": portfolio_2024[:10],
        },
        "targets_and_progress": {
            "full_targets_available": bool(targets),
            "target_summary_available": bool(target_summary),
            "target_count": len(target_rows),
            "targets": target_rows,
        },
        "methodology_and_data_quality": {
            "methodology_records_available": bool(methodology_records),
            "methodology_records": methodology_records[:10],
        },
        "evidence_boundaries": {
            "operational_scope_1_2_3_emissions_available": bool(ghg_2024),
            "financed_emissions_total_available": _mt_present(cross_industry_metrics.get("financed_emissions_2024_tco2e")) or bool(financed_2024),
            "carbon_intensity_available": _mt_present(cross_industry_metrics.get("carbon_intensity_2024_tco2e_per_meur")),
            "high_carbon_exposure_available": _mt_present(cross_industry_metrics.get("high_carbon_sector_exposure_pct")) or _mt_present(cross_industry_metrics.get("high_carbon_sector_exposure_meur")),
            "fossil_fuel_exposure_available": _mt_present(cross_industry_metrics.get("fossil_fuel_exposure_pct")) or _mt_present(cross_industry_metrics.get("fossil_fuel_exposure_meur")),
            "green_finance_metrics_available": _mt_present(cross_industry_metrics.get("green_loans_pct_2024")) or _mt_present(cross_industry_metrics.get("green_loans_meur_2024")),
            "climate_capex_opex_available": _mt_present(cross_industry_metrics.get("climate_capex_2024_meur")) or _mt_present(cross_industry_metrics.get("climate_opex_2024_meur")),
            "full_targets_available": bool(targets),
            "target_summary_available": bool(target_summary),
            "methodology_records_available": bool(methodology_records),
            "target_boundary_instruction": (
                "Full target records are available; use only explicitly provided fields."
                if targets else
                "Only target summary evidence is available; do not invent baselines, validation bodies, milestones, gross/net status or planned carbon-credit details."
                if target_summary else
                "No target records are available; state the boundary and do not invent targets."
            ),
            "methodology_boundary_instruction": "Use methodology records only where provided. If no methodology records are available, state that detailed calculation methodology, data quality and assumptions are not described in the available evidence.",
            "do_not_invent_missing_metric_breakdowns": True,
        },
    }
    compact["scope3_categories"] = payload.get("scope3_categories", [])
    compact["ghg_methodology"] = payload.get("ghg_methodology", [])
    compact["scope12_consolidation"] = payload.get("scope12_consolidation", [])
    compact["payload_profile"]["scope3_categories_available"] = bool(payload.get("scope3_categories"))
    compact["payload_profile"]["ghg_methodology_available"] = bool(payload.get("ghg_methodology"))
    compact["payload_profile"]["scope12_consolidation_available"] = bool(payload.get("scope12_consolidation"))
    return _mt_clean(compact)


metrics_targets_evidence = summarize_metrics_targets_evidence(metrics_targets_payload)
raw_size = len(json.dumps(metrics_targets_payload, ensure_ascii=False))
compact_size = len(json.dumps(metrics_targets_evidence, ensure_ascii=False))
print("Compact Metrics & Targets evidence ready")
print(f"Raw Metrics & Targets payload size: {raw_size:,} chars")
print(f"Compact evidence size: {compact_size:,} chars")
print(f"Reduction: {round((1 - compact_size / max(raw_size, 1)) * 100, 1)}%")
print(json.dumps({
    "bank": metrics_targets_evidence["bank"],
    "full_targets_available": metrics_targets_evidence["payload_profile"].get("full_targets_available"),
    "target_summary_available": metrics_targets_evidence["payload_profile"].get("target_summary_available"),
    "target_count": metrics_targets_evidence["targets_and_progress"].get("target_count"),
    "financed_emissions_total_available": metrics_targets_evidence["evidence_boundaries"].get("financed_emissions_total_available"),
    "operational_ghg_records_available": metrics_targets_evidence["evidence_boundaries"].get("operational_scope_1_2_3_emissions_available"),
    "methodology_records_available": metrics_targets_evidence["evidence_boundaries"].get("methodology_records_available"),
}, indent=2, ensure_ascii=False))

Compact Metrics & Targets evidence ready
Raw Metrics & Targets payload size: 34,120 chars
Compact evidence size: 5,831 chars
Reduction: 82.9%
{
  "bank": {
    "bank_id": "BANK01",
    "bank_name": "Eurolux Universal Bank AG",
    "country": "DE",
    "reporting_year": 2024,
    "reporting_currency": "EUR",
    "regulatory_regime": "CSRD",
    "total_assets_meur": 850000,
    "total_loans_meur": 31150.64
  },
  "full_targets_available": true,
  "target_summary_available": true,
  "target_count": 3,
  "financed_emissions_total_available": true,
  "operational_ghg_records_available": false,
  "methodology_records_available": false
}


## Metrics and Targets evidence availability, traceability and saving


In [44]:
# ── METRICS AND TARGETS AVAILABILITY + SAVING ───────────────

def build_metrics_targets_availability_profile(evidence: dict) -> dict:
    metrics = evidence.get("cross_industry_metrics", {})
    boundaries = evidence.get("evidence_boundaries", {})
    targets = evidence.get("targets_and_progress", {})

    def present(v) -> bool:
        return v is not None and str(v).strip().lower() not in {"", "none", "null", "nan"}

    return {
        "reporting_kpis_available": bool(evidence.get("payload_profile", {}).get("reporting_kpis_available")),
        "operational_ghg_records_available": bool(boundaries.get("operational_scope_1_2_3_emissions_available")),
        "financed_emissions_total_available": bool(boundaries.get("financed_emissions_total_available")),
        "carbon_intensity_available": bool(boundaries.get("carbon_intensity_available")),
        "high_carbon_exposure_available": bool(boundaries.get("high_carbon_exposure_available")),
        "fossil_fuel_exposure_available": bool(boundaries.get("fossil_fuel_exposure_available")),
        "green_finance_metrics_available": bool(boundaries.get("green_finance_metrics_available")),
        "climate_capex_opex_available": bool(boundaries.get("climate_capex_opex_available")),
        "full_targets_available": bool(boundaries.get("full_targets_available")),
        "target_summary_available": bool(boundaries.get("target_summary_available")),
        "targets_available": bool(targets.get("targets")),
        "methodology_records_available": bool(boundaries.get("methodology_records_available")),
        "metric_values_present": {key: present(value) for key, value in metrics.items()},
        "writer_policy": {
            "use_available_evidence": "Use all material Metrics and Targets evidence that is available and relevant.",
            "handle_unavailable_evidence": "When a metric, target detail, methodology, data quality item or breakdown is not evidenced, state the boundary once and do not invent it.",
            "target_policy": boundaries.get("target_boundary_instruction"),
            "methodology_policy": boundaries.get("methodology_boundary_instruction"),
            "final_language": "Write in the Bank's own voice and state facts directly. Do not refer to the report's evidence base: never use 'available evidence', 'available documentation', 'documentation reviewed', 'source data', 'internally generated analytical outputs' or 'payload' in the final disclosure. Frame genuine gaps as the Bank's position - 'the Bank does not currently disclose X', 'X is outside the scope of this report', or 'X has not yet been quantified'.",
        },
    }


def add_metrics_targets_traceability(evidence: dict, source_payload_path: Path) -> dict:
    evidence = dict(evidence)
    evidence["availability_profile"] = build_metrics_targets_availability_profile(evidence)
    evidence["source_traceability"] = {
        "source_payload_path": str(source_payload_path),
        "source_tables": [key for key in [
            "bank", "metadata", "reporting_kpis", "financial_summary", "targets",
            "ghg_emissions", "emissions", "scope_emissions", "operational_emissions",
            "financed_emissions", "portfolio_metrics", "climate_metrics", "methodology", "data_quality", "assumptions",
        ] if key in metrics_targets_payload],
        "target_refs": [
            item.get("target_id") or item.get("target_type") or item.get("scope")
            for item in evidence.get("targets_and_progress", {}).get("targets", [])
            if isinstance(item, dict)
        ],
    }
    return evidence


metrics_targets_evidence = add_metrics_targets_traceability(metrics_targets_evidence, METRICS_TARGETS_PAYLOAD_PATH)

raw_metrics_targets_payload = {key: metrics_targets_payload.get(key) for key in [
    "metadata", "bank", "reporting_kpis", "financial_summary", "targets", "ghg_emissions", "emissions",
    "scope_emissions", "operational_emissions", "financed_emissions", "portfolio_metrics", "climate_metrics",
    "methodology", "data_quality", "assumptions",
] if key in metrics_targets_payload}

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
RAW_METRICS_TARGETS_PATH = output_dir / "payload_BANK01_metrics_targets_raw.json"
COMPACT_METRICS_TARGETS_PATH = output_dir / "compact_metrics_targets_evidence_BANK01.json"

with open(RAW_METRICS_TARGETS_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_metrics_targets_payload, f, indent=2, ensure_ascii=False)
with open(COMPACT_METRICS_TARGETS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_targets_evidence, f, indent=2, ensure_ascii=False)

print("Metrics & Targets evidence prepared")
print(f"- Raw Metrics & Targets payload: {RAW_METRICS_TARGETS_PATH}")
print(f"- Compact Metrics & Targets evidence: {COMPACT_METRICS_TARGETS_PATH}")
print("- Availability profile:")
print(json.dumps(metrics_targets_evidence["availability_profile"], indent=2, ensure_ascii=False))

Metrics & Targets evidence prepared
- Raw Metrics & Targets payload: outputs\payload_BANK01_metrics_targets_raw.json
- Compact Metrics & Targets evidence: outputs\compact_metrics_targets_evidence_BANK01.json
- Availability profile:
{
  "reporting_kpis_available": true,
  "operational_ghg_records_available": false,
  "financed_emissions_total_available": true,
  "carbon_intensity_available": true,
  "high_carbon_exposure_available": true,
  "fossil_fuel_exposure_available": true,
  "green_finance_metrics_available": true,
  "climate_capex_opex_available": true,
  "full_targets_available": true,
  "target_summary_available": true,
  "targets_available": true,
  "methodology_records_available": false,
  "metric_values_present": {
    "financed_emissions_2024_tco2e": true,
    "carbon_intensity_2024_tco2e_per_meur": true,
    "high_carbon_sector_exposure_pct": true,
    "high_carbon_sector_exposure_meur": true,
    "fossil_fuel_exposure_pct": true,
    "fossil_fuel_exposure_meur": true,
  

## Metrics and Targets state definition


In [45]:
# ── METRICS AND TARGETS STATE DEFINITION ────────────────────
class MetricsTargetsState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

## Metrics and Targets requirements — Emirates-style structure


In [46]:
# ── METRICS AND TARGETS REQUIREMENTS ────────────────────────
METRICS_TARGETS_REQUIREMENTS = """
STRICT METRICS AND TARGETS DISCLOSURE REQUIREMENTS FOR THIS SECTION:

Presentation style:
- Write like a dedicated bank IFRS S1/S2 Metrics and targets section, not like a checklist answer.
- Follow an Emirates NBD-style flow: overview, sustainability-related metrics and targets, greenhouse gas emissions, financed emissions exposure, financial metrics and resource allocation, climate targets, then limitations.
- Keep financial and emissions figures evidence-based.
- Do not overuse "available evidence" in the main narrative. Use it mainly in the limitations subsection.
- Do not use the word "payload" in the final report.

Final section title and headings must be exactly:
### Metrics and targets
#### Overview
#### Sustainability-related metrics and targets
#### Greenhouse gas emissions
#### Managing exposure towards financed emissions
#### Climate-related financial metrics and resource allocation
#### Climate targets and progress
#### Metrics and targets limitations and evidence boundaries

Internal coverage map:
1. Metrics used to assess and manage climate-related risks and opportunities:
   - Use reported climate KPIs and portfolio metrics where available.
   - Use high-carbon sector exposure, fossil-fuel exposure, green loans, climate capex and climate opex where available.
2. Greenhouse gas emissions:
   - Use operational Scope 1/2/3 records only if available.
   - If operational GHG records are unavailable, state this boundary.
   - Do not invent operational emissions or location-based/market-based breakdowns.
3. Financed emissions:
   - Use financed emissions total and carbon intensity where available.
   - Do not invent asset-class, sector, PCAF score or methodology breakdowns unless evidenced.
4. Targets:
   - Use full targets if available.
   - If only target_summary is available, use only target type, scope, status, target year, framework and progress.
   - Do not invent baselines, validation bodies, milestones, gross/net status or planned carbon-credit details unless evidenced.
5. Methodology and boundaries:
   - Use calculation methodology, data quality and assumptions only where provided.
   - If methodology records are unavailable, state that detailed calculation methodology, data quality and assumptions are not described in the available evidence.
6. Do not include visible IFRS paragraph references, internal IDs or bracketed evidence tags.
7. Do not use the technical word "payload" in the final report.

Target-record parameter rules:
- Target baseline values, target values and interim milestones may be disclosed only as target-record parameters.
- Do not present target baseline or milestone tCO₂e values as current-year operational emissions.
- Keep the 2024 operational emissions boundary only in the Greenhouse gas emissions subsection.
- Progress percentages must be described as reported indicators from target records; do not imply independent verification.
- If validation fields are available, write them as record fields, for example: "the target record indicates third_party_validated=true, with validation body [name]." Do not use assurance-style wording.

Data-aware requirements:
- Use exact figures where available.
- Distinguish operational emissions from financed emissions.
- Distinguish metrics from targets.
- Treat targets and progress as reported evidence, not assurance that targets will be achieved.
- If a metric is unavailable, state the boundary and do not invent values.
- Use cautious wording: "reported", "available evidence shows", "source data indicates".
- Avoid unsupported strong claims such as "validated", "achieved", "assured", "on track" or "Paris-aligned". Use target-record field wording instead of assurance-style wording.
""".strip()
print("Metrics and Targets requirements ready — Emirates-style structure")

Metrics and Targets requirements ready — Emirates-style structure


## Metrics and Targets writer and judge prompts


In [47]:
# ── METRICS AND TARGETS WRITER / JUDGE PROMPTS ──────────────
METRICS_TARGETS_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the Metrics and targets section of an IFRS S1/S2-aligned climate disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style similar to a dedicated bank IFRS S1/S2 report.

Evidence rules:
- In the Greenhouse gas emissions subsection: state that operational Scope 1/2/3 records are unavailable. Do not include any target baseline, milestone, or tCO2e table there.
- Put target baseline and milestone parameters only in Climate targets and progress.
- Avoid the words "assurance", "assured", and "independent assurance" in this section. Use "third-party validation" for targets and "not independently verified" for current-year metrics.
- State clearly that third-party validation fields are target-record attributes only; they do not verify current-year emissions, progress, Paris alignment, or future achievement.
- If target-setting oversight body, owner, or remuneration linkage is not evidenced in this section, add a boundary statement instead of implying management integration.

- Avoid raw target-record field names and key=value syntax; translate target attributes into public report wording.
- Translate raw data field names into public report wording; avoid snake_case and key=value syntax in the final section.
- Never end the section mid-sentence; complete every required subsection, especially the final evidence-boundary subsection.
- Use the clean style guide only for writing style, tone, structure and formatting; never use it as factual evidence.
- Use only the compact Metrics and Targets evidence supplied by the user prompt.
- Do not invent operational emissions, financed-emissions breakdowns, PCAF scores, methodology assumptions, target baselines, milestones, validation bodies, gross/net status, planned credits or assurance status.
- When evidence is missing, state any genuine gap once, in the Bank's own voice as the Bank's position - for example "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".
- Do not use the word "payload" in the final section.
- Do not include IFRS paragraph references, internal references or bracketed evidence tags.
- Do not use markdown tables.

Target wording rules:

GREENHOUSE GAS EMISSIONS SUBSECTION HARD RULE:
- In "#### Greenhouse gas emissions", write only that detailed 2024 operational Scope 1, Scope 2 and Scope 3 emissions records are unavailable.
- Do not include a target table, baseline parameter, milestone value, or operational tCO2e value in that subsection.
- Discuss target baseline/milestone parameters only under "#### Climate targets and progress".

PUBLIC TARGET WORDING RULES:
- Never write "target framework=..." in the final text. Write "the target record identifies [framework] as the target framework."
- Never write "third-party validation is recorded" in the final text. Write "the target record records third-party validation, with [body] identified as the validation body."
- Never write "validation body" in the final text. Write "validation body".
- Do not use raw target IDs as section labels unless needed for traceability; prefer descriptive target names.
- Keep the target section narrative-style. If using bullets, keep them short and public-facing, not a raw data extract.

- Do not expose raw field names such as baseline value, target percentage reduction, target framework, third-party validation field, Paris-alignment field, gross or net basis, planned carbon credits percentage.
- Do not use key=value syntax or underscore-form values in the final report.
- Convert framework names to public wording, for example "SBTi 1.5°C pathway" and "UNEP FI".
- Convert "on track" into public wording such as "recorded as on track in the target record".
- Convert "technology-removal credits" into normal prose.
- If a target table is used, keep it concise with public columns only: Target, Scope, Baseline year, Target year, Reduction target, Progress indicator, Framework, Validation body.
- Prefer a short summary table plus narrative, not a dense data extract.

- Do not expose raw target field names such as baseline value, target percentage reduction, target framework, third-party validation field, Paris-alignment field, gross or net basis or planned carbon credits percentage.
- Avoid the words "validated", "achieved", "assured", "on track" and "Paris-aligned" in the final text.
- If validation fields exist, write in public report wording: "the target record indicates that third-party validation is recorded, with [name] identified as the validation body."
- If progress/status fields exist, write in public report wording: "the target record reports [status] status and [value]% progress."
- Do not imply independent assurance or future achievement.
- When disclosing target baselines, target values or interim milestones in tCO₂e, explicitly label them as target-record parameters, not current-year operational emissions.

Emirates-style structure rules:
- Use the exact seven required subsections.
- Start with an Overview.
- Present sustainability-related metrics and targets before detailed emissions subsections.
- Use a dedicated subsection for managing exposure towards financed emissions.
- Keep limitations and evidence boundaries as the final subsection.
- Do not invent operational Scope 1/2/3 values if detailed operational GHG records are unavailable.

Additional precision rules (avoid the recurring overclaims):
- Never present target status codes (for example "on track") or progress percentages as a substantiated assessment of performance or likelihood of achievement. Where a target record carries a status, describe it neutrally as an internal target-record label (for example, "the internal target record classifies this target as on track") and pair it with the boundary that progress is unverified. Prefer omitting a status descriptor over implying assured performance.
- Do not imply methodological robustness for financed emissions or other metrics when methodology records are unavailable; describe the figure and state the methodology boundary.

- Do not use the literal phrase "on track" anywhere, even attributed to a target record. Describe progress neutrally (for example, "the internal target record indicates progress against the target"), state that progress is unverified, or omit the status entirely.

- Present third-party validation strictly as a recorded target-record field. Say, for example, "the target record lists UNEP FI in its validation_body field" — never "third-party validation by UNEP FI" or wording that implies an independent validation or assurance was performed. Apply the same caution to SBTi: describe it as the recorded target framework, not as evidence of pathway alignment or validation.

REPORTING VOICE (this overrides any other wording guidance, including wording in the evidence): Write strictly as the Bank's own published disclosure, in the Bank's voice, and state facts directly. The following are BANNED from the final section: "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", "based on available evidence/documentation", and any phrasing that narrates what "the documentation" or "the evidence" does or does not show. Write "The Board met eight times in 2024", never "Available documentation indicates the Board met eight times". For genuine gaps, write the Bank's position: "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".

- Do not state that no governance body oversees target-setting or that target ownership is undisclosed. Oversight of climate-related targets by the Board and the ESG & Sustainability Committee is described in the Governance section; cross-refer to it. You may note only that target-level internal owner roles are not separately detailed, without implying that governance oversight of targets is absent.

Output rules:
- Return only the complete Metrics and targets section.
- Keep exactly the seven required subsections.
""".strip()


def build_metrics_targets_writer_prompt(evidence: dict, judge_feedback: str | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    boundaries = evidence.get("evidence_boundaries", {})
    instructions = []
    instructions.append("- Do not state the section or its metrics are 'in line with', 'compliant with' or 'aligned with' IFRS S1/S2. Where referencing the frameworks, say the disclosures are 'prepared with reference to' IFRS S1 and IFRS S2, or state the boundary instead.")
    instructions.append("- Do not assert scoping exclusions for financed emissions (e.g. excluding retail mortgage or revolving-credit products) unless that exclusion is explicitly evidenced; if the perimeter is not evidenced, state that the basis of the financed-emissions perimeter is not separately disclosed.")

    if profile.get("operational_ghg_records_available"):
        instructions.append("- Use operational GHG emission records exactly as provided.")
    else:
        instructions.append("- Operational Scope 1/2/3 GHG records are not available as detailed records. State this boundary and do not invent operational emissions.")

    if profile.get("financed_emissions_total_available"):
        instructions.append("- Use financed emissions and carbon intensity values as portfolio emissions metrics.")
    else:
        instructions.append("- Financed-emissions total is unavailable. State this boundary if financed emissions are discussed.")

    if profile.get("high_carbon_exposure_available"):
        instructions.append("- Use high-carbon sector exposure values.")
    if profile.get("fossil_fuel_exposure_available"):
        instructions.append("- Use fossil-fuel exposure values.")
    if profile.get("green_finance_metrics_available"):
        instructions.append("- Use green loans / green finance values.")
    if profile.get("climate_capex_opex_available"):
        instructions.append("- Use climate capex and climate opex values.")

    if profile.get("full_targets_available"):
        instructions.append("- Full target records are available. Use only the explicitly provided target fields.")
    elif profile.get("target_summary_available"):
        instructions.append("- Only target summary evidence is available. Use target type, scope, status, target year, framework and progress only; do not invent baselines, validation bodies, milestones, gross/net status or planned credits.")
    else:
        instructions.append("- Target records are unavailable. State this boundary and do not invent targets.")

    if profile.get("methodology_records_available"):
        instructions.append("- Use methodology/data quality records exactly as provided.")
    else:
        instructions.append("- Detailed methodology/data quality records are unavailable. State this boundary once in the limitations subsection.")

    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION POLICY:
- Fix omissions using available evidence.
- Preserve accurate evidence-boundary statements where evidence is unavailable.
- Never invent evidence to satisfy the judge.
""".strip()

    return f"""
{METRICS_TARGETS_REQUIREMENTS}

WRITING STYLE GUIDE — STYLE ONLY, NOT EVIDENCE:
{build_style_context("metrics_and_targets")}

STYLE APPLICATION RULES:
- Use the style guide only for tone, flow, structure and formatting.
- Use compact Metrics and Targets evidence for facts.
- Use IFRS requirements only to decide what topics to cover.
- Do not copy reference-report wording or mention forbidden reference terms.

BANK:
{evidence['bank']['bank_name']} ({evidence['bank'].get('bank_id')})
REPORTING YEAR: {evidence['bank'].get('reporting_year')}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DATA-AWARE WRITING INSTRUCTIONS:
{chr(10).join(instructions)}

SECTION BLUEPRINT:
### Metrics and targets

#### Overview
Summarise the metrics and targets architecture: operational emissions boundaries, financed emissions, portfolio exposure, green loans, resource allocation and climate targets.

#### Sustainability-related metrics and targets
Present key metrics from the evidence in a concise report-style narrative.

#### Greenhouse gas emissions
Discuss operational GHG emissions only if operational GHG records are available. If unavailable, state the boundary without using target baseline or milestone tCO2e values as current-year operational emissions. If scope3_categories evidence is present, disclose Scope 3 emissions by GHG Protocol category, identifying which categories are included and stating explicitly which categories are excluded and why. If ghg_methodology evidence is present, describe the measurement approach, key inputs and assumptions for each scope and the reason for that approach. If scope12_consolidation evidence is present, state the consolidation basis (for example operational control) and how Scope 1 and Scope 2 emissions are split between the consolidated accounting group and other investees.

#### Managing exposure towards financed emissions
Present 2024 financed emissions, carbon intensity and portfolio exposure indicators. State assurance boundary where applicable.

#### Climate-related financial metrics and resource allocation
Present climate capex, climate opex, green loans and exposure metrics as financial/resource allocation indicators.

#### Climate targets and progress
Use target records carefully. Frame baseline, target, milestone, progress and validation fields as target-record parameters or reported source fields, not as assurance or proof of achievement.

#### Metrics and targets limitations and evidence boundaries
State boundaries around operational emissions records, methodology/data-quality records, assurance scope, target-record interpretation and disaggregation.

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("metrics_targets")}

REQUIREMENT-TO-EVIDENCE POLICY:
- Use IFRS requirements to decide which metrics and targets topics must be addressed.
- Use compact bank evidence to write bank-specific facts.
- If an IFRS requirement is relevant but evidence is unavailable, disclose the evidence boundary instead of inventing operational GHG values, methodologies, target details, validation, assurance or disaggregation.
- Treat core_standard items as mandatory disclosure guidance.
- Treat industry_guidance and industry_metrics items as banking-specific application guidance.
- Do not display IFRS paragraph references or requirement IDs in the final report.

COMPACT METRICS AND TARGETS EVIDENCE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

IMPORTANT DATA BOUNDARIES:
- Operational GHG records available: {boundaries.get('operational_scope_1_2_3_emissions_available')}
- Financed emissions total available: {boundaries.get('financed_emissions_total_available')}
- Carbon intensity available: {boundaries.get('carbon_intensity_available')}
- High-carbon exposure available: {boundaries.get('high_carbon_exposure_available')}
- Fossil-fuel exposure available: {boundaries.get('fossil_fuel_exposure_available')}
- Full targets available: {boundaries.get('full_targets_available')}
- Target summary available: {boundaries.get('target_summary_available')}
- Target instruction: {boundaries.get('target_boundary_instruction')}
- Methodology records available: {boundaries.get('methodology_records_available')}
- Methodology instruction: {boundaries.get('methodology_boundary_instruction')}

GENERAL WRITING RULES:
- Use exact figures where available.
- Avoid the words "validated", "achieved", "assured", "on track" and "Paris-aligned"; use target-record field wording instead.
- Do not invent operational GHG metrics or financed-emissions breakdowns.
- Do not invent PCAF data quality scores or methodology assumptions.
- Mention limitations only in the relevant subsection and in the final limitations subsection.

{feedback_block}

Write the complete Metrics and targets section only.
""".strip()


METRICS_TARGETS_JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-aligned bank climate Metrics and Targets disclosure.

You evaluate whether the Metrics and targets section is supported by compact evidence, aligned with the checklist, transparent about missing evidence, and free from unsupported metric values, target claims, methodology claims, assurance claims and emissions breakdowns.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings such as "true" or "false".
""".strip()


def build_metrics_targets_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    profile = evidence.get("availability_profile", {})
    deterministic_checks = deterministic_checks or {}
    return f"""
Evaluate the Metrics and targets draft against the compact evidence, availability profile and deterministic pre-checks.

DRAFT:
{draft}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(profile, indent=2, ensure_ascii=False)}

REPORTING VOICE: The section must read as the Bank's own published disclosure. Treat as a material wording defect any report-machinery language - "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", or phrasing that narrates what "the documentation"/"the evidence" does or does not show. Facts must be stated directly; genuine gaps framed as the Bank's position ("the Bank does not currently disclose ...", "... is outside the scope of this report", "... has not yet been quantified"). List any such phrasing in material_unsupported_claims so it is corrected.

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("metrics_targets")}

COMPACT METRICS AND TARGETS EVIDENCE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
0A. Check the IFRS requirements pack and the ifrs_requirement_coverage pre-check. Penalise missing mandatory core IFRS topics unless the draft clearly discloses an evidence boundary.
0. Verify that the section follows the Emirates-style metrics and targets structure: overview, sustainability-related metrics and targets, greenhouse gas emissions, financed emissions exposure, financial metrics/resource allocation, targets/progress and limitations.
1. Penalise any claim that contradicts evidence or invents unavailable metric/target information.
2. Penalise omission when material evidence is available but not used.
3. Do not demand unavailable evidence. Correctly disclosed evidence boundaries are acceptable but lower completeness.
4. Verify that operational emissions and financed emissions are distinguished.
5. If operational GHG records are unavailable, the draft must not invent Scope 1, Scope 2 or Scope 3 operational values.
6. If financed emissions and carbon intensity are available, verify they are used.
7. If high-carbon/fossil-fuel/green loans/capex/opex metrics are available, verify they are used or explain any omission.
8. If full target records are available, target baseline or milestone tCO₂e values may be used only as target-record parameters, not as current-year operational emissions. If only target_summary is available, verify no target baseline, validation body, milestone, gross/net status or carbon-credit details are invented.
9. If methodology records are unavailable, verify the section states the methodology/data quality boundary and does not invent methodology assumptions.
10. Verify no visible IFRS paragraph references, internal refs or bracketed evidence tags appear.
11. Verify no unsupported strong claims such as validated, achieved, assured, on track, Paris-aligned or guaranteed. Target validation/status fields must be framed strictly as target-record fields, not assurance or future achievement.

SCORING:
Score on completeness RELATIVE TO AVAILABLE EVIDENCE, not on how much data the bank happens to hold. Correctly disclosed evidence boundaries (genuine unavailability, accurately stated) are good IFRS practice and MUST NOT reduce the score; the number of disclosed boundaries is irrelevant to the score. Penalise ONLY: unsupported or overclaimed statements, omitted AVAILABLE evidence, factual inconsistencies with the evidence, hallucinations, and structural or deterministic failures.
- 9-10: Materially complete relative to available evidence, report-ready structure, no unsupported or overclaimed statements, no omitted available evidence, and all genuine data gaps correctly disclosed as boundaries.
- 8: As 9-10 but with one minor evidence-use issue or one slightly loose statement to tighten.
- 7: Usable but contains an unsupported/overclaimed statement, omits available evidence, or has a structural/keyword gap.
- 6: Revision required: available evidence omitted, unsupported wording remains, factual inconsistency with evidence, or required structure/boundary keywords missing.
- 5 or below: Major evidence failure, hallucination, contradiction, or missing core subsection.

When listing omitted evidence, separate MATERIAL omissions (which must be incorporated) from OPTIONAL ones (acceptable to leave out); never mark internal record IDs or optional traceability codes as material. Likewise, separate unsupported claims into material_unsupported_claims (misrepresent the evidence or overstate scope/assurance/compliance — must fix) and minor_wording_to_tighten (defensible phrasing such as 'covers' vs 'includes' or singular/plural that does not misrepresent the evidence — an acceptable minor limitation). Do not list trivial wording as material. Return valid JSON only. Keep arrays concise:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "overview_present": <true/false>,
    "report_style_structure_present": <true/false>,
    "emirates_style_metrics_flow_present": <true/false>,
    "operational_ghg_handled_according_to_availability": <true/false>,
    "financed_emissions_used_correctly": <true/false>,
    "portfolio_intensity_used_correctly": <true/false>,
    "financial_metrics_used_correctly": <true/false>,
    "high_carbon_and_fossil_exposure_used_if_available": <true/false>,
    "green_finance_capex_opex_used_if_available": <true/false>,
    "targets_handled_according_to_availability": <true/false>,
    "methodology_boundaries_handled_correctly": <true/false>,
    "evidence_boundaries_present": <true/false>,
    "no_visible_ifrs_refs_or_internal_refs": <true/false>,
    "no_unsupported_strong_claims": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "material_evidence_omitted": [<omitted available evidence whose absence materially weakens the disclosure; MUST be incorporated as report prose>],
  "optional_evidence_omitted": [<omitted available evidence that is acceptable to leave out; do not force inclusion>],
  "unsupported_claims": [<specific unsupported claims>],
  "material_unsupported_claims": [<claims that misrepresent the evidence, assert unevidenced facts, or overstate scope/assurance/compliance; these MUST be removed or reframed>],
  "minor_wording_to_tighten": [<defensible phrasing that could be tightened but does NOT misrepresent the evidence; acceptable as a minor limitation>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()
print("Metrics and Targets prompts ready — Emirates-style structure")

Metrics and Targets prompts ready — Emirates-style structure


## Metrics and Targets evaluation mode


In [48]:
# ── METRICS AND TARGETS EVALUATION MODE ─────────────────────
print("Metrics and Targets evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks")

Metrics and Targets evaluation mode: GPT-5.2 judge + data-aware deterministic pre-checks


## Metrics and Targets LangGraph nodes and deterministic checks


In [49]:
# ── METRICS AND TARGETS LANGGRAPH NODES ─────────────────────

def _mt_number_present(text: str, expected: float, tolerance: float = 0.2) -> bool:
    nums = re.findall(r"\b\d+(?:\.\d+)?\b", text.replace(",", ""))
    nums = [float(n) for n in nums]
    return any(abs(n - expected) <= tolerance for n in nums)


def run_metrics_targets_deterministic_checks(draft: str, evidence: dict) -> dict:
    text = draft or ""
    text_l = text.lower()
    profile = evidence.get("availability_profile", {})
    metrics = evidence.get("cross_industry_metrics", {})
    required_headings = [
        "#### Overview",
        "#### Sustainability-related metrics and targets",
        "#### Greenhouse gas emissions",
        "#### Managing exposure towards financed emissions",
        "#### Climate-related financial metrics and resource allocation",
        "#### Climate targets and progress",
        "#### Metrics and targets limitations and evidence boundaries",
    ]
    failures, warnings = [], []

    missing_headings = [h for h in required_headings if h not in text]
    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})
    if re.search(r"IFRS\s*S?[12]?\s*§|§\s*\d|\[IFRS", text):
        failures.append({"check": "visible_ifrs_references", "details": "Visible IFRS references/tags found."})
    if re.search(r"\[REF:|MTG-BANK\d+", text):
        failures.append({"check": "internal_refs_visible", "details": "Internal references should not appear in final text."})

    if not profile.get("operational_ghg_records_available"):
        ghg_section = ""
        if "#### Greenhouse gas emissions" in text:
            ghg_section = text.split("#### Greenhouse gas emissions", 1)[1]
            if "#### Managing exposure towards financed emissions" in ghg_section:
                ghg_section = ghg_section.split("#### Managing exposure towards financed emissions", 1)[0]
        ghg_l = ghg_section.lower()
        if re.search(r"\|\s*target\s*\|\s*scope\s*\|\s*baseline", ghg_l) or "target parameters (not current-year emissions)" in ghg_l:
            failures.append({
                "check": "target_parameter_table_in_ghg_subsection",
                "details": "Do not place target baseline/milestone values in the Greenhouse gas emissions subsection."
            })
        operational_value_patterns = [
            r"scope\s*1[^.\n]{0,50}\b\d{2,}(?:[,.]\d+)?\b",
            r"scope\s*2[^.\n]{0,50}\b\d{2,}(?:[,.]\d+)?\b",
            r"scope\s*3[^.\n]{0,50}\b\d{2,}(?:[,.]\d+)?\b",
        ]
        if any(re.search(pattern, ghg_l) for pattern in operational_value_patterns):
            failures.append({
                "check": "operational_ghg_values_invented",
                "details": "Operational Scope 1/2/3 values appear in the GHG subsection although detailed operational GHG records are unavailable."
            })

    numeric_expectations = {
        "financed_emissions_2024_tco2e": metrics.get("financed_emissions_2024_tco2e"),
        "carbon_intensity_2024_tco2e_per_meur": metrics.get("carbon_intensity_2024_tco2e_per_meur"),
        "high_carbon_sector_exposure_pct": metrics.get("high_carbon_sector_exposure_pct"),
        "fossil_fuel_exposure_pct": metrics.get("fossil_fuel_exposure_pct"),
        "green_loans_pct_2024": metrics.get("green_loans_pct_2024"),
        "climate_capex_2024_meur": metrics.get("climate_capex_2024_meur"),
        "climate_opex_2024_meur": metrics.get("climate_opex_2024_meur"),
    }
    for name, expected in numeric_expectations.items():
        if expected is None:
            continue
        try:
            expected_f = float(expected)
        except Exception:
            continue
        if not _mt_number_present(text, expected_f, tolerance=max(0.2, abs(expected_f) * 0.001)):
            warnings.append({"check": f"{name}_may_be_omitted", "details": f"Expected numeric value may be missing: {expected_f}"})

    if profile.get("target_summary_available") and not profile.get("full_targets_available"):
        unsupported_target_terms = ["baseline year", "baseline value", "validation body", "third-party validated", "gross target", "net target", "interim milestone", "planned carbon credits", "carbon-credit use"]
        found_target_terms = [p for p in unsupported_target_terms if p in text_l]
        if found_target_terms:
            failures.append({"check": "target_details_invented", "details": found_target_terms})

    unsupported_strong_phrases = [
        "on track",
        "validated",
        "third-party validated",
        "achieved",
        "assured",
        "guaranteed",
        "paris-aligned",
        "paris aligned",
    ]

    allowed_record_field_contexts = [
        "target record indicates third_party_validated=true",
        "target record reports status=",
        "reported indicator from the target record",
        "no assurance statement",
        "does not constitute assurance",
        "future target outcomes will be met",
    ]

    found_strong = []
    for phrase in unsupported_strong_phrases:
        if phrase in text_l:
            # Avoid false positives for cautious limitation wording.
            if phrase == "achieved" and ("not constitute assurance" in text_l or "future target outcomes will be met" in text_l):
                continue
            if phrase in {"validated", "third-party validated"} and any(ctx in text_l for ctx in allowed_record_field_contexts):
                continue
            found_strong.append(phrase)

    if found_strong:
        warnings.append({"check": "possible_unsupported_strong_language", "details": found_strong})


    if term_in_claiming_context(text_l, "assur"):
        warnings.append({
            "check": "metrics_assurance_vocabulary_present",
            "details": "Use validation/verification wording in Metrics and targets; avoid assurance terminology."
        })

    if "monitored by management" in text_l and "available evidence does not specify" not in text_l:
        warnings.append({
            "check": "metrics_governance_linkage_boundary_missing",
            "details": "Add a boundary if target-setting oversight body/remuneration linkage is not evidenced in this section."
        })

    if not profile.get("methodology_records_available"):
        if "methodology" not in text_l and "data quality" not in text_l and "assumption" not in text_l:
            failures.append({"check": "methodology_boundary_missing", "details": "Methodology/data quality boundary should be stated when methodology records are unavailable."})

    limitations_heading = "#### Metrics and targets limitations and evidence boundaries"
    if limitations_heading in text:
        limitations_text = text.split(limitations_heading, 1)[1].lower()
        if len(limitations_text.split()) < 60:
            failures.append({"check": "limitations_section_too_short", "details": "Metrics and targets limitations section is present but too short."})
    else:
        failures.append({"check": "limitations_section_missing", "details": "Metrics, targets and methodology limitations subsection is missing."})

    style_leak = check_reference_identity_leak(text)
    if not style_leak.get("passed", True):
        failures.append({"check": "reference_identity_leak", "details": style_leak.get("matches", [])})

    completeness = check_section_completeness(text, "metrics_and_targets")
    failures.extend(completeness.get("failures", []))
    warnings.extend(completeness.get("warnings", []))

    return {"failures": failures, "warnings": warnings, "failure_count": len(failures), "warning_count": len(warnings)}


def metrics_targets_writer_node(state: MetricsTargetsState) -> MetricsTargetsState:
    is_revision = state["revision_count"] > 0
    feedback = None
    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
        feedback = "REQUIRED FIXES:\n" + "\n".join(f"- {fix}" for fix in issues) + "\n\nFAILED CHECKLIST ITEMS:\n" + "\n".join(f"- {item}" for item in false_items)
    prompt = build_metrics_targets_writer_prompt(state["evidence"], judge_feedback=feedback)
    print("Metrics & Targets writer prompt approx tokens:", round(len(prompt) / 4))
    draft = call_writer_llm(system_prompt=METRICS_TARGETS_WRITER_SYSTEM, user_prompt=prompt)
    print(f"\n{'='*50}\nMETRICS & TARGETS WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words\n{'='*50}")
    return {**state, "draft": draft.strip(), "status": "judging"}


def metrics_targets_judge_node(state: MetricsTargetsState) -> MetricsTargetsState:
    draft = state["draft"]
    deterministic_checks = run_metrics_targets_deterministic_checks(draft, state["evidence"])
    deterministic_checks["ifrs_requirement_coverage"] = check_ifrs_requirement_coverage(draft, "metrics_targets")
    prompt = build_metrics_targets_judge_prompt(draft, state["evidence"], deterministic_checks=deterministic_checks)
    judge_result = call_judge_llm_json(system_prompt=METRICS_TARGETS_JUDGE_SYSTEM, user_prompt=prompt)
    try:
        judge_result = normalize_json_booleans(judge_result)
    except NameError:
        pass
    judge_result.setdefault("approved", False)
    judge_result.setdefault("approval_status", "approved" if judge_result.get("approved") else "revision_required")
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    judge_result["_best_so_far"] = track_best(state, judge_result)  # non-regression: keep best draft across iterations
    print("\nMETRICS & TARGETS JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))
    return {**state, "judge_result": judge_result, "status": "approved" if bool(judge_result.get("approved")) else "revising"}


METRICS_TARGETS_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing Metrics and targets section using only the supplied compact evidence, availability profile, deterministic pre-checks and judge fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct content that was not criticised.
- Never invent evidence.
- Keep the exact seven-subsection Emirates-style Metrics and targets structure.
- Do not add visible IFRS paragraph references or internal refs.
- Return only the complete revised Metrics and targets section.
""".strip()


def metrics_targets_reviser_node(state: MetricsTargetsState) -> MetricsTargetsState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}
    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False and k != "false_count"]
    revision_prompt = f"""
STRICT METRICS AND TARGETS REQUIREMENTS:
{METRICS_TARGETS_REQUIREMENTS}

AVAILABLE-EVIDENCE PROFILE:
{json.dumps(state["evidence"].get("availability_profile", {}), indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("metrics_targets")}

COMPACT METRICS AND TARGETS EVIDENCE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

REVISION BOUNDARY:
- Use available evidence when the judge identifies an omission.
- Preserve or improve accurate boundary statements where evidence is unavailable.
- Never invent missing metric values, operational emissions, financed-emissions breakdowns, target details, methodology assumptions, data quality statements or assurance claims.

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete Metrics and targets section.
""".strip()
    revised_draft = call_reviser_llm(system_prompt=METRICS_TARGETS_REVISER_SYSTEM, user_prompt=(revision_prompt + "\n\nPRIORITISED REVISION BRIEF (address every item using available evidence only):\n" + build_reviser_focus(judge)))
    new_revision_count = state["revision_count"] + 1
    print(f"\nMetrics & Targets revised with GPT-5.1 | revision {new_revision_count}")
    return {**state, "draft": revised_draft.strip(), "revision_count": new_revision_count, "status": "judging"}


def metrics_targets_finalize_node(state: MetricsTargetsState) -> MetricsTargetsState:
    new_state = finalize_state(state, target_score=TARGET_SCORE)
    gate = new_state["judge_result"]["finalize_gate"]
    print(f"\n{'='*50}")
    print("FINALIZED METRICS AND TARGETS")
    print(f"  Score: {gate['score']}/{gate['target_score']} | publishable={gate['publishable']}")
    print(f"  Status: {new_state['judge_result']['approval_status']} | revisions: {state['revision_count']}")
    if not gate["publishable"]:
        print("  Blocking reasons:", gate["blocking_reasons"])
    print(f"{'='*50}")
    return new_state


def metrics_targets_should_continue(state: MetricsTargetsState) -> str:
    return route_after_judge_generic(state, target_score=TARGET_SCORE, revise_label="reviser")




## Build and compile Metrics and Targets graph


In [50]:
# ── BUILD AND COMPILE METRICS AND TARGETS GRAPH ─────────────
metrics_targets_builder = StateGraph(MetricsTargetsState)
metrics_targets_builder.add_node("writer", metrics_targets_writer_node)
metrics_targets_builder.add_node("judge", metrics_targets_judge_node)
metrics_targets_builder.add_node("reviser", metrics_targets_reviser_node)
metrics_targets_builder.add_node("finalize", metrics_targets_finalize_node)
metrics_targets_builder.add_edge(START, "writer")
metrics_targets_builder.add_edge("writer", "judge")
metrics_targets_builder.add_conditional_edges("judge", metrics_targets_should_continue, {"finalize": "finalize", "reviser": "reviser"})
metrics_targets_builder.add_edge("reviser", "judge")
metrics_targets_builder.add_edge("finalize", END)
metrics_targets_graph = metrics_targets_builder.compile()
print("Metrics and Targets graph compiled")

Metrics and Targets graph compiled


## Run Metrics and Targets generation


In [51]:
# ── RUN METRICS AND TARGETS SECTION ─────────────────────────
metrics_targets_initial_state: MetricsTargetsState = {
    "bank_name": bank_name,
    "evidence": metrics_targets_evidence,
    "draft": "",
    "judge_result": {},
    "revision_count": 0,
    "max_revisions": 3,
    "status": "drafting",
    "final_section": "",
    "token_usage": {},
}
print(f"Starting Metrics and Targets generation for: {bank_name}\n")
metrics_targets_result = run_section_with_floor(metrics_targets_graph, metrics_targets_initial_state, label="metrics_targets")

Starting Metrics and Targets generation for: Eurolux Universal Bank AG

Metrics & Targets writer prompt approx tokens: 12848

METRICS & TARGETS WRITER (initial draft)
Draft length: 2100 words

METRICS & TARGETS JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS
{
  "overall_score": 8,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 7,
  "specificity_score": 8,
  "hallucination_risk": "low",
  "approval_status": "approved_with_limitations",
  "approved": true,
  "checklist": {
    "required_structure_present": true,
    "overview_present": true,
    "report_style_structure_present": true,
    "emirates_style_metrics_flow_present": true,
    "operational_ghg_handled_according_to_availability": true,
    "financed_emissions_used_correctly": true,
    "portfolio_intensity_used_correctly": true,
    "financial_metrics_used_correctly": true,
    "high_carbon_and_fossil_exposure_used_if_available": true,
    "green_finance_capex_opex_used_if_available": true,
    "targets_handled_accordin

## Print and save Metrics and Targets output


In [52]:
# ── METRICS AND TARGETS OUTPUT ──────────────────────────────
print("\n" + "="*60)
print("FINAL METRICS AND TARGETS JUDGE RESULT")
print("="*60)
print(json.dumps(metrics_targets_result["judge_result"], indent=2, ensure_ascii=False))
print("\n" + "="*60)
print("METRICS AND TARGETS SECTION")
print("="*60)
print(metrics_targets_result["final_section"])

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
with open(output_dir / "metrics_targets_BANK01.md", "w", encoding="utf-8") as f:
    f.write(metrics_targets_result["final_section"])
with open(output_dir / "metrics_targets_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id": "BANK01",
        "bank_name": bank_name,
        "section": "metrics_targets",
        "status": metrics_targets_result["status"],
        "approval_status": metrics_targets_result["judge_result"].get("approval_status"),
        "final_score": metrics_targets_result["judge_result"].get("overall_score"),
        "revisions": metrics_targets_result["revision_count"],
        "approved": metrics_targets_result["judge_result"].get("approved"),
        "checklist": metrics_targets_result["judge_result"].get("checklist"),
        "issues": metrics_targets_result["judge_result"].get("main_issues"),
        "available_evidence_omitted": metrics_targets_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": metrics_targets_result["judge_result"].get("unsupported_claims"),
        "material_unsupported_claims": metrics_targets_result["judge_result"].get("material_unsupported_claims"),
        "minor_wording_to_tighten": metrics_targets_result["judge_result"].get("minor_wording_to_tighten"),
        "correctly_disclosed_evidence_boundaries": metrics_targets_result["judge_result"].get("correctly_disclosed_evidence_boundaries"),
        "raw_metrics_targets_payload_path": str(RAW_METRICS_TARGETS_PATH),
        "compact_metrics_targets_evidence_path": str(COMPACT_METRICS_TARGETS_PATH),
    }, f, indent=2, ensure_ascii=False)
print("\nSaved to outputs/metrics_targets_BANK01.md")
print(f"Raw Metrics & Targets payload saved to: {RAW_METRICS_TARGETS_PATH}")
print(f"Compact Metrics & Targets evidence saved to: {COMPACT_METRICS_TARGETS_PATH}")


FINAL METRICS AND TARGETS JUDGE RESULT
{
  "overall_score": 8,
  "evidence_support_score": 9,
  "ifrs_alignment_score": 7,
  "specificity_score": 8,
  "hallucination_risk": "low",
  "approval_status": "needs_human_review",
  "approved": false,
  "checklist": {
    "required_structure_present": true,
    "overview_present": true,
    "report_style_structure_present": true,
    "emirates_style_metrics_flow_present": true,
    "operational_ghg_handled_according_to_availability": true,
    "financed_emissions_used_correctly": true,
    "portfolio_intensity_used_correctly": true,
    "financial_metrics_used_correctly": true,
    "high_carbon_and_fossil_exposure_used_if_available": true,
    "green_finance_capex_opex_used_if_available": true,
    "targets_handled_according_to_availability": true,
    "methodology_boundaries_handled_correctly": true,
    "evidence_boundaries_present": true,
    "no_visible_ifrs_refs_or_internal_refs": true,
    "no_unsupported_strong_claims": false
  },
  "a

# General Requirements section

This report-level section is generated after the four core sections so it can reflect the overall report scope, structure, standards, evidence boundaries and methodology limitations.


## General Requirements evidence builder


In [53]:
# ── GENERAL REQUIREMENTS EVIDENCE BUILDER ───────────────────
# This section uses the four section payloads already loaded in the notebook:
# governance_payload, strategy_payload, risk_management_payload, metrics_targets_payload.
#
# It does not invent compliance, assurance, materiality or reporting scope.
# It builds a compact evidence package for a report-level General Requirements section.

def _gr_present(value) -> bool:
    if value is None:
        return False
    if isinstance(value, str):
        return value.strip().lower() not in {"", "none", "null", "nan"}
    return True


def _gr_get_bank_profile() -> dict:
    sources = [
        globals().get("governance_payload", {}),
        globals().get("strategy_payload", {}),
        globals().get("risk_management_payload", {}),
        globals().get("metrics_targets_payload", {}),
    ]

    bank = {}
    for src in sources:
        if isinstance(src, dict) and isinstance(src.get("bank"), dict):
            bank.update({k: v for k, v in src["bank"].items() if _gr_present(v)})

    return {
        "bank_id": bank.get("bank_id"),
        "bank_name": bank.get("bank_name") or bank.get("name"),
        "country": bank.get("country"),
        "reporting_year": bank.get("reporting_year"),
        "reporting_currency": bank.get("reporting_currency"),
        "regulatory_regime": bank.get("regulatory_regime"),
        "total_assets_meur": bank.get("total_assets_meur"),
        "total_loans_meur": bank.get("total_loans_meur"),
    }


def _gr_collect_top_level_keys() -> dict:
    return {
        "governance_payload_keys": list(globals().get("governance_payload", {}).keys()) if isinstance(globals().get("governance_payload", {}), dict) else [],
        "strategy_payload_keys": list(globals().get("strategy_payload", {}).keys()) if isinstance(globals().get("strategy_payload", {}), dict) else [],
        "risk_management_payload_keys": list(globals().get("risk_management_payload", {}).keys()) if isinstance(globals().get("risk_management_payload", {}), dict) else [],
        "metrics_targets_payload_keys": list(globals().get("metrics_targets_payload", {}).keys()) if isinstance(globals().get("metrics_targets_payload", {}), dict) else [],
    }


def _gr_safe_get(path: list[str], source: dict, default=None):
    cur = source
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def build_general_requirements_evidence() -> dict:
    gov = globals().get("governance_payload", {}) if isinstance(globals().get("governance_payload", {}), dict) else {}
    strat = globals().get("strategy_payload", {}) if isinstance(globals().get("strategy_payload", {}), dict) else {}
    rm = globals().get("risk_management_payload", {}) if isinstance(globals().get("risk_management_payload", {}), dict) else {}
    mt = globals().get("metrics_targets_payload", {}) if isinstance(globals().get("metrics_targets_payload", {}), dict) else {}

    bank = _gr_get_bank_profile()

    # Pull known evidence groups when present.
    reporting_basis = {}
    for src in [gov, strat, rm, mt]:
        for key in ["reporting_basis", "basis_of_preparation", "general_requirements", "reporting_scope"]:
            if isinstance(src.get(key), dict):
                reporting_basis.update(src[key])

    materiality = {}
    for src in [strat, gov, rm, mt]:
        for key in ["materiality_assessment", "material_topics", "materiality"]:
            if isinstance(src.get(key), dict):
                materiality.update(src[key])
            elif isinstance(src.get(key), list):
                materiality[key] = src[key]

    sources_of_guidance = []
    for src in [gov, strat, rm, mt]:
        for key in ["sources_of_guidance", "standards", "frameworks", "reporting_frameworks"]:
            val = src.get(key)
            if isinstance(val, list):
                sources_of_guidance.extend(val)
            elif isinstance(val, str):
                sources_of_guidance.append(val)

    # Use section-level evidence objects if they exist from previous cells.
    generated_section_status = {}
    for var_names, label in [
        (["governance_result", "result"], "Governance"),
        (["strategy_result"], "Strategy"),
        (["risk_management_result"], "Risk management"),
        (["metrics_targets_result"], "Metrics and targets"),
    ]:
        res = {}
        for _vn in var_names:
            _cand = globals().get(_vn)
            if isinstance(_cand, dict) and (_cand.get("final_section") or _cand.get("draft") or _cand.get("judge_result")):
                res = _cand
                break
        if res:
            judge = res.get("judge_result", {}) or {}
            generated_section_status[label] = {
                "available": bool(res.get("final_section") or res.get("draft")),
                "approval_status": judge.get("approval_status"),
                "overall_score": judge.get("overall_score"),
                "approved": judge.get("approved"),
            }
        else:
            generated_section_status[label] = {"available": False}

    # Common report boundaries based on current project design.
    boundary_indicators = {
        "uses_synthetic_or_generated_data": True,
        "not_external_assurance_for_whole_report": True,
        "generated_sections_are_evidence_based": True,
        "general_requirements_generated_after_core_sections": True,
        "final_report_should_display_general_requirements_before_core_sections": True,
    }

    # Metrics / assurance boundaries.
    metrics_boundaries = {}
    if isinstance(globals().get("metrics_targets_evidence", {}), dict):
        metrics_boundaries = globals()["metrics_targets_evidence"].get("evidence_boundaries", {}) or {}

    # Strategy / scenario boundaries.
    strategy_boundaries = {}
    if isinstance(globals().get("strategy_evidence", {}), dict):
        strategy_boundaries = globals()["strategy_evidence"].get("evidence_boundaries", {}) or {}

    # Governance / assurance boundaries.
    governance_profile = {}
    if isinstance(globals().get("evidence", {}), dict):
        governance_profile = globals()["evidence"].get("availability_profile", {}) or {}

    evidence_package = {
        "bank": bank,
        "reporting_period": {
            "reporting_year": bank.get("reporting_year"),
            "period_start": reporting_basis.get("period_start"),
            "period_end": reporting_basis.get("period_end"),
            "financial_year": reporting_basis.get("financial_year") or bank.get("reporting_year"),
        },
        "reporting_entity_and_boundaries": {
            "reporting_entity": reporting_basis.get("reporting_entity") or bank.get("bank_name"),
            "country": bank.get("country"),
            "currency": reporting_basis.get("currency") or bank.get("reporting_currency"),
            "consolidation_boundary": reporting_basis.get("consolidation_boundary"),
            "subsidiaries_included": reporting_basis.get("subsidiaries_included"),
            "financial_statement_boundary_alignment": reporting_basis.get("financial_statement_boundary_alignment"),
        },
        "business_model_and_value_chain": {
            "business_model": strat.get("business_model"),
            "value_chain": strat.get("value_chain") or strat.get("value_chain_map"),
            "portfolio_summary": strat.get("portfolio_summary") or strat.get("portfolio_kpis"),
            "total_assets_meur": bank.get("total_assets_meur"),
            "total_loans_meur": bank.get("total_loans_meur"),
        },
        "sources_of_guidance": sorted({str(x) for x in sources_of_guidance if _gr_present(x)}),
        "materiality": materiality,
        "section_status": generated_section_status,
        "available_payload_keys": _gr_collect_top_level_keys(),
        "report_level_boundaries": boundary_indicators,
        "known_evidence_boundaries": {
            "governance_profile": governance_profile,
            "strategy_boundaries": strategy_boundaries,
            "metrics_boundaries": metrics_boundaries,
            "scope1_scope2_assurance_only": governance_profile.get("assurance_scope_is_scope1_scope2_only"),
            "operational_ghg_records_available": metrics_boundaries.get("operational_scope_1_2_3_emissions_available"),
            "methodology_records_available": metrics_boundaries.get("methodology_records_available"),
        },
        "disclosure_structure": [
            "General Requirements",
            "Governance",
            "Strategy",
            "Risk management",
            "Metrics and targets",
        ],
        "strict_boundaries": {
            "do_not_claim_full_ifrs_compliance_unless_statement_evidenced": True,
            "do_not_claim_external_assurance_of_whole_report": True,
            "do_not_claim_financed_emissions_assurance_if_scope_is_only_scope1_scope2": True,
            "do_not_use_word_payload_in_final_report": True,
            "do_not_invent_materiality_survey_or_stakeholders": True,
            "do_not_invent_subsidiaries_or_consolidation_scope": True,
            "do_not_invent_accounting_policy_alignment": True,
        },
    }

    return evidence_package


general_requirements_evidence = build_general_requirements_evidence()

print("General Requirements evidence package ready")
print(json.dumps({
    "bank": general_requirements_evidence.get("bank"),
    "section_status": general_requirements_evidence.get("section_status"),
    "sources_of_guidance": general_requirements_evidence.get("sources_of_guidance"),
    "known_boundaries": general_requirements_evidence.get("known_evidence_boundaries"),
}, indent=2, ensure_ascii=False))

General Requirements evidence package ready
{
  "bank": {
    "bank_id": "BANK01",
    "bank_name": "Eurolux Universal Bank AG",
    "country": "DE",
    "reporting_year": null,
    "reporting_currency": "EUR",
    "regulatory_regime": "CSRD",
    "total_assets_meur": 850000,
    "total_loans_meur": 31150.64
  },
  "section_status": {
    "Governance": {
      "available": true,
      "approval_status": "needs_human_review",
      "overall_score": 8,
      "approved": false
    },
    "Strategy": {
      "available": true,
      "approval_status": "needs_human_review",
      "overall_score": 8,
      "approved": false
    },
    "Risk management": {
      "available": true,
      "approval_status": "needs_human_review",
      "overall_score": 8,
      "approved": false
    },
    "Metrics and targets": {
      "available": true,
      "approval_status": "needs_human_review",
      "overall_score": 8,
      "approved": false
    }
  },
  "sources_of_guidance": [],
  "known_boundaries": 

## General Requirements state definition


In [54]:
# ── GENERAL REQUIREMENTS STATE DEFINITION ───────────────────
class GeneralRequirementsState(TypedDict):
    bank_name:        str
    evidence:         dict
    draft:            str
    judge_result:     dict
    revision_count:   int
    max_revisions:    int
    status:           Literal["drafting", "judging", "revising", "approved", "failed"]
    final_section:    str
    token_usage:      dict

## General Requirements requirements


In [55]:
# ── GENERAL REQUIREMENTS REQUIREMENTS ───────────────────────
GENERAL_REQUIREMENTS_REQUIREMENTS = """
STRICT GENERAL REQUIREMENTS DISCLOSURE REQUIREMENTS:

Presentation style:
- Write like the General Requirements chapter of a dedicated bank IFRS S1/S2 report.
- Use an Emirates NBD-style flow: approach to IFRS S1/S2, fair presentation, connected information, comparative information, timing and location, reporting entity/business model/value chain, sources of guidance, statement of compliance, and materiality.
- Keep the section evidence-based and transparent about boundaries.
- Do not use the word "payload" in the final report.
- Do not include markdown tables.

Final section title and headings must be exactly:
### General Requirements
#### Understanding the Bank’s approach towards implementing IFRS S1 and IFRS S2 requirements
#### Fair presentation
#### Connected information
#### Comparative information
#### Timing and location of disclosure
#### Reporting entity, business model and value chain
#### Sources of guidance
#### Statement of compliance
#### Materiality assessment

Content rules:
1. Understanding the Bank’s approach:
   - Explain that the report is structured around IFRS S1/S2-style climate and sustainability-related financial disclosure areas.
   - Describe the purpose as providing decision-useful information about sustainability-related and climate-related risks and opportunities.
   - Do not claim full legal compliance unless an explicit compliance statement is evidenced.
2. Fair presentation:
   - State the basis of report-style preparation in careful wording.
   - Use boundaries where the evidence is synthetic/generated or where source data is limited.
   - Do not claim the information is complete, neutral and free from material error unless evidenced.
3. Connected information:
   - Explain the connection between Governance, Strategy, Risk management and Metrics and targets.
   - Connect climate risk register, scenario analysis, financial metrics, targets and governance oversight where evidenced.
4. Comparative information:
   - Use comparative years or trends only where available.
   - Do not invent prior-year restatements.
5. Timing and location:
   - State reporting year and that the disclosure is part of the generated sustainability report.
   - Do not invent publication date, annual report integration or external filing location.
6. Reporting entity, business model and value chain:
   - Use bank name, country, currency, total assets/loans and business/value-chain evidence when available.
   - If consolidation/subsidiary scope is unavailable, state that boundary.
7. Sources of guidance:
   - Mention IFRS S1 and IFRS S2 as the primary disclosure framing.
   - Mention other frameworks only if present in evidence.
8. Statement of compliance:
   - If no explicit compliance statement exists, state alignment-style wording rather than full compliance.
   - Do not overclaim statutory adoption or jurisdictional endorsement.
9. Materiality assessment:
   - Use materiality evidence only when present.
   - If no detailed materiality survey/stakeholder evidence exists, state that the report uses the generated evidence base and climate-risk/material metrics to identify disclosure topics, without inventing a formal survey.

Boundary rules:
- Do not claim external assurance over the full report.
- Do not claim financed-emissions assurance if assurance scope is Scope 1 and Scope 2 only.
- Do not invent materiality survey participants.
- Do not invent subsidiaries, consolidation boundaries or value-chain details.
- Do not invent accounting policy alignment with financial statements.
- Do not show internal references, code names or file names.
""".strip()

print("General Requirements requirements ready — Emirates-style structure")

General Requirements requirements ready — Emirates-style structure


## General Requirements writer and judge prompts


In [56]:
# ── GENERAL REQUIREMENTS WRITER / JUDGE PROMPTS ─────────────
GENERAL_REQUIREMENTS_WRITER_SYSTEM = """
You are a senior sustainability reporting specialist writing the General Requirements section of a dedicated IFRS S1/S2-style climate and sustainability disclosure report for a commercial bank.

Write in a formal, third-person, publication-ready style similar to a dedicated bank IFRS S1/S2 report.

Evidence rules:
- Never mention that a section has failed, requires revision, is in scope, was assessed, or was generated by a system.
- Use this exact public wording when discussing estimates: "Parts of the report rely on modelled estimates, internally generated analytical outputs and available source data, particularly for climate scenario analysis and financed emissions."
- Do not repeat the modelled-estimates sentence.
- Do not state that Metrics and targets requires revision or validation.
- In Materiality assessment, describe the approach as risk- and portfolio-driven screening using available risk register, exposure mapping and scenario outputs. Do not imply a formal stakeholder materiality process unless evidenced.
- For own operations, use generic wording such as "physical disruption" or "physical chronic risk"; do not name hazards such as heat, flooding or wildfire for own operations unless hazard-level evidence is provided.

- Do not say Governance disclosures are limited if the Governance section exists; instead refer to Governance evidence boundaries.
- Do not describe the final report as synthetic, generated, AI-produced, or built from generated sections/data.
- Do not mention synthetic data, generated data, notebooks, payloads or model-building internals in the public section.
- If European/CSRD context is not explicitly evidenced, use softened wording such as 'European sustainability reporting environment' rather than claiming a specific applicable CSRD regime.
- Translate raw data field names into public report wording; avoid snake_case and key=value syntax in the final section.
- Never end the section mid-sentence; complete every required subsection, especially the final evidence-boundary subsection.
- Use the clean style guide only for writing style, tone, structure and formatting; never use it as factual evidence.
- Use only the General Requirements evidence package supplied by the user prompt.
- Do not invent reporting scope, subsidiaries, materiality process, assurance, compliance statement, publication location or accounting policy alignment.
- If evidence is missing, state the boundary carefully.
- Use alignment-style wording unless a full compliance statement is explicitly evidenced.
- Do not use the word "payload".
- Do not use markdown tables.
- Do not include visible paragraph references, internal IDs, file names or code references.

Additional precision rules (avoid the recurring overclaims):
- In the materiality subsection, do not mention stakeholder surveys, materiality surveys, or stakeholder-engagement processes at all — not even to say they were not performed. State only that topic selection was based on the available evidence (risk register, value-chain mapping, portfolio exposures) and that no documented IFRS S1 materiality process artefact was available.
- In the comparative-information subsection, describe which comparatives are available — the governance trend metrics (ESG committee meetings, board climate expertise %, climate-linked remuneration %, climate-on-agenda %, board meeting frequency) — but do not say the comparative figures are "provided" or "presented" in this section; they appear in the Governance and Metrics sections. State that comparatives for portfolio-exposure and climate metrics are not available.
- Do not state or imply that governance evidence or trend metrics are unavailable for this report. Governance core metrics, trend metrics and board decisions ARE available and are presented in the Governance section; acknowledge their existence rather than implying the governance evidence set is absent.
- In the sources-of-guidance subsection, explicitly state that no other framework or guidance (such as SASB or other standards) is documented as having been applied or considered beyond IFRS S1 and IFRS S2, rather than leaving it ambiguous.

- Do not assert a specific reporting period or that the report "corresponds to a single financial year" when the reporting-period fields are not populated in the evidence; state that the specific reporting period is not specified in the available evidence.
- Do not use IFRS faithful-representation or audit-style phrases such as "fair presentation", "faithful representation" or "free from material error" — even to qualify or deny them. Describe the approach in plain terms (transparent, balanced, evidence-based) without these regulated phrases.

- The available evidence provides only a country code (DE). State this as "the country code recorded in the available evidence is DE (Germany)"; do not assert the country of incorporation or the country of operation, as the evidence does not distinguish these.

REPORTING VOICE (this overrides any other wording guidance, including wording in the evidence): Write strictly as the Bank's own published disclosure, in the Bank's voice, and state facts directly. The following are BANNED from the final section: "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", "based on available evidence/documentation", and any phrasing that narrates what "the documentation" or "the evidence" does or does not show. Write "The Board met eight times in 2024", never "Available documentation indicates the Board met eight times". For genuine gaps, write the Bank's position: "the Bank does not currently disclose ...", "... is outside the scope of this report", or "... has not yet been quantified".

- State that this report covers the financial year ended 31 December 2024 (consistent with the metrics and targets, which are reported for 2024). Do not state that the reporting period is unspecified or not determined.

Output rules:
- Return only the complete General Requirements section.
- Keep exactly the nine required subsections.
""".strip()


def build_general_requirements_writer_prompt(evidence: dict, judge_feedback: str | None = None) -> str:
    feedback_block = ""
    if judge_feedback:
        feedback_block = f"""
JUDGE FEEDBACK TO ADDRESS:
{judge_feedback}

REVISION POLICY:
- Fix omissions using available evidence.
- Preserve accurate evidence-boundary statements.
- Never invent evidence to satisfy the judge.
""".strip()

    return f"""
{GENERAL_REQUIREMENTS_REQUIREMENTS}

WRITING STYLE GUIDE — STYLE ONLY, NOT EVIDENCE:
{build_style_context("general")}

STYLE APPLICATION RULES:
- Use the style guide only for tone, flow, structure and formatting.
- Use General Requirements evidence for facts.
- Use IFRS requirements only to decide what topics to cover.
- Do not copy reference-report wording or mention forbidden reference terms.

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("general_requirements")}

REQUIREMENT-TO-EVIDENCE POLICY:
- Use IFRS requirements to decide which General Requirements topics must be addressed.
- Use the evidence package to write bank-specific facts.
- If an IFRS requirement is relevant but evidence is unavailable, disclose the evidence boundary instead of inventing compliance, assurance, materiality, consolidation scope or publication-location details.
- Treat core_standard items as mandatory disclosure guidance.
- Treat industry_guidance items as banking-specific application guidance.
- Do not display IFRS paragraph references or requirement IDs in the final report.

GENERAL REQUIREMENTS EVIDENCE PACKAGE — USE ONLY THIS DATA:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

SECTION BLUEPRINT:
### General Requirements

#### Understanding the Bank’s approach towards implementing IFRS S1 and IFRS S2 requirements
Explain the IFRS S1/S2-style reporting approach, the focus on sustainability-related and climate-related financial disclosures, and the report's core structure. Do not describe the report as synthetic, generated, or AI-produced.

#### Fair presentation
Explain careful preparation boundaries and evidence-based limitations without claiming full completeness, external assurance or absence of material error unless evidenced.

#### Connected information
Explain how governance, strategy, risk management, metrics and targets connect across the report.

#### Comparative information
Use available trend/comparative evidence. If limited, state that comparative information is provided where available.

#### Timing and location of disclosure
State the reporting year and generated report context. Do not invent publication date or filing location.

#### Reporting entity, business model and value chain
Describe bank profile and business/value-chain information where available. State boundaries where consolidation or subsidiary scope is unavailable.

#### Sources of guidance
Mention IFRS S1 and IFRS S2 as the primary framing. Do not state that no other external frameworks exist if target records mention SBTi, NZBA or validation bodies. Clarify that no other report-preparation standards or disclosure frameworks are evidenced, while target records may reference climate-target frameworks where explicitly provided.

#### Statement of compliance
Use careful alignment wording unless explicit compliance evidence is available.

#### Materiality assessment
Use materiality evidence only if present. Otherwise state that disclosure topics are derived from available climate-risk, scenario, governance and metrics evidence.

IMPORTANT:

PUBLIC WORDING RULES:
- Never write that the report is prepared using synthetic or generated sections/data. Use: "Parts of the report rely on modelled estimates, internally generated analytical outputs and available source data."
- Do not write that Governance disclosures are limited if a Governance section is generated. Use: "Governance disclosures are based on the governance evidence currently available and are subject to the evidence boundaries described in the Governance section."
- Avoid duplicated wording such as "modelled estimates ... and modelled estimates"; write it once cleanly.
- Do not write "synthetic data" or "generated data" in the final report. Use "modelled estimates", "internally generated analytical outputs" or "available source data" where needed.
- Do not claim that a specific CSRD regime applies unless the evidence explicitly supports it. Prefer "European sustainability reporting environment" or "sustainability reporting requirements that may apply to banks operating in Europe".
- Do not expose internal project language, file names, code references or data-field names.

- Do not use the word "payload".
- Do not claim whole-report assurance.
- Do not claim financed emissions assurance when assurance is limited to Scope 1 and Scope 2.
- Do not invent materiality surveys, stakeholders, subsidiaries or accounting policy alignment.

{feedback_block}

Write the complete General Requirements section now.
Return only the final General Requirements section.
""".strip()


GENERAL_REQUIREMENTS_JUDGE_SYSTEM = """
You are a strict sustainability-reporting judge for an IFRS S1/S2-style bank General Requirements section.

You evaluate whether the section is:
- supported by the evidence package;
- structured like a dedicated bank IFRS S1/S2 General Requirements chapter;
- transparent about reporting scope and evidence boundaries;
- free from invented compliance, assurance, materiality, consolidation or publication claims.
- free from reference-bank identity leakage.

Return valid JSON only.
All checklist values must be JSON booleans true or false, not strings.
""".strip()


def build_general_requirements_judge_prompt(draft: str, evidence: dict, deterministic_checks: dict | None = None) -> str:
    deterministic_checks = deterministic_checks or {}

    return f"""
Evaluate the General Requirements draft against the evidence package and deterministic pre-checks.

DRAFT:
{draft}

REPORTING VOICE: The section must read as the Bank's own published disclosure. Treat as a material wording defect any report-machinery language - "available evidence", "available documentation", "documentation reviewed", "source data", "internally generated analytical outputs", "evidence package", or phrasing that narrates what "the documentation"/"the evidence" does or does not show. Facts must be stated directly; genuine gaps framed as the Bank's position ("the Bank does not currently disclose ...", "... is outside the scope of this report", "... has not yet been quantified"). List any such phrasing in material_unsupported_claims so it is corrected.

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic_checks, indent=2, ensure_ascii=False)}

SECTION IFRS S1/S2 + COMMERCIAL BANKS REQUIREMENTS PACK:
{build_ifrs_requirements_text("general_requirements")}

GENERAL REQUIREMENTS EVIDENCE PACKAGE:
{json.dumps(evidence, indent=2, ensure_ascii=False)}

EVALUATION PRINCIPLES:
0. Check the IFRS requirements pack and the ifrs_requirement_coverage pre-check. Penalise missing mandatory core IFRS topics unless the draft clearly discloses an evidence boundary.
1. Penalise invented compliance, assurance, publication location, consolidation scope, subsidiaries, materiality survey or accounting policy alignment.
2. Penalise the use of "complete, neutral and free from material error" unless explicitly evidenced.
3. Penalise full IFRS compliance claims unless explicit compliance evidence exists.
4. Reward careful alignment wording and transparent evidence boundaries.
5. Verify the Emirates-style General Requirements structure with all nine required headings.
6. Do not penalise honest boundaries where evidence is missing.

SCORING:
Score on completeness RELATIVE TO AVAILABLE EVIDENCE, not on how much data the bank happens to hold. Correctly disclosed evidence boundaries (genuine unavailability, accurately stated) are good IFRS practice and MUST NOT reduce the score; the number of disclosed boundaries is irrelevant to the score. Penalise ONLY: unsupported or overclaimed statements, omitted AVAILABLE evidence, factual inconsistencies with the evidence, hallucinations, and structural or deterministic failures.
- 9-10: Materially complete relative to available evidence, report-ready structure, no unsupported or overclaimed statements, no omitted available evidence, and all genuine data gaps correctly disclosed as boundaries.
- 8: As 9-10 but with one minor evidence-use issue or one slightly loose statement to tighten.
- 7: Usable but contains an unsupported/overclaimed statement, omits available evidence, or has a structural/keyword gap.
- 6: Revision required: available evidence omitted, unsupported wording remains, factual inconsistency with evidence, or required structure/boundary keywords missing.
- 5 or below: Major evidence failure, hallucination, contradiction, or missing core subsection.

When listing omitted evidence, separate MATERIAL omissions (which must be incorporated) from OPTIONAL ones (acceptable to leave out); never mark internal record IDs or optional traceability codes as material. Likewise, separate unsupported claims into material_unsupported_claims (misrepresent the evidence or overstate scope/assurance/compliance — must fix) and minor_wording_to_tighten (defensible phrasing such as 'covers' vs 'includes' or singular/plural that does not misrepresent the evidence — an acceptable minor limitation). Do not list trivial wording as material. Return valid JSON only:
{{
  "overall_score": <integer 1-10>,
  "evidence_support_score": <integer 1-10>,
  "ifrs_alignment_score": <integer 1-10>,
  "specificity_score": <integer 1-10>,
  "hallucination_risk": "<low|medium|high>",
  "approval_status": "<approved|approved_with_limitations|revision_required|rejected>",
  "approved": <true if approved or approved_with_limitations, otherwise false>,
  "checklist": {{
    "required_structure_present": <true/false>,
    "report_style_structure_present": <true/false>,
    "approach_to_ifrs_s1_s2_present": <true/false>,
    "fair_presentation_boundary_correct": <true/false>,
    "connected_information_present": <true/false>,
    "comparative_information_handled_correctly": <true/false>,
    "timing_and_location_handled_correctly": <true/false>,
    "reporting_entity_business_model_value_chain_present": <true/false>,
    "sources_of_guidance_handled_correctly": <true/false>,
    "statement_of_compliance_not_overclaimed": <true/false>,
    "materiality_not_invented": <true/false>,
    "no_whole_report_assurance_overclaim": <true/false>,
    "no_financed_emissions_assurance_overclaim": <true/false>,
    "no_visible_internal_refs_or_file_names": <true/false>
  }},
  "available_evidence_omitted": [<specific available evidence omitted>],
  "material_evidence_omitted": [<omitted available evidence whose absence materially weakens the disclosure; MUST be incorporated as report prose>],
  "optional_evidence_omitted": [<omitted available evidence that is acceptable to leave out; do not force inclusion>],
  "unsupported_claims": [<specific unsupported claims>],
  "material_unsupported_claims": [<claims that misrepresent the evidence, assert unevidenced facts, or overstate scope/assurance/compliance; these MUST be removed or reframed>],
  "minor_wording_to_tighten": [<defensible phrasing that could be tightened but does NOT misrepresent the evidence; acceptable as a minor limitation>],
  "correctly_disclosed_evidence_boundaries": [<accurate boundary statements>],
  "main_issues": [<specific issues>],
  "required_fixes": [<actionable evidence-aware fixes>]
}}
""".strip()

print("General Requirements prompts ready — Emirates-style structure")

General Requirements prompts ready — Emirates-style structure


## General Requirements LangGraph nodes and deterministic checks


In [57]:
# ── GENERAL REQUIREMENTS DETERMINISTIC CHECKS + NODES ───────
def run_general_requirements_deterministic_checks(draft: str, evidence: dict) -> dict:
    text = draft or ""
    text_l = text.lower()

    required_headings = [
        "#### Understanding the Bank’s approach towards implementing IFRS S1 and IFRS S2 requirements",
        "#### Fair presentation",
        "#### Connected information",
        "#### Comparative information",
        "#### Timing and location of disclosure",
        "#### Reporting entity, business model and value chain",
        "#### Sources of guidance",
        "#### Statement of compliance",
        "#### Materiality assessment",
    ]

    failures = []
    warnings = []

    missing_headings = [h for h in required_headings if h not in text]
    if missing_headings:
        failures.append({"check": "missing_required_headings", "details": missing_headings})

    forbidden_terms = ["payload", ".json", "notebook", "cell_", "outputs/"]
    found_forbidden = [term for term in forbidden_terms if term in text_l]
    if re.search(r"\bfile(name| path| reference| system)?\b", text_l):
        found_forbidden.append("file")
    if found_forbidden:
        failures.append({"check": "visible_internal_or_file_references", "details": found_forbidden})

    strong_compliance_patterns = [
        "fully complies with ifrs",
        "complies with ifrs s1 and ifrs s2",
        "in full compliance",
        "complete, neutral and accurate",
        "free from material error",
        "externally assured report",
        "assured sustainability report",
    ]
    found_strong = [p for p in strong_compliance_patterns if p in text_l]
    if found_strong:
        warnings.append({
            "check": "possible_compliance_or_assurance_overclaim",
            "details": found_strong,
        })

    if "financed emissions" in text_l and "assured" in text_l and "outside" not in text_l and "not" not in text_l:
        warnings.append({
            "check": "possible_financed_emissions_assurance_overclaim",
            "details": "Financed emissions appear near assurance language without a clear exclusion."
        })

    if "materiality assessment" in text_l and any(word in text_l for word in ["survey", "stakeholders", "interviews"]) and not evidence.get("materiality"):
        warnings.append({
            "check": "possible_materiality_process_invented",
            "details": "Materiality survey/stakeholder process language appears although detailed materiality evidence is limited."
        })

    if "#### Fair presentation" in text:
        fair_section = text.split("#### Fair presentation", 1)[1].split("#### Connected information", 1)[0].lower()
        if "boundary" not in fair_section and "limitation" not in fair_section and "source data" not in fair_section and "available" not in fair_section:
            warnings.append({
                "check": "fair_presentation_boundary_may_be_weak",
                "details": "Fair presentation subsection may need clearer evidence-boundary wording."
            })


    public_blockers = [
        "synthetic data",
        "generated data",
        "synthetic or generated data",
        "synthetic or generated sections",
        "generated sections",
        "assessed as requiring revision",
        "requires revision",
        "section is in scope",
    ]
    found_public_blockers = [p for p in public_blockers if p in text_l]
    if found_public_blockers:
        failures.append({"check": "general_public_blocker_terms", "details": found_public_blockers})

    if "modelled estimates, internally generated analytical outputs and available source data is used" in text_l:
        failures.append({"check": "awkward_modelled_estimates_grammar", "details": "Use plural verb: are used."})

    if re.search(r"own operations[^.\n]*(heat|flooding|wildfire)", text_l):
        warnings.append({
            "check": "own_operations_hazard_specificity_may_exceed_evidence",
            "details": "Use generic physical disruption wording unless hazard-level own-operations evidence exists."
        })

    style_leak = check_reference_identity_leak(text)
    if not style_leak.get("passed", True):
        failures.append({"check": "reference_identity_leak", "details": style_leak.get("matches", [])})

    completeness = check_section_completeness(text, "general_requirements")
    failures.extend(completeness.get("failures", []))
    warnings.extend(completeness.get("warnings", []))

    return {
        "failures": failures,
        "warnings": warnings,
        "failure_count": len(failures),
        "warning_count": len(warnings),
    }


def general_requirements_writer_node(state: GeneralRequirementsState) -> GeneralRequirementsState:
    is_revision = state["revision_count"] > 0
    feedback = None

    if is_revision:
        judge = state.get("judge_result", {})
        issues = judge.get("required_fixes", [])
        checklist = judge.get("checklist", {})
        false_items = [k for k, v in checklist.items() if v is False]
        feedback = (
            "REQUIRED FIXES:\n" +
            "\n".join(f"- {fix}" for fix in issues) +
            "\n\nFAILED CHECKLIST ITEMS:\n" +
            "\n".join(f"- {item}" for item in false_items)
        )

    prompt = build_general_requirements_writer_prompt(state["evidence"], judge_feedback=feedback)
    print("General Requirements writer prompt approx tokens:", round(len(prompt) / 4))

    draft = call_writer_llm(
        system_prompt=GENERAL_REQUIREMENTS_WRITER_SYSTEM,
        user_prompt=prompt,
    )

    print(f"\n{'='*50}")
    print(f"GENERAL REQUIREMENTS WRITER {'(revision ' + str(state['revision_count']) + ')' if is_revision else '(initial draft)'}")
    print(f"Draft length: {len(draft.split())} words")
    print(f"{'='*50}")

    return {**state, "draft": draft.strip(), "status": "judging"}


def general_requirements_judge_node(state: GeneralRequirementsState) -> GeneralRequirementsState:
    draft = state["draft"]
    deterministic_checks = run_general_requirements_deterministic_checks(draft, state["evidence"])
    deterministic_checks["ifrs_requirement_coverage"] = check_ifrs_requirement_coverage(draft, "general_requirements")

    prompt = build_general_requirements_judge_prompt(
        draft,
        state["evidence"],
        deterministic_checks=deterministic_checks,
    )

    judge_result = call_judge_llm_json(
        system_prompt=GENERAL_REQUIREMENTS_JUDGE_SYSTEM,
        user_prompt=prompt,
    )

    try:
        judge_result = normalize_json_booleans(judge_result)
    except NameError:
        pass

    judge_result.setdefault("approved", False)
    judge_result.setdefault(
        "approval_status",
        "approved" if judge_result.get("approved") else "revision_required",
    )
    judge_result.setdefault("main_issues", [])
    judge_result.setdefault("required_fixes", [])
    judge_result.setdefault("checklist", {})
    judge_result["deterministic_prechecks"] = deterministic_checks
    judge_result["_best_so_far"] = track_best(state, judge_result)  # non-regression: keep best draft across iterations

    print("\nGENERAL REQUIREMENTS JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS")
    print(json.dumps(judge_result, indent=2, ensure_ascii=False))

    status = "approved" if bool(judge_result.get("approved")) else "revising"
    return {**state, "judge_result": judge_result, "status": status}


GENERAL_REQUIREMENTS_REVISER_SYSTEM = """
You are a precise sustainability disclosure reviser.

Revise the existing General Requirements section using only:
- the supplied evidence package;
- the deterministic pre-checks; and
- the judge's required fixes.

Rules:
- Fix every judge issue and failed checklist item.
- Preserve correct evidence-boundary statements.
- Never invent evidence.
- Keep the exact nine-subsection Emirates-style General Requirements structure.
- Do not add visible IFRS paragraph references, internal refs, file names or code references.
- Return only the complete revised General Requirements section.
""".strip()


def general_requirements_reviser_node(state: GeneralRequirementsState) -> GeneralRequirementsState:
    if state["revision_count"] >= state["max_revisions"]:
        return {**state, "status": "failed", "final_section": state["draft"]}

    judge = state.get("judge_result", {})
    issues = judge.get("required_fixes", [])
    checklist = judge.get("checklist", {})
    deterministic = judge.get("deterministic_prechecks", {})
    false_items = [k for k, v in checklist.items() if v is False]

    revision_prompt = f"""
STRICT GENERAL REQUIREMENTS REQUIREMENTS:
{GENERAL_REQUIREMENTS_REQUIREMENTS}

EVIDENCE PACKAGE:
{json.dumps(state["evidence"], indent=2, ensure_ascii=False)}

DETERMINISTIC PRE-CHECKS:
{json.dumps(deterministic, indent=2, ensure_ascii=False)}

CURRENT DRAFT:
{state["draft"]}

JUDGE REQUIRED FIXES:
{json.dumps(issues, indent=2, ensure_ascii=False)}

FAILED CHECKLIST ITEMS:
{json.dumps(false_items, indent=2, ensure_ascii=False)}

Revise the current draft and return only the complete General Requirements section.
""".strip()

    revised_draft = call_reviser_llm(
        system_prompt=GENERAL_REQUIREMENTS_REVISER_SYSTEM,
        user_prompt=(revision_prompt + "\n\nPRIORITISED REVISION BRIEF (address every item using available evidence only):\n" + build_reviser_focus(judge)),
    )

    new_revision_count = state["revision_count"] + 1
    print(f"\nGeneral Requirements revised with GPT-5.1 | revision {new_revision_count}")

    return {
        **state,
        "draft": revised_draft.strip(),
        "revision_count": new_revision_count,
        "status": "judging",
    }


def general_requirements_finalize_node(state: GeneralRequirementsState) -> GeneralRequirementsState:
    new_state = finalize_state(state, target_score=TARGET_SCORE)
    gate = new_state["judge_result"]["finalize_gate"]
    print(f"\n{'='*50}")
    print("FINALIZED GENERAL REQUIREMENTS")
    print(f"  Score: {gate['score']}/{gate['target_score']} | publishable={gate['publishable']}")
    print(f"  Status: {new_state['judge_result']['approval_status']} | revisions: {state['revision_count']}")
    if not gate["publishable"]:
        print("  Blocking reasons:", gate["blocking_reasons"])
    print(f"{'='*50}")
    return new_state


def general_requirements_should_continue(state: GeneralRequirementsState) -> str:
    return route_after_judge_generic(state, target_score=TARGET_SCORE, revise_label="reviser")




## Build and compile General Requirements graph


In [58]:
# ── BUILD AND COMPILE GENERAL REQUIREMENTS GRAPH ────────────
general_requirements_builder = StateGraph(GeneralRequirementsState)

general_requirements_builder.add_node("writer",   general_requirements_writer_node)
general_requirements_builder.add_node("judge",    general_requirements_judge_node)
general_requirements_builder.add_node("reviser",  general_requirements_reviser_node)
general_requirements_builder.add_node("finalize", general_requirements_finalize_node)

general_requirements_builder.add_edge(START, "writer")
general_requirements_builder.add_edge("writer", "judge")
general_requirements_builder.add_conditional_edges(
    "judge",
    general_requirements_should_continue,
    {
        "finalize": "finalize",
        "reviser": "reviser",
    }
)
general_requirements_builder.add_edge("reviser", "judge")
general_requirements_builder.add_edge("finalize", END)

general_requirements_graph = general_requirements_builder.compile()
print("General Requirements graph compiled")

General Requirements graph compiled


## Run General Requirements generation


In [59]:
# ── RUN GENERAL REQUIREMENTS SECTION ────────────────────────
general_requirements_initial_state: GeneralRequirementsState = {
    "bank_name":      bank_name,
    "evidence":       general_requirements_evidence,
    "draft":          "",
    "judge_result":   {},
    "revision_count": 0,
    "max_revisions": 3,
    "status":         "drafting",
    "final_section":  "",
    "token_usage":    {},
}

print(f"Starting General Requirements generation for: {bank_name}\n")
general_requirements_result = run_section_with_floor(general_requirements_graph, general_requirements_initial_state, label="general_requirements")

Starting General Requirements generation for: Eurolux Universal Bank AG

General Requirements writer prompt approx tokens: 13710


GPT-5.1 writer: connection reset/timeout (ConnectionResetError); retrying attempt 2/6 in 1.3s...
GPT-5.1 writer: connection reset/timeout (ConnectionResetError); retrying attempt 3/6 in 2.6s...

GENERAL REQUIREMENTS WRITER (initial draft)
Draft length: 2647 words

GENERAL REQUIREMENTS JUDGE RESULT — GPT-5.2 + DATA-AWARE PRECHECKS
{
  "overall_score": 6,
  "evidence_support_score": 6,
  "ifrs_alignment_score": 8,
  "specificity_score": 7,
  "hallucination_risk": "medium",
  "approval_status": "revision_required",
  "approved": false,
  "checklist": {
    "required_structure_present": false,
    "report_style_structure_present": false,
    "approach_to_ifrs_s1_s2_present": true,
    "fair_presentation_boundary_correct": true,
    "connected_information_present": true,
    "comparative_information_handled_correctly": true,
    "timing_and_location_handled_correctly": true,
    "reporting_entity_business_model_value_chain_present": true,
    "sources_of_guidance_handled_correctly": false,


## Print and save General Requirements output


In [60]:
# ── GENERAL REQUIREMENTS OUTPUT ─────────────────────────────
print("\n" + "="*60)
print("FINAL GENERAL REQUIREMENTS JUDGE RESULT")
print("="*60)
print(json.dumps(general_requirements_result["judge_result"], indent=2, ensure_ascii=False))

print("\n" + "="*60)
print("GENERAL REQUIREMENTS SECTION")
print("="*60)
print(general_requirements_result["final_section"])

# Save outputs
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "general_requirements_BANK01.md", "w", encoding="utf-8") as f:
    f.write(general_requirements_result["final_section"])

with open(output_dir / "general_requirements_BANK01_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "bank_id":        "BANK01",
        "bank_name":      bank_name,
        "section":        "general_requirements",
        "status":         general_requirements_result["status"],
        "approval_status": general_requirements_result["judge_result"].get("approval_status"),
        "final_score":    general_requirements_result["judge_result"].get("overall_score"),
        "revisions":      general_requirements_result["revision_count"],
        "approved":       general_requirements_result["judge_result"].get("approved"),
        "checklist":      general_requirements_result["judge_result"].get("checklist"),
        "issues":         general_requirements_result["judge_result"].get("main_issues"),
        "available_evidence_omitted": general_requirements_result["judge_result"].get("available_evidence_omitted"),
        "unsupported_claims": general_requirements_result["judge_result"].get("unsupported_claims"),
        "material_unsupported_claims": general_requirements_result["judge_result"].get("material_unsupported_claims"),
        "minor_wording_to_tighten": general_requirements_result["judge_result"].get("minor_wording_to_tighten"),
    }, f, indent=2, ensure_ascii=False)

print("\nSaved to outputs/general_requirements_BANK01.md")


FINAL GENERAL REQUIREMENTS JUDGE RESULT
{
  "overall_score": 7,
  "evidence_support_score": 7,
  "ifrs_alignment_score": 8,
  "specificity_score": 6,
  "hallucination_risk": "medium",
  "approval_status": "needs_human_review",
  "approved": false,
  "checklist": {
    "required_structure_present": false,
    "report_style_structure_present": true,
    "approach_to_ifrs_s1_s2_present": true,
    "fair_presentation_boundary_correct": true,
    "connected_information_present": true,
    "comparative_information_handled_correctly": true,
    "timing_and_location_handled_correctly": true,
    "reporting_entity_business_model_value_chain_present": true,
    "sources_of_guidance_handled_correctly": false,
    "statement_of_compliance_not_overclaimed": true,
    "materiality_not_invented": true,
    "no_whole_report_assurance_overclaim": true,
    "no_financed_emissions_assurance_overclaim": true,
    "no_visible_internal_refs_or_file_names": true
  },
  "available_evidence_omitted": [
    "R

## Optional: one-click full pipeline runner

Use this only after all evidence builder cells and all graph compilation cells have been run. It runs missing sections and then assembles the full report.


In [61]:
# ── OPTIONAL ONE-CLICK FULL PIPELINE RUNNER ──────────────────
# Run only after all evidence packages and compiled graphs exist.
# This cell skips any section result already present in memory.
# After this patch, rerun affected truncated sections by deleting their result variables or restarting the kernel.

def run_full_generation_pipeline_if_needed():
    required_objects = [
        "graph",
        "strategy_graph",
        "risk_management_graph",
        "metrics_targets_graph",
        "general_requirements_graph",
        "evidence",
        "strategy_evidence",
        "risk_management_evidence",
        "metrics_targets_evidence",
        "general_requirements_evidence",
    ]
    missing = [name for name in required_objects if name not in globals()]
    if missing:
        raise RuntimeError(
            "Cannot run full pipeline yet. Run the setup/evidence/graph compile cells first. "
            "Missing: " + ", ".join(missing)
        )

    if "result" not in globals():
        print("Running Governance...")
        globals()["result"] = graph.invoke({
            "bank_name": bank_name,
            "evidence": evidence,
            "draft": "",
            "judge_result": {},
            "revision_count": 0,
            "max_revisions": 3,
            "status": "drafting",
            "final_section": "",
            "token_usage": {},
        })
    else:
        print("Governance already exists; skipping.")

    if "strategy_result" not in globals():
        print("Running Strategy...")
        globals()["strategy_result"] = strategy_graph.invoke({
            "bank_name": bank_name,
            "evidence": strategy_evidence,
            "draft": "",
            "judge_result": {},
            "revision_count": 0,
            "max_revisions": 3,
            "status": "drafting",
            "final_section": "",
            "token_usage": {},
        })
    else:
        print("Strategy already exists; skipping.")

    if "risk_management_result" not in globals():
        print("Running Risk Management...")
        globals()["risk_management_result"] = risk_management_graph.invoke({
            "bank_name": bank_name,
            "evidence": risk_management_evidence,
            "draft": "",
            "judge_result": {},
            "revision_count": 0,
            "max_revisions": 3,
            "status": "drafting",
            "final_section": "",
            "token_usage": {},
        })
    else:
        print("Risk Management already exists; skipping.")

    if "metrics_targets_result" not in globals():
        print("Running Metrics and Targets...")
        globals()["metrics_targets_result"] = metrics_targets_graph.invoke({
            "bank_name": bank_name,
            "evidence": metrics_targets_evidence,
            "draft": "",
            "judge_result": {},
            "revision_count": 0,
            "max_revisions": 3,
            "status": "drafting",
            "final_section": "",
            "token_usage": {},
        })
    else:
        print("Metrics and Targets already exists; skipping.")

    if "general_requirements_result" not in globals():
        print("Running General Requirements...")
        globals()["general_requirements_result"] = general_requirements_graph.invoke({
            "bank_name": bank_name,
            "evidence": general_requirements_evidence,
            "draft": "",
            "judge_result": {},
            "revision_count": 0,
            "max_revisions": 3,
            "status": "drafting",
            "final_section": "",
            "token_usage": {},
        })
    else:
        print("General Requirements already exists; skipping.")

    print("All section results are available. Now run the Full Report Assembly cell.")


# Uncomment to run:
# run_full_generation_pipeline_if_needed()

# Full report assembly

This final step assembles the generated sections into one public Markdown report and saves a separate metadata file with judge results and deterministic checks.


## Senior final patch notes

This version blocks full-report assembly if any section is failed, unapproved, missing judge metadata, or below the minimum section score. It also adds senior-level QA for internal wording, target-parameter placement, raw field syntax, Strategy/Metrics contradictions, and overbroad Risk Management language.

Recommended clean rerun order after restarting the kernel: General Requirements → Governance → Strategy → Risk Management → Metrics and Targets → Full report assembly → Optional final public wording QA.


In [62]:
# ── REPORT EXHIBITS: deterministic tables & charts from the compact evidence ──
# Built from the same evidence the writer used, so figures are exact, consistent
# with the prose, and generalise across banks. No LLM, no invented data.
import json as _ejson
from pathlib import Path as _EPath
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as _eplt

_FIG_DIR = _EPath("outputs") / "figures"
_FIG_DIR.mkdir(parents=True, exist_ok=True)
_PALETTE = ["#1f4e5f", "#2e7d7d", "#c08a3e", "#7a9e7e", "#9b3a3a"]

_CAT = {"transition_technology": "Transition - technology", "transition_market": "Transition - market",
        "transition_policy": "Transition - policy", "transition_reputational": "Transition - reputational",
        "physical_acute": "Physical - acute", "physical_chronic": "Physical - chronic"}
_HORIZON = {"short_0_2yr": "Short (0-2 yr)", "medium_2_5yr": "Medium (2-5 yr)", "long_5yr_plus": "Long (>5 yr)"}
_FRAMEWORK = {"SBTi_1.5C": "SBTi 1.5\u00b0C", "NZBA": "NZBA"}


def _exhibit_bank_id(bank_id=None):
    if bank_id:
        return bank_id
    if "_safe_bank_id" in globals():
        try:
            return _safe_bank_id()
        except Exception:
            pass
    return globals().get("bank_id", "BANK01")


def _load_compact_evidence(section_key, bank_id=None):
    p = _EPath("outputs") / f"compact_{section_key}_evidence_{_exhibit_bank_id(bank_id)}.json"
    try:
        return _ejson.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return {}


def _cap(text):
    """Caption that stays attached to its table/figure across page breaks."""
    return f"**{text}**\n{{: .exhibit-caption }}"


def _num(v, fmt="{:,.2f}", suffix=""):
    try:
        return fmt.format(float(v)) + suffix
    except Exception:
        return "\u2014"


def _governance_exhibits(ev):
    trend = [r for r in (ev.get("governance_trend") or []) if isinstance(r, dict)]
    g24 = ev.get("governance_2024") or {}
    years = {r.get("year"): dict(r) for r in trend}
    if g24 and 2024 not in years:
        years[2024] = {
            "year": 2024,
            "board_full_meeting_frequency": g24.get("board_full_meeting_frequency"),
            "esg_committee_meetings": g24.get("esg_committee_meetings_per_year"),
            "board_climate_expertise_pct": g24.get("board_climate_expertise_pct"),
            "climate_on_board_agenda_pct": g24.get("climate_on_board_agenda_pct"),
            "ceo_esg_compensation_pct": g24.get("ceo_esg_compensation_pct"),
            "all_exec_climate_remuneration_pct": g24.get("all_exec_climate_remuneration_pct"),
        }
    rows = [years[y] for y in sorted(years) if y is not None]
    if not rows:
        return ""
    yrs = [r["year"] for r in rows]
    hdr = "| Indicator | " + " | ".join(str(y) for y in yrs) + " |"
    sep = "|---|" + "".join([" ---: |"] * len(rows))

    def line(label, key, pct=False):
        cells = []
        for r in rows:
            v = r.get(key)
            cells.append("\u2014" if v is None else (f"{float(v):.1f}%" if pct else f"{v}"))
        return f"| {label} | " + " | ".join(cells) + " |"

    table = "\n".join([
        _cap("Table G1 \u2014 Climate governance indicators (three-year trend)"), "",
        hdr, sep,
        line("Full Board meetings", "board_full_meeting_frequency"),
        line("ESG & Sustainability Committee meetings", "esg_committee_meetings"),
        line("Board members with climate expertise", "board_climate_expertise_pct", True),
        line("Meetings with climate on the agenda", "climate_on_board_agenda_pct", True),
        line("CEO remuneration linked to ESG", "ceo_esg_compensation_pct", True),
        line("Executive remuneration linked to climate", "all_exec_climate_remuneration_pct", True),
    ])

    chart_md = ""
    try:
        series = [("Board climate expertise", "board_climate_expertise_pct"),
                  ("Climate on Board agenda", "climate_on_board_agenda_pct"),
                  ("CEO ESG-linked remuneration", "ceo_esg_compensation_pct"),
                  ("Exec climate-linked remuneration", "all_exec_climate_remuneration_pct")]
        fig, ax = _eplt.subplots(figsize=(6.4, 3.4))
        plotted = False
        for i, (lbl, key) in enumerate(series):
            ys = [r.get(key) for r in rows]
            if all(v is not None for v in ys):
                ax.plot(yrs, ys, marker="o", color=_PALETTE[i % len(_PALETTE)], label=lbl)
                plotted = True
        if plotted:
            ax.set_ylabel("Per cent"); ax.set_xticks(yrs)
            ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=7, frameon=False); ax.grid(True, alpha=0.3)
            for s in ("top", "right"): ax.spines[s].set_visible(False)
            fig.tight_layout()
            fig.savefig(_FIG_DIR / "governance_trend.png", dpi=150, bbox_inches="tight")
            chart_md = (_cap(f"Figure G1 \u2014 Climate governance indicators, {yrs[0]}\u2013{yrs[-1]}")
                        + "\n\n![Figure G1](figures/governance_trend.png)")
        _eplt.close(fig)
    except Exception:
        pass
    return "\n\n".join([x for x in [table, chart_md] if x])


def _risk_exhibits(ev):
    risks = ((ev.get("risk_identification_and_assessment") or {}).get("top_risks_by_financial_impact")
             or (ev.get("risk_mitigation_and_controls") or {}).get("material_risk_examples") or [])
    risks = [r for r in risks if isinstance(r, dict)]
    if not risks:
        return ""
    hdr = "| Risk | Category | Rating | Time horizon | Financial impact | Monitoring | ERM-integrated | Scenario-linked |"
    sep = "|---|---|---|---| ---: |---|:---:|:---:|"
    lines = [hdr, sep]
    for r in risks:
        name = r.get("risk_name", "\u2014")
        cat = _CAT.get(r.get("risk_category"), str(r.get("risk_category", "\u2014")).replace("_", " "))
        rating = str(r.get("risk_rating", "\u2014")).capitalize()
        hor = _HORIZON.get(r.get("time_horizon"), str(r.get("time_horizon", "\u2014")).replace("_", " "))
        fin = r.get("financial_impact_meur")
        fin = "\u2014" if fin in (None, "") else f"EUR {float(fin):,.1f}m"
        mon = str(r.get("monitoring_frequency", "\u2014")).replace("_", " ").capitalize()
        erm = "Yes" if r.get("erm_integrated_flag") else "No"
        scn = "Linked" if r.get("scenario_analysis_link") else "\u2014"
        lines.append(f"| {name} | {cat} | {rating} | {hor} | {fin} | {mon} | {erm} | {scn} |")
    return _cap("Table R1 \u2014 2024 climate risk register (summary)") + "\n\n" + "\n".join(lines)


def _metrics_exhibits(ev):
    cim = ev.get("cross_industry_metrics") or {}
    out = []
    if cim:
        m1 = [_cap("Table M1 \u2014 Key climate-related metrics (2024)"), "", "| Metric | Value |", "|---| ---: |"]
        fe = cim.get("financed_emissions_2024_tco2e")
        m1.append(f"| Financed emissions (loan portfolio) | {_num(fe)} tCO\u2082e |")
        ci = cim.get("carbon_intensity_2024_tco2e_per_meur")
        m1.append(f"| Portfolio carbon intensity | {_num(ci)} tCO\u2082e per EUR m lending |")
        m1.append(f"| High-carbon sector exposure | EUR {_num(cim.get('high_carbon_sector_exposure_meur'))}m ({_num(cim.get('high_carbon_sector_exposure_pct'),'{:.1f}')}% of loans) |")
        m1.append(f"| Fossil-fuel exposure | EUR {_num(cim.get('fossil_fuel_exposure_meur'))}m ({_num(cim.get('fossil_fuel_exposure_pct'),'{:.1f}')}% of loans) |")
        m1.append(f"| Green loans | EUR {_num(cim.get('green_loans_meur_2024'))}m ({_num(cim.get('green_loans_pct_2024'),'{:.1f}')}% of loans) |")
        m1.append(f"| Climate-related capital expenditure | EUR {_num(cim.get('climate_capex_2024_meur'))}m |")
        m1.append(f"| Climate-related operating expenditure | EUR {_num(cim.get('climate_opex_2024_meur'))}m |")
        m1.append(f"| Total loans | EUR {_num(cim.get('total_loans_meur'))}m |")
        m1.append(f"| Total assets | EUR {_num(cim.get('total_assets_meur'))}m |")
        out.append("\n".join(m1))

    tgts = [t for t in ((ev.get("targets_and_progress") or {}).get("targets") or []) if isinstance(t, dict)]
    if tgts:
        t2 = [_cap("Table M2 \u2014 Climate targets (as recorded in internal target records)"), "",
              "| Target | Scope | Baseline | Target year | Reduction | Framework | 2024 progress (internal, unverified) |",
              "|---|---|---|---|---|---| ---: |"]
        for t in tgts:
            ttype = str(t.get("target_type", "\u2014")).replace("_", " ").capitalize()
            scope = str(t.get("scope", "\u2014")).replace("_", " ").replace("scope", "Scope")
            base = f"{_num(t.get('baseline_value'),'{:,.0f}')} ({t.get('baseline_year','\u2014')})"
            ty = t.get("target_year", "\u2014")
            red = f"{t.get('target_value_pct_reduction','\u2014')}%"
            fw = _FRAMEWORK.get(t.get("target_framework"), str(t.get("target_framework", "\u2014")))
            prog = t.get("target_progress_pct_2024")
            prog = "\u2014" if prog is None else f"{float(prog):.1f}%"
            t2.append(f"| {ttype} | {scope} | {base} | {ty} | {red} | {fw} | {prog} |")
        out.append("\n".join(t2))

    chart_md = ""
    try:
        labels = ["Green loans", "High-carbon", "Fossil-fuel"]
        vals = [cim.get("green_loans_pct_2024"), cim.get("high_carbon_sector_exposure_pct"), cim.get("fossil_fuel_exposure_pct")]
        if all(v is not None for v in vals):
            fig, ax = _eplt.subplots(figsize=(5.6, 3.2))
            bars = ax.bar(labels, [float(v) for v in vals], color=[_PALETTE[3], _PALETTE[2], _PALETTE[4]])
            ax.set_ylabel("Per cent of loan portfolio")
            for b, v in zip(bars, vals):
                ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f"{float(v):.1f}%", ha="center", va="bottom", fontsize=8)
            for s in ("top", "right"): ax.spines[s].set_visible(False)
            fig.tight_layout(); fig.savefig(_FIG_DIR / "portfolio_composition.png", dpi=150, bbox_inches="tight")
            chart_md = _cap("Figure M1 \u2014 Loan portfolio composition by climate-related category (2024)") + "\n\n![Figure M1](figures/portfolio_composition.png)"
        _eplt.close(fig)
    except Exception:
        pass
    if chart_md:
        out.append(chart_md)
    return "\n\n".join(out)


_EXHIBIT_BUILDERS = {
    "governance": _governance_exhibits,
    "risk_management": _risk_exhibits,
    "metrics_targets": _metrics_exhibits,
}


def build_section_exhibits(section_key, bank_id=None):
    """Return markdown (tables + chart refs) for a section, or '' if none apply."""
    builder = _EXHIBIT_BUILDERS.get(section_key)
    if not builder:
        return ""
    ev = _load_compact_evidence(section_key, bank_id)
    if not ev:
        return ""
    try:
        body = builder(ev)
    except Exception:
        return ""
    if not body:
        return ""
    return "#### Supporting data and exhibits\n\n" + body


In [63]:
# === Report appendices (deterministic, evidence-bounded) ===
import json as _json, glob as _glob

_ABBREV = {
    "CSRD": "Corporate Sustainability Reporting Directive",
    "ERM": "Enterprise Risk Management",
    "ESG": "Environmental, Social and Governance",
    "GHG": "Greenhouse gas",
    "ICAAP": "Internal Capital Adequacy Assessment Process",
    "IFRS S1": "IFRS S1 General Requirements for Disclosure of Sustainability-related Financial Information",
    "IFRS S2": "IFRS S2 Climate-related Disclosures",
    "ISAE 3000": "International Standard on Assurance Engagements 3000 (assurance of non-financial information)",
    "ISSB": "International Sustainability Standards Board",
    "KPI": "Key performance indicator",
    "NGFS": "Network for Greening the Financial System",
    "NZBA": "Net-Zero Banking Alliance",
    "PCAF": "Partnership for Carbon Accounting Financials",
    "SBTi": "Science Based Targets initiative",
    "Scope 1": "Direct greenhouse gas emissions from owned or controlled sources",
    "Scope 2": "Indirect greenhouse gas emissions from purchased energy",
    "Scope 3": "Other indirect greenhouse gas emissions across the value chain, including financed emissions",
    "TCFD": "Task Force on Climate-related Financial Disclosures",
    "tCO2e": "Tonnes of carbon dioxide equivalent",
    "UNEP FI": "United Nations Environment Programme Finance Initiative",
}
# frameworks/standards that may be referenced; included only if present in the report text
_FRAMEWORKS = {
    "IFRS S1": "IFRS S1 General Requirements for Disclosure of Sustainability-related Financial Information (ISSB).",
    "IFRS S2": "IFRS S2 Climate-related Disclosures (ISSB).",
    "TCFD": "Recommendations of the Task Force on Climate-related Financial Disclosures.",
    "ISAE 3000": "International Standard on Assurance Engagements 3000 (Revised).",
    "SBTi": "Science Based Targets initiative target-setting framework.",
    "NZBA": "Net-Zero Banking Alliance.",
    "NGFS": "Network for Greening the Financial System climate scenarios.",
    "PCAF": "Partnership for Carbon Accounting Financials Global GHG Accounting and Reporting Standard.",
    "GHG Protocol": "Greenhouse Gas Protocol Corporate Accounting and Reporting Standard.",
    "CSRD": "EU Corporate Sustainability Reporting Directive.",
    "UNEP FI": "United Nations Environment Programme Finance Initiative.",
}
_JURISDICTION = {"DE": "Germany", "FR": "France", "NL": "the Netherlands", "ES": "Spain", "IT": "Italy"}

# Map availability-flag keys (value False = genuine gap) to plain bank-voice limitation statements.
_LIMITATION_LABELS = {
    "operational_scope_1_2_3_emissions_available":
        "The Bank's operational Scope 1, Scope 2 and Scope 3 greenhouse gas emissions are not separately quantified in this report.",
    "financed_emissions_total_available":
        "A consolidated financed-emissions total is not reported.",
    "carbon_intensity_available":
        "Portfolio carbon-intensity metrics are not reported.",
    "green_finance_metrics_available":
        "Green-finance and sustainable-finance volume metrics are not separately reported.",
    "climate_capex_opex_available":
        "Climate-related capital and operating expenditure is not separately quantified.",
    "full_targets_available":
        "Full target attributes (baselines, interim milestones and gross or net status) are not disclosed for every target.",
    "formal_risk_appetite_thresholds_available":
        "Formal climate-related risk-appetite thresholds are not disclosed.",
    "formal_risk_policy_documents_available":
        "Standalone climate-risk policy documents are not separately disclosed.",
    "formal_controls_procedures_available":
        "Detailed formal control procedures, including individual control owners, are not disclosed.",
    "formal_escalation_thresholds_available":
        "Formal escalation thresholds are not disclosed.",
}
# Known disclosures the Bank does not currently provide (confirmed gaps; not fabricated into the body).
_KNOWN_GAPS = [
    "A breakdown of financed emissions by sector or asset class is not currently reported.",
    "A detailed description of the financed-emissions measurement methodology is not provided.",
    "A formal three-lines-of-defence assignment of climate-risk responsibilities is not described.",
]

def _load_boundaries(bank_id):
    flags = {}
    for sec in ("strategy", "risk_management", "metrics_targets"):
        for path in (f"outputs/compact_{sec}_evidence_{bank_id}.json",
                     f"/mnt/project/compact_{sec}_evidence_{bank_id}.json",
                     f"compact_{sec}_evidence_{bank_id}.json"):
            try:
                ev = _json.load(open(path)); break
            except Exception:
                ev = None
        if isinstance(ev, dict):
            eb = ev.get("evidence_boundaries") or {}
            if isinstance(eb, dict):
                flags.update(eb)
    return flags

def _derived_limitations(bank_id):
    flags = _load_boundaries(bank_id)
    out = []
    for k, v in flags.items():
        if k.endswith("_available") and v is False and k in _LIMITATION_LABELS:
            out.append(_LIMITATION_LABELS[k])
    # de-duplicate while preserving order, then add confirmed known gaps
    seen, ordered = set(), []
    for s in out + _KNOWN_GAPS:
        if s not in seen:
            seen.add(s); ordered.append(s)
    return ordered

def build_report_appendices(bank_id, bank_name="the Bank", reporting_year=2024, report_text=""):
    rt = (report_text or "")
    rl = rt.lower().replace("\u2082", "2")  # normalise CO2 subscript for matching
    parts = ["## Appendices"]

    # Appendix A - Abbreviations (only terms used in the report)
    used = [(k, v) for k, v in _ABBREV.items() if k.lower().replace("\u2082","2") in rl]
    if used:
        rows = "\n".join(f"| {k} | {v} |" for k, v in sorted(used))
        parts.append(
            "### Appendix A - Abbreviations and defined terms\n\n"
            "| Term | Definition |\n|---|---|\n" + rows
        )

    # Appendix B - Basis of preparation
    juris = _JURISDICTION.get((rl and None) or "", None)
    # jurisdiction is derived from data elsewhere; default to the reporting entity's stated country if present
    juris = _JURISDICTION.get("DE", "its home jurisdiction") if "germany" in rl or " ag" in (bank_name or "").lower() else "its home jurisdiction"
    currency = "euro (EUR)" if ("eur" in rl or "\u20ac" in rt) else "its functional currency"
    bop = [
        "### Appendix B - Basis of preparation",
        "",
        f"This report presents the sustainability-related and climate-related financial disclosures of "
        f"{bank_name} for the financial year ended 31 December {reporting_year}, prepared with reference to "
        f"IFRS S1 and IFRS S2. The reporting entity is {bank_name}, headquartered in {juris}. Monetary amounts "
        f"are presented in {currency} unless otherwise stated.",
        "",
        "Quantitative climate-related metrics, in particular forward-looking scenario outputs and financed "
        "emissions, represent management's best estimates and are subject to measurement uncertainty. Except "
        "where an assurance statement is expressly provided, the metrics in this report have not been externally "
        "audited or assured.",
        "",
        "**Methodology limitations.** The following matters are not addressed in this report for the current "
        "reporting year, and are areas the Bank intends to develop in future reporting cycles:",
        "",
    ]
    lims = _derived_limitations(bank_id)
    bop += [f"- {s}" for s in lims]
    parts.append("\n".join(bop))

    # Appendix C - Referenced information and frameworks (only those referenced)
    refs = [(k, v) for k, v in _FRAMEWORKS.items() if k.lower().replace("\u2082","2") in rl]
    if refs:
        rows = "\n".join(f"| {k} | {v} |" for k, v in refs)
        parts.append(
            "### Appendix C - Referenced information and frameworks\n\n"
            "This report refers to the following frameworks, standards and initiatives:\n\n"
            "| Reference | Description |\n|---|---|\n" + rows
        )

    # Appendix D - IFRS S1 / S2 disclosure index (content-area level)
    index_rows = [
        ("IFRS S1 - General requirements (governance, strategy, risk management, metrics and targets; "
         "connected information; judgements and uncertainties)", "General Requirements"),
        ("IFRS S2 - Governance", "Governance"),
        ("IFRS S2 - Strategy (business model and value chain, climate-related risks and opportunities, "
         "scenario analysis, financial effects)", "Strategy"),
        ("IFRS S2 - Risk management", "Risk management"),
        ("IFRS S2 - Metrics and targets (cross-industry metrics, greenhouse gas emissions, climate-related "
         "targets)", "Metrics and targets"),
        ("IFRS S2 - Industry-based metrics (commercial banks)",
         "Metrics and targets (financed-emissions and exposure metrics)"),
    ]
    rows = "\n".join(f"| {a} | {b} |" for a, b in index_rows)
    parts.append(
        "### Appendix D - IFRS S1 and IFRS S2 disclosure index\n\n"
        "The following index indicates the section of this report in which each IFRS S1 and IFRS S2 content "
        "area is addressed. Where a content area is addressed only in part, the relevant limitation is set out "
        "in the section concerned and summarised in Appendix B.\n\n"
        "| IFRS S1 / S2 content area | Where addressed in this report |\n|---|---|\n" + rows
    )

    return "\n\n".join(parts).strip()

print("build_report_appendices defined.")


build_report_appendices defined.


In [64]:
# ── FULL REPORT ASSEMBLY ────────────────────────────────────
# Senior final version:
# - assembles only approved section outputs;
# - blocks failed / unapproved sections;
# - runs completeness, public wording and senior QA checks before saving.

from datetime import datetime, timezone
import re
import json
from pathlib import Path


PUBLIC_SECTION_ORDER = [
    ("general_requirements", "General Requirements", "general_requirements_BANK01.md", "general_requirements_BANK01_meta.json"),
    ("governance", "Governance", "governance_BANK01.md", "governance_BANK01_meta.json"),
    ("strategy", "Strategy", "strategy_BANK01.md", "strategy_BANK01_meta.json"),
    ("risk_management", "Risk Management", "risk_management_BANK01.md", "risk_management_BANK01_meta.json"),
    ("metrics_targets", "Metrics and Targets", "metrics_targets_BANK01.md", "metrics_targets_BANK01_meta.json"),
]

SECTION_COMPLETENESS_KEYS = {
    "general_requirements": "general_requirements",
    "governance": "governance",
    "strategy": "strategy",
    "risk_management": "risk_management",
    "metrics_targets": "metrics_and_targets",
}

RESULT_VARIABLES = {
    "general_requirements": "general_requirements_result",
    "governance": "result",
    "strategy": "strategy_result",
    "risk_management": "risk_management_result",
    "metrics_targets": "metrics_targets_result",
}

REQUIRE_APPROVED_SECTIONS_FOR_FULL_REPORT = True
MIN_SECTION_SCORE_FOR_FULL_REPORT = 7


def _safe_bank_id() -> str:
    for candidate in [globals().get("bank_id"), globals().get("BANK_ID"), "BANK01"]:
        if candidate:
            return str(candidate)
    return "BANK01"


def _safe_bank_name() -> str:
    for candidate in [globals().get("bank_name"), globals().get("BANK_NAME")]:
        if candidate:
            return str(candidate)

    for evidence_name in ["evidence", "strategy_evidence", "risk_management_evidence", "metrics_targets_evidence", "general_requirements_evidence"]:
        ev = globals().get(evidence_name)
        if isinstance(ev, dict):
            bank = ev.get("bank", {})
            if isinstance(bank, dict):
                for key in ["bank_name", "name"]:
                    if bank.get(key):
                        return str(bank[key])

    return "BANK01"


def _safe_reporting_year() -> str:
    for candidate in [globals().get("reporting_year"), globals().get("REPORTING_YEAR")]:
        if candidate:
            return str(candidate)

    for evidence_name in ["evidence", "strategy_evidence", "risk_management_evidence", "metrics_targets_evidence", "general_requirements_evidence"]:
        ev = globals().get(evidence_name)
        if isinstance(ev, dict):
            if ev.get("reporting_year"):
                return str(ev["reporting_year"])
            bank = ev.get("bank", {})
            if isinstance(bank, dict) and bank.get("reporting_year"):
                return str(bank["reporting_year"])

    return "2024"


def _tidy_report_numbers(text: str) -> str:
    """Cosmetic only: round machine-precision decimals (3+ places) to 2 so figures
    like 35,973,167.6878965 render consistently as 35,973,167.69, and normalise the
    CO2 notation to a subscript. Does not change values used in calculations - it only
    formats the rendered report text."""
    if not isinstance(text, str) or not text:
        return text

    def _round(m):
        intpart = m.group(1).replace(",", "")
        dec = m.group(2)
        try:
            val = round(float(f"{intpart}.{dec}"), 2)
        except ValueError:
            return m.group(0)
        return f"{val:,.2f}"

    text = re.sub(r"(\d[\d,]*)\.(\d{3,})", _round, text)
    text = text.replace("CO2", "CO\u2082")  # normalise tCO2e / CO2e to subscript
    return text


def _read_json_if_exists(path: Path) -> dict:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _get_section_from_result_or_disk(section_key: str, md_filename: str) -> str:
    result_var = RESULT_VARIABLES.get(section_key)
    result_obj = globals().get(result_var)

    if isinstance(result_obj, dict):
        final_section = result_obj.get("final_section")
        if isinstance(final_section, str) and final_section.strip():
            return final_section.strip()

    output_dir = Path("outputs")
    md_path = output_dir / md_filename
    if md_path.exists():
        return md_path.read_text(encoding="utf-8").strip()

    return ""


def _get_section_meta(section_key: str, meta_filename: str) -> dict:
    result_var = RESULT_VARIABLES.get(section_key)
    result_obj = globals().get(result_var)

    if isinstance(result_obj, dict):
        judge = result_obj.get("judge_result", {}) or {}
        return {
            "section": section_key,
            "status": result_obj.get("status"),
            "approved": judge.get("approved"),
            "approval_status": judge.get("approval_status"),
            "overall_score": judge.get("overall_score"),
            "revisions": result_obj.get("revision_count"),
            "main_issues": judge.get("main_issues"),
            "required_fixes": judge.get("required_fixes"),
            "unsupported_claims": judge.get("unsupported_claims"),
            "material_unsupported_claims": judge.get("material_unsupported_claims"),
            "minor_wording_to_tighten": judge.get("minor_wording_to_tighten"),
            "available_evidence_omitted": judge.get("available_evidence_omitted"),
            "deterministic_checks": judge.get("deterministic_checks"),
        }

    output_dir = Path("outputs")
    meta = _read_json_if_exists(output_dir / meta_filename)
    return meta if isinstance(meta, dict) else {}


def _normalise_section_title(section_text: str, section_title: str) -> str:
    text = (section_text or "").strip()
    if not text:
        return ""

    heading_patterns = [
        rf"^###\s+{re.escape(section_title)}\s*$",
        rf"^##\s+{re.escape(section_title)}\s*$",
        rf"^#\s+{re.escape(section_title)}\s*$",
    ]

    if any(re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE) for pattern in heading_patterns):
        return text

    return f"### {section_title}\n\n{text}"


def build_main_section_toc() -> str:
    toc_lines = ["## Table of contents", ""]
    for _, section_title, _, _ in PUBLIC_SECTION_ORDER:
        slug = re.sub(r"[^\w\s-]", "", section_title.strip().lower())
        slug = re.sub(r"\s+", "-", slug)
        toc_lines.append(f"- [{section_title}](#{slug})")
    return "\n".join(toc_lines)


def validate_section_approval_before_assembly(section_meta: dict) -> dict:
    failures = {}
    warnings = {}

    if not REQUIRE_APPROVED_SECTIONS_FOR_FULL_REPORT:
        return {"passed": True, "failures": failures, "warnings": warnings}

    for section_key, meta in section_meta.items():
        meta = meta or {}
        approved = meta.get("approved")
        status = str(meta.get("status", "")).lower()
        approval_status = str(meta.get("approval_status", "")).lower()
        score = meta.get("overall_score")

        section_failures = []
        # Sections at 'needs_human_review' are accepted into the report (signed off via the
        # accompanying data-limitations & expert-review log). Only genuinely failed or
        # revision-required sections block assembly.
        hard_fail = (status in ("failed", "error")) or (approval_status in ("revision_required", "rejected"))
        if hard_fail:
            section_failures.append({
                "check": "section_not_approved",
                "approved": approved,
                "status": status,
                "approval_status": approval_status,
                "overall_score": score,
            })

        try:
            if score is not None and float(score) < MIN_SECTION_SCORE_FOR_FULL_REPORT:
                section_failures.append({
                    "check": "section_score_below_threshold",
                    "overall_score": score,
                    "minimum_required": MIN_SECTION_SCORE_FOR_FULL_REPORT,
                })
        except Exception:
            warnings.setdefault(section_key, []).append({
                "check": "section_score_not_numeric",
                "details": score,
            })

        # Missing metadata is also a problem because the report would be unaudited.
        if not meta:
            section_failures.append({
                "check": "section_metadata_missing",
                "details": "No judge metadata was found for this section."
            })

        if section_failures:
            failures[section_key] = section_failures

    if failures:
        raise RuntimeError(
            "Full report assembly stopped because one or more sections are not approved. "
            "Rerun/fix the failed sections before assembling the final report.\n\n"
            + json.dumps(failures, indent=2, ensure_ascii=False)
        )

    return {"passed": True, "failures": failures, "warnings": warnings}


def validate_sections_before_assembly(sections: dict) -> dict:
    checks = {}
    failures = {}

    for section_key, text in sections.items():
        completeness_key = SECTION_COMPLETENESS_KEYS.get(section_key, section_key)
        if "check_section_completeness" in globals():
            check = check_section_completeness(text, completeness_key)
        else:
            check = {"passed": True, "failures": [], "warnings": []}

        checks[section_key] = check

        if not check.get("passed", True):
            failures[section_key] = check.get("failures", [])

    if failures:
        raise RuntimeError(
            "Full report assembly stopped because one or more sections are incomplete/truncated.\n"
            "Fix by rerunning the affected section generation cells.\n\n"
            + json.dumps(failures, indent=2, ensure_ascii=False)
        )

    return checks


def build_full_report_markdown(include_toc: bool = True) -> tuple[str, dict]:
    bank_id = _safe_bank_id()
    bank_name = _safe_bank_name()
    reporting_year = _safe_reporting_year()

    output_dir = Path("outputs")
    output_dir.mkdir(exist_ok=True)

    sections = {}
    section_meta = {}
    missing_sections = []

    for section_key, section_title, md_filename, meta_filename in PUBLIC_SECTION_ORDER:
        section_text = _get_section_from_result_or_disk(section_key, md_filename)
        section_meta[section_key] = _get_section_meta(section_key, meta_filename)

        if not section_text.strip():
            missing_sections.append(section_key)
            continue

        section_text = _normalise_section_title(section_text, section_title)
        if "polish_public_report_wording" in globals():
            section_text = polish_public_report_wording(section_text)
        if "polish_public_report_wording_final" in globals():
            section_text = polish_public_report_wording_final(section_text)

        section_text = _tidy_report_numbers(section_text)

        # Append deterministic exhibits (tables + charts) built from this section's evidence
        if "build_section_exhibits" in globals():
            try:
                _exhibits_md = build_section_exhibits(section_key, bank_id)
            except Exception:
                _exhibits_md = ""
            if _exhibits_md:
                section_text = section_text.rstrip() + "\n\n" + _exhibits_md

        sections[section_key] = section_text

    if missing_sections:
        raise RuntimeError(
            "Cannot build full report. Missing final sections: "
            + ", ".join(missing_sections)
            + ". Run those section generation cells or make sure their markdown files exist in outputs/."
        )

    approval_check = validate_section_approval_before_assembly(section_meta)
    completeness_checks = validate_sections_before_assembly(sections)

    title = f"# {bank_name}\n## Sustainability-related Financial Disclosures (prepared with reference to IFRS S1 and IFRS S2)"
    cover = f"""{title}

**Reporting year:** {reporting_year}
""".strip()

    section_body = "\n\n---\n\n".join([sections[key] for key, *_ in PUBLIC_SECTION_ORDER])

    if include_toc:
        full_report = cover + "\n\n---\n\n" + build_main_section_toc() + "\n\n---\n\n" + section_body
    else:
        full_report = cover + "\n\n---\n\n" + section_body

    if "build_report_appendices" in globals():
        _appx_md = build_report_appendices(bank_id, bank_name, reporting_year, section_body)
        if _appx_md:
            full_report = full_report + "\n\n---\n\n" + _appx_md

    if "polish_public_report_wording" in globals():
        full_report = polish_public_report_wording(full_report)
    if "polish_public_report_wording_final" in globals():
        full_report = polish_public_report_wording_final(full_report)

    full_report = _tidy_report_numbers(full_report)

    metadata = {
        "bank_id": bank_id,
        "bank_name": bank_name,
        "reporting_year": reporting_year,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "section_order": [key for key, *_ in PUBLIC_SECTION_ORDER],
        "style_guide_used": bool(globals().get("STYLE_REFERENCE") is not None),
        "style_guide_policy": "style_only_not_evidence",
        "ifrs_requirements_used": "build_ifrs_requirements_text" in globals(),
        "section_approval_check": approval_check,
        "pre_assembly_completeness_checks": completeness_checks,
        "sections": {
            key: {
                "word_count": len(sections[key].split()),
                "metadata": section_meta.get(key, {}),
            }
            for key, *_ in PUBLIC_SECTION_ORDER
        },
    }

    # Public wording QA
    if "check_public_report_wording" in globals():
        public_report_wording_check = check_public_report_wording(full_report)
        metadata["public_report_wording_check"] = public_report_wording_check
        if not public_report_wording_check.get("passed", True):
            raise RuntimeError(
                "Full report contains internal/raw public wording issues.\n"
                + json.dumps(public_report_wording_check, indent=2, ensure_ascii=False)
            )

    # Senior final QA
    if "senior_public_report_quality_check" in globals():
        senior_final_quality_check = senior_public_report_quality_check(full_report, metadata)
        metadata["senior_final_quality_check"] = senior_final_quality_check
        if not senior_final_quality_check.get("passed", True):
            raise RuntimeError(
                "Senior final QA failed. Fix the issues before using the report.\n"
                + json.dumps(senior_final_quality_check, indent=2, ensure_ascii=False)
            )

    return full_report, metadata


full_report_markdown, full_report_meta = build_full_report_markdown(include_toc=True)

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

bank_id = full_report_meta["bank_id"]
full_report_md_path = output_dir / f"full_report_{bank_id}.md"
full_report_json_path = output_dir / f"full_report_{bank_id}.json"
full_report_meta_path = output_dir / f"full_report_{bank_id}_meta.json"

full_report_md_path.write_text(full_report_markdown, encoding="utf-8")

with open(full_report_json_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "metadata": full_report_meta,
            "markdown": full_report_markdown,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

with open(full_report_meta_path, "w", encoding="utf-8") as f:
    json.dump(full_report_meta, f, indent=2, ensure_ascii=False)

print("Full report created.")
print("Markdown:", full_report_md_path.resolve())
print("JSON:", full_report_json_path.resolve())
print("Metadata:", full_report_meta_path.resolve())

print("\nSection word counts:")
for section_key, info in full_report_meta["sections"].items():
    meta = info.get("metadata", {})
    print(
        f"- {section_key}: {info['word_count']} words | "
        f"score={meta.get('overall_score')} | approved={meta.get('approved')} | status={meta.get('status')}"
    )

print("\nSenior QA:")
print(json.dumps(full_report_meta.get("senior_final_quality_check", {}), indent=2, ensure_ascii=False))

print("\nPreview:")
print(full_report_markdown[:2500])

Full report created.
Markdown: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\outputs\full_report_BANK01.md
JSON: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\outputs\full_report_BANK01.json
Metadata: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\outputs\full_report_BANK01_meta.json

Section word counts:
- general_requirements: 2777 words | score=7 | approved=False | status=needs_human_review
- governance: 1462 words | score=8 | approved=False | status=needs_human_review
- strategy: 2538 words | score=8 | approved=False | status=needs_human_review
- risk_management: 3129 words | score=8 | approved=False | status=needs_human_review
- metrics_targets: 2315 words | score=8 | approved=False | status=needs_human_review

Senior QA:
{
  "passed": true,
  "failures": [],
  "warnings": [
    {
      "check": "metrics_assurance_vocabulary_present",
      "details": "Prefer validation/verification wording in Metrics and targets."
    }
  ],
  "fa

In [65]:
# ── DATA LIMITATIONS & EXPERT-REVIEW LOG ─────────────────────
# Internal QA companion to the full report. For each section it records:
#  (a) IFRS S1/S2 requirement areas limited by available data (disclosed boundaries),
#  (b) material items requiring expert correction, and
#  (c) minor wording for optional editorial review.
# This file is NOT part of the published report.

from pathlib import Path
import json

_output_dir = Path("outputs")
_output_dir.mkdir(exist_ok=True)
_bank_id = full_report_meta["bank_id"] if "full_report_meta" in globals() else _safe_bank_id()
_bank_name = full_report_meta["bank_name"] if "full_report_meta" in globals() else _safe_bank_name()
_order = PUBLIC_SECTION_ORDER if "PUBLIC_SECTION_ORDER" in globals() else [
    ("general_requirements", "General Requirements", "general_requirements_BANK01.md", "general_requirements_BANK01_meta.json"),
    ("governance", "Governance", "governance_BANK01.md", "governance_BANK01_meta.json"),
    ("strategy", "Strategy", "strategy_BANK01.md", "strategy_BANK01_meta.json"),
    ("risk_management", "Risk Management", "risk_management_BANK01.md", "risk_management_BANK01_meta.json"),
    ("metrics_targets", "Metrics and Targets", "metrics_targets_BANK01.md", "metrics_targets_BANK01_meta.json"),
]


def _load_review_meta(section_key, meta_filename):
    # Prefer the on-disk section meta (it carries the disclosed boundaries); fall back to in-memory.
    disk = _read_json_if_exists(_output_dir / meta_filename) if "_read_json_if_exists" in globals() else {}
    if disk:
        return disk
    if "full_report_meta" in globals():
        return (full_report_meta.get("sections", {}).get(section_key, {}) or {}).get("metadata", {}) or {}
    return {}


def _as_list(v):
    if not v:
        return []
    return v if isinstance(v, list) else [v]


review_payload = {"bank_id": _bank_id, "bank_name": _bank_name, "sections": {}}
L = []
L.append(f"# {_bank_name} - Climate Disclosure: Data Limitations & Expert-Review Log")
L.append("")
L.append("**Internal quality-assurance document - not part of the published report.**")
L.append("")
L.append("For each disclosure section this log records (a) the IFRS S1 / S2 requirement areas that could not be "
         "fully satisfied because the underlying data does not contain the relevant information, and (b) residual "
         "items flagged for expert review before the report is finalised. Sections are generated under an "
         "evidence-bounded policy: where a requirement cannot be supported by the available data it is disclosed "
         "as a boundary rather than estimated. None of the items below indicate fabricated content; they indicate "
         "where expert judgement and/or additional source data are required.")
L.append("")
L.append("---")

for section_key, section_title, _md, meta_filename in _order:
    m = _load_review_meta(section_key, meta_filename)
    score = m.get("final_score", m.get("overall_score"))
    status = str(m.get("status", "needs_human_review")).replace("_", " ")
    boundaries = _as_list(m.get("correctly_disclosed_evidence_boundaries"))
    material = _as_list(m.get("material_unsupported_claims"))
    minor = _as_list(m.get("minor_wording_to_tighten"))
    review_payload["sections"][section_key] = {
        "final_score": score,
        "status": m.get("status"),
        "data_limited_requirements": boundaries,
        "material_items_for_expert_correction": material,
        "minor_wording_for_editorial_review": minor,
    }
    L.append(f"\n## {section_title}  \n*Automated score: {score}/10 - {status}*\n")
    L.append("### IFRS S1/S2 requirement areas limited by available data")
    if boundaries:
        L.append("_Disclosed in the report as evidence boundaries because the source data does not contain the "
                 "underlying information. An expert should confirm the boundary wording and supply the data if it "
                 "exists elsewhere._\n")
        for x in boundaries:
            L.append(f"- {x}")
    else:
        L.append("_No data-driven requirement gaps recorded for this section._")
    L.append("\n### Items requiring expert correction before finalisation (material)")
    if material:
        for x in material:
            L.append(f"- [ ] {x}")
    else:
        L.append("- None. No material unsupported statements identified.")
    L.append("\n### Minor wording for editorial review (optional)")
    if minor:
        for x in minor:
            L.append(f"- {x}")
    else:
        L.append("- None.")
    L.append("\n---")

L.append("\n## Reviewer sign-off")
L.append("")
L.append("| Section | Data gaps confirmed | Material items resolved | Approved for publication | Reviewer / date |")
L.append("|---|---|---|---|---|")
for _k, section_title, *_ in _order:
    L.append(f"| {section_title} |  |  |  |  |")

review_markdown = "\n".join(L) + "\n"
review_md_path = _output_dir / f"{_bank_id}_data_limitations_and_expert_review.md"
review_json_path = _output_dir / f"{_bank_id}_data_limitations_and_expert_review.json"
review_md_path.write_text(review_markdown, encoding="utf-8")
with open(review_json_path, "w", encoding="utf-8") as f:
    json.dump(review_payload, f, indent=2, ensure_ascii=False)

print("Data limitations & expert-review log created.")
print("Markdown:", review_md_path.resolve())
print("JSON:", review_json_path.resolve())
for _k, _v in review_payload["sections"].items():
    print(f"- {_k}: score={_v['final_score']} | data-gaps={len(_v['data_limited_requirements'])} "
          f"| material={len(_v['material_items_for_expert_correction'])} "
          f"| minor={len(_v['minor_wording_for_editorial_review'])}")


Data limitations & expert-review log created.
Markdown: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\outputs\BANK01_data_limitations_and_expert_review.md
JSON: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\outputs\BANK01_data_limitations_and_expert_review.json
- general_requirements: score=7 | data-gaps=0 | material=2 | minor=3
- governance: score=8 | data-gaps=6 | material=3 | minor=3
- strategy: score=8 | data-gaps=7 | material=2 | minor=3
- risk_management: score=8 | data-gaps=7 | material=1 | minor=3
- metrics_targets: score=8 | data-gaps=6 | material=2 | minor=3


In [66]:
# ── STYLED PDF (WeasyPrint) ─────────────────────────────────────────────
# Renders the assembled markdown report (prose + tables + charts) into a
# designed PDF. Requires: markdown, weasyprint (and system pango/cairo).
from pathlib import Path as _PPath
import markdown as _md
import weasyprint as _wp

_PDF_CSS = """
@page {
    size: A4;
    margin: 22mm 20mm 20mm 20mm;
    @bottom-center { content: counter(page) " / " counter(pages); font-size: 8pt; color: #888; }
}
html { font-family: 'Helvetica Neue', Arial, sans-serif; font-size: 10.5pt; color: #1c2b2f; line-height: 1.5; }
h1 { font-size: 23pt; color: #14323b; margin: 0 0 2pt 0; }
h2 { font-size: 13pt; font-weight: 500; color: #2e7d7d; margin: 0 0 14pt 0; }
h3 { font-size: 15pt; color: #14323b; border-bottom: 2px solid #2e7d7d; padding-bottom: 4px;
     margin: 26pt 0 12pt 0; page-break-after: avoid; page-break-before: always; }
h3:first-of-type { page-break-before: avoid; }
h4 { font-size: 11.5pt; color: #1f4e5f; margin: 16pt 0 6pt 0; page-break-after: avoid; }
p { margin: 0 0 8pt 0; text-align: justify; }
ul { margin: 0 0 8pt 0; }
li { margin: 0 0 3pt 0; }
strong { color: #14323b; }
table { border-collapse: collapse; width: 100%; margin: 4pt 0 14pt 0; font-size: 8.6pt; }
thead { display: table-header-group; }
.exhibit-caption { page-break-after: avoid; margin: 14pt 0 4pt 0; font-weight: 700; color: #14323b; }
th { background: #14323b; color: #fff; text-align: left; padding: 5px 7px; font-weight: 600; }
td { border-bottom: 1px solid #d6dee0; padding: 4px 7px; vertical-align: top; }
tr:nth-child(even) td { background: #f3f6f6; }
img { max-width: 100%; margin: 4pt auto 12pt auto; display: block; }
hr { border: none; border-top: 1px solid #d6dee0; margin: 16pt 0; }
a { color: #1f4e5f; text-decoration: none; }
ul#toc, .toc ul { list-style: none; padding-left: 0; }
"""


def build_full_report_pdf(md_path="outputs/full_report_BANK01.md",
                          pdf_path="outputs/full_report_BANK01.pdf"):
    md_path = _PPath(md_path); pdf_path = _PPath(pdf_path)
    text = md_path.read_text(encoding="utf-8")
    html_body = _md.markdown(text, extensions=["tables", "toc", "sane_lists", "attr_list"])
    html_doc = f"<!DOCTYPE html><html><head><meta charset='utf-8'></head><body>{html_body}</body></html>"
    # base_url = the report's directory so 'figures/...' image refs resolve
    _wp.HTML(string=html_doc, base_url=str(md_path.parent)).write_pdf(
        str(pdf_path), stylesheets=[_wp.CSS(string=_PDF_CSS)])
    return str(pdf_path)


pdf_output_path = build_full_report_pdf()
print("Styled PDF written:", pdf_output_path)
try:
    import os as _os
    print("PDF size:", _os.path.getsize(pdf_output_path), "bytes")
except Exception:
    pass


Styled PDF written: outputs\full_report_BANK01.pdf
PDF size: 178814 bytes


## Optional final public wording QA


## Final public-ready patch notes

This version includes stronger public wording cleanup for General Requirements and Metrics and Targets. Restart the kernel or rerun affected sections so old in-memory drafts do not remain.

Recommended rerun order: General Requirements → Metrics and Targets → Full report assembly → Optional final public wording QA.


## Final notebook patch — public-ready wording

This final version removes public-report blocker wording such as synthetic/generated sections, fixes the Governance-disclosure consistency sentence, and prevents Strategy limitations from contradicting Metrics and targets.

Recommended rerun order after restarting the kernel: General Requirements → Strategy → Metrics and Targets → Full report assembly → Optional final public wording QA.


In [67]:
# ── OPTIONAL FINAL PUBLIC WORDING QA ─────────────────────────
# Run after full report assembly to inspect public wording and senior QA issues.

if "full_report_markdown" not in globals():
    raise RuntimeError("Run the Full report assembly cell first.")

qa = check_public_report_wording(full_report_markdown) if "check_public_report_wording" in globals() else {"passed": True}
senior_qa = senior_public_report_quality_check(full_report_markdown, full_report_meta if "full_report_meta" in globals() else None) if "senior_public_report_quality_check" in globals() else {"passed": True}

print("PUBLIC WORDING QA")
print(json.dumps(qa, indent=2, ensure_ascii=False))

print("\nSENIOR FINAL QA")
print(json.dumps(senior_qa, indent=2, ensure_ascii=False))

if qa.get("passed") and senior_qa.get("passed"):
    print("Final public wording and senior QA passed.")
else:
    print("Final QA failed. Review the failures above before using the report.")

PUBLIC WORDING QA
{
  "passed": true,
  "failures": [],
  "warnings": [
    {
      "check": "snake_case_tokens_visible",
      "details": [
        "governance_trend",
        "portfolio_composition"
      ]
    }
  ],
  "failure_count": 0,
  "warning_count": 1
}

SENIOR FINAL QA
{
  "passed": true,
  "failures": [],
  "warnings": [
    {
      "check": "metrics_assurance_vocabulary_present",
      "details": "Prefer validation/verification wording in Metrics and targets."
    }
  ],
  "failure_count": 0,
  "warning_count": 1
}
Final public wording and senior QA passed.
